In [1]:
# ============================================================
# TRUST-STROKE (CDE-TT) — JOURNAL-READY NOTEBOOK
# Block 0: Setup + Reproducibility + Run Metadata
# ============================================================

import os, sys, gc, math, json, time, random, platform
from dataclasses import dataclass, asdict
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit
from sklearn.metrics import average_precision_score, roc_auc_score, brier_score_loss

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

# -------------------------
# 0.1 Reproducibility helpers
# -------------------------
def set_global_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    # Determinism flags (good for journal reproducibility)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def get_device():
    return "cuda" if torch.cuda.is_available() else "cpu"

# -------------------------
# 0.2 Central config (single source of truth)
# -------------------------
@dataclass
class CFG:
    # core
    seed: int = 42
    device: str = get_device()

    # data
    data_path: str = "/kaggle/input/stroke-dataset/Stroke.csv"
    target_col: str = "stroke"
    drop_id_col: str = "id"

    # CV
    n_splits: int = 5
    n_repeats: int = 2

    # training
    ensemble_M: int = 3     # increase to 5+ on GPU
    epochs: int = 25        # increase to 35+ on GPU
    batch_size: int = 256
    pos_frac: float = 0.30  # balanced batch positive fraction
    lr: float = 2e-3
    wd: float = 1e-5

    # model
    d_model: int = 64
    n_heads: int = 4
    n_layers: int = 3
    dropout: float = 0.20

    # metrics
    ece_bins: int = 15

    # artifacts
    out_dir: str = "./trust_stroke_artifacts"

cfg = CFG()
os.makedirs(cfg.out_dir, exist_ok=True)

set_global_seed(cfg.seed)

# -------------------------
# 0.3 Run metadata (for paper reproducibility)
# -------------------------
run_meta = {
    "timestamp_local": time.strftime("%Y-%m-%d %H:%M:%S"),
    "python_version": sys.version.split(" ")[0],
    "platform": platform.platform(),
    "torch_version": torch.__version__,
    "cuda_available": torch.cuda.is_available(),
    "device": cfg.device,
}

print("=== RUN METADATA ===")
for k, v in run_meta.items():
    print(f"{k:>16}: {v}")

print("\n=== CONFIG ===")
print(json.dumps(asdict(cfg), indent=2))

# Save config + meta immediately
with open(os.path.join(cfg.out_dir, "run_metadata.json"), "w") as f:
    json.dump(run_meta, f, indent=2)

with open(os.path.join(cfg.out_dir, "config.json"), "w") as f:
    json.dump(asdict(cfg), f, indent=2)

print(f"\nArtifacts directory: {os.path.abspath(cfg.out_dir)}")

=== RUN METADATA ===
 timestamp_local: 2026-02-02 15:56:26
  python_version: 3.12.12
        platform: Linux-6.6.113+-x86_64-with-glibc2.35
   torch_version: 2.8.0+cu126
  cuda_available: False
          device: cpu

=== CONFIG ===
{
  "seed": 42,
  "device": "cpu",
  "data_path": "/kaggle/input/stroke-dataset/Stroke.csv",
  "target_col": "stroke",
  "drop_id_col": "id",
  "n_splits": 5,
  "n_repeats": 2,
  "ensemble_M": 3,
  "epochs": 25,
  "batch_size": 256,
  "pos_frac": 0.3,
  "lr": 0.002,
  "wd": 1e-05,
  "d_model": 64,
  "n_heads": 4,
  "n_layers": 3,
  "dropout": 0.2,
  "ece_bins": 15,
  "out_dir": "./trust_stroke_artifacts"
}

Artifacts directory: /kaggle/working/trust_stroke_artifacts


In [2]:
# ============================================================
# Block 1: Data Load + Basic Audit (journal-grade)
# ============================================================

import os
import numpy as np
import pandas as pd

# -------------------------
# 1.1 Load
# -------------------------
assert os.path.exists(cfg.data_path), f"DATA_PATH not found: {cfg.data_path}"

df = pd.read_csv(cfg.data_path)

# normalize column names
df.columns = [str(c).strip() for c in df.columns]

# drop id if present
if cfg.drop_id_col in df.columns:
    df = df.drop(columns=[cfg.drop_id_col])

# target checks
assert cfg.target_col in df.columns, f"TARGET_COL '{cfg.target_col}' not found in columns: {df.columns.tolist()}"

# move target out
y = df[cfg.target_col].copy()
X = df.drop(columns=[cfg.target_col]).copy()

# coerce target to 0/1 int safely
# handles cases like "1"/"0", True/False, etc.
y = pd.to_numeric(y, errors="coerce")

# if there are NaNs in target, that's an issue
if y.isna().any():
    bad_n = int(y.isna().sum())
    raise ValueError(f"Target column has {bad_n} NaNs after coercion. Please clean target labels.")

# force to int 0/1
y = y.astype(int)
uniq_y = sorted(y.unique().tolist())
assert uniq_y in ([0,1], [0], [1]), f"Target must be binary 0/1, got unique values: {uniq_y}"

y = y.values.astype(int)

print("=== DATA SUMMARY ===")
print("X shape:", X.shape)
print("y shape:", y.shape)
print("Positive rate:", float(y.mean()))
print("Positives:", int(y.sum()), "Negatives:", int((y==0).sum()))

# duplicates
dup_rows = int(X.duplicated().sum())
print("Duplicate feature-rows:", dup_rows)

# -------------------------
# 1.2 Missingness report
# -------------------------
miss_ct = X.isna().sum().sort_values(ascending=False)
miss_pct = (miss_ct / len(X) * 100.0).round(3)
miss_df = pd.DataFrame({"missing_count": miss_ct, "missing_pct": miss_pct})
miss_df = miss_df[miss_df["missing_count"] > 0]

print("\n=== MISSINGNESS (non-zero only) ===")
if len(miss_df) == 0:
    print("No missing values detected in X.")
else:
    display(miss_df.head(30))

# -------------------------
# 1.3 Schema summary (for paper transparency)
# -------------------------
schema_df = pd.DataFrame({
    "column": X.columns,
    "dtype": [str(X[c].dtype) for c in X.columns],
    "n_unique": [int(X[c].nunique(dropna=True)) for c in X.columns],
})
schema_df = schema_df.sort_values(["dtype", "n_unique"], ascending=[True, True]).reset_index(drop=True)

print("\n=== SCHEMA SUMMARY (first 40 rows) ===")
display(schema_df.head(40))

# -------------------------
# 1.4 Save audit artifacts
# -------------------------
audit_dir = os.path.join(cfg.out_dir, "audit")
os.makedirs(audit_dir, exist_ok=True)

schema_df.to_csv(os.path.join(audit_dir, "schema_summary.csv"), index=False)

if len(miss_df) > 0:
    miss_df.to_csv(os.path.join(audit_dir, "missingness_report.csv"), index=True)

pd.DataFrame({
    "n_rows": [len(X)],
    "n_features": [X.shape[1]],
    "positive_rate": [float(y.mean())],
    "n_pos": [int(y.sum())],
    "n_neg": [int((y==0).sum())],
    "n_duplicate_rows": [dup_rows],
}).to_csv(os.path.join(audit_dir, "dataset_summary.csv"), index=False)

print(f"\nSaved audit artifacts to: {os.path.abspath(audit_dir)}")

=== DATA SUMMARY ===
X shape: (4603, 35)
y shape: (4603,)
Positive rate: 0.07864436237236584
Positives: 362 Negatives: 4241
Duplicate feature-rows: 0

=== MISSINGNESS (non-zero only) ===
No missing values detected in X.

=== SCHEMA SUMMARY (first 40 rows) ===


,column,dtype,n_unique
0,sleep time,float64,23
1,Glycohemoglobin,float64,106
2,High-density lipoprotein,float64,110
3,Dietary fiber,float64,484
4,Waist Circumference,float64,776
5,Low-density lipoprotein,float64,2414
6,Fasting Glucose,float64,2492
7,Triglyceride,float64,2528
8,protein,float64,3864
9,Total fat,float64,3933



Saved audit artifacts to: /kaggle/working/trust_stroke_artifacts/audit


In [3]:
# ============================================================
# Block 2: Explicit Column Schema (Cat / Num / Sensitive)
# ============================================================

# -------------------------
# 2.1 Explicit schema definition
# -------------------------
# IMPORTANT:
# - Numerical = continuous or ordinal with magnitude meaning
# - Categorical = nominal / discrete groups
# - Sensitive = used ONLY for fairness analysis (not dropped)

NUM_COLS = [
    # continuous biomarkers / measurements
    "sleep time",
    "Glycohemoglobin",
    "High-density lipoprotein",
    "Dietary fiber",
    "Waist Circumference",
    "Low-density lipoprotein",
    "Fasting Glucose",
    "Triglyceride",
    "protein",
    "Total fat",
    "Total polyunsaturated fatty acids",
    "Carbohydrate",
    "Total saturated fatty acids",
    "Total monounsaturated fatty acids",
    "Minutes sedentary activity",
    "Diastolic blood pressure",
    "Systolic blood pressure",
    "energy",
    "Potassium",
    "Sodium",
]

CAT_COLS = [
    # binary lifestyle / clinical indicators
    "gender",
    "alcohol",
    "smoke",
    "sleep disorder",
    "Health Insurance",
    "diabetes",
    "hypertension",
    "high cholesterol",
    "Coronary Heart Disease",

    # ordinal / discrete health & demographic groups
    "age",
    "depression",
    "Body Mass Index",
    "Race",
    "General health condition",
    "Marital status",
]

# Sensitive attributes for fairness analysis
# (model will still use them; we only *measure* disparities)
SENSITIVE_COLS = [
    "gender",
    "Race",
    "age",
    "Health Insurance",
]

# -------------------------
# 2.2 Sanity checks (fail fast)
# -------------------------
all_cols = set(X.columns)

assert set(NUM_COLS).issubset(all_cols), f"Missing NUM_COLS: {set(NUM_COLS) - all_cols}"
assert set(CAT_COLS).issubset(all_cols), f"Missing CAT_COLS: {set(CAT_COLS) - all_cols}"
assert set(SENSITIVE_COLS).issubset(all_cols), f"Missing SENSITIVE_COLS: {set(SENSITIVE_COLS) - all_cols}"

# no overlap between num & cat
overlap = set(NUM_COLS).intersection(CAT_COLS)
assert len(overlap) == 0, f"Columns cannot be both num and cat: {overlap}"

# ensure full coverage
unused = all_cols - set(NUM_COLS) - set(CAT_COLS)
assert len(unused) == 0, f"Unassigned columns found: {unused}"

print("=== COLUMN SCHEMA CONFIRMED ===")
print(f"Numerical cols ({len(NUM_COLS)}):", NUM_COLS)
print(f"\nCategorical cols ({len(CAT_COLS)}):", CAT_COLS)
print(f"\nSensitive cols ({len(SENSITIVE_COLS)}):", SENSITIVE_COLS)

# -------------------------
# 2.3 Schema table for paper / supplement
# -------------------------
schema_final = []

for c in NUM_COLS:
    schema_final.append({
        "column": c,
        "role": "numerical",
        "dtype": str(X[c].dtype),
        "n_unique": int(X[c].nunique()),
        "sensitive": c in SENSITIVE_COLS
    })

for c in CAT_COLS:
    schema_final.append({
        "column": c,
        "role": "categorical",
        "dtype": str(X[c].dtype),
        "n_unique": int(X[c].nunique()),
        "sensitive": c in SENSITIVE_COLS
    })

schema_final_df = pd.DataFrame(schema_final).sort_values(
    ["role", "n_unique"], ascending=[True, True]
).reset_index(drop=True)

print("\n=== FINAL SCHEMA TABLE (first 30 rows) ===")
display(schema_final_df.head(30))

# -------------------------
# 2.4 Save schema artifacts
# -------------------------
schema_dir = os.path.join(cfg.out_dir, "schema")
os.makedirs(schema_dir, exist_ok=True)

schema_final_df.to_csv(os.path.join(schema_dir, "final_feature_schema.csv"), index=False)

with open(os.path.join(schema_dir, "sensitive_columns.json"), "w") as f:
    json.dump(SENSITIVE_COLS, f, indent=2)

print(f"\nSaved schema artifacts to: {os.path.abspath(schema_dir)}")

=== COLUMN SCHEMA CONFIRMED ===
Numerical cols (20): ['sleep time', 'Glycohemoglobin', 'High-density lipoprotein', 'Dietary fiber', 'Waist Circumference', 'Low-density lipoprotein', 'Fasting Glucose', 'Triglyceride', 'protein', 'Total fat', 'Total polyunsaturated fatty acids', 'Carbohydrate', 'Total saturated fatty acids', 'Total monounsaturated fatty acids', 'Minutes sedentary activity', 'Diastolic blood pressure', 'Systolic blood pressure', 'energy', 'Potassium', 'Sodium']

Categorical cols (15): ['gender', 'alcohol', 'smoke', 'sleep disorder', 'Health Insurance', 'diabetes', 'hypertension', 'high cholesterol', 'Coronary Heart Disease', 'age', 'depression', 'Body Mass Index', 'Race', 'General health condition', 'Marital status']

Sensitive cols (4): ['gender', 'Race', 'age', 'Health Insurance']

=== FINAL SCHEMA TABLE (first 30 rows) ===


,column,role,dtype,n_unique,sensitive
0,gender,categorical,int64,2,True
1,alcohol,categorical,int64,2,False
2,smoke,categorical,int64,2,False
3,sleep disorder,categorical,int64,2,False
4,Health Insurance,categorical,int64,2,True
5,diabetes,categorical,int64,2,False
6,hypertension,categorical,int64,2,False
7,high cholesterol,categorical,int64,2,False
8,Coronary Heart Disease,categorical,int64,2,False
9,age,categorical,int64,3,True



Saved schema artifacts to: /kaggle/working/trust_stroke_artifacts/schema


In [4]:
# ============================================================
# Block 3: Fold Preprocessor (train-only fit, journal-safe)
# ============================================================

from typing import List, Dict, Tuple

# -------------------------
# 3.1 Fold Preprocessor
# -------------------------
class FoldPreprocessor:
    """
    Journal-safe preprocessing:
    - Categorical: integer encoding with UNK=0
    - Numerical: median imputation + z-score (fit on train only)
    """

    def __init__(self, cat_cols: List[str], num_cols: List[str]):
        self.cat_cols = list(cat_cols)
        self.num_cols = list(num_cols)

        self.cat_maps: Dict[str, Dict] = {}
        self.cat_sizes: List[int] = []
        self.num_stats: Dict[str, Tuple[float, float, float]] = {}

        self._is_fitted = False

    # -------------------------
    def fit(self, X_tr: pd.DataFrame):
        Xtr = X_tr.copy()

        # ---- categorical mappings
        self.cat_maps = {}
        self.cat_sizes = []

        for c in self.cat_cols:
            s = Xtr[c].astype("object").fillna("MISSING").astype(str)
            uniq = pd.Index(s.unique())
            mp = {k: i + 1 for i, k in enumerate(uniq)}  # 0 reserved for UNK
            self.cat_maps[c] = mp
            self.cat_sizes.append(len(mp) + 1)

        # ---- numerical statistics
        self.num_stats = {}
        for c in self.num_cols:
            s = pd.to_numeric(Xtr[c], errors="coerce")
            med = float(s.median())
            s = s.fillna(med).astype(np.float32)
            mu = float(s.mean())
            sd = float(s.std(ddof=0) + 1e-6)
            self.num_stats[c] = (med, mu, sd)

        self._is_fitted = True
        return self

    # -------------------------
    def transform(self, X_any: pd.DataFrame):
        assert self._is_fitted, "FoldPreprocessor must be fitted before transform()"
        Xa = X_any.copy()

        # ---- categorical
        X_cat = None
        if len(self.cat_cols) > 0:
            cat_arr = []
            for c in self.cat_cols:
                s = Xa[c].astype("object").fillna("MISSING").astype(str)
                mp = self.cat_maps[c]
                codes = s.map(mp).fillna(0).astype(np.int64).values
                cat_arr.append(codes)
            X_cat = np.stack(cat_arr, axis=1)

        # ---- numerical
        X_num = None
        if len(self.num_cols) > 0:
            num_arr = []
            for c in self.num_cols:
                s = pd.to_numeric(Xa[c], errors="coerce")
                med, mu, sd = self.num_stats[c]
                s = s.fillna(med).astype(np.float32)
                s = (s - mu) / sd
                num_arr.append(s.values.astype(np.float32))
            X_num = np.stack(num_arr, axis=1)

        # ---- safety checks
        if X_cat is not None:
            assert not np.isnan(X_cat).any(), "NaNs found in X_cat"
        if X_num is not None:
            assert not np.isnan(X_num).any(), "NaNs found in X_num"

        return X_cat, X_num

    # -------------------------
    def summary(self):
        return {
            "n_cat_cols": len(self.cat_cols),
            "n_num_cols": len(self.num_cols),
            "cat_sizes": self.cat_sizes,
        }


# -------------------------
# 3.2 Smoke test on a single split
# -------------------------
print("=== PREPROCESSOR SMOKE TEST ===")

sss = StratifiedShuffleSplit(n_splits=1, test_size=0.25, random_state=cfg.seed)
tr_idx, va_idx = next(sss.split(X, y))

X_tr, y_tr = X.iloc[tr_idx], y[tr_idx]
X_va, y_va = X.iloc[va_idx], y[va_idx]

prep = FoldPreprocessor(CAT_COLS, NUM_COLS).fit(X_tr)
Xtr_cat, Xtr_num = prep.transform(X_tr)
Xva_cat, Xva_num = prep.transform(X_va)

print("Train shapes:")
print("  Xtr_cat:", None if Xtr_cat is None else Xtr_cat.shape)
print("  Xtr_num:", None if Xtr_num is None else Xtr_num.shape)

print("Val shapes:")
print("  Xva_cat:", None if Xva_cat is None else Xva_cat.shape)
print("  Xva_num:", None if Xva_num is None else Xva_num.shape)

print("\nCategorical vocab sizes:", prep.cat_sizes)
print("Total categorical tokens:", sum(prep.cat_sizes))

# -------------------------
# 3.3 Save example preprocessor metadata (for reproducibility)
# -------------------------
prep_dir = os.path.join(cfg.out_dir, "preprocessor_example")
os.makedirs(prep_dir, exist_ok=True)

with open(os.path.join(prep_dir, "cat_sizes.json"), "w") as f:
    json.dump(prep.cat_sizes, f, indent=2)

with open(os.path.join(prep_dir, "num_stats.json"), "w") as f:
    json.dump({k: list(v) for k, v in prep.num_stats.items()}, f, indent=2)

print(f"\nSaved example preprocessor metadata to: {os.path.abspath(prep_dir)}")

=== PREPROCESSOR SMOKE TEST ===
Train shapes:
  Xtr_cat: (3452, 15)
  Xtr_num: (3452, 20)
Val shapes:
  Xva_cat: (1151, 15)
  Xva_num: (1151, 20)

Categorical vocab sizes: [3, 3, 3, 3, 3, 3, 3, 3, 3, 4, 4, 5, 6, 6, 7]
Total categorical tokens: 59

Saved example preprocessor metadata to: /kaggle/working/trust_stroke_artifacts/preprocessor_example


In [5]:
# ============================================================
# Block 4: Dataset + Balanced Batch Sampler (with unit test)
# ============================================================

from torch.utils.data import Dataset, DataLoader, Sampler

# -------------------------
# 4.1 Dataset
# -------------------------
class TabDataset(Dataset):
    def __init__(self, X_cat, X_num, y):
        self.X_cat = X_cat
        self.X_num = X_num
        self.y = np.asarray(y).astype(np.float32)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, i):
        yi = torch.tensor(self.y[i], dtype=torch.float32)
        x_cat = torch.tensor(self.X_cat[i], dtype=torch.long) if self.X_cat is not None else None
        x_num = torch.tensor(self.X_num[i], dtype=torch.float32) if self.X_num is not None else None
        return x_cat, x_num, yi


# -------------------------
# 4.2 Balanced sampler
# -------------------------
class BalancedBatchSampler(Sampler):
    """
    Creates batches with approx pos_frac positives and (1-pos_frac) negatives.
    Uses replacement to handle rare positives safely.

    NOTE: We yield indices; DataLoader(batch_size=..., sampler=...) will collect them into batches.
    """
    def __init__(self, y, batch_size=256, pos_frac=0.30, seed=0):
        self.y = np.asarray(y).astype(int)
        self.batch_size = int(batch_size)

        self.pos_bs = max(1, int(round(self.batch_size * float(pos_frac))))
        self.neg_bs = self.batch_size - self.pos_bs

        self.pos_idx = np.where(self.y == 1)[0]
        self.neg_idx = np.where(self.y == 0)[0]

        assert len(self.pos_idx) > 0, "No positive samples in y."
        assert len(self.neg_idx) > 0, "No negative samples in y."

        self.rng = np.random.RandomState(seed)

    def __iter__(self):
        n_batches = int(np.ceil(len(self.y) / self.batch_size))
        for _ in range(n_batches):
            pos = self.rng.choice(self.pos_idx, size=self.pos_bs, replace=True)
            neg = self.rng.choice(self.neg_idx, size=self.neg_bs, replace=True)
            idx = np.concatenate([pos, neg])
            self.rng.shuffle(idx)
            for j in idx.tolist():
                yield j

    def __len__(self):
        n_batches = int(np.ceil(len(self.y) / self.batch_size))
        return n_batches * self.batch_size


# -------------------------
# 4.3 Unit test: verify batch positive fraction
# -------------------------
print("=== BALANCED SAMPLER UNIT TEST ===")

ds_tr = TabDataset(Xtr_cat, Xtr_num, y_tr)
sampler = BalancedBatchSampler(y_tr, batch_size=cfg.batch_size, pos_frac=cfg.pos_frac, seed=cfg.seed)

dl_tr = DataLoader(
    ds_tr,
    batch_size=cfg.batch_size,
    sampler=sampler,
    drop_last=True,
    num_workers=0
)

# check first K batches
K = 10
pos_rates = []
for bi, (x_cat, x_num, yb) in enumerate(dl_tr):
    if bi >= K:
        break
    pos_rates.append(float((yb.numpy() == 1).mean()))

print(f"Target pos_frac: {cfg.pos_frac:.3f}")
print(f"Observed pos_frac (first {K} batches): mean={np.mean(pos_rates):.3f}, std={np.std(pos_rates):.3f}")
print("Per-batch pos rates:", [round(r, 3) for r in pos_rates])

# quick shape sanity
print("\nBatch shapes:")
print("  x_cat:", None if x_cat is None else tuple(x_cat.shape))
print("  x_num:", None if x_num is None else tuple(x_num.shape))
print("  y:", tuple(yb.shape))

=== BALANCED SAMPLER UNIT TEST ===
Target pos_frac: 0.300
Observed pos_frac (first 10 batches): mean=0.301, std=0.000
Per-batch pos rates: [0.301, 0.301, 0.301, 0.301, 0.301, 0.301, 0.301, 0.301, 0.301, 0.301]

Batch shapes:
  x_cat: (256, 15)
  x_num: (256, 20)
  y: (256,)


In [6]:
# ============================================================
# Block 5: TabTransformer Model + Forward Tests
# ============================================================

cfg.device = get_device()  # re-evaluate device (still CPU for you)
DEVICE = cfg.device
print("DEVICE:", DEVICE)

# -------------------------
# 5.1 Transformer block
# -------------------------
class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, dropout):
        super().__init__()
        self.attn = nn.MultiheadAttention(
            embed_dim=d_model,
            num_heads=n_heads,
            dropout=dropout,
            batch_first=True
        )
        self.ln1 = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(
            nn.Linear(d_model, 4 * d_model),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(4 * d_model, d_model),
        )
        self.ln2 = nn.LayerNorm(d_model)
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        h, _ = self.attn(x, x, x, need_weights=False)
        x = self.ln1(x + self.drop(h))
        h = self.ff(x)
        x = self.ln2(x + self.drop(h))
        return x


# -------------------------
# 5.2 TabTransformer (cat+num tokens + CLS)
# -------------------------
class TabTransformer(nn.Module):
    def __init__(self, cat_sizes, n_num, d_model=64, n_heads=4, n_layers=3, dropout=0.20):
        super().__init__()
        self.n_cat = len(cat_sizes)
        self.n_num = int(n_num)
        self.d_model = int(d_model)

        self.cat_embeds = nn.ModuleList([nn.Embedding(sz, d_model) for sz in cat_sizes]) if self.n_cat > 0 else None
        self.num_proj = nn.Linear(1, d_model) if self.n_num > 0 else None

        self.cls = nn.Parameter(torch.zeros(1, 1, d_model))
        nn.init.normal_(self.cls, std=0.02)

        self.blocks = nn.ModuleList([TransformerBlock(d_model, n_heads, dropout) for _ in range(n_layers)])
        self.drop = nn.Dropout(dropout)

        self.head = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Dropout(dropout),
            nn.Linear(d_model, 1),
        )

    def forward(self, x_cat, x_num):
        tokens = []

        # cat tokens
        if self.n_cat > 0:
            assert x_cat is not None, "x_cat is None but model expects categorical inputs."
            for j, emb in enumerate(self.cat_embeds):
                tokens.append(emb(x_cat[:, j]))  # (B, D)

        # num tokens
        if self.n_num > 0:
            assert x_num is not None, "x_num is None but model expects numerical inputs."
            for j in range(self.n_num):
                v = x_num[:, j:j+1]          # (B, 1)
                tokens.append(self.num_proj(v))  # (B, D)

        # stack tokens (B, T, D)
        if len(tokens) == 0:
            raise ValueError("No tokens created: check cat_sizes/n_num.")
        x = torch.stack(tokens, dim=1)

        # add CLS
        B = x.shape[0]
        cls = self.cls.expand(B, -1, -1)
        x = torch.cat([cls, x], dim=1)  # (B, 1+T, D)

        x = self.drop(x)
        for blk in self.blocks:
            x = blk(x)

        z = x[:, 0, :]                  # CLS pooled representation
        logit = self.head(z).squeeze(-1)  # (B,)
        return logit


# -------------------------
# 5.3 Instantiate model using current preprocessor
# -------------------------
cat_sizes = prep.cat_sizes
n_num = Xtr_num.shape[1]

model = TabTransformer(
    cat_sizes=cat_sizes,
    n_num=n_num,
    d_model=cfg.d_model,
    n_heads=cfg.n_heads,
    n_layers=cfg.n_layers,
    dropout=cfg.dropout
).to(DEVICE)

# -------------------------
# 5.4 Parameter count (paper-friendly)
# -------------------------
def count_params(m):
    total = sum(p.numel() for p in m.parameters())
    trainable = sum(p.numel() for p in m.parameters() if p.requires_grad)
    return total, trainable

total_p, trainable_p = count_params(model)
print("=== MODEL SUMMARY ===")
print("TabTransformer params:", f"{total_p:,} (trainable {trainable_p:,})")
print("Tokens per sample: cat =", len(cat_sizes), ", num =", n_num, ", total =", len(cat_sizes) + n_num, "+ CLS")

# -------------------------
# 5.5 Forward pass smoke test
# -------------------------
model.eval()
x_cat_b, x_num_b, y_b = next(iter(dl_tr))
x_cat_b = x_cat_b.to(DEVICE)
x_num_b = x_num_b.to(DEVICE)

with torch.no_grad():
    logits = model(x_cat_b, x_num_b)

print("\n=== FORWARD TEST ===")
print("logits shape:", tuple(logits.shape))
print("logits stats: mean =", float(logits.mean().cpu()), "std =", float(logits.std().cpu()))

DEVICE: cpu
=== MODEL SUMMARY ===
TabTransformer params: 154,113 (trainable 154,113)
Tokens per sample: cat = 15 , num = 20 , total = 35 + CLS

=== FORWARD TEST ===
logits shape: (256,)
logits stats: mean = -0.11059747636318207 std = 0.22367030382156372


In [7]:
# ============================================================
# Block 6: Training Loop (Early stop on PR-AUC) + Predict utils
# ============================================================

from sklearn.metrics import average_precision_score, roc_auc_score
import torch.optim as optim

# -------------------------
# 6.1 Focal loss (binary)
# -------------------------
class FocalBCE(nn.Module):
    def __init__(self, gamma=2.0, alpha=0.25):
        super().__init__()
        self.gamma = float(gamma)
        self.alpha = float(alpha)

    def forward(self, logits, y):
        # logits: (B,), y: (B,)
        bce = F.binary_cross_entropy_with_logits(logits, y, reduction="none")
        p = torch.sigmoid(logits)
        pt = torch.where(y == 1, p, 1 - p)
        w = (1 - pt).pow(self.gamma)
        a = torch.where(
            y == 1,
            torch.tensor(self.alpha, device=y.device),
            torch.tensor(1 - self.alpha, device=y.device),
        )
        return (w * a * bce).mean()


# -------------------------
# 6.2 Predict utilities
# -------------------------
@torch.no_grad()
def predict_logits(model, X_cat, X_num, batch_size=2048):
    """
    Returns logits (numpy array shape [N]) for calibration/uncertainty workflows.
    """
    n = len(X_cat) if X_cat is not None else len(X_num)
    ds = TabDataset(X_cat, X_num, np.zeros(n, dtype=np.float32))
    dl = DataLoader(ds, batch_size=batch_size, shuffle=False, num_workers=0)

    model.eval()
    out = []
    for x_cat, x_num, _ in dl:
        x_cat = x_cat.to(DEVICE) if x_cat is not None else None
        x_num = x_num.to(DEVICE) if x_num is not None else None
        z = model(x_cat, x_num).detach().cpu().numpy()
        out.append(z)
    return np.concatenate(out)


def sigmoid_np(x):
    return 1.0 / (1.0 + np.exp(-x))


def predict_proba(model, X_cat, X_num, batch_size=2048):
    """
    Returns probabilities (numpy array shape [N]).
    """
    z = predict_logits(model, X_cat, X_num, batch_size=batch_size)
    return sigmoid_np(z)


# -------------------------
# 6.3 One-model training (journal-safe)
# -------------------------
def train_one_model(
    Xtr_cat, Xtr_num, ytr,
    Xva_cat, Xva_num, yva,
    cat_sizes,
    seed: int = 0,
    verbose: bool = True,
):
    """
    Trains ONE TabTransformer with balanced batches + hybrid loss.
    Early-stops on validation PR-AUC.

    Returns:
      model (best-state loaded),
      history (pd.DataFrame with epoch logs),
      best_val_pr (float)
    """
    set_global_seed(seed)

    ds_tr = TabDataset(Xtr_cat, Xtr_num, ytr)
    ds_va = TabDataset(Xva_cat, Xva_num, yva)

    sampler = BalancedBatchSampler(ytr, batch_size=cfg.batch_size, pos_frac=cfg.pos_frac, seed=seed)

    dl_tr = DataLoader(
        ds_tr,
        batch_size=cfg.batch_size,
        sampler=sampler,
        drop_last=True,
        num_workers=0
    )

    dl_va = DataLoader(
        ds_va,
        batch_size=2048,
        shuffle=False,
        num_workers=0
    )

    model = TabTransformer(
        cat_sizes=cat_sizes,
        n_num=(0 if Xtr_num is None else Xtr_num.shape[1]),
        d_model=cfg.d_model,
        n_heads=cfg.n_heads,
        n_layers=cfg.n_layers,
        dropout=cfg.dropout
    ).to(DEVICE)

    focal = FocalBCE(gamma=2.0, alpha=0.25)
    opt = optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.wd)

    # Early stopping
    best_pr = -1.0
    best_state = None
    patience = 5
    bad = 0

    rows = []
    t0 = time.time()

    for ep in range(1, cfg.epochs + 1):
        # ---- train
        model.train()
        tr_losses = []

        for x_cat, x_num, yb in dl_tr:
            yb = yb.to(DEVICE)
            x_cat = x_cat.to(DEVICE) if x_cat is not None else None
            x_num = x_num.to(DEVICE) if x_num is not None else None

            opt.zero_grad(set_to_none=True)
            logits = model(x_cat, x_num)

            loss_bce = F.binary_cross_entropy_with_logits(logits, yb)
            loss_focal = focal(logits, yb)
            loss = 0.7 * loss_bce + 0.3 * loss_focal

            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()

            tr_losses.append(float(loss.detach().cpu().item()))

        # ---- validate: PR-AUC + ROC-AUC on probs
        model.eval()
        p_list, y_list = [], []
        with torch.no_grad():
            for x_cat, x_num, yb in dl_va:
                x_cat = x_cat.to(DEVICE) if x_cat is not None else None
                x_num = x_num.to(DEVICE) if x_num is not None else None
                z = model(x_cat, x_num).detach().cpu().numpy()
                p_list.append(sigmoid_np(z))
                y_list.append(yb.numpy())

        pva = np.concatenate(p_list)
        yva_np = np.concatenate(y_list).astype(int)

        pr = float(average_precision_score(yva_np, pva))
        roc = float(roc_auc_score(yva_np, pva)) if len(np.unique(yva_np)) > 1 else float("nan")

        row = {
            "epoch": ep,
            "train_loss": float(np.mean(tr_losses)) if len(tr_losses) else float("nan"),
            "val_pr_auc": pr,
            "val_roc_auc": roc,
            "elapsed_sec": float(time.time() - t0),
        }
        rows.append(row)

        if verbose:
            print(f"[ep {ep:02d}/{cfg.epochs}] "
                  f"loss={row['train_loss']:.4f}  valPR={pr:.4f}  valROC={roc:.4f}")

        # ---- early stopping on PR-AUC
        if pr > best_pr + 1e-4:
            best_pr = pr
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            bad = 0
        else:
            bad += 1
            if bad >= patience:
                if verbose:
                    print(f"Early stopping triggered at epoch {ep} (best val PR-AUC={best_pr:.4f})")
                break

    # restore best weights
    if best_state is not None:
        model.load_state_dict(best_state)

    history = pd.DataFrame(rows)
    return model, history, best_pr


# -------------------------
# 6.4 Smoke-run training on current train/val split
# -------------------------
print("=== TRAINING SMOKE RUN (1 model) ===")

model_1, hist_1, best_pr_1 = train_one_model(
    Xtr_cat, Xtr_num, y_tr,
    Xva_cat, Xva_num, y_va,
    cat_sizes=prep.cat_sizes,
    seed=cfg.seed,
    verbose=True
)

print("\nBest val PR-AUC:", round(best_pr_1, 4))
display(hist_1.tail(8))

# Save logs
train_dir = os.path.join(cfg.out_dir, "train_smoke")
os.makedirs(train_dir, exist_ok=True)
hist_1.to_csv(os.path.join(train_dir, "train_history_smoke.csv"), index=False)

print(f"\nSaved training log to: {os.path.abspath(train_dir)}")

=== TRAINING SMOKE RUN (1 model) ===
[ep 01/25] loss=0.4845  valPR=0.0938  valROC=0.5400
[ep 02/25] loss=0.4470  valPR=0.1008  valROC=0.5535
[ep 03/25] loss=0.4369  valPR=0.1071  valROC=0.5998
[ep 04/25] loss=0.4105  valPR=0.1368  valROC=0.6402
[ep 05/25] loss=0.3982  valPR=0.1203  valROC=0.6211
[ep 06/25] loss=0.3916  valPR=0.1349  valROC=0.6476
[ep 07/25] loss=0.3899  valPR=0.1328  valROC=0.6567
[ep 08/25] loss=0.3907  valPR=0.1459  valROC=0.6552
[ep 09/25] loss=0.3825  valPR=0.1298  valROC=0.6449
[ep 10/25] loss=0.3803  valPR=0.1292  valROC=0.6332
[ep 11/25] loss=0.3742  valPR=0.1231  valROC=0.6311
[ep 12/25] loss=0.3769  valPR=0.1237  valROC=0.6216
[ep 13/25] loss=0.3719  valPR=0.1200  valROC=0.6194
Early stopping triggered at epoch 13 (best val PR-AUC=0.1459)

Best val PR-AUC: 0.1459


,epoch,train_loss,val_pr_auc,val_roc_auc,elapsed_sec
5,6,0.391585,0.134892,0.647636,43.691181
6,7,0.389867,0.132766,0.656666,50.929950
7,8,0.390678,0.145902,0.655246,58.164922
8,9,0.382485,0.129820,0.644941,65.439392
9,10,0.380259,0.129248,0.633216,72.694824
10,11,0.374220,0.123141,0.631122,79.952403
11,12,0.376919,0.123721,0.621626,87.129671
12,13,0.371894,0.120023,0.619438,94.431012



Saved training log to: /kaggle/working/trust_stroke_artifacts/train_smoke


In [8]:
# ============================================================
# Block 7: Temperature Scaling Calibration (logits-based)
# ============================================================

from sklearn.metrics import log_loss

# -------------------------
# 7.1 Helpers
# -------------------------
def ece_score(y_true, p, n_bins=15):
    y_true = np.asarray(y_true).astype(int)
    p = np.asarray(p).astype(float)
    bins = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    for i in range(n_bins):
        lo, hi = bins[i], bins[i + 1]
        if i < n_bins - 1:
            m = (p >= lo) & (p < hi)
        else:
            m = (p >= lo) & (p <= hi)
        if m.sum() == 0:
            continue
        acc = y_true[m].mean()
        conf = p[m].mean()
        ece += (m.sum() / len(p)) * abs(acc - conf)
    return float(ece)

def reliability_bins(y_true, p, n_bins=15):
    """
    Returns a dataframe with per-bin:
    - count
    - mean predicted probability (confidence)
    - empirical accuracy (event rate)
    """
    y_true = np.asarray(y_true).astype(int)
    p = np.asarray(p).astype(float)
    bins = np.linspace(0, 1, n_bins + 1)

    rows = []
    for i in range(n_bins):
        lo, hi = bins[i], bins[i + 1]
        if i < n_bins - 1:
            m = (p >= lo) & (p < hi)
        else:
            m = (p >= lo) & (p <= hi)

        cnt = int(m.sum())
        if cnt == 0:
            rows.append({"bin": i, "lo": lo, "hi": hi, "count": 0, "conf": np.nan, "acc": np.nan})
        else:
            rows.append({
                "bin": i, "lo": lo, "hi": hi,
                "count": cnt,
                "conf": float(p[m].mean()),
                "acc": float(y_true[m].mean())
            })
    return pd.DataFrame(rows)

# -------------------------
# 7.2 Fit Temperature Scaling on validation logits
# -------------------------
def fit_temperature_scaling_from_logits(z_val, y_val, max_iter=300, lr=0.05):
    """
    Fit scalar temperature T>0 to minimize NLL:
      p = sigmoid(z / T)
    """
    z = torch.tensor(z_val.astype(np.float32), device=DEVICE)
    y = torch.tensor(y_val.astype(np.float32), device=DEVICE)

    logT = torch.zeros((), device=DEVICE, requires_grad=True)
    opt = torch.optim.Adam([logT], lr=lr)

    for _ in range(max_iter):
        T = torch.exp(logT) + 1e-6
        loss = F.binary_cross_entropy_with_logits(z / T, y)
        opt.zero_grad()
        loss.backward()
        opt.step()

    T = float((torch.exp(logT) + 1e-6).detach().cpu().item())
    return T

def apply_temperature_to_logits(z, T):
    return z / (T + 1e-12)

# -------------------------
# 7.3 Calibrate the current smoke-run model on the same val split
# -------------------------
print("=== CALIBRATION SMOKE RUN (Temperature Scaling) ===")

# logits and probs BEFORE calibration
z_val = predict_logits(model_1, Xva_cat, Xva_num, batch_size=2048)
p_val = sigmoid_np(z_val)

# fit temperature on val
T_hat = fit_temperature_scaling_from_logits(z_val, y_va, max_iter=300, lr=0.05)

# probs AFTER calibration
z_val_cal = apply_temperature_to_logits(z_val, T_hat)
p_val_cal = sigmoid_np(z_val_cal)

# -------------------------
# 7.4 Metrics before vs after (val)
# -------------------------
def safe_log_loss(y_true, p):
    p = np.clip(p, 1e-6, 1 - 1e-6)
    return float(log_loss(y_true, p))

report = {
    "T_hat": T_hat,
    "val_PR_AUC": float(average_precision_score(y_va, p_val_cal)),
    "val_ROC_AUC": float(roc_auc_score(y_va, p_val_cal)) if len(np.unique(y_va)) > 1 else float("nan"),

    "NLL_before": safe_log_loss(y_va, p_val),
    "NLL_after":  safe_log_loss(y_va, p_val_cal),

    "Brier_before": float(brier_score_loss(y_va, p_val)),
    "Brier_after":  float(brier_score_loss(y_va, p_val_cal)),

    "ECE_before": ece_score(y_va, p_val, n_bins=cfg.ece_bins),
    "ECE_after":  ece_score(y_va, p_val_cal, n_bins=cfg.ece_bins),
}

print(json.dumps(report, indent=2))

# reliability tables (for plotting later)
rel_before = reliability_bins(y_va, p_val, n_bins=cfg.ece_bins)
rel_after  = reliability_bins(y_va, p_val_cal, n_bins=cfg.ece_bins)

print("\nReliability bins (after calibration) — first 10:")
display(rel_after.head(10))

# -------------------------
# 7.5 Save calibration artifacts
# -------------------------
cal_dir = os.path.join(cfg.out_dir, "calibration_smoke")
os.makedirs(cal_dir, exist_ok=True)

pd.DataFrame([report]).to_csv(os.path.join(cal_dir, "val_calibration_report.csv"), index=False)
rel_before.to_csv(os.path.join(cal_dir, "reliability_bins_before.csv"), index=False)
rel_after.to_csv(os.path.join(cal_dir, "reliability_bins_after.csv"), index=False)

with open(os.path.join(cal_dir, "temperature_T_hat.json"), "w") as f:
    json.dump({"T_hat": T_hat}, f, indent=2)

print(f"\nSaved calibration artifacts to: {os.path.abspath(cal_dir)}")

=== CALIBRATION SMOKE RUN (Temperature Scaling) ===
{
  "T_hat": 0.7580705285072327,
  "val_PR_AUC": 0.14590171492886894,
  "val_ROC_AUC": 0.6552456976985279,
  "NLL_before": 0.36468826025420775,
  "NLL_after": 0.3549111004500216,
  "Brier_before": 0.11130249329025915,
  "Brier_after": 0.10804700369973567,
  "ECE_before": 0.14041710587025827,
  "ECE_after": 0.11947730098479631
}

Reliability bins (after calibration) — first 10:


,bin,lo,hi,count,conf,acc
0,0,0.000000,0.066667,505,0.030271,0.045545
1,1,0.066667,0.133333,150,0.093408,0.080000
2,2,0.133333,0.200000,84,0.167296,0.035714
3,3,0.200000,0.266667,91,0.232540,0.120879
4,4,0.266667,0.333333,70,0.293771,0.157143
5,5,0.333333,0.400000,60,0.361529,0.083333
6,6,0.400000,0.466667,48,0.434246,0.041667
7,7,0.466667,0.533333,46,0.501368,0.130435
8,8,0.533333,0.600000,34,0.563599,0.147059
9,9,0.600000,0.666667,24,0.628690,0.208333



Saved calibration artifacts to: /kaggle/working/trust_stroke_artifacts/calibration_smoke


In [9]:
# ============================================================
# Block 8: Deep Ensemble + Epistemic Uncertainty (Smoke)
# ============================================================

# -------------------------
# 8.1 Train a small ensemble on the SAME train/val split
#      (this is a smoke test; CV comes later)
# -------------------------
print("=== ENSEMBLE SMOKE RUN ===")

M = 3  # keep small on CPU
ensemble_models = []
ensemble_histories = []

for m in range(M):
    seed_m = cfg.seed + 1000 + m
    print(f"\n--- Training ensemble member {m+1}/{M} (seed={seed_m}) ---")

    model_m, hist_m, best_pr_m = train_one_model(
        Xtr_cat, Xtr_num, y_tr,
        Xva_cat, Xva_num, y_va,
        cat_sizes=prep.cat_sizes,
        seed=seed_m,
        verbose=False
    )

    print(f"Best val PR-AUC (member {m+1}): {best_pr_m:.4f}")
    ensemble_models.append(model_m)
    ensemble_histories.append(hist_m)

# -------------------------
# 8.2 Predict logits for each member
# -------------------------
print("\n=== ENSEMBLE PREDICTION ===")

Z_val_members = []
for m, model_m in enumerate(ensemble_models):
    z_val_m = predict_logits(model_m, Xva_cat, Xva_num, batch_size=2048)
    Z_val_members.append(z_val_m)

Z_val = np.stack(Z_val_members, axis=0)  # (M, N)

# -------------------------
# 8.3 Ensemble mean + epistemic uncertainty
# -------------------------
z_mean = Z_val.mean(axis=0)          # mean logit
z_var  = Z_val.var(axis=0, ddof=0)   # epistemic variance

p_mean = sigmoid_np(z_mean)

print("Logit mean stats:", "mean =", float(z_mean.mean()), "std =", float(z_mean.std()))
print("Epistemic var stats:", "mean =", float(z_var.mean()), "std =", float(z_var.std()))

# -------------------------
# 8.4 Metrics with ensemble mean
# -------------------------
pr_ens = average_precision_score(y_va, p_mean)
roc_ens = roc_auc_score(y_va, p_mean)
brier_ens = brier_score_loss(y_va, p_mean)
ece_ens = ece_score(y_va, p_mean, n_bins=cfg.ece_bins)

print("\n=== ENSEMBLE (UNCALIBRATED) METRICS ===")
print(f"PR-AUC:  {pr_ens:.4f}")
print(f"ROC-AUC: {roc_ens:.4f}")
print(f"Brier:   {brier_ens:.4f}")
print(f"ECE:     {ece_ens:.4f}")

# -------------------------
# 8.5 Simple uncertainty sanity checks
# -------------------------
# High-uncertainty samples should be harder
top_u = np.argsort(-z_var)[:100]
low_u = np.argsort(z_var)[:100]

err_top = np.mean((p_mean[top_u] > 0.5) != y_va[top_u])
err_low = np.mean((p_mean[low_u] > 0.5) != y_va[low_u])

print("\nError rate (top 100 uncertainty):", round(err_top, 3))
print("Error rate (low 100 uncertainty):", round(err_low, 3))

# -------------------------
# 8.6 Save ensemble smoke artifacts
# -------------------------
ens_dir = os.path.join(cfg.out_dir, "ensemble_smoke")
os.makedirs(ens_dir, exist_ok=True)

np.save(os.path.join(ens_dir, "z_val_members.npy"), Z_val)
np.save(os.path.join(ens_dir, "z_val_mean.npy"), z_mean)
np.save(os.path.join(ens_dir, "z_val_var.npy"), z_var)
np.save(os.path.join(ens_dir, "p_val_mean.npy"), p_mean)

print(f"\nSaved ensemble smoke artifacts to: {os.path.abspath(ens_dir)}")

=== ENSEMBLE SMOKE RUN ===

--- Training ensemble member 1/3 (seed=1042) ---
Best val PR-AUC (member 1): 0.1476

--- Training ensemble member 2/3 (seed=1043) ---
Best val PR-AUC (member 2): 0.1474

--- Training ensemble member 3/3 (seed=1044) ---
Best val PR-AUC (member 3): 0.1556

=== ENSEMBLE PREDICTION ===
Logit mean stats: mean = -1.1795696020126343 std = 0.8619080781936646
Epistemic var stats: mean = 0.13667060434818268 std = 0.13577473163604736

=== ENSEMBLE (UNCALIBRATED) METRICS ===
PR-AUC:  0.1570
ROC-AUC: 0.6628
Brier:   0.1181
ECE:     0.1868

Error rate (top 100 uncertainty): 0.07
Error rate (low 100 uncertainty): 0.18

Saved ensemble smoke artifacts to: /kaggle/working/trust_stroke_artifacts/ensemble_smoke


In [10]:
# ============================================================
# Block 9: Selective Prediction / Risk–Coverage Analysis
# ============================================================

# -------------------------
# 9.1 Prepare uncertainty + predictions
# -------------------------
y_val = y_va.copy()
p_val = p_mean.copy()
u_val = z_var.copy()   # epistemic uncertainty

assert len(y_val) == len(p_val) == len(u_val)

# -------------------------
# 9.2 Risk–coverage computation
# -------------------------
def risk_coverage_curve(y, p, u, coverages):
    """
    Reject highest-uncertainty samples first.
    Returns coverage, risk (error rate), PR-AUC on retained set.
    """
    order = np.argsort(u)  # low uncertainty first
    n = len(y)

    rows = []
    for cov in coverages:
        k = int(np.floor(cov * n))
        idx = order[:k]

        if len(np.unique(y[idx])) < 2:
            pr = np.nan
        else:
            pr = average_precision_score(y[idx], p[idx])

        err = np.mean((p[idx] >= 0.5) != y[idx])

        rows.append({
            "coverage": cov,
            "n_kept": k,
            "risk_error_rate": err,
            "pr_auc": pr
        })
    return pd.DataFrame(rows)

coverages = np.linspace(0.2, 1.0, 17)  # 20% → 100%
rc_df = risk_coverage_curve(y_val, p_val, u_val, coverages)

print("=== RISK–COVERAGE TABLE ===")
display(rc_df)

# -------------------------
# 9.3 Sanity check
# -------------------------
print("\nAt full coverage (no rejection):")
display(rc_df.tail(1))

print("\nAt lowest coverage (most conservative):")
display(rc_df.head(1))

# -------------------------
# 9.4 Save artifacts
# -------------------------
rc_dir = os.path.join(cfg.out_dir, "risk_coverage_smoke")
os.makedirs(rc_dir, exist_ok=True)

rc_df.to_csv(os.path.join(rc_dir, "risk_coverage_table.csv"), index=False)

print(f"\nSaved risk–coverage artifacts to: {os.path.abspath(rc_dir)}")

=== RISK–COVERAGE TABLE ===


,coverage,n_kept,risk_error_rate,pr_auc
0,0.20,230,0.169565,0.131137
1,0.25,287,0.198606,0.110821
2,0.30,345,0.214493,0.144945
3,0.35,402,0.201493,0.185045
4,0.40,460,0.204348,0.184890
5,0.45,517,0.195358,0.180123
6,0.50,575,0.184348,0.172337
7,0.55,633,0.184834,0.162424
8,0.60,690,0.176812,0.166336
9,0.65,748,0.171123,0.167357



At full coverage (no rejection):


,coverage,n_kept,risk_error_rate,pr_auc
16,1.0,1151,0.140747,0.156952



At lowest coverage (most conservative):


,coverage,n_kept,risk_error_rate,pr_auc
0,0.2,230,0.169565,0.131137



Saved risk–coverage artifacts to: /kaggle/working/trust_stroke_artifacts/risk_coverage_smoke


In [11]:
# ============================================================
# Block 10: Risk Stratification (Lift, Capture, FIXED BSS)
# ============================================================

from sklearn.metrics import brier_score_loss

# -------------------------
# 10.1 Risk stratification metrics (journal-correct)
# -------------------------
def risk_stratification_metrics_fixed_bss(y, p, top_fracs):
    """
    Computes:
    - risk (event rate) in top-K% predicted risk group
    - lift vs population prevalence
    - stroke capture rate
    - Brier Skill Score (null = population prevalence)
    """
    y = np.asarray(y).astype(int)
    p = np.asarray(p).astype(float)

    n = len(y)
    base_rate = float(y.mean())
    total_pos = max(1, int(y.sum()))

    order = np.argsort(-p)
    rows = []

    for frac in top_fracs:
        k = int(np.ceil(frac * n))
        idx = order[:k]

        y_g = y[idx]
        p_g = p[idx]

        risk = float(y_g.mean())
        lift = risk / base_rate if base_rate > 0 else np.nan
        capture = float(y_g.sum()) / total_pos

        # Brier Skill Score
        brier_model = brier_score_loss(y_g, p_g)
        p_null = np.full_like(y_g, fill_value=base_rate, dtype=float)
        brier_null = brier_score_loss(y_g, p_null)
        bss = 1.0 - (brier_model / (brier_null + 1e-12))

        rows.append({
            "top_frac": frac,
            "n": k,
            "base_rate": base_rate,
            "risk_in_group": risk,
            "lift": lift,
            "stroke_capture_rate": capture,
            "brier_model": brier_model,
            "brier_null": brier_null,
            "brier_skill": bss
        })

    return pd.DataFrame(rows)


# -------------------------
# 10.2 Apply on ensemble validation predictions
# -------------------------
TOP_FRACS = [0.05, 0.10, 0.20, 0.30, 0.50]

risk_df = risk_stratification_metrics_fixed_bss(
    y=y_val,
    p=p_val,
    top_fracs=TOP_FRACS
)

print("=== RISK STRATIFICATION (FIXED BSS) ===")
display(risk_df)

# -------------------------
# 10.3 Save artifacts
# -------------------------
risk_dir = os.path.join(cfg.out_dir, "risk_stratification_smoke")
os.makedirs(risk_dir, exist_ok=True)

risk_df.to_csv(os.path.join(risk_dir, "risk_stratification_table.csv"), index=False)

print(f"\nSaved risk stratification artifacts to: {os.path.abspath(risk_dir)}")

=== RISK STRATIFICATION (FIXED BSS) ===


,top_frac,n,base_rate,risk_in_group,lift,stroke_capture_rate,brier_model,brier_null,brier_skill
0,0.05,58,0.079062,0.172414,2.180750,0.109890,0.323894,0.151402,-1.139299
1,0.10,116,0.079062,0.146552,1.853638,0.186813,0.292692,0.129629,-1.257919
2,0.20,231,0.079062,0.125541,1.587888,0.318681,0.247735,0.111941,-1.213089
3,0.30,346,0.079062,0.130058,1.645017,0.494505,0.223055,0.115743,-0.927149
4,0.50,576,0.079062,0.111111,1.405372,0.703297,0.183377,0.099793,-0.837577



Saved risk stratification artifacts to: /kaggle/working/trust_stroke_artifacts/risk_stratification_smoke


In [12]:
# ============================================================
# Block 10b: Risk Stratification AFTER Calibration (BSS fixed)
# ============================================================

# -------------------------
# 10b.1 Calibrate ensemble logits on validation
# -------------------------
T_ens = fit_temperature_scaling_from_logits(z_mean, y_val, max_iter=300, lr=0.05)
print("Ensemble temperature T:", round(T_ens, 4))

z_mean_cal = apply_temperature_to_logits(z_mean, T_ens)
p_mean_cal = sigmoid_np(z_mean_cal)

# -------------------------
# 10b.2 Risk stratification with calibrated probs
# -------------------------
risk_df_cal = risk_stratification_metrics_fixed_bss(
    y=y_val,
    p=p_mean_cal,
    top_fracs=TOP_FRACS
)

print("\n=== RISK STRATIFICATION (CALIBRATED ENSEMBLE) ===")
display(risk_df_cal)

# -------------------------
# 10b.3 Compare uncalibrated vs calibrated BSS
# -------------------------
cmp = risk_df[["top_frac", "brier_skill"]].merge(
    risk_df_cal[["top_frac", "brier_skill"]],
    on="top_frac",
    suffixes=("_uncal", "_cal")
)

print("\n=== BSS COMPARISON ===")
display(cmp)

# -------------------------
# 10b.4 Save artifacts
# -------------------------
risk_cal_dir = os.path.join(cfg.out_dir, "risk_stratification_calibrated")
os.makedirs(risk_cal_dir, exist_ok=True)

risk_df_cal.to_csv(os.path.join(risk_cal_dir, "risk_stratification_calibrated.csv"), index=False)
cmp.to_csv(os.path.join(risk_cal_dir, "bss_comparison.csv"), index=False)

print(f"\nSaved calibrated risk stratification artifacts to: {os.path.abspath(risk_cal_dir)}")

Ensemble temperature T: 0.5062

=== RISK STRATIFICATION (CALIBRATED ENSEMBLE) ===


,top_frac,n,base_rate,risk_in_group,lift,stroke_capture_rate,brier_model,brier_null,brier_skill
0,0.05,58,0.079062,0.172414,2.180750,0.109890,0.408931,0.151402,-1.700965
1,0.10,116,0.079062,0.146552,1.853638,0.186813,0.341225,0.129629,-1.632312
2,0.20,231,0.079062,0.125541,1.587888,0.318681,0.254719,0.111941,-1.275476
3,0.30,346,0.079062,0.130058,1.645017,0.494505,0.214798,0.115743,-0.855811
4,0.50,576,0.079062,0.111111,1.405372,0.703297,0.162243,0.099793,-0.625804



=== BSS COMPARISON ===


,top_frac,brier_skill_uncal,brier_skill_cal
0,0.05,-1.139299,-1.700965
1,0.10,-1.257919,-1.632312
2,0.20,-1.213089,-1.275476
3,0.30,-0.927149,-0.855811
4,0.50,-0.837577,-0.625804



Saved calibrated risk stratification artifacts to: /kaggle/working/trust_stroke_artifacts/risk_stratification_calibrated


In [13]:
# ============================================================
# Block 11: Decision Curve Analysis (Net Benefit)
# ============================================================

# -------------------------
# 11.1 Decision curve function
# -------------------------
def decision_curve_net_benefit(y, p, thresholds):
    """
    Net Benefit = TP/N − FP/N × (pt / (1 − pt))
    """
    y = np.asarray(y).astype(int)
    p = np.asarray(p).astype(float)
    N = len(y)

    out = []
    for pt in thresholds:
        pred = (p >= pt).astype(int)
        tp = np.sum((pred == 1) & (y == 1))
        fp = np.sum((pred == 1) & (y == 0))
        nb = (tp / N) - (fp / N) * (pt / (1 - pt + 1e-12))
        out.append(nb)
    return np.array(out)


# -------------------------
# 11.2 Compute DCA curves
# -------------------------
thresholds = np.linspace(0.01, 0.40, 40)

nb_model = decision_curve_net_benefit(y_val, p_mean_cal, thresholds)

prev = float(y_val.mean())
nb_all = prev - (1 - prev) * (thresholds / (1 - thresholds + 1e-12))
nb_none = np.zeros_like(thresholds)

dca_df = pd.DataFrame({
    "threshold": thresholds,
    "nb_model": nb_model,
    "nb_all": nb_all,
    "nb_none": nb_none
})

print("=== DECISION CURVE (FIRST 10 ROWS) ===")
display(dca_df.head(10))

# -------------------------
# 11.3 Key comparisons
# -------------------------
better_than_all = np.mean(nb_model > nb_all)
better_than_none = np.mean(nb_model > nb_none)

print(f"\nModel beats 'treat all' in {better_than_all*100:.1f}% of thresholds")
print(f"Model beats 'treat none' in {better_than_none*100:.1f}% of thresholds")

# -------------------------
# 11.4 Save artifacts
# -------------------------
dca_dir = os.path.join(cfg.out_dir, "decision_curve_smoke")
os.makedirs(dca_dir, exist_ok=True)

dca_df.to_csv(os.path.join(dca_dir, "decision_curve_table.csv"), index=False)

print(f"\nSaved decision curve artifacts to: {os.path.abspath(dca_dir)}")

=== DECISION CURVE (FIRST 10 ROWS) ===


,threshold,nb_model,nb_all,nb_none
0,0.01,0.069294,0.069759,0.0
1,0.02,0.059593,0.060267,0.0
2,0.03,0.052335,0.050579,0.0
3,0.04,0.043332,0.040689,0.0
4,0.05,0.036764,0.030591,0.0
5,0.06,0.030797,0.020278,0.0
6,0.07,0.024429,0.009744,0.0
7,0.08,0.019227,-0.001020,0.0
8,0.09,0.013891,-0.012020,0.0
9,0.10,0.007530,-0.023265,0.0



Model beats 'treat all' in 95.0% of thresholds
Model beats 'treat none' in 27.5% of thresholds

Saved decision curve artifacts to: /kaggle/working/trust_stroke_artifacts/decision_curve_smoke


In [14]:
# ============================================================
# Block 12: Fairness Evaluation (group-wise metrics)
# ============================================================

from collections import defaultdict

# -------------------------
# 12.1 Helper: group metrics
# -------------------------
def group_metrics(y, p, u, groups, group_name):
    """
    Computes group-wise:
    - size
    - prevalence
    - PR-AUC
    - TPR (recall for positive class, threshold=0.5)
    - FNR
    - ECE
    - mean epistemic uncertainty
    """
    rows = []
    for g in sorted(np.unique(groups)):
        idx = np.where(groups == g)[0]
        if len(idx) < 20:
            continue  # too small to be reliable

        y_g = y[idx]
        p_g = p[idx]
        u_g = u[idx]

        # metrics
        pr = average_precision_score(y_g, p_g) if len(np.unique(y_g)) > 1 else np.nan
        pred = (p_g >= 0.5).astype(int)

        tp = np.sum((pred == 1) & (y_g == 1))
        fn = np.sum((pred == 0) & (y_g == 1))
        tpr = tp / (tp + fn + 1e-12)
        fnr = fn / (tp + fn + 1e-12)

        rows.append({
            "group_attr": group_name,
            "group_value": g,
            "n": len(idx),
            "prevalence": float(y_g.mean()),
            "pr_auc": pr,
            "tpr": tpr,
            "fnr": fnr,
            "ece": ece_score(y_g, p_g, n_bins=cfg.ece_bins),
            "mean_uncertainty": float(np.mean(u_g)),
        })

    return pd.DataFrame(rows)


# -------------------------
# 12.2 Run fairness analysis for each sensitive attribute
# -------------------------
fairness_tables = []

for attr in SENSITIVE_COLS:
    print(f"\n=== FAIRNESS ANALYSIS: {attr} ===")
    groups = X_va[attr].values
    df_f = group_metrics(
        y=y_val,
        p=p_mean_cal,
        u=z_var,
        groups=groups,
        group_name=attr
    )
    display(df_f)
    fairness_tables.append(df_f)

fairness_df = pd.concat(fairness_tables, axis=0, ignore_index=True)

# -------------------------
# 12.3 Gap summary (worst-case disparity)
# -------------------------
gap_rows = []

for attr in fairness_df["group_attr"].unique():
    df_a = fairness_df[fairness_df["group_attr"] == attr]

    gap_rows.append({
        "group_attr": attr,
        "TPR_gap": float(df_a["tpr"].max() - df_a["tpr"].min()),
        "FNR_gap": float(df_a["fnr"].max() - df_a["fnr"].min()),
        "PR_AUC_gap": float(df_a["pr_auc"].max() - df_a["pr_auc"].min()),
        "ECE_gap": float(df_a["ece"].max() - df_a["ece"].min()),
        "Uncertainty_gap": float(df_a["mean_uncertainty"].max() - df_a["mean_uncertainty"].min()),
    })

gap_df = pd.DataFrame(gap_rows)

print("\n=== FAIRNESS GAP SUMMARY ===")
display(gap_df)

# -------------------------
# 12.4 Save artifacts
# -------------------------
fair_dir = os.path.join(cfg.out_dir, "fairness_smoke")
os.makedirs(fair_dir, exist_ok=True)

fairness_df.to_csv(os.path.join(fair_dir, "fairness_group_metrics.csv"), index=False)
gap_df.to_csv(os.path.join(fair_dir, "fairness_gap_summary.csv"), index=False)

print(f"\nSaved fairness artifacts to: {os.path.abspath(fair_dir)}")


=== FAIRNESS ANALYSIS: gender ===


,group_attr,group_value,n,prevalence,pr_auc,tpr,fnr,ece,mean_uncertainty
0,gender,1,523,0.076482,0.190184,0.175000,0.825000,0.115055,0.146954
1,gender,2,628,0.081210,0.140624,0.156863,0.843137,0.113670,0.128107



=== FAIRNESS ANALYSIS: Race ===


,group_attr,group_value,n,prevalence,pr_auc,tpr,fnr,ece,mean_uncertainty
0,Race,1,126,0.071429,0.092774,0.111111,0.888889,0.120747,0.125205
1,Race,2,120,0.025000,0.024503,0.000000,1.000000,0.145183,0.136023
2,Race,3,579,0.101900,0.204395,0.186441,0.813559,0.105724,0.119841
3,Race,4,256,0.062500,0.102025,0.125000,0.875000,0.130745,0.169181
4,Race,5,70,0.057143,0.451149,0.250000,0.750000,0.078162,0.178726



=== FAIRNESS ANALYSIS: age ===


...


=== FAIRNESS ANALYSIS: Health Insurance ===


,group_attr,group_value,n,prevalence,pr_auc,tpr,fnr,ece,mean_uncertainty
0,Health Insurance,1,1007,0.082423,0.161021,0.168675,0.831325,0.116234,0.131003
1,Health Insurance,2,144,0.055556,0.221010,0.125000,0.875000,0.062359,0.176304



=== FAIRNESS GAP SUMMARY ===


,group_attr,TPR_gap,FNR_gap,PR_AUC_gap,ECE_gap,Uncertainty_gap
0,gender,0.018137,0.018137,0.049560,0.001385,0.018847
1,Race,0.250000,0.250000,0.426647,0.067021,0.058885
2,age,0.227273,0.227273,0.201825,0.211641,0.186235
3,Health Insurance,0.043675,0.043675,0.059990,0.053875,0.045301



Saved fairness artifacts to: /kaggle/working/trust_stroke_artifacts/fairness_smoke


In [15]:
# ============================================================
# Block 13a: Permutation Importance (Global Interpretability)
# ============================================================

from sklearn.metrics import average_precision_score
import copy

# -------------------------
# 13a.1 Helper: permutation importance
# -------------------------
def permutation_importance(
    model,
    X_cat,
    X_num,
    y,
    feature_names_cat,
    feature_names_num,
    n_repeats=5,
    seed=0
):
    """
    Computes permutation importance using PR-AUC drop.
    """
    rng = np.random.RandomState(seed)

    # baseline performance
    p_base = predict_proba(model, X_cat, X_num)
    base_score = average_precision_score(y, p_base)

    rows = []

    # categorical features
    for j, fname in enumerate(feature_names_cat):
        scores = []
        for _ in range(n_repeats):
            Xc_perm = X_cat.copy()
            rng.shuffle(Xc_perm[:, j])
            p_perm = predict_proba(model, Xc_perm, X_num)
            scores.append(base_score - average_precision_score(y, p_perm))

        rows.append({
            "feature": fname,
            "type": "categorical",
            "importance_mean": float(np.mean(scores)),
            "importance_std": float(np.std(scores)),
        })

    # numerical features
    for j, fname in enumerate(feature_names_num):
        scores = []
        for _ in range(n_repeats):
            Xn_perm = X_num.copy()
            rng.shuffle(Xn_perm[:, j])
            p_perm = predict_proba(model, X_cat, Xn_perm)
            scores.append(base_score - average_precision_score(y, p_perm))

        rows.append({
            "feature": fname,
            "type": "numerical",
            "importance_mean": float(np.mean(scores)),
            "importance_std": float(np.std(scores)),
        })

    imp_df = pd.DataFrame(rows).sort_values(
        "importance_mean", ascending=False
    ).reset_index(drop=True)

    return imp_df, base_score


# -------------------------
# 13a.2 Run permutation importance on calibrated ensemble mean
# -------------------------
print("=== PERMUTATION IMPORTANCE (VAL SET) ===")

# We use a *single representative model* (member 1) for interpretability
# This is standard practice and reviewer-acceptable
model_for_pi = ensemble_models[0]

imp_df, base_pr = permutation_importance(
    model=model_for_pi,
    X_cat=Xva_cat,
    X_num=Xva_num,
    y=y_val,
    feature_names_cat=CAT_COLS,
    feature_names_num=NUM_COLS,
    n_repeats=5,
    seed=cfg.seed
)

print("Baseline PR-AUC:", round(base_pr, 4))
display(imp_df.head(15))

# -------------------------
# 13a.3 Save artifacts
# -------------------------
interp_dir = os.path.join(cfg.out_dir, "interpretability_smoke")
os.makedirs(interp_dir, exist_ok=True)

imp_df.to_csv(os.path.join(interp_dir, "permutation_importance.csv"), index=False)

print(f"\nSaved interpretability artifacts to: {os.path.abspath(interp_dir)}")

=== PERMUTATION IMPORTANCE (VAL SET) ===
Baseline PR-AUC: 0.1476


,feature,type,importance_mean,importance_std
0,depression,categorical,0.029432,0.004642
1,Coronary Heart Disease,categorical,0.026197,0.007489
2,General health condition,categorical,0.025156,0.009257
3,age,categorical,0.023362,0.006256
4,smoke,categorical,0.015566,0.005720
5,Race,categorical,0.008689,0.006718
6,sleep time,numerical,0.007360,0.001807
7,diabetes,categorical,0.007343,0.004568
8,High-density lipoprotein,numerical,0.007038,0.001534
9,Systolic blood pressure,numerical,0.004696,0.001549



Saved interpretability artifacts to: /kaggle/working/trust_stroke_artifacts/interpretability_smoke


In [16]:
# ============================================================
# Block 13b: SHAP Explanations (Baseline Model)
# ============================================================

import shap
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

# -------------------------
# 13b.1 Prepare tabular matrix for baseline model
# -------------------------
# Encode categoricals as integers (already done), concatenate with numericals
Xtr_tab = np.hstack([Xtr_cat, Xtr_num])
Xva_tab = np.hstack([Xva_cat, Xva_num])

feature_names_tab = CAT_COLS + NUM_COLS

# Standardize for logistic regression
scaler = StandardScaler()
Xtr_tab_std = scaler.fit_transform(Xtr_tab)
Xva_tab_std = scaler.transform(Xva_tab)

# -------------------------
# 13b.2 Train baseline logistic regression
# -------------------------
lr_base = LogisticRegression(
    penalty="l2",
    C=1.0,
    class_weight="balanced",
    solver="liblinear",
    max_iter=1000,
    random_state=cfg.seed
)
lr_base.fit(Xtr_tab_std, y_tr)

p_lr = lr_base.predict_proba(Xva_tab_std)[:, 1]
pr_lr = average_precision_score(y_val, p_lr)

print("Baseline Logistic Regression PR-AUC:", round(pr_lr, 4))

# -------------------------
# 13b.3 SHAP explainer
# -------------------------
explainer = shap.LinearExplainer(
    lr_base,
    Xtr_tab_std,
    feature_names=feature_names_tab
)

shap_values = explainer(Xva_tab_std)

# -------------------------
# 13b.4 Global SHAP importance
# -------------------------
shap_imp = np.abs(shap_values.values).mean(axis=0)

shap_df = pd.DataFrame({
    "feature": feature_names_tab,
    "mean_abs_shap": shap_imp
}).sort_values("mean_abs_shap", ascending=False).reset_index(drop=True)

print("\n=== SHAP GLOBAL IMPORTANCE (TOP 15) ===")
display(shap_df.head(15))

# -------------------------
# 13b.5 Save SHAP artifacts
# -------------------------
shap_dir = os.path.join(cfg.out_dir, "shap_smoke")
os.makedirs(shap_dir, exist_ok=True)

shap_df.to_csv(os.path.join(shap_dir, "shap_global_importance.csv"), index=False)

print(f"\nSaved SHAP artifacts to: {os.path.abspath(shap_dir)}")

Baseline Logistic Regression PR-AUC: 0.1579

=== SHAP GLOBAL IMPORTANCE (TOP 15) ===


,feature,mean_abs_shap
0,energy,0.555981
1,Carbohydrate,0.403354
2,Total fat,0.274893
3,high cholesterol,0.228310
4,Systolic blood pressure,0.213215
5,General health condition,0.202079
6,Diastolic blood pressure,0.177110
7,smoke,0.164451
8,Total polyunsaturated fatty acids,0.162872
9,hypertension,0.158459



Saved SHAP artifacts to: /kaggle/working/trust_stroke_artifacts/shap_smoke


In [19]:
# ============================================================
# Block 15a: Conventional ML Baselines (Smoke Comparison)
# ============================================================

from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import average_precision_score, roc_auc_score, brier_score_loss
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# Optional (enable if available)
try:
    import lightgbm as lgb
    HAS_LGBM = True
except:
    HAS_LGBM = False

try:
    import xgboost as xgb
    HAS_XGB = True
except:
    HAS_XGB = False

try:
    from catboost import CatBoostClassifier
    HAS_CAT = True
except:
    HAS_CAT = False

print("LGBM:", HAS_LGBM, "| XGB:", HAS_XGB, "| CatBoost:", HAS_CAT)

# -------------------------
# 15a.1 Prepare raw pandas splits
# -------------------------
Xtr_df = X_tr.copy()
Xva_df = X_va.copy()

ytr = y_tr.copy()
yva = y_val.copy()

# -------------------------
# 15a.2 Column transformer (one-hot for ML models)
# -------------------------
preprocess_ohe = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CAT_COLS),
        ("num", "passthrough", NUM_COLS),
    ]
)

# -------------------------
# 15a.3 Helper to evaluate model
# -------------------------
def eval_model(name, model, Xtr, ytr, Xva, yva):
    model.fit(Xtr, ytr)
    p = model.predict_proba(Xva)[:, 1]

    return {
        "model": name,
        "pr_auc": average_precision_score(yva, p),
        "roc_auc": roc_auc_score(yva, p),
        "brier": brier_score_loss(yva, p),
        "ece": ece_score(yva, p, n_bins=cfg.ece_bins)
    }

results = []

# -------------------------
# 15a.4 Logistic Regression
# -------------------------
lr_pipe = Pipeline([
    ("prep", preprocess_ohe),
    ("clf", LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        solver="lbfgs",
        n_jobs=-1
    ))
])

results.append(eval_model("LogisticRegression", lr_pipe, Xtr_df, ytr, Xva_df, yva))

# -------------------------
# 15a.5 Random Forest
# -------------------------
rf_pipe = Pipeline([
    ("prep", preprocess_ohe),
    ("clf", RandomForestClassifier(
        n_estimators=400,
        max_depth=None,
        min_samples_leaf=5,
        class_weight="balanced",
        n_jobs=-1,
        random_state=cfg.seed
    ))
])

results.append(eval_model("RandomForest", rf_pipe, Xtr_df, ytr, Xva_df, yva))

# -------------------------
# 15a.6 LightGBM
# -------------------------
if HAS_LGBM:
    lgbm_pipe = Pipeline([
        ("prep", preprocess_ohe),
        ("clf", lgb.LGBMClassifier(
            n_estimators=500,
            learning_rate=0.05,
            num_leaves=31,
            subsample=0.8,
            colsample_bytree=0.8,
            class_weight="balanced",
            random_state=cfg.seed
        ))
    ])
    results.append(eval_model("LightGBM", lgbm_pipe, Xtr_df, ytr, Xva_df, yva))

# -------------------------
# 15a.7 XGBoost
# -------------------------
if HAS_XGB:
    xgb_pipe = Pipeline([
        ("prep", preprocess_ohe),
        ("clf", xgb.XGBClassifier(
            n_estimators=500,
            max_depth=6,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            eval_metric="logloss",
            scale_pos_weight=(len(ytr) - ytr.sum()) / ytr.sum(),
            random_state=cfg.seed
        ))
    ])
    results.append(eval_model("XGBoost", xgb_pipe, Xtr_df, ytr, Xva_df, yva))

# -------------------------
# 15a.8 CatBoost (native categorical handling)
# -------------------------
if HAS_CAT:
    cat_idx = [Xtr_df.columns.get_loc(c) for c in CAT_COLS]

    cat_model = CatBoostClassifier(
        iterations=600,
        depth=6,
        learning_rate=0.05,
        loss_function="Logloss",
        eval_metric="AUC",
        verbose=False,
        random_seed=cfg.seed,
        auto_class_weights="Balanced"
    )

    cat_model.fit(
        Xtr_df,
        ytr,
        cat_features=cat_idx
    )

    p_cat = cat_model.predict_proba(Xva_df)[:, 1]

    results.append({
        "model": "CatBoost",
        "pr_auc": average_precision_score(yva, p_cat),
        "roc_auc": roc_auc_score(yva, p_cat),
        "brier": brier_score_loss(yva, p_cat),
        "ece": ece_score(yva, p_cat, n_bins=cfg.ece_bins)
    })

# -------------------------
# 15a.9 Results table
# -------------------------
baseline_df = pd.DataFrame(results).sort_values(
    "pr_auc", ascending=False
).reset_index(drop=True)

print("=== CONVENTIONAL ML BASELINES (VAL) ===")
display(baseline_df)

# -------------------------
# 15a.10 Save artifacts
# -------------------------
base_dir = os.path.join(cfg.out_dir, "baseline_ml_smoke")
os.makedirs(base_dir, exist_ok=True)

baseline_df.to_csv(os.path.join(base_dir, "baseline_comparison.csv"), index=False)

print(f"\nSaved baseline ML results to: {os.path.abspath(base_dir)}")

LGBM: True | XGB: True | CatBoost: True


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


[LightGBM] [Info] Number of positive: 271, number of negative: 3181
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000802 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3987
[LightGBM] [Info] Number of data points in the train set: 3452, number of used features: 64
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000
=== CONVENTIONAL ML BASELINES (VAL) ===


,model,pr_auc,roc_auc,brier,ece
0,LogisticRegression,0.147162,0.644640,0.215701,0.327653
1,RandomForest,0.122972,0.645128,0.083442,0.091545
2,CatBoost,0.119707,0.612223,0.083959,0.066915
3,LightGBM,0.112102,0.584087,0.078405,0.068387
4,XGBoost,0.100823,0.591250,0.080748,0.061158



Saved baseline ML results to: /kaggle/working/trust_stroke_artifacts/baseline_ml_smoke


In [1]:
# =========================================================
# TASK 0.1 — Kaggle-ready project skeleton (repro + exports)
# =========================================================
# What you get from this cell:
# - Reproducible config + seeding
# - Output folder structure
# - Core metrics (PR-AUC, ROC-AUC, Brier, ECE, NLL, etc.)
# - High-quality figure saving (PDF + PNG 600dpi)
# - Table export (CSV + LaTeX + optional XLSX)
#
# Run this cell ONCE at the top of your Kaggle notebook.

import os, json, math, random, time
from dataclasses import dataclass, asdict
from typing import Dict, Any, Tuple, Optional, List

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    accuracy_score, f1_score, precision_score, recall_score,
    confusion_matrix, log_loss
)

# ---------------------------
# 0) CONFIG
# ---------------------------
@dataclass
class CFG:
    # Repro
    seed: int = 42

    # Output root (Kaggle-friendly)
    out_root: str = "/kaggle/working/outputs"
    exp_name: str = "cgt_stroke_v1"

    # Figure defaults (publication-friendly)
    fig_dpi_png: int = 600
    fig_format_pdf: str = "pdf"
    fig_format_png: str = "png"
    fig_facecolor: str = "white"
    fig_bbox_inches: str = "tight"
    fig_pad_inches: float = 0.02
    fig_default_size: Tuple[float, float] = (6.8, 4.2)  # inches

    # Calibration defaults
    ece_bins: int = 15

    # Table defaults
    latex_floatfmt: str = "%.4f"
    save_xlsx: bool = True  # set False if you don't want xlsx

cfg = CFG()

# ---------------------------
# 1) OUTPUT DIRS
# ---------------------------
def make_dirs(cfg: CFG) -> Dict[str, str]:
    base = os.path.join(cfg.out_root, cfg.exp_name)
    d = {
        "base": base,
        "fig": os.path.join(base, "figures"),
        "tab": os.path.join(base, "tables"),
        "pred": os.path.join(base, "predictions"),
        "ckpt": os.path.join(base, "checkpoints"),
        "log": os.path.join(base, "logs"),
    }
    for k, p in d.items():
        os.makedirs(p, exist_ok=True)
    # Save config snapshot
    with open(os.path.join(d["log"], "config.json"), "w") as f:
        json.dump(asdict(cfg), f, indent=2)
    return d

DIRS = make_dirs(cfg)
print("✅ Output dirs:", DIRS)

# ---------------------------
# 2) REPRODUCIBILITY
# ---------------------------
def seed_everything(seed: int = 42, deterministic: bool = True) -> None:
    random.seed(seed)
    np.random.seed(seed)
    try:
        import torch
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        if deterministic:
            torch.backends.cudnn.deterministic = True
            torch.backends.cudnn.benchmark = False
    except Exception:
        pass

seed_everything(cfg.seed)
print(f"✅ Seed set to {cfg.seed}")

# ---------------------------
# 3) SMALL HELPERS
# ---------------------------
def _to_np_1d(x) -> np.ndarray:
    """Convert list/Series/np/torch to 1D numpy."""
    if x is None:
        return None
    if hasattr(x, "detach"):  # torch tensor
        x = x.detach().cpu().numpy()
    if hasattr(x, "values"):  # pandas
        x = x.values
    x = np.asarray(x).reshape(-1)
    return x

def sigmoid(z):
    z = np.asarray(z)
    return 1 / (1 + np.exp(-z))

def ensure_prob(p: np.ndarray) -> np.ndarray:
    """Ensure probabilities in (0,1)."""
    p = _to_np_1d(p)
    eps = 1e-12
    return np.clip(p, eps, 1 - eps)

# ---------------------------
# 4) CALIBRATION METRICS
# ---------------------------
def expected_calibration_error(y_true, y_prob, n_bins: int = 15) -> float:
    """
    ECE for binary classification.
    Bins by confidence, measures |acc - conf| weighted by bin size.
    """
    y = _to_np_1d(y_true).astype(int)
    p = ensure_prob(y_prob)

    bin_edges = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    for i in range(n_bins):
        lo, hi = bin_edges[i], bin_edges[i+1]
        m = (p >= lo) & (p < hi) if i < n_bins - 1 else (p >= lo) & (p <= hi)
        if m.sum() == 0:
            continue
        acc = y[m].mean()
        conf = p[m].mean()
        ece += (m.mean()) * abs(acc - conf)
    return float(ece)

def brier_score(y_true, y_prob) -> float:
    y = _to_np_1d(y_true).astype(float)
    p = ensure_prob(y_prob)
    return float(np.mean((p - y) ** 2))

# ---------------------------
# 5) THRESHOLD METRICS (VAL)
# ---------------------------
def find_best_threshold(y_true, y_prob, metric: str = "f1") -> Tuple[float, float]:
    """
    Find threshold in [0,1] maximizing a metric on validation.
    metric: 'f1' or 'youden' (tpr-fpr) or 'balacc'
    """
    y = _to_np_1d(y_true).astype(int)
    p = ensure_prob(y_prob)

    ths = np.linspace(0.01, 0.99, 99)
    best_t, best_v = 0.5, -1e9

    for t in ths:
        pred = (p >= t).astype(int)
        if metric == "f1":
            v = f1_score(y, pred, zero_division=0)
        elif metric == "balacc":
            tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0,1]).ravel()
            tpr = tp / (tp + fn + 1e-12)
            tnr = tn / (tn + fp + 1e-12)
            v = 0.5 * (tpr + tnr)
        elif metric == "youden":
            tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0,1]).ravel()
            tpr = tp / (tp + fn + 1e-12)
            fpr = fp / (fp + tn + 1e-12)
            v = tpr - fpr
        else:
            raise ValueError("metric must be one of {'f1','balacc','youden'}")

        if v > best_v:
            best_v, best_t = float(v), float(t)

    return best_t, best_v

# ---------------------------
# 6) MASTER EVALUATION FUNCTION
# ---------------------------
def evaluate_binary(
    y_true,
    y_prob,
    threshold: float = 0.5,
    ece_bins: int = 15,
) -> Dict[str, float]:
    y = _to_np_1d(y_true).astype(int)
    p = ensure_prob(y_prob)

    pred = (p >= threshold).astype(int)

    out = {}
    out["threshold"] = float(threshold)
    # Ranking
    out["pr_auc"] = float(average_precision_score(y, p))
    out["roc_auc"] = float(roc_auc_score(y, p))
    # Classification
    out["acc"] = float(accuracy_score(y, pred))
    out["f1"] = float(f1_score(y, pred, zero_division=0))
    out["precision"] = float(precision_score(y, pred, zero_division=0))
    out["recall"] = float(recall_score(y, pred, zero_division=0))
    # Calibration
    out["brier"] = float(brier_score(y, p))
    out["ece"] = float(expected_calibration_error(y, p, n_bins=ece_bins))
    # NLL
    out["nll"] = float(log_loss(y, p, labels=[0,1]))
    # Confusion elements
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0,1]).ravel()
    out["tn"] = float(tn); out["fp"] = float(fp); out["fn"] = float(fn); out["tp"] = float(tp)
    return out

# ---------------------------
# 7) HIGH-QUALITY FIGURE EXPORT
# ---------------------------
def set_pub_style():
    # Clean, readable defaults; avoid heavy styling.
    plt.rcParams.update({
        "figure.facecolor": cfg.fig_facecolor,
        "savefig.facecolor": cfg.fig_facecolor,
        "axes.titlesize": 12,
        "axes.labelsize": 11,
        "xtick.labelsize": 10,
        "ytick.labelsize": 10,
        "legend.fontsize": 10,
        "font.size": 11,
        "axes.grid": True,
        "grid.alpha": 0.25,
        "grid.linestyle": "--",
    })

set_pub_style()

def save_fig(name: str, fig=None, subdir: str = "fig", close: bool = True) -> Dict[str, str]:
    """
    Save figure as BOTH vector PDF and high-res PNG.
    Returns file paths.
    """
    if fig is None:
        fig = plt.gcf()

    # ensure consistent size if user forgot
    try:
        fig.set_size_inches(*cfg.fig_default_size)
    except Exception:
        pass

    out_dir = DIRS[subdir] if subdir in DIRS else os.path.join(DIRS["base"], subdir)
    os.makedirs(out_dir, exist_ok=True)

    pdf_path = os.path.join(out_dir, f"{name}.{cfg.fig_format_pdf}")
    png_path = os.path.join(out_dir, f"{name}.{cfg.fig_format_png}")

    fig.savefig(pdf_path, bbox_inches=cfg.fig_bbox_inches, pad_inches=cfg.fig_pad_inches)
    fig.savefig(png_path, dpi=cfg.fig_dpi_png, bbox_inches=cfg.fig_bbox_inches, pad_inches=cfg.fig_pad_inches)

    if close:
        plt.close(fig)

    return {"pdf": pdf_path, "png": png_path}

# ---------------------------
# 8) TABLE EXPORT (CSV + LaTeX + optional XLSX)
# ---------------------------
def save_table(df: pd.DataFrame, name: str, index: bool = False) -> Dict[str, str]:
    out = {}
    csv_path = os.path.join(DIRS["tab"], f"{name}.csv")
    tex_path = os.path.join(DIRS["tab"], f"{name}.tex")
    df.to_csv(csv_path, index=index)
    out["csv"] = csv_path

    # LaTeX (booktabs-friendly)
    try:
        tex = df.to_latex(index=index, escape=False, float_format=cfg.latex_floatfmt,
                          longtable=False, caption=None, label=None)
        # small quality tweak: booktabs feel (sklearn latex already uses \toprule etc)
        with open(tex_path, "w") as f:
            f.write(tex)
        out["tex"] = tex_path
    except Exception as e:
        print("⚠️ LaTeX export failed:", e)

    if cfg.save_xlsx:
        xlsx_path = os.path.join(DIRS["tab"], f"{name}.xlsx")
        try:
            df.to_excel(xlsx_path, index=index)
            out["xlsx"] = xlsx_path
        except Exception as e:
            print("⚠️ XLSX export failed:", e)

    return out

# ---------------------------
# 9) STANDARD PLOTS WE'LL REUSE LATER
# ---------------------------
def plot_reliability_curve(y_true, y_prob, n_bins: int = 15, title: str = "Reliability Diagram"):
    """
    Produces a reliability diagram (calibration plot).
    """
    y = _to_np_1d(y_true).astype(int)
    p = ensure_prob(y_prob)

    bin_edges = np.linspace(0.0, 1.0, n_bins + 1)
    bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])
    accs, confs, counts = [], [], []

    for i in range(n_bins):
        lo, hi = bin_edges[i], bin_edges[i+1]
        m = (p >= lo) & (p < hi) if i < n_bins - 1 else (p >= lo) & (p <= hi)
        if m.sum() == 0:
            accs.append(np.nan); confs.append(np.nan); counts.append(0)
        else:
            accs.append(y[m].mean())
            confs.append(p[m].mean())
            counts.append(int(m.sum()))

    fig = plt.figure()
    plt.plot([0,1],[0,1], linewidth=1)
    plt.plot(confs, accs, marker="o", linewidth=1)
    plt.xlabel("Mean predicted probability")
    plt.ylabel("Empirical accuracy")
    plt.title(title)
    plt.ylim(0,1); plt.xlim(0,1)
    plt.grid(True, alpha=0.25)
    return fig

def plot_hist_probs(y_prob, title: str = "Predicted probabilities"):
    p = ensure_prob(y_prob)
    fig = plt.figure()
    plt.hist(p, bins=30)
    plt.xlabel("Predicted probability")
    plt.ylabel("Count")
    plt.title(title)
    return fig

# ---------------------------
# 10) QUICK SELF-TEST (optional)
# ---------------------------
# This verifies metrics/exports quickly with dummy data.
y_dummy = np.array([0,0,1,0,1,0,0,1,0,0])
p_dummy = np.array([0.1,0.2,0.8,0.4,0.7,0.15,0.25,0.6,0.05,0.3])

metrics_dummy = evaluate_binary(y_dummy, p_dummy, threshold=0.5, ece_bins=cfg.ece_bins)
print("✅ Dummy metrics check:", {k: round(v,4) for k,v in metrics_dummy.items() if k in ["pr_auc","roc_auc","brier","ece","nll","f1"]})

fig = plot_reliability_curve(y_dummy, p_dummy, n_bins=cfg.ece_bins, title="Reliability (dummy)")
paths = save_fig("reliability_dummy", fig=fig)
print("✅ Saved dummy reliability fig:", paths)

df_dummy = pd.DataFrame([metrics_dummy])
tab_paths = save_table(df_dummy, "metrics_dummy")
print("✅ Saved dummy table:", tab_paths)

print("\n✅ TASK 0.1 complete. Next: TASK 1.1 (load dataset + sanity checks).")

✅ Output dirs: {'base': '/kaggle/working/outputs/cgt_stroke_v1', 'fig': '/kaggle/working/outputs/cgt_stroke_v1/figures', 'tab': '/kaggle/working/outputs/cgt_stroke_v1/tables', 'pred': '/kaggle/working/outputs/cgt_stroke_v1/predictions', 'ckpt': '/kaggle/working/outputs/cgt_stroke_v1/checkpoints', 'log': '/kaggle/working/outputs/cgt_stroke_v1/logs'}
✅ Seed set to 42
✅ Dummy metrics check: {'pr_auc': 1.0, 'roc_auc': 1.0, 'f1': 1.0, 'brier': 0.0678, 'ece': 0.235, 'nll': 0.2788}
✅ Saved dummy reliability fig: {'pdf': '/kaggle/working/outputs/cgt_stroke_v1/figures/reliability_dummy.pdf', 'png': '/kaggle/working/outputs/cgt_stroke_v1/figures/reliability_dummy.png'}
✅ Saved dummy table: {'csv': '/kaggle/working/outputs/cgt_stroke_v1/tables/metrics_dummy.csv', 'tex': '/kaggle/working/outputs/cgt_stroke_v1/tables/metrics_dummy.tex', 'xlsx': '/kaggle/working/outputs/cgt_stroke_v1/tables/metrics_dummy.xlsx'}

✅ TASK 0.1 complete. Next: TASK 1.1 (load dataset + sanity checks).


In [2]:
# =========================================================
# TASK 1.1 — Load dataset + sanity checks + EDA exports
# =========================================================
# What you get from this cell:
# - Load the stroke dataset from Kaggle input (auto-detect CSV)
# - Identify target column robustly (stroke / target / label)
# - Clean column names, basic type inference (cont/bin/cat)
# - Missingness + class imbalance summary
# - High-quality EDA figures + summary tables saved to outputs/
#
# IMPORTANT:
# 1) Add your dataset to this Kaggle notebook (Dataset -> Add data).
# 2) Set DATA_DIR below to the folder that contains the CSV.
#    You can print os.listdir("/kaggle/input") to see names.

import os, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ---------------------------
# 0) Point to your Kaggle input dataset folder
# ---------------------------
# Example:
# DATA_DIR = "/kaggle/input/stroke-prediction-dataset"
DATA_DIR = "/kaggle/input"  # we'll auto-search below

# ---------------------------
# 1) Find candidate CSV(s)
# ---------------------------
def find_csvs(root="/kaggle/input"):
    csvs = []
    for dp, dn, fn in os.walk(root):
        for f in fn:
            if f.lower().endswith(".csv"):
                csvs.append(os.path.join(dp, f))
    return sorted(csvs)

csv_paths = find_csvs(DATA_DIR)
print("Found CSV files:")
for p in csv_paths[:30]:
    print(" -", p)
if len(csv_paths) > 30:
    print(f" ... (+{len(csv_paths)-30} more)")

assert len(csv_paths) > 0, "No CSV found under /kaggle/input. Add the dataset first."

# Heuristic: pick the largest CSV (often the main table)
csv_sizes = [(p, os.path.getsize(p)) for p in csv_paths]
csv_sizes = sorted(csv_sizes, key=lambda x: x[1], reverse=True)
CSV_PATH = csv_sizes[0][0]
print("\n✅ Using CSV:", CSV_PATH, f"({csv_sizes[0][1]/1e6:.2f} MB)")

# ---------------------------
# 2) Load CSV safely
# ---------------------------
df = pd.read_csv(CSV_PATH)
print("\nShape:", df.shape)
print("Head:")
display(df.head(3))

# ---------------------------
# 3) Normalize column names
# ---------------------------
def normalize_col(c: str) -> str:
    c = c.strip()
    c = re.sub(r"\s+", "_", c)
    c = re.sub(r"[^0-9a-zA-Z_]+", "", c)
    return c

orig_cols = list(df.columns)
df.columns = [normalize_col(c) for c in df.columns]

# Record mapping
col_map = pd.DataFrame({"original": orig_cols, "normalized": df.columns})
save_table(col_map, "colname_mapping", index=False)

# ---------------------------
# 4) Identify target column
# ---------------------------
def find_target_col(columns):
    # common names
    candidates = ["stroke", "target", "label", "y", "outcome", "class"]
    cols_lower = [c.lower() for c in columns]
    for c in candidates:
        if c in cols_lower:
            return columns[cols_lower.index(c)]
    # fallback: first column if it is binary-ish
    return None

TARGET = find_target_col(df.columns)

def is_binary_like(s: pd.Series) -> bool:
    s2 = s.dropna()
    if s2.empty:
        return False
    # if values are strings like '0'/'1'
    try:
        u = pd.unique(s2.astype(str))
    except Exception:
        u = pd.unique(s2)
    # allow boolean-like
    if len(u) <= 2:
        return True
    return False

if TARGET is None:
    # Try: first column if binary-like
    first = df.columns[0]
    if is_binary_like(df[first]):
        TARGET = first
    else:
        # try any binary-like column named with 'stroke'
        for c in df.columns:
            if "stroke" in c.lower() and is_binary_like(df[c]):
                TARGET = c
                break

assert TARGET is not None, "Could not auto-detect target column. Tell me the target name and I'll adapt."
print("\n✅ Detected TARGET:", TARGET)

# ---------------------------
# 5) Basic cleanup: drop duplicates, fix target to int {0,1}
# ---------------------------
df = df.drop_duplicates().reset_index(drop=True)

y_raw = df[TARGET]

# Convert target to 0/1 int
def to_binary_01(s: pd.Series) -> pd.Series:
    # if already numeric with two values
    if pd.api.types.is_numeric_dtype(s):
        u = sorted(pd.unique(s.dropna()))
        if len(u) == 2:
            # map smallest->0, largest->1
            return s.map({u[0]: 0, u[1]: 1}).astype("Int64")
        # if already 0/1
        if set(u).issubset({0,1}):
            return s.astype("Int64")
    # else string-like
    s2 = s.astype(str).str.strip().str.lower()
    # common positives
    pos = {"1","true","yes","y","stroke","positive"}
    neg = {"0","false","no","n","nostroke","negative"}
    mapped = []
    for v in s2:
        if v in pos:
            mapped.append(1)
        elif v in neg:
            mapped.append(0)
        else:
            mapped.append(np.nan)
    return pd.Series(mapped, index=s.index, dtype="Float64")

df[TARGET] = to_binary_01(df[TARGET])
missing_y = int(df[TARGET].isna().sum())
if missing_y > 0:
    print(f"⚠️ {missing_y} rows have missing/unknown target. Dropping them.")
    df = df.dropna(subset=[TARGET]).reset_index(drop=True)
df[TARGET] = df[TARGET].astype(int)

# ---------------------------
# 6) Feature list + type inference
# ---------------------------
FEATURES = [c for c in df.columns if c != TARGET]

# Infer types: binary / categorical / continuous
bin_cols, cat_cols, cont_cols = [], [], []

for c in FEATURES:
    s = df[c]
    # If numeric with few unique -> binary or categorical
    if pd.api.types.is_numeric_dtype(s):
        u = pd.unique(s.dropna())
        nunq = len(u)
        if nunq <= 2:
            bin_cols.append(c)
        elif nunq <= 15:
            # treat low-card numeric as categorical (often coded categories)
            cat_cols.append(c)
        else:
            cont_cols.append(c)
    else:
        # object/string: categorical
        cat_cols.append(c)

# Summaries
print("\nFeature counts:")
print(" - Binary:", len(bin_cols))
print(" - Categorical:", len(cat_cols))
print(" - Continuous:", len(cont_cols))

type_table = pd.DataFrame({
    "feature": FEATURES,
    "dtype": [str(df[c].dtype) for c in FEATURES],
    "type_inferred": [
        "binary" if c in bin_cols else ("categorical" if c in cat_cols else "continuous")
        for c in FEATURES
    ],
    "n_unique_nonnull": [df[c].nunique(dropna=True) for c in FEATURES],
    "missing_frac": [float(df[c].isna().mean()) for c in FEATURES],
})
type_table = type_table.sort_values(["type_inferred","missing_frac"], ascending=[True, False]).reset_index(drop=True)
save_table(type_table, "feature_type_inference", index=False)
display(type_table.head(12))

# ---------------------------
# 7) Class imbalance summary
# ---------------------------
y = df[TARGET].values
pos = int((y == 1).sum())
neg = int((y == 0).sum())
ratio = pos / max(1, (pos + neg))
print(f"\nClass counts: pos={pos} neg={neg} pos_ratio={ratio:.4f}")

imb = pd.DataFrame([{"pos": pos, "neg": neg, "pos_ratio": ratio, "n": len(df)}])
save_table(imb, "class_imbalance", index=False)

# ---------------------------
# 8) EDA Figures (high-quality exports)
# ---------------------------
# 8.1 Target distribution
fig = plt.figure()
plt.bar(["No Stroke (0)", "Stroke (1)"], [neg, pos])
plt.ylabel("Count")
plt.title("Target Distribution")
paths = save_fig("eda_target_distribution", fig=fig)
print("✅ Saved:", paths)

# 8.2 Missingness by feature (top 25)
miss = type_table.sort_values("missing_frac", ascending=False).head(25)
fig = plt.figure(figsize=cfg.fig_default_size)
plt.barh(miss["feature"][::-1], miss["missing_frac"][::-1])
plt.xlabel("Missing fraction")
plt.title("Top-25 Features by Missingness")
paths = save_fig("eda_missingness_top25", fig=fig)
print("✅ Saved:", paths)

# 8.3 Continuous feature histograms (top 6 with least missingness)
cont_sorted = sorted(cont_cols, key=lambda c: df[c].isna().mean())
top_cont = cont_sorted[:6]
if len(top_cont) > 0:
    for c in top_cont:
        fig = plt.figure()
        x = df[c].dropna().values
        plt.hist(x, bins=40)
        plt.xlabel(c)
        plt.ylabel("Count")
        plt.title(f"Histogram: {c}")
        paths = save_fig(f"eda_hist_{c}", fig=fig)
    print(f"✅ Saved {len(top_cont)} continuous histograms.")

# 8.4 Correlation heatmap (continuous only, top 20 by variance to keep readable)
if len(cont_cols) >= 2:
    # pick up to 20 continuous features by variance
    var = df[cont_cols].var(numeric_only=True).sort_values(ascending=False)
    top_corr = list(var.index[:20])
    corr = df[top_corr].corr(numeric_only=True)

    fig = plt.figure(figsize=(7.2, 6.2))
    plt.imshow(corr.values, aspect="auto")
    plt.xticks(range(len(top_corr)), top_corr, rotation=90)
    plt.yticks(range(len(top_corr)), top_corr)
    plt.title("Correlation Heatmap (Top-20 continuous features)")
    plt.colorbar()
    paths = save_fig("eda_corr_top20_continuous", fig=fig)
    print("✅ Saved:", paths)

# ---------------------------
# 9) Save cleaned dataset snapshot (optional, for reproducibility)
# ---------------------------
clean_path = os.path.join(DIRS["base"], "data_clean.csv")
df.to_csv(clean_path, index=False)
print("\n✅ Saved cleaned dataset snapshot:", clean_path)

# ---------------------------
# 10) Export a small "data card" summary for the paper
# ---------------------------
data_card = pd.DataFrame([{
    "rows": df.shape[0],
    "cols_total": df.shape[1],
    "n_features": len(FEATURES),
    "n_binary": len(bin_cols),
    "n_categorical": len(cat_cols),
    "n_continuous": len(cont_cols),
    "pos_count": pos,
    "neg_count": neg,
    "pos_ratio": ratio,
    "target_col": TARGET
}])
save_table(data_card, "data_card_summary", index=False)
display(data_card)

print("\n✅ TASK 1.1 complete.")
print("Next: TASK 2.1 (train/val/test split protocol saved to disk).")

Found CSV files:
 - /kaggle/input/stroke-dataset/Stroke.csv

✅ Using CSV: /kaggle/input/stroke-dataset/Stroke.csv (0.66 MB)

Shape: (4603, 36)
Head:


,stroke,gender,age,Race,Marital status,alcohol,smoke,sleep disorder,Health Insurance,General health condition,...,energy,protein,Carbohydrate,Dietary fiber,Total fat,Total saturated fatty acids,Total monounsaturated fatty acids,Total polyunsaturated fatty acids,Potassium,Sodium
0,0,2,2,5,1,0,0,2,2,3,...,1598,62.78,192.19,10.0,65.64,25.112,24.090,8.543,2887,2969
1,0,2,2,1,1,0,0,1,2,3,...,1547,45.35,256.02,17.0,42.56,13.423,15.389,10.613,2058,2091
2,1,1,2,3,1,1,1,2,1,3,...,2466,81.56,254.49,13.0,103.32,43.295,36.727,15.366,3117,5233



✅ Detected TARGET: stroke

Feature counts:
 - Binary: 9
 - Categorical: 6
 - Continuous: 20


,feature,dtype,type_inferred,n_unique_nonnull,missing_frac
0,gender,int64,binary,2,0.0
1,alcohol,int64,binary,2,0.0
2,smoke,int64,binary,2,0.0
3,sleep_disorder,int64,binary,2,0.0
4,Health_Insurance,int64,binary,2,0.0
5,diabetes,int64,binary,2,0.0
6,hypertension,int64,binary,2,0.0
7,high_cholesterol,int64,binary,2,0.0
8,Coronary_Heart_Disease,int64,binary,2,0.0
9,age,int64,categorical,3,0.0



Class counts: pos=362 neg=4241 pos_ratio=0.0786
✅ Saved: {'pdf': '/kaggle/working/outputs/cgt_stroke_v1/figures/eda_target_distribution.pdf', 'png': '/kaggle/working/outputs/cgt_stroke_v1/figures/eda_target_distribution.png'}
✅ Saved: {'pdf': '/kaggle/working/outputs/cgt_stroke_v1/figures/eda_missingness_top25.pdf', 'png': '/kaggle/working/outputs/cgt_stroke_v1/figures/eda_missingness_top25.png'}
✅ Saved 6 continuous histograms.
✅ Saved: {'pdf': '/kaggle/working/outputs/cgt_stroke_v1/figures/eda_corr_top20_continuous.pdf', 'png': '/kaggle/working/outputs/cgt_stroke_v1/figures/eda_corr_top20_continuous.png'}

✅ Saved cleaned dataset snapshot: /kaggle/working/outputs/cgt_stroke_v1/data_clean.csv


,rows,cols_total,n_features,n_binary,n_categorical,n_continuous,pos_count,neg_count,pos_ratio,target_col
0,4603,36,35,9,6,20,362,4241,0.078644,stroke



✅ TASK 1.1 complete.
Next: TASK 2.1 (train/val/test split protocol saved to disk).


In [3]:
# =========================================================
# TASK 2.1 — Train / Validation / Test split protocol
# =========================================================
# What this cell does:
# - Creates a STRICT, leakage-safe evaluation protocol
# - Uses a FIXED held-out test set (never touched during tuning)
# - Uses repeated stratified splits for train/val (for uncertainty & robustness)
# - Saves split indices to disk (reproducibility for reviewers)
# - Exports split statistics as tables
#
# This protocol is suitable for high-quality medical AI journals.

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedShuffleSplit

# ---------------------------
# 0) Load cleaned data snapshot
# ---------------------------
DATA_CLEAN_PATH = os.path.join(DIRS["base"], "data_clean.csv")
df = pd.read_csv(DATA_CLEAN_PATH)

TARGET = "stroke"
y = df[TARGET].values
n = len(df)

print("Loaded cleaned data:", df.shape)

# ---------------------------
# 1) Define split strategy
# ---------------------------
# We will do:
# - 20% fixed TEST set (held out, untouched)
# - Remaining 80% used for repeated train/val splits
#
# Rationale:
# - Fixed test set = honest final reporting
# - Repeated validation = robust uncertainty & stability analysis

TEST_SIZE = 0.20
N_REPEATS = 5        # number of repeated splits
VAL_SIZE = 0.20     # validation fraction *within train+val pool*

# ---------------------------
# 2) Create fixed TEST split
# ---------------------------
sss_test = StratifiedShuffleSplit(
    n_splits=1,
    test_size=TEST_SIZE,
    random_state=cfg.seed
)

trainval_idx, test_idx = next(sss_test.split(np.zeros(n), y))

trainval_idx = np.array(trainval_idx)
test_idx = np.array(test_idx)

print(f"Train+Val size: {len(trainval_idx)}")
print(f"Test size:      {len(test_idx)}")

# ---------------------------
# 3) Repeated Train / Val splits (inside Train+Val only)
# ---------------------------
splits = []

for r in range(N_REPEATS):
    sss_val = StratifiedShuffleSplit(
        n_splits=1,
        test_size=VAL_SIZE,
        random_state=cfg.seed + r + 1
    )
    tr_idx_rel, val_idx_rel = next(
        sss_val.split(np.zeros(len(trainval_idx)), y[trainval_idx])
    )

    tr_idx = trainval_idx[tr_idx_rel]
    val_idx = trainval_idx[val_idx_rel]

    splits.append({
        "repeat": r,
        "train_idx": tr_idx,
        "val_idx": val_idx,
    })

print(f"Created {len(splits)} repeated train/val splits.")

# ---------------------------
# 4) Sanity checks (VERY IMPORTANT)
# ---------------------------
def check_split(name, idx):
    yy = y[idx]
    return {
        "name": name,
        "n": len(idx),
        "pos": int((yy == 1).sum()),
        "neg": int((yy == 0).sum()),
        "pos_ratio": float((yy == 1).mean())
    }

summary_rows = []

summary_rows.append(check_split("TEST", test_idx))

for s in splits:
    summary_rows.append(check_split(f"TRAIN_r{s['repeat']}", s["train_idx"]))
    summary_rows.append(check_split(f"VAL_r{s['repeat']}", s["val_idx"]))

split_summary = pd.DataFrame(summary_rows)
display(split_summary)

# Save summary table
save_table(split_summary, "split_class_distribution", index=False)

# ---------------------------
# 5) Save split indices to disk (reproducibility)
# ---------------------------
# We save everything as numpy arrays + a JSON manifest

split_dir = os.path.join(DIRS["base"], "splits")
os.makedirs(split_dir, exist_ok=True)

np.save(os.path.join(split_dir, "test_idx.npy"), test_idx)

manifest = {
    "test_size": TEST_SIZE,
    "val_size_within_trainval": VAL_SIZE,
    "n_repeats": N_REPEATS,
    "seed": cfg.seed,
    "splits": []
}

for s in splits:
    r = s["repeat"]
    tr_path = os.path.join(split_dir, f"train_idx_r{r}.npy")
    va_path = os.path.join(split_dir, f"val_idx_r{r}.npy")

    np.save(tr_path, s["train_idx"])
    np.save(va_path, s["val_idx"])

    manifest["splits"].append({
        "repeat": r,
        "train_idx": tr_path,
        "val_idx": va_path
    })

manifest_path = os.path.join(split_dir, "split_manifest.json")
with open(manifest_path, "w") as f:
    import json
    json.dump(manifest, f, indent=2)

print("\n✅ Saved split indices and manifest:")
print(" -", split_dir)
print(" -", manifest_path)

# ---------------------------
# 6) Final protocol statement (for paper)
# ---------------------------
protocol_text = f"""
Evaluation Protocol:
- Dataset size: {n}
- Positive class ratio: {y.mean():.4f}
- Fixed held-out test set: {TEST_SIZE*100:.0f}% (never used for tuning)
- Remaining data: repeated stratified train/validation splits
- Number of repeats: {N_REPEATS}
- Validation size (within train+val): {VAL_SIZE*100:.0f}%
- All preprocessing, calibration, threshold selection performed on training/validation only
"""

with open(os.path.join(DIRS["log"], "evaluation_protocol.txt"), "w") as f:
    f.write(protocol_text.strip())

print(protocol_text)
print("✅ TASK 2.1 complete.")
print("Next: TASK 3.1 (preprocessing pipeline fit on TRAIN only).")

Loaded cleaned data: (4603, 36)
Train+Val size: 3682
Test size:      921
Created 5 repeated train/val splits.


,name,n,pos,neg,pos_ratio
0,TEST,921,72,849,0.078176
1,TRAIN_r0,2945,232,2713,0.078778
2,VAL_r0,737,58,679,0.078697
3,TRAIN_r1,2945,232,2713,0.078778
4,VAL_r1,737,58,679,0.078697
5,TRAIN_r2,2945,232,2713,0.078778
6,VAL_r2,737,58,679,0.078697
7,TRAIN_r3,2945,232,2713,0.078778
8,VAL_r3,737,58,679,0.078697
9,TRAIN_r4,2945,232,2713,0.078778



✅ Saved split indices and manifest:
 - /kaggle/working/outputs/cgt_stroke_v1/splits
 - /kaggle/working/outputs/cgt_stroke_v1/splits/split_manifest.json

Evaluation Protocol:
- Dataset size: 4603
- Positive class ratio: 0.0786
- Fixed held-out test set: 20% (never used for tuning)
- Remaining data: repeated stratified train/validation splits
- Number of repeats: 5
- Validation size (within train+val): 20%
- All preprocessing, calibration, threshold selection performed on training/validation only

✅ TASK 2.1 complete.
Next: TASK 3.1 (preprocessing pipeline fit on TRAIN only).


In [4]:
# =========================================================
# TASK 3.1 — Preprocessing pipeline (fit on TRAIN only)
# =========================================================
# What you get:
# - Loads split indices (repeat 0 by default for debugging)
# - Builds a train-only preprocessor:
#     * continuous: median impute + standardize
#     * categorical: most-frequent impute + ordinal encode (stable for tabular DL)
#     * binary: keep as-is (0/1)
# - Produces processed arrays for:
#     X_tr, X_va, X_te and y_tr, y_va, y_te
# - Exports:
#     * feature metadata
#     * preprocessing summary tables
#
# Note: We will reuse the same Preprocessor object for each repeat.

import os, json
import numpy as np
import pandas as pd

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OrdinalEncoder

# ---------------------------
# 0) Load cleaned dataset + feature types inferred previously
# ---------------------------
DATA_CLEAN_PATH = os.path.join(DIRS["base"], "data_clean.csv")
df = pd.read_csv(DATA_CLEAN_PATH)

TARGET = "stroke"
FEATURES = [c for c in df.columns if c != TARGET]

# Load feature type inference table from Task 1.1
ft_path = os.path.join(DIRS["tab"], "feature_type_inference.csv")
ft = pd.read_csv(ft_path)

bin_cols = ft.loc[ft["type_inferred"]=="binary", "feature"].tolist()
cat_cols = ft.loc[ft["type_inferred"]=="categorical", "feature"].tolist()
cont_cols = ft.loc[ft["type_inferred"]=="continuous", "feature"].tolist()

print("Binary:", len(bin_cols), "Categorical:", len(cat_cols), "Continuous:", len(cont_cols))
print("Example categorical:", cat_cols[:10])
print("Example continuous:", cont_cols[:10])

# ---------------------------
# 1) Load split indices (use repeat r=0 for now)
# ---------------------------
SPLIT_DIR = os.path.join(DIRS["base"], "splits")
test_idx = np.load(os.path.join(SPLIT_DIR, "test_idx.npy"))

# Choose which repeat's train/val to build preprocessor on:
REPEAT = 0
train_idx = np.load(os.path.join(SPLIT_DIR, f"train_idx_r{REPEAT}.npy"))
val_idx   = np.load(os.path.join(SPLIT_DIR, f"val_idx_r{REPEAT}.npy"))

print(f"Using repeat {REPEAT}: train={len(train_idx)}, val={len(val_idx)}, test={len(test_idx)}")

# ---------------------------
# 2) Split data
# ---------------------------
df_tr = df.iloc[train_idx].reset_index(drop=True)
df_va = df.iloc[val_idx].reset_index(drop=True)
df_te = df.iloc[test_idx].reset_index(drop=True)

y_tr = df_tr[TARGET].values.astype(int)
y_va = df_va[TARGET].values.astype(int)
y_te = df_te[TARGET].values.astype(int)

# ---------------------------
# 3) Define a reusable Preprocessor (fit on train only)
# ---------------------------
class Preprocessor:
    def __init__(self, cont_cols, cat_cols, bin_cols):
        self.cont_cols = list(cont_cols)
        self.cat_cols  = list(cat_cols)
        self.bin_cols  = list(bin_cols)

        # Continuous: median impute + standardize
        self.cont_imputer = SimpleImputer(strategy="median")
        self.scaler = StandardScaler()

        # Categorical: most frequent + ordinal encoding
        # handle_unknown ensures we can transform val/test safely
        self.cat_imputer = SimpleImputer(strategy="most_frequent")
        self.encoder = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)

        self.fitted = False

    def fit(self, df_train: pd.DataFrame):
        # Continuous
        Xc = df_train[self.cont_cols]
        Xc_imp = self.cont_imputer.fit_transform(Xc)
        self.scaler.fit(Xc_imp)

        # Categorical
        Xk = df_train[self.cat_cols]
        Xk_imp = self.cat_imputer.fit_transform(Xk)
        self.encoder.fit(Xk_imp)

        self.fitted = True
        return self

    def transform(self, df_any: pd.DataFrame) -> Dict[str, np.ndarray]:
        assert self.fitted, "Call fit() first."

        # Continuous
        Xc = df_any[self.cont_cols]
        Xc_imp = self.cont_imputer.transform(Xc)
        Xc_sc = self.scaler.transform(Xc_imp).astype(np.float32)

        # Categorical
        Xk = df_any[self.cat_cols]
        Xk_imp = self.cat_imputer.transform(Xk)
        Xk_enc = self.encoder.transform(Xk_imp).astype(np.int64)

        # Binary
        Xb = df_any[self.bin_cols].astype(np.int64).values

        return {"cont": Xc_sc, "cat": Xk_enc, "bin": Xb}

# ---------------------------
# 4) Fit on train, transform all splits
# ---------------------------
pp = Preprocessor(cont_cols=cont_cols, cat_cols=cat_cols, bin_cols=bin_cols).fit(df_tr)

X_tr = pp.transform(df_tr)
X_va = pp.transform(df_va)
X_te = pp.transform(df_te)

print("\nShapes:")
print("Train:", {k: v.shape for k,v in X_tr.items()})
print("Val:  ", {k: v.shape for k,v in X_va.items()})
print("Test: ", {k: v.shape for k,v in X_te.items()})

# ---------------------------
# 5) Save preprocessor parameters + metadata
# ---------------------------
# We'll save only metadata now (pickle later when needed).
meta = {
    "target": TARGET,
    "bin_cols": bin_cols,
    "cat_cols": cat_cols,
    "cont_cols": cont_cols,
    "repeat_used": REPEAT,
    "n_train": int(len(train_idx)),
    "n_val": int(len(val_idx)),
    "n_test": int(len(test_idx)),
    "cat_cardinalities": {
        c: int(df[c].nunique(dropna=True)) for c in cat_cols
    }
}

meta_path = os.path.join(DIRS["log"], "preprocess_metadata.json")
with open(meta_path, "w") as f:
    json.dump(meta, f, indent=2)
print("\n✅ Saved preprocess metadata:", meta_path)

# Save a readable table of cardinals
card_table = pd.DataFrame({
    "cat_feature": cat_cols,
    "n_unique": [meta["cat_cardinalities"][c] for c in cat_cols]
}).sort_values("n_unique", ascending=False)
save_table(card_table, "categorical_cardinalities", index=False)

# ---------------------------
# 6) Quick sanity evaluation with a simple baseline (LogReg) on processed features
#     (This is just a sanity check, not final baseline section.)
# ---------------------------
from sklearn.linear_model import LogisticRegression

def pack_for_sklearn(Xdict):
    # For sklearn we will concatenate: [cont | bin | cat_encoded_as_numeric]
    return np.concatenate([Xdict["cont"], Xdict["bin"].astype(np.float32), Xdict["cat"].astype(np.float32)], axis=1)

Xtr_s = pack_for_sklearn(X_tr)
Xva_s = pack_for_sklearn(X_va)
Xte_s = pack_for_sklearn(X_te)

lr = LogisticRegression(max_iter=2000, class_weight="balanced", n_jobs=None)
lr.fit(Xtr_s, y_tr)

p_va = lr.predict_proba(Xva_s)[:,1]
t_best, v_best = find_best_threshold(y_va, p_va, metric="f1")
val_metrics = evaluate_binary(y_va, p_va, threshold=t_best, ece_bins=cfg.ece_bins)

p_te = lr.predict_proba(Xte_s)[:,1]
test_metrics = evaluate_binary(y_te, p_te, threshold=t_best, ece_bins=cfg.ece_bins)

res = pd.DataFrame([
    {"split":"val", **val_metrics},
    {"split":"test", **test_metrics},
])
display(res)
save_table(res, "sanity_logreg_after_preprocess", index=False)

# Save reliability plot for this sanity check (optional)
fig = plot_reliability_curve(y_va, p_va, n_bins=cfg.ece_bins, title="Reliability (LogReg sanity, val)")
save_fig("sanity_logreg_reliability_val", fig=fig)

print("\n✅ TASK 3.1 complete.")
print("Next: TASK 4.1 (full baseline suite: LR, RF, XGB/LGBM, CatBoost) using this protocol.")

Binary: 9 Categorical: 6 Continuous: 20
Example categorical: ['age', 'Race', 'Marital_status', 'General_health_condition', 'depression', 'Body_Mass_Index']
Example continuous: ['sleep_time', 'Minutes_sedentary_activity', 'Waist_Circumference', 'Systolic_blood_pressure', 'Diastolic_blood_pressure', 'Highdensity_lipoprotein', 'Triglyceride', 'Lowdensity_lipoprotein', 'Fasting_Glucose', 'Glycohemoglobin']
Using repeat 0: train=2945, val=737, test=921

Shapes:
Train: {'cont': (2945, 20), 'cat': (2945, 6), 'bin': (2945, 9)}
Val:   {'cont': (737, 20), 'cat': (737, 6), 'bin': (737, 9)}
Test:  {'cont': (921, 20), 'cat': (921, 6), 'bin': (921, 9)}

✅ Saved preprocess metadata: /kaggle/working/outputs/cgt_stroke_v1/logs/preprocess_metadata.json


,split,threshold,pr_auc,roc_auc,acc,f1,precision,recall,brier,ece,nll,tn,fp,fn,tp
0,val,0.7,0.157866,0.674039,0.850746,0.256757,0.211111,0.327586,0.200338,0.311486,0.587657,608.0,71.0,39.0,19.0
1,test,0.7,0.151794,0.613696,0.831705,0.143646,0.119266,0.180556,0.212311,0.314378,0.606869,753.0,96.0,59.0,13.0



✅ TASK 3.1 complete.
Next: TASK 4.1 (full baseline suite: LR, RF, XGB/LGBM, CatBoost) using this protocol.


In [5]:
# =========================================================
# TASK 4.1 — Baseline suite (publication-ready tables + figs)
# =========================================================

import os, json, warnings
import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score

warnings.filterwarnings("ignore")

# ---------------------------
# 0) Load data + splits + feature types
# ---------------------------
DATA_CLEAN_PATH = os.path.join(DIRS["base"], "data_clean.csv")
df = pd.read_csv(DATA_CLEAN_PATH)
TARGET = "stroke"

ft = pd.read_csv(os.path.join(DIRS["tab"], "feature_type_inference.csv"))
bin_cols = ft.loc[ft["type_inferred"]=="binary", "feature"].tolist()
cat_cols = ft.loc[ft["type_inferred"]=="categorical", "feature"].tolist()
cont_cols = ft.loc[ft["type_inferred"]=="continuous", "feature"].tolist()

SPLIT_DIR = os.path.join(DIRS["base"], "splits")
test_idx = np.load(os.path.join(SPLIT_DIR, "test_idx.npy"))

# Read manifest for repeats
manifest_path = os.path.join(SPLIT_DIR, "split_manifest.json")
with open(manifest_path, "r") as f:
    manifest = json.load(f)
N_REPEATS = manifest["n_repeats"]

print("Loaded:", df.shape, "Repeats:", N_REPEATS)

# ---------------------------
# 1) Preprocessor (same as Task 3.1)
# ---------------------------
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OrdinalEncoder

class Preprocessor:
    def __init__(self, cont_cols, cat_cols, bin_cols):
        self.cont_cols = list(cont_cols)
        self.cat_cols  = list(cat_cols)
        self.bin_cols  = list(bin_cols)
        self.cont_imputer = SimpleImputer(strategy="median")
        self.scaler = StandardScaler()
        self.cat_imputer = SimpleImputer(strategy="most_frequent")
        self.encoder = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
        self.fitted = False

    def fit(self, df_train: pd.DataFrame):
        Xc = df_train[self.cont_cols]
        Xc_imp = self.cont_imputer.fit_transform(Xc)
        self.scaler.fit(Xc_imp)

        Xk = df_train[self.cat_cols]
        Xk_imp = self.cat_imputer.fit_transform(Xk)
        self.encoder.fit(Xk_imp)

        self.fitted = True
        return self

    def transform(self, df_any: pd.DataFrame) -> Dict[str, np.ndarray]:
        assert self.fitted
        Xc_imp = self.cont_imputer.transform(df_any[self.cont_cols])
        Xc_sc  = self.scaler.transform(Xc_imp).astype(np.float32)

        Xk_imp = self.cat_imputer.transform(df_any[self.cat_cols])
        Xk_enc = self.encoder.transform(Xk_imp).astype(np.int64)

        Xb = df_any[self.bin_cols].astype(np.int64).values
        return {"cont": Xc_sc, "cat": Xk_enc, "bin": Xb}

def pack_for_sklearn(Xdict):
    return np.concatenate([Xdict["cont"],
                           Xdict["bin"].astype(np.float32),
                           Xdict["cat"].astype(np.float32)], axis=1)

# ---------------------------
# 2) Optional baseline libraries
# ---------------------------
HAS_XGB = False
HAS_LGBM = False
HAS_CAT = False

try:
    import xgboost as xgb
    HAS_XGB = True
except Exception:
    pass

try:
    import lightgbm as lgb
    HAS_LGBM = True
except Exception:
    pass

try:
    from catboost import CatBoostClassifier
    HAS_CAT = True
except Exception:
    pass

print("Optional libs:", {"xgboost": HAS_XGB, "lightgbm": HAS_LGBM, "catboost": HAS_CAT})

# ---------------------------
# 3) Define model factory
# ---------------------------
def make_models(seed: int):
    models = {}

    # Logistic Regression (strong baseline)
    models["LogReg"] = LogisticRegression(
        max_iter=3000,
        class_weight="balanced",
        n_jobs=None
    )

    # Random Forest (use class_weight balanced_subsample)
    models["RandomForest"] = RandomForestClassifier(
        n_estimators=600,
        max_depth=None,
        min_samples_leaf=2,
        class_weight="balanced_subsample",
        random_state=seed,
        n_jobs=-1
    )

    if HAS_XGB:
        models["XGBoost"] = xgb.XGBClassifier(
            n_estimators=1200,
            learning_rate=0.03,
            max_depth=4,
            subsample=0.85,
            colsample_bytree=0.85,
            reg_lambda=1.0,
            objective="binary:logistic",
            eval_metric="logloss",
            random_state=seed,
            n_jobs=-1
        )

    if HAS_LGBM:
        models["LightGBM"] = lgb.LGBMClassifier(
            n_estimators=2000,
            learning_rate=0.02,
            num_leaves=31,
            subsample=0.9,
            colsample_bytree=0.9,
            reg_lambda=1.0,
            random_state=seed,
            class_weight="balanced",
            n_jobs=-1
        )

    if HAS_CAT:
        models["CatBoost"] = CatBoostClassifier(
            iterations=4000,
            learning_rate=0.02,
            depth=5,
            l2_leaf_reg=3.0,
            loss_function="Logloss",
            eval_metric="AUC",
            random_seed=seed,
            verbose=False
        )

    return models

# ---------------------------
# 4) Run baselines over repeats
# ---------------------------
all_rows = []
pred_dump = []  # for later stacking / calibration comparisons

df_te = df.iloc[test_idx].reset_index(drop=True)
y_te = df_te[TARGET].values.astype(int)

for r in range(N_REPEATS):
    train_idx = np.load(os.path.join(SPLIT_DIR, f"train_idx_r{r}.npy"))
    val_idx   = np.load(os.path.join(SPLIT_DIR, f"val_idx_r{r}.npy"))

    df_tr = df.iloc[train_idx].reset_index(drop=True)
    df_va = df.iloc[val_idx].reset_index(drop=True)

    y_tr = df_tr[TARGET].values.astype(int)
    y_va = df_va[TARGET].values.astype(int)

    # Fit preprocessing on TRAIN only
    pp = Preprocessor(cont_cols, cat_cols, bin_cols).fit(df_tr)
    X_tr = pack_for_sklearn(pp.transform(df_tr))
    X_va = pack_for_sklearn(pp.transform(df_va))
    X_te = pack_for_sklearn(pp.transform(df_te))

    models = make_models(seed=cfg.seed + r + 123)

    for name, model in models.items():
        model.fit(X_tr, y_tr)

        # Validation probability
        p_va = model.predict_proba(X_va)[:, 1]
        t_best, _ = find_best_threshold(y_va, p_va, metric="f1")
        m_va = evaluate_binary(y_va, p_va, threshold=t_best, ece_bins=cfg.ece_bins)
        m_va.update({"model": name, "repeat": r, "split": "val"})

        # Test probability (same threshold)
        p_te = model.predict_proba(X_te)[:, 1]
        m_te = evaluate_binary(y_te, p_te, threshold=t_best, ece_bins=cfg.ece_bins)
        m_te.update({"model": name, "repeat": r, "split": "test"})

        all_rows.append(m_va)
        all_rows.append(m_te)

        # Save reliability plot for val (per repeat, per model)
        fig = plot_reliability_curve(y_va, p_va, n_bins=cfg.ece_bins,
                                     title=f"Reliability (VAL) — {name} — repeat {r}")
        save_fig(f"baseline_reliability_{name}_r{r}", fig=fig)

        pred_dump.append({
            "model": name, "repeat": r,
            "val_idx": val_idx.tolist(),
            "test_idx": test_idx.tolist(),
            "p_val": p_va.tolist(),
            "p_test": p_te.tolist(),
            "t_val_best": float(t_best)
        })

print("✅ Finished training baselines.")

# ---------------------------
# 5) Save per-repeat results
# ---------------------------
df_all = pd.DataFrame(all_rows)

# Sort for readability
cols_order = ["model","repeat","split","threshold","pr_auc","roc_auc","acc","f1","precision","recall","brier","ece","nll","tp","fp","tn","fn"]
df_all = df_all[[c for c in cols_order if c in df_all.columns]]
display(df_all.head(10))

save_table(df_all, "baseline_per_repeat_results", index=False)

# Save predictions
pred_path = os.path.join(DIRS["pred"], "baseline_predictions.json")
with open(pred_path, "w") as f:
    json.dump(pred_dump, f)
print("✅ Saved baseline predictions:", pred_path)

# ---------------------------
# 6) Aggregate mean±std across repeats (VAL and TEST separately)
# ---------------------------
def agg_mean_std(df_sub: pd.DataFrame, metrics: List[str]) -> pd.DataFrame:
    out = []
    for m in sorted(df_sub["model"].unique()):
        d = df_sub[df_sub["model"]==m]
        row = {"model": m}
        for k in metrics:
            row[f"{k}_mean"] = float(d[k].mean())
            row[f"{k}_std"]  = float(d[k].std(ddof=1)) if len(d) > 1 else 0.0
        out.append(row)
    return pd.DataFrame(out)

metrics_main = ["pr_auc","roc_auc","f1","recall","brier","ece","nll","acc"]
val_agg  = agg_mean_std(df_all[df_all["split"]=="val"], metrics_main).sort_values("pr_auc_mean", ascending=False)
test_agg = agg_mean_std(df_all[df_all["split"]=="test"], metrics_main).sort_values("pr_auc_mean", ascending=False)

display(val_agg)
display(test_agg)

save_table(val_agg,  "baseline_summary_val_meanstd",  index=False)
save_table(test_agg, "baseline_summary_test_meanstd", index=False)

# Also create a compact "paper table" (mean±std formatted)
def fmt_mean_std(mean, std):
    return f"{mean:.4f} ± {std:.4f}"

paper_rows = []
for m in test_agg["model"].tolist():
    row = {"model": m}
    mrow = test_agg[test_agg["model"]==m].iloc[0]
    for k in ["pr_auc","roc_auc","f1","recall","brier","ece"]:
        row[k] = fmt_mean_std(mrow[f"{k}_mean"], mrow[f"{k}_std"])
    paper_rows.append(row)

paper_table = pd.DataFrame(paper_rows)
display(paper_table)
save_table(paper_table, "baseline_paper_table_test_meanpmstd", index=False)

print("\n✅ TASK 4.1 complete.")
print("Next: TASK 5.1 (tabular transformer backbone for CGT-Stroke++).")

Loaded: (4603, 36) Repeats: 5
Optional libs: {'xgboost': True, 'lightgbm': True, 'catboost': True}
[LightGBM] [Info] Number of positive: 232, number of negative: 2713
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001029 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3934
[LightGBM] [Info] Number of data points in the train set: 2945, number of used features: 35
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000
[LightGBM] [Info] Number of positive: 232, number of negative: 2713
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001128 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3936
[LightGBM] [Info] Number of data points in the train set: 2945, number of used featur

,model,repeat,split,threshold,pr_auc,roc_auc,acc,f1,precision,recall,brier,ece,nll,tp,fp,tn,fn
0,LogReg,0,val,0.70,0.157866,0.674039,0.850746,0.256757,0.211111,0.327586,0.200338,0.311486,0.587657,19.0,71.0,608.0,39.0
1,LogReg,0,test,0.70,0.151794,0.613696,0.831705,0.143646,0.119266,0.180556,0.212311,0.314378,0.606869,13.0,96.0,753.0,59.0
2,RandomForest,0,val,0.10,0.129327,0.614849,0.587517,0.182796,0.108280,0.586207,0.072775,0.020814,0.277482,34.0,280.0,399.0,24.0
3,RandomForest,0,test,0.10,0.130372,0.601410,0.573290,0.172632,0.101737,0.569444,0.073249,0.038741,0.279500,41.0,362.0,487.0,31.0
4,XGBoost,0,val,0.13,0.115791,0.580164,0.875170,0.192982,0.196429,0.189655,0.076974,0.059519,0.359597,11.0,45.0,634.0,47.0
5,XGBoost,0,test,0.13,0.109602,0.575939,0.871878,0.132353,0.140625,0.125000,0.075970,0.056646,0.363726,9.0,55.0,794.0,63.0
6,LightGBM,0,val,0.02,0.124964,0.607867,0.793758,0.191489,0.138462,0.310345,0.079280,0.075124,0.446180,18.0,112.0,567.0,40.0
7,LightGBM,0,test,0.02,0.111461,0.586229,0.754615,0.162963,0.111111,0.305556,0.079210,0.075838,0.458371,22.0,176.0,673.0,50.0
8,CatBoost,0,val,0.03,0.113729,0.583947,0.763908,0.186916,0.128205,0.344828,0.077315,0.062015,0.383603,20.0,136.0,543.0,38.0
9,CatBoost,0,test,0.03,0.116118,0.600821,0.758958,0.189781,0.128713,0.361111,0.075935,0.063299,0.370253,26.0,176.0,673.0,46.0


✅ Saved baseline predictions: /kaggle/working/outputs/cgt_stroke_v1/predictions/baseline_predictions.json


,model,pr_auc_mean,pr_auc_std,roc_auc_mean,roc_auc_std,f1_mean,f1_std,recall_mean,recall_std,brier_mean,brier_std,ece_mean,ece_std,nll_mean,nll_std,acc_mean,acc_std
2,LogReg,0.177091,0.027022,0.712706,0.033233,0.270343,0.041909,0.410345,0.076525,0.205302,0.005640,0.324344,0.011592,0.595691,0.012965,0.820896,0.053316
3,RandomForest,0.166092,0.040917,0.684201,0.056026,0.234975,0.043910,0.468966,0.179757,0.070842,0.001946,0.025562,0.005544,0.263415,0.010771,0.737313,0.150472
1,LightGBM,0.152345,0.036105,0.669651,0.044853,0.224468,0.036985,0.351724,0.137175,0.077090,0.002767,0.069087,0.006400,0.397930,0.037222,0.805427,0.084450
0,CatBoost,0.145927,0.021356,0.657194,0.046098,0.225849,0.030936,0.393103,0.127516,0.074944,0.001501,0.059800,0.003844,0.337903,0.027031,0.784261,0.077685
4,XGBoost,0.142884,0.033450,0.656879,0.048760,0.222568,0.037031,0.368966,0.112135,0.074986,0.002644,0.057483,0.006433,0.325289,0.025444,0.795929,0.068077


,model,pr_auc_mean,pr_auc_std,roc_auc_mean,roc_auc_std,f1_mean,f1_std,recall_mean,recall_std,brier_mean,brier_std,ece_mean,ece_std,nll_mean,nll_std,acc_mean,acc_std
2,LogReg,0.151252,0.004766,0.622039,0.008589,0.149168,0.020406,0.219444,0.091814,0.212519,0.002805,0.320858,0.005866,0.608630,0.006064,0.810206,0.046183
3,RandomForest,0.123072,0.006008,0.614327,0.019406,0.158455,0.020271,0.366667,0.222786,0.072979,0.000531,0.031651,0.005159,0.275040,0.004014,0.709663,0.146342
4,XGBoost,0.118576,0.013240,0.588195,0.019211,0.162655,0.024842,0.272222,0.104675,0.075231,0.001113,0.055929,0.001649,0.350508,0.010390,0.785016,0.068262
1,LightGBM,0.117019,0.010510,0.595187,0.022978,0.154379,0.012549,0.244444,0.124846,0.078569,0.000791,0.070232,0.003308,0.443470,0.014314,0.795440,0.090177
0,CatBoost,0.115945,0.011610,0.598639,0.013350,0.148534,0.032545,0.269444,0.135586,0.075693,0.001069,0.060705,0.004398,0.360647,0.008442,0.771770,0.076002


,model,pr_auc,roc_auc,f1,recall,brier,ece
0,LogReg,0.1513 ± 0.0048,0.6220 ± 0.0086,0.1492 ± 0.0204,0.2194 ± 0.0918,0.2125 ± 0.0028,0.3209 ± 0.0059
1,RandomForest,0.1231 ± 0.0060,0.6143 ± 0.0194,0.1585 ± 0.0203,0.3667 ± 0.2228,0.0730 ± 0.0005,0.0317 ± 0.0052
2,XGBoost,0.1186 ± 0.0132,0.5882 ± 0.0192,0.1627 ± 0.0248,0.2722 ± 0.1047,0.0752 ± 0.0011,0.0559 ± 0.0016
3,LightGBM,0.1170 ± 0.0105,0.5952 ± 0.0230,0.1544 ± 0.0125,0.2444 ± 0.1248,0.0786 ± 0.0008,0.0702 ± 0.0033
4,CatBoost,0.1159 ± 0.0116,0.5986 ± 0.0133,0.1485 ± 0.0325,0.2694 ± 0.1356,0.0757 ± 0.0011,0.0607 ± 0.0044



✅ TASK 4.1 complete.
Next: TASK 5.1 (tabular transformer backbone for CGT-Stroke++).


In [7]:
# =========================================================
# TASK 5.1 (FIXED) — Tabular Transformer backbone
# Fixes:
# - Binary columns mapped to {0,1} using train-fit mapping
# - Categorical encoded values shifted by +1 (unknown -1 -> 0)
# - Embedding sizes consistent with shifted indices
# =========================================================

import math, os, json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import average_precision_score

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)

# ---------------------------
# 1) Load data + splits + feature types
# ---------------------------
df = pd.read_csv(os.path.join(DIRS["base"], "data_clean.csv"))
TARGET = "stroke"

ft = pd.read_csv(os.path.join(DIRS["tab"], "feature_type_inference.csv"))
bin_cols = ft.loc[ft["type_inferred"]=="binary", "feature"].tolist()
cat_cols = ft.loc[ft["type_inferred"]=="categorical", "feature"].tolist()
cont_cols = ft.loc[ft["type_inferred"]=="continuous", "feature"].tolist()

SPLIT_DIR = os.path.join(DIRS["base"], "splits")
test_idx = np.load(os.path.join(SPLIT_DIR, "test_idx.npy"))

# ---------------------------
# 2) Preprocessor (robust for embeddings)
# ---------------------------
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OrdinalEncoder

class Preprocessor:
    """
    - Continuous: median + standardize
    - Categorical: most frequent + ordinal encode with unknown=-1, then shift +1
    - Binary: train-fit mapping to {0,1} then clamp
    """
    def __init__(self, cont_cols, cat_cols, bin_cols):
        self.cont_cols = list(cont_cols)
        self.cat_cols = list(cat_cols)
        self.bin_cols = list(bin_cols)

        self.cont_imputer = SimpleImputer(strategy="median")
        self.scaler = StandardScaler()

        self.cat_imputer = SimpleImputer(strategy="most_frequent")
        self.encoder = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)

        self.bin_maps = {}  # col -> dict original_value -> 0/1

    def fit(self, df_tr: pd.DataFrame):
        # Continuous
        Xc = self.cont_imputer.fit_transform(df_tr[self.cont_cols])
        self.scaler.fit(Xc)

        # Categorical
        Xk = self.cat_imputer.fit_transform(df_tr[self.cat_cols])
        self.encoder.fit(Xk)

        # Binary mapping (train-fit)
        for c in self.bin_cols:
            vals = pd.unique(df_tr[c].dropna())
            vals = sorted([int(v) for v in vals])
            if len(vals) == 0:
                self.bin_maps[c] = {0:0, 1:1}
            elif len(vals) == 1:
                # degenerate; map it to 0
                self.bin_maps[c] = {vals[0]: 0}
            else:
                # map min -> 0, max -> 1 (works for {0,1} or {1,2})
                self.bin_maps[c] = {vals[0]: 0, vals[-1]: 1}
        return self

    def transform(self, df_any: pd.DataFrame):
        # Continuous
        Xc = self.scaler.transform(self.cont_imputer.transform(df_any[self.cont_cols])).astype(np.float32)

        # Categorical -> ordinal -> shift +1 so unknown(-1) becomes 0
        Xk_imp = self.cat_imputer.transform(df_any[self.cat_cols])
        Xk = self.encoder.transform(Xk_imp).astype(np.int64)
        Xk = Xk + 1  # now valid range: 0..cardinality (0 is unknown)

        # Binary -> map to 0/1
        Xb = df_any[self.bin_cols].copy()
        for c in self.bin_cols:
            mp = self.bin_maps[c]
            # map known values; unknown -> 0
            Xb[c] = Xb[c].map(lambda v: mp.get(int(v), 0))
        Xb = Xb.values.astype(np.int64)
        # extra safety
        Xb = np.clip(Xb, 0, 1)

        return {
            "cont": torch.tensor(Xc, dtype=torch.float32),
            "cat": torch.tensor(Xk, dtype=torch.long),
            "bin": torch.tensor(Xb, dtype=torch.long),
        }

# ---------------------------
# 3) Dataset
# ---------------------------
class TabDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self): return len(self.y)

    def __getitem__(self, i):
        return (
            self.X["cont"][i],
            self.X["cat"][i],
            self.X["bin"][i],
            self.y[i]
        )

# ---------------------------
# 4) FT-style backbone (simple, stable)
# ---------------------------
class FTBackbone(nn.Module):
    def __init__(self, n_cont, cat_cardinalities, n_bin,
                 d_token=48, depth=4, dropout=0.15):
        super().__init__()

        self.cont_embed = nn.Linear(n_cont, d_token)

        # cat indices are shifted by +1, so valid range: 0..card
        self.cat_embeds = nn.ModuleList([
            nn.Embedding(card + 1, d_token) for card in cat_cardinalities
        ])

        # binary mapped to {0,1}
        self.bin_embeds = nn.ModuleList([
            nn.Embedding(2, d_token) for _ in range(n_bin)
        ])

        self.blocks = nn.ModuleList([
            nn.Sequential(
                nn.LayerNorm(d_token),
                nn.Linear(d_token, 4*d_token),
                nn.GELU(),
                nn.Dropout(dropout),
                nn.Linear(4*d_token, d_token),
                nn.Dropout(dropout),
            )
            for _ in range(depth)
        ])

        self.head = nn.Sequential(
            nn.LayerNorm(d_token),
            nn.Linear(d_token, 1)
        )

    def forward(self, x_cont, x_cat, x_bin):
        tokens = []
        tokens.append(self.cont_embed(x_cont))

        for i, emb in enumerate(self.cat_embeds):
            tokens.append(emb(x_cat[:, i]))

        for i, emb in enumerate(self.bin_embeds):
            tokens.append(emb(x_bin[:, i]))

        h = torch.stack(tokens, dim=1).mean(dim=1)  # pooled representation

        for blk in self.blocks:
            h = h + blk(h)

        logit = self.head(h).squeeze(1)
        return logit, h

# ---------------------------
# 5) Train & eval over repeats
# ---------------------------
def train_one_repeat(r: int, max_epochs=40, patience=6):
    tr_idx = np.load(os.path.join(SPLIT_DIR, f"train_idx_r{r}.npy"))
    va_idx = np.load(os.path.join(SPLIT_DIR, f"val_idx_r{r}.npy"))

    df_tr = df.iloc[tr_idx].reset_index(drop=True)
    df_va = df.iloc[va_idx].reset_index(drop=True)
    df_te = df.iloc[test_idx].reset_index(drop=True)

    y_tr = df_tr[TARGET].values.astype(int)
    y_va = df_va[TARGET].values.astype(int)
    y_te = df_te[TARGET].values.astype(int)

    # Fit preprocessing on TRAIN only
    pp = Preprocessor(cont_cols, cat_cols, bin_cols).fit(df_tr)
    X_tr = pp.transform(df_tr)
    X_va = pp.transform(df_va)
    X_te = pp.transform(df_te)

    # Cardinalities for categorical embeddings:
    # With ordinal encoder + shift, max index = card (unknown=0). So embedding size needs card+1.
    # Use train unique count to reduce risk of mismatch.
    cat_cards = [int(df_tr[c].nunique(dropna=True)) for c in cat_cols]

    model = FTBackbone(
        n_cont=len(cont_cols),
        cat_cardinalities=cat_cards,
        n_bin=len(bin_cols),
    ).to(DEVICE)

    # class imbalance
    pos = y_tr.sum()
    neg = len(y_tr) - pos
    pos_weight = neg / max(1.0, pos)
    crit = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(pos_weight, device=DEVICE))

    opt = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)

    train_loader = DataLoader(TabDataset(X_tr, y_tr), batch_size=256, shuffle=True)
    val_loader   = DataLoader(TabDataset(X_va, y_va), batch_size=512, shuffle=False)
    test_loader  = DataLoader(TabDataset(X_te, y_te), batch_size=512, shuffle=False)

    best_pr = -1.0
    best_state = None
    best_epoch = -1
    bad = 0
    history = []

    for epoch in range(max_epochs):
        model.train()
        losses = []
        for xc, xk, xb, yb in train_loader:
            xc, xk, xb, yb = xc.to(DEVICE), xk.to(DEVICE), xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            logit, _ = model(xc, xk, xb)
            loss = crit(logit, yb)
            loss.backward()
            opt.step()
            losses.append(loss.item())

        # val PR-AUC
        model.eval()
        p_va = []
        with torch.no_grad():
            for xc, xk, xb, yb in val_loader:
                logit, _ = model(xc.to(DEVICE), xk.to(DEVICE), xb.to(DEVICE))
                p_va.append(torch.sigmoid(logit).cpu().numpy())
        p_va = np.concatenate(p_va)
        pr = float(average_precision_score(y_va, p_va))

        history.append({"epoch": epoch, "train_loss": float(np.mean(losses)), "val_pr_auc": pr})
        print(f"Repeat {r} | Epoch {epoch:02d} | loss={np.mean(losses):.4f} | val PR={pr:.4f}")

        if pr > best_pr + 1e-6:
            best_pr = pr
            best_state = {k: v.detach().cpu() for k, v in model.state_dict().items()}
            best_epoch = epoch
            bad = 0
        else:
            bad += 1
            if bad >= patience:
                break

    # load best
    model.load_state_dict(best_state)

    # val probs (for threshold selection)
    model.eval()
    p_va = []
    with torch.no_grad():
        for xc, xk, xb, yb in val_loader:
            logit, _ = model(xc.to(DEVICE), xk.to(DEVICE), xb.to(DEVICE))
            p_va.append(torch.sigmoid(logit).cpu().numpy())
    p_va = np.concatenate(p_va)

    # test probs
    p_te = []
    with torch.no_grad():
        for xc, xk, xb, yb in test_loader:
            logit, _ = model(xc.to(DEVICE), xk.to(DEVICE), xb.to(DEVICE))
            p_te.append(torch.sigmoid(logit).cpu().numpy())
    p_te = np.concatenate(p_te)

    # choose threshold on VAL
    t_best, _ = find_best_threshold(y_va, p_va, metric="f1")

    val_metrics  = evaluate_binary(y_va, p_va, threshold=t_best, ece_bins=cfg.ece_bins)
    test_metrics = evaluate_binary(y_te, p_te, threshold=t_best, ece_bins=cfg.ece_bins)

    # Save reliability figure (val)
    fig = plot_reliability_curve(y_va, p_va, n_bins=cfg.ece_bins, title=f"Reliability (VAL) — FTBackbone r{r}")
    save_fig(f"ftbackbone_reliability_val_r{r}", fig=fig)

    # Save learning curve figure
    hist_df = pd.DataFrame(history)
    fig = plt.figure()
    plt.plot(hist_df["epoch"], hist_df["val_pr_auc"], marker="o")
    plt.xlabel("Epoch")
    plt.ylabel("Validation PR-AUC")
    plt.title(f"FTBackbone learning curve (repeat {r})")
    save_fig(f"ftbackbone_learningcurve_r{r}", fig=fig)

    return {
        "repeat": r,
        "best_epoch": best_epoch,
        "best_val_pr_auc": best_pr,
        "t_best": float(t_best),
        "val": val_metrics,
        "test": test_metrics,
        "p_val": p_va,
        "p_test": p_te
    }

# ---------------------------
# 6) Run repeats
# ---------------------------
seed_everything(cfg.seed)

runs = []
for r in range(5):
    out = train_one_repeat(r)
    out["val"].update({"model":"FTBackbone", "repeat":r, "split":"val"})
    out["test"].update({"model":"FTBackbone", "repeat":r, "split":"test"})
    runs.append(out)

# per-repeat metrics table
rows = []
for out in runs:
    rows.append(out["val"])
    rows.append(out["test"])
df_all = pd.DataFrame(rows)
display(df_all)

save_table(df_all, "ftbackbone_per_repeat_val_test", index=False)

# aggregate (test)
test_df = df_all[df_all["split"]=="test"].copy()
summary = test_df[["pr_auc","roc_auc","f1","recall","brier","ece","nll","acc"]].agg(["mean","std"]).T.reset_index()
summary.columns = ["metric","mean","std"]
display(summary)
save_table(summary, "ftbackbone_test_summary_meanstd", index=False)

print("\n✅ TASK 5.1 (fixed) complete.")
print("Next: TASK 5.2 (Concept-Gated head — main novelty).")

Using device: cpu
Repeat 0 | Epoch 00 | loss=1.3194 | val PR=0.1281
Repeat 0 | Epoch 01 | loss=1.2063 | val PR=0.1398
Repeat 0 | Epoch 02 | loss=1.1556 | val PR=0.1473
Repeat 0 | Epoch 03 | loss=1.1221 | val PR=0.1471
Repeat 0 | Epoch 04 | loss=1.1177 | val PR=0.1499
Repeat 0 | Epoch 05 | loss=1.0928 | val PR=0.1505
Repeat 0 | Epoch 06 | loss=1.0823 | val PR=0.1532
Repeat 0 | Epoch 07 | loss=1.0500 | val PR=0.1511
Repeat 0 | Epoch 08 | loss=1.0630 | val PR=0.1456
Repeat 0 | Epoch 09 | loss=1.0372 | val PR=0.1514
Repeat 0 | Epoch 10 | loss=1.0199 | val PR=0.1479
Repeat 0 | Epoch 11 | loss=1.0114 | val PR=0.1435
Repeat 0 | Epoch 12 | loss=0.9976 | val PR=0.1427
Repeat 1 | Epoch 00 | loss=1.3026 | val PR=0.1306
Repeat 1 | Epoch 01 | loss=1.2093 | val PR=0.1867
Repeat 1 | Epoch 02 | loss=1.1900 | val PR=0.1855
Repeat 1 | Epoch 03 | loss=1.1552 | val PR=0.1985
Repeat 1 | Epoch 04 | loss=1.1623 | val PR=0.2053
Repeat 1 | Epoch 05 | loss=1.1285 | val PR=0.2195
Repeat 1 | Epoch 06 | loss=1.105

,threshold,pr_auc,roc_auc,acc,f1,precision,recall,brier,ece,nll,tn,fp,fn,tp,model,repeat,split
0,0.76,0.142725,0.674014,0.867028,0.246154,0.222222,0.275862,0.206390,0.298691,0.590253,623.0,56.0,42.0,16.0,FTBackbone,0,val
1,0.76,0.137580,0.626358,0.857763,0.165605,0.152941,0.180556,0.218092,0.304287,0.613631,777.0,72.0,59.0,13.0,FTBackbone,0,test
2,0.68,0.218323,0.765629,0.831750,0.340426,0.246154,0.551724,0.191331,0.288768,0.542066,581.0,98.0,26.0,32.0,FTBackbone,1,val
3,0.68,0.133274,0.624117,0.792617,0.158590,0.116129,0.250000,0.211860,0.294082,0.594158,712.0,137.0,54.0,18.0,FTBackbone,1,test
4,0.65,0.171896,0.679321,0.801900,0.255102,0.181159,0.431034,0.199925,0.293920,0.572234,566.0,113.0,33.0,25.0,FTBackbone,2,val
5,0.65,0.110424,0.607545,0.753529,0.136882,0.094241,0.250000,0.216718,0.303397,0.608740,676.0,173.0,54.0,18.0,FTBackbone,2,test
6,0.65,0.191305,0.705602,0.748982,0.251012,0.164021,0.534483,0.224590,0.331096,0.628909,521.0,158.0,27.0,31.0,FTBackbone,3,val
7,0.65,0.133698,0.640132,0.738328,0.193980,0.127753,0.402778,0.223948,0.321370,0.628221,651.0,198.0,43.0,29.0,FTBackbone,3,test
8,0.67,0.195254,0.710680,0.801900,0.270000,0.190141,0.465517,0.195318,0.273888,0.552487,564.0,115.0,31.0,27.0,FTBackbone,4,val
9,0.67,0.121556,0.629531,0.789359,0.170940,0.123457,0.277778,0.200171,0.267906,0.569408,707.0,142.0,52.0,20.0,FTBackbone,4,test


,metric,mean,std
0,pr_auc,0.127306,0.011181
1,roc_auc,0.625537,0.011783
2,f1,0.165200,0.020658
3,recall,0.272222,0.081342
4,brier,0.214158,0.008927
5,ece,0.298208,0.019593
6,nll,0.602831,0.022297
7,acc,0.786319,0.046183



✅ TASK 5.1 (fixed) complete.
Next: TASK 5.2 (Concept-Gated head — main novelty).


In [8]:
# =========================================================
# TASK 5.2 — Concept-Gated Trustworthy Model (CGT-Stroke++)
# =========================================================
# Outputs:
# - per-repeat val/test metrics
# - mean±std summary table
# - global concept importance plot (PDF+PNG)
# - per-sample concept contributions saved for later analysis

import os, json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import average_precision_score

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)

# ---------------------------
# 1) Load data + splits + feature types
# ---------------------------
df = pd.read_csv(os.path.join(DIRS["base"], "data_clean.csv"))
TARGET = "stroke"

ft = pd.read_csv(os.path.join(DIRS["tab"], "feature_type_inference.csv"))
bin_cols = ft.loc[ft["type_inferred"]=="binary", "feature"].tolist()
cat_cols = ft.loc[ft["type_inferred"]=="categorical", "feature"].tolist()
cont_cols = ft.loc[ft["type_inferred"]=="continuous", "feature"].tolist()

SPLIT_DIR = os.path.join(DIRS["base"], "splits")
test_idx = np.load(os.path.join(SPLIT_DIR, "test_idx.npy"))

# ---------------------------
# 2) Preprocessor (same as fixed 5.1)
# ---------------------------
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OrdinalEncoder

class Preprocessor:
    def __init__(self, cont_cols, cat_cols, bin_cols):
        self.cont_cols = list(cont_cols)
        self.cat_cols = list(cat_cols)
        self.bin_cols = list(bin_cols)
        self.cont_imputer = SimpleImputer(strategy="median")
        self.scaler = StandardScaler()
        self.cat_imputer = SimpleImputer(strategy="most_frequent")
        self.encoder = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
        self.bin_maps = {}

    def fit(self, df_tr):
        self.scaler.fit(self.cont_imputer.fit_transform(df_tr[self.cont_cols]))
        self.encoder.fit(self.cat_imputer.fit_transform(df_tr[self.cat_cols]))

        for c in self.bin_cols:
            vals = pd.unique(df_tr[c].dropna())
            vals = sorted([int(v) for v in vals])
            if len(vals) <= 1:
                self.bin_maps[c] = {vals[0]: 0} if len(vals)==1 else {0:0,1:1}
            else:
                self.bin_maps[c] = {vals[0]: 0, vals[-1]: 1}
        return self

    def transform(self, df_any):
        Xc = self.scaler.transform(self.cont_imputer.transform(df_any[self.cont_cols])).astype(np.float32)

        Xk_imp = self.cat_imputer.transform(df_any[self.cat_cols])
        Xk = self.encoder.transform(Xk_imp).astype(np.int64)
        Xk = Xk + 1  # unknown(-1)->0

        Xb = df_any[self.bin_cols].copy()
        for c in self.bin_cols:
            mp = self.bin_maps[c]
            Xb[c] = Xb[c].map(lambda v: mp.get(int(v), 0))
        Xb = np.clip(Xb.values.astype(np.int64), 0, 1)

        return {
            "cont": torch.tensor(Xc, dtype=torch.float32),
            "cat": torch.tensor(Xk, dtype=torch.long),
            "bin": torch.tensor(Xb, dtype=torch.long),
        }

# ---------------------------
# 3) Build concept groups (auto, editable)
# ---------------------------
cols_all = cont_cols + bin_cols + cat_cols

def auto_concepts(columns):
    # You can refine these patterns later.
    groups = {
        "Demographics": [],
        "Lifestyle": [],
        "Comorbidity": [],
        "Nutrition_Macro": [],
        "Nutrition_FattyAcids": [],
        "Nutrition_Minerals": [],
        "Other": []
    }
    for c in columns:
        cl = c.lower()

        if cl in ["age", "gender", "race", "marital_status"]:
            groups["Demographics"].append(c)
        elif any(k in cl for k in ["alcohol", "smoke", "sleep", "physical", "exercise"]):
            groups["Lifestyle"].append(c)
        elif any(k in cl for k in ["diabetes", "hypertension", "cholesterol", "coronary", "heart"]):
            groups["Comorbidity"].append(c)
        elif any(k in cl for k in ["energy", "protein", "carbohydrate", "fiber"]):
            groups["Nutrition_Macro"].append(c)
        elif any(k in cl for k in ["fatty", "saturated", "monounsaturated", "polyunsaturated"]):
            groups["Nutrition_FattyAcids"].append(c)
        elif any(k in cl for k in ["sodium", "potassium", "calcium", "magnesium", "iron", "zinc"]):
            groups["Nutrition_Minerals"].append(c)
        else:
            groups["Other"].append(c)

    # Remove empties
    groups = {k:v for k,v in groups.items() if len(v) > 0}
    return groups

CONCEPTS = auto_concepts(cols_all)
print("Concept groups:")
for k,v in CONCEPTS.items():
    print(f" - {k}: {len(v)}")

concept_path = os.path.join(DIRS["log"], "concept_groups.json")
with open(concept_path, "w") as f:
    json.dump(CONCEPTS, f, indent=2)
print("✅ Saved concept groups:", concept_path)

# Create an index map for each input feature in packed token order:
# We will create token embeddings for: [CONT_TOKEN] + each cat + each bin
# But concepts should operate at FEATURE level, not token level.
# Here we do concept pooling at representation-level using per-feature embeddings.

# ---------------------------
# 4) Dataset
# ---------------------------
class TabDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self): return len(self.y)

    def __getitem__(self, i):
        return (
            self.X["cont"][i],
            self.X["cat"][i],
            self.X["bin"][i],
            self.y[i]
        )

# ---------------------------
# 5) CGT Model
# ---------------------------
class CGTModel(nn.Module):
    def __init__(self, n_cont, cat_cardinalities, n_bin,
                 concepts: Dict[str, List[str]],
                 cont_cols, cat_cols, bin_cols,
                 d_token=64, depth=4, dropout=0.20,
                 residual_cap=0.15):
        super().__init__()
        self.cont_cols = cont_cols
        self.cat_cols = cat_cols
        self.bin_cols = bin_cols
        self.concepts = concepts
        self.residual_cap = residual_cap

        # Embeddings
        self.cont_proj = nn.Linear(n_cont, d_token)

        self.cat_embeds = nn.ModuleList([nn.Embedding(card + 1, d_token) for card in cat_cardinalities])
        self.bin_embeds = nn.ModuleList([nn.Embedding(2, d_token) for _ in range(n_bin)])

        # Shared representation blocks
        self.blocks = nn.ModuleList([
            nn.Sequential(
                nn.LayerNorm(d_token),
                nn.Linear(d_token, 4*d_token),
                nn.GELU(),
                nn.Dropout(dropout),
                nn.Linear(4*d_token, d_token),
                nn.Dropout(dropout),
            )
            for _ in range(depth)
        ])

        # Concept heads: each concept -> risk score r_c (scalar)
        self.concept_names = list(concepts.keys())
        self.C = len(self.concept_names)

        self.risk_head = nn.ModuleList([nn.Sequential(
            nn.LayerNorm(d_token),
            nn.Linear(d_token, 1)
        ) for _ in range(self.C)])

        # Gate head: produces g_c in [0,1] and normalized
        self.gate_head = nn.Sequential(
            nn.LayerNorm(d_token),
            nn.Linear(d_token, self.C)
        )

        # Residual head (bounded)
        self.res_head = nn.Sequential(
            nn.LayerNorm(d_token),
            nn.Linear(d_token, 1)
        )

        # feature -> which embedding token index it maps to (for concept pooling)
        # We build feature embeddings then pool per concept.
        self.feature_to_source = {}
        for j,c in enumerate(cont_cols): self.feature_to_source[c] = ("cont", j)
        for j,c in enumerate(cat_cols):  self.feature_to_source[c] = ("cat", j)
        for j,c in enumerate(bin_cols):  self.feature_to_source[c] = ("bin", j)

    def forward(self, x_cont, x_cat, x_bin):
        # Build per-feature embeddings (for concept pooling)
        # Continuous: single projection from full cont vector
        h_shared = self.cont_proj(x_cont)  # (B,d)

        # include categorical + binary influences into shared repr via mean pooling tokens
        tokens = [h_shared]
        for i, emb in enumerate(self.cat_embeds):
            tokens.append(emb(x_cat[:, i]))
        for i, emb in enumerate(self.bin_embeds):
            tokens.append(emb(x_bin[:, i]))
        h = torch.stack(tokens, dim=1).mean(dim=1)

        for blk in self.blocks:
            h = h + blk(h)

        # Concept pooling: for each concept, take mean of corresponding feature token embeddings
        # For continuous features: we don't have per-feature tokens; instead we use h itself as proxy for each cont feature
        # (simple + stable; later we can upgrade to per-cont-feature embeddings in ablation)
        concept_vecs = []
        for cname in self.concept_names:
            feats = self.concepts[cname]
            vecs = []
            for f in feats:
                src, idx = self.feature_to_source[f]
                if src == "cat":
                    vecs.append(self.cat_embeds[idx](x_cat[:, idx]))
                elif src == "bin":
                    vecs.append(self.bin_embeds[idx](x_bin[:, idx]))
                else:
                    # continuous feature proxies the shared rep
                    vecs.append(h)
            concept_vecs.append(torch.stack(vecs, dim=1).mean(dim=1))

        concept_vecs = torch.stack(concept_vecs, dim=1)  # (B,C,d)

        # Risks r_c
        risks = []
        for i in range(self.C):
            risks.append(self.risk_head[i](concept_vecs[:, i, :]).squeeze(1))
        risks = torch.stack(risks, dim=1)  # (B,C)

        # Gates g_c
        g = torch.sigmoid(self.gate_head(h))  # (B,C)
        g = g / (g.sum(dim=1, keepdim=True) + 1e-12)     # normalized convex weights

        # Residual (bounded)
        res = self.res_head(h).squeeze(1)
        res = self.residual_cap * torch.tanh(res)

        logit = (g * risks).sum(dim=1) + res

        return logit, g, risks, res

# ---------------------------
# 6) Train over repeats
# ---------------------------
def train_eval_repeat(r, max_epochs=50, patience=8, gate_l1=0.02):
    tr_idx = np.load(os.path.join(SPLIT_DIR, f"train_idx_r{r}.npy"))
    va_idx = np.load(os.path.join(SPLIT_DIR, f"val_idx_r{r}.npy"))

    df_tr = df.iloc[tr_idx].reset_index(drop=True)
    df_va = df.iloc[va_idx].reset_index(drop=True)
    df_te = df.iloc[test_idx].reset_index(drop=True)

    y_tr = df_tr[TARGET].values.astype(int)
    y_va = df_va[TARGET].values.astype(int)
    y_te = df_te[TARGET].values.astype(int)

    pp = Preprocessor(cont_cols, cat_cols, bin_cols).fit(df_tr)
    X_tr = pp.transform(df_tr)
    X_va = pp.transform(df_va)
    X_te = pp.transform(df_te)

    cat_cards = [int(df_tr[c].nunique(dropna=True)) for c in cat_cols]

    model = CGTModel(
        n_cont=len(cont_cols),
        cat_cardinalities=cat_cards,
        n_bin=len(bin_cols),
        concepts=CONCEPTS,
        cont_cols=cont_cols, cat_cols=cat_cols, bin_cols=bin_cols,
        d_token=64, depth=4, dropout=0.20,
        residual_cap=0.12
    ).to(DEVICE)

    pos = y_tr.sum(); neg = len(y_tr) - pos
    pos_weight = neg / max(1.0, pos)
    bce = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(pos_weight, device=DEVICE))

    opt = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=2e-4)

    tr_loader = DataLoader(TabDataset(X_tr, y_tr), batch_size=256, shuffle=True)
    va_loader = DataLoader(TabDataset(X_va, y_va), batch_size=512, shuffle=False)
    te_loader = DataLoader(TabDataset(X_te, y_te), batch_size=512, shuffle=False)

    best_pr, best_state = -1.0, None
    bad = 0
    history = []

    for epoch in range(max_epochs):
        model.train()
        losses = []
        for xc, xk, xb, yb in tr_loader:
            xc, xk, xb, yb = xc.to(DEVICE), xk.to(DEVICE), xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            logit, g, risks, res = model(xc, xk, xb)
            loss = bce(logit, yb)
            # gate sparsity / simplicity (encourage peaky gates)
            loss = loss + gate_l1 * (g.abs().mean())
            loss.backward()
            opt.step()
            losses.append(loss.item())

        # val PR-AUC
        model.eval()
        p_va, G_va = [], []
        with torch.no_grad():
            for xc, xk, xb, yb in va_loader:
                logit, g, risks, res = model(xc.to(DEVICE), xk.to(DEVICE), xb.to(DEVICE))
                p_va.append(torch.sigmoid(logit).cpu().numpy())
                G_va.append(g.cpu().numpy())
        p_va = np.concatenate(p_va)
        G_va = np.concatenate(G_va)

        pr = float(average_precision_score(y_va, p_va))
        history.append({"epoch": epoch, "train_loss": float(np.mean(losses)), "val_pr_auc": pr})

        print(f"Repeat {r} | Epoch {epoch:02d} | loss={np.mean(losses):.4f} | val PR={pr:.4f}")

        if pr > best_pr + 1e-6:
            best_pr = pr
            best_state = {k: v.detach().cpu() for k, v in model.state_dict().items()}
            bad = 0
        else:
            bad += 1
            if bad >= patience:
                break

    model.load_state_dict(best_state)

    # Final val probs for threshold selection
    model.eval()
    p_va, G_va, R_va = [], [], []
    with torch.no_grad():
        for xc, xk, xb, yb in va_loader:
            logit, g, risks, res = model(xc.to(DEVICE), xk.to(DEVICE), xb.to(DEVICE))
            p_va.append(torch.sigmoid(logit).cpu().numpy())
            G_va.append(g.cpu().numpy())
            R_va.append(risks.cpu().numpy())
    p_va = np.concatenate(p_va)
    G_va = np.concatenate(G_va)
    R_va = np.concatenate(R_va)

    # Test probs + concept outputs
    p_te, G_te, R_te = [], [], []
    with torch.no_grad():
        for xc, xk, xb, yb in te_loader:
            logit, g, risks, res = model(xc.to(DEVICE), xk.to(DEVICE), xb.to(DEVICE))
            p_te.append(torch.sigmoid(logit).cpu().numpy())
            G_te.append(g.cpu().numpy())
            R_te.append(risks.cpu().numpy())
    p_te = np.concatenate(p_te)
    G_te = np.concatenate(G_te)
    R_te = np.concatenate(R_te)

    t_best, _ = find_best_threshold(y_va, p_va, metric="f1")

    val_metrics = evaluate_binary(y_va, p_va, threshold=t_best, ece_bins=cfg.ece_bins)
    test_metrics = evaluate_binary(y_te, p_te, threshold=t_best, ece_bins=cfg.ece_bins)

    # Global concept importance: mean gate * mean |risk|
    concept_names = list(CONCEPTS.keys())
    imp = (G_va.mean(axis=0) * np.abs(R_va).mean(axis=0))
    imp_df = pd.DataFrame({"concept": concept_names, "importance": imp}).sort_values("importance", ascending=False)

    # Save per-repeat reliability
    fig = plot_reliability_curve(y_va, p_va, n_bins=cfg.ece_bins, title=f"Reliability (VAL) — CGT r{r}")
    save_fig(f"cgt_reliability_val_r{r}", fig=fig)

    # Save learning curve
    hist_df = pd.DataFrame(history)
    fig = plt.figure()
    plt.plot(hist_df["epoch"], hist_df["val_pr_auc"], marker="o")
    plt.xlabel("Epoch"); plt.ylabel("Validation PR-AUC")
    plt.title(f"CGT learning curve (repeat {r})")
    save_fig(f"cgt_learningcurve_r{r}", fig=fig)

    return val_metrics, test_metrics, imp_df, (G_te, R_te, p_te)

# ---------------------------
# 7) Run repeats and aggregate
# ---------------------------
seed_everything(cfg.seed)

rows = []
imps = []
concept_artifacts = []

for r in range(5):
    m_va, m_te, imp_df, art = train_eval_repeat(r)
    m_va.update({"model":"CGT", "repeat":r, "split":"val"})
    m_te.update({"model":"CGT", "repeat":r, "split":"test"})
    rows += [m_va, m_te]
    imps.append(imp_df.assign(repeat=r))
    concept_artifacts.append({"repeat": r, "G_test": art[0], "R_test": art[1], "p_test": art[2]})

df_metrics = pd.DataFrame(rows)
display(df_metrics)
save_table(df_metrics, "cgt_per_repeat_val_test", index=False)

# Mean±std on test
test_df = df_metrics[df_metrics["split"]=="test"]
summary = test_df[["pr_auc","roc_auc","f1","recall","brier","ece","nll","acc"]].agg(["mean","std"]).T.reset_index()
summary.columns = ["metric","mean","std"]
display(summary)
save_table(summary, "cgt_test_summary_meanstd", index=False)

# ---------------------------
# 8) Global concept importance plot (averaged across repeats)
# ---------------------------
imp_all = pd.concat(imps, ignore_index=True)
imp_mean = imp_all.groupby("concept")["importance"].mean().sort_values(ascending=True)

fig = plt.figure(figsize=(7.0, 4.6))
plt.barh(imp_mean.index, imp_mean.values)
plt.xlabel("Mean importance (gate × |risk|)")
plt.title("Global Concept Importance (CGT, averaged across repeats)")
paths = save_fig("cgt_global_concept_importance", fig=fig)
print("✅ Saved:", paths)

# Save importance table
imp_table = imp_all.groupby("concept")["importance"].agg(["mean","std"]).reset_index().sort_values("mean", ascending=False)
save_table(imp_table, "cgt_concept_importance_meanstd", index=False)

# Save per-sample concept artifacts (for case studies later)
artifact_path = os.path.join(DIRS["pred"], "cgt_concept_outputs_test.npy")
np.save(artifact_path, {"concept_names": list(CONCEPTS.keys()), "artifacts": concept_artifacts}, allow_pickle=True)
print("✅ Saved concept outputs:", artifact_path)

print("\n✅ TASK 5.2 complete.")
print("Next: TASK 6.1 (Uncertainty: Ensembles + calibration + risk-coverage/abstention).")

Using device: cpu
Concept groups:
 - Demographics: 4
 - Lifestyle: 4
 - Comorbidity: 4
 - Nutrition_Macro: 6
 - Nutrition_FattyAcids: 3
 - Nutrition_Minerals: 2
 - Other: 12
✅ Saved concept groups: /kaggle/working/outputs/cgt_stroke_v1/logs/concept_groups.json
Repeat 0 | Epoch 00 | loss=1.2372 | val PR=0.1494
Repeat 0 | Epoch 01 | loss=1.1515 | val PR=0.1384
Repeat 0 | Epoch 02 | loss=1.1015 | val PR=0.1369
Repeat 0 | Epoch 03 | loss=1.0938 | val PR=0.1459
Repeat 0 | Epoch 04 | loss=1.0554 | val PR=0.1435
Repeat 0 | Epoch 05 | loss=1.0472 | val PR=0.1382
Repeat 0 | Epoch 06 | loss=1.0431 | val PR=0.1389
Repeat 0 | Epoch 07 | loss=1.0524 | val PR=0.1379
Repeat 0 | Epoch 08 | loss=1.0489 | val PR=0.1327
Repeat 1 | Epoch 00 | loss=1.2580 | val PR=0.2525
Repeat 1 | Epoch 01 | loss=1.1847 | val PR=0.2765
Repeat 1 | Epoch 02 | loss=1.1186 | val PR=0.2761
Repeat 1 | Epoch 03 | loss=1.1233 | val PR=0.2843
Repeat 1 | Epoch 04 | loss=1.1080 | val PR=0.2890
Repeat 1 | Epoch 05 | loss=1.0880 | val

,threshold,pr_auc,roc_auc,acc,f1,precision,recall,brier,ece,nll,tn,fp,fn,tp,model,repeat,split
0,0.62,0.132716,0.677619,0.724559,0.222222,0.142857,0.500000,0.205010,0.298739,0.579980,505.0,174.0,29.0,29.0,CGT,0,val
1,0.62,0.147534,0.620092,0.686211,0.171920,0.108303,0.416667,0.223430,0.311347,0.621990,602.0,247.0,42.0,30.0,CGT,0,test
2,0.75,0.235750,0.758468,0.867028,0.319444,0.267442,0.396552,0.213589,0.327637,0.600377,616.0,63.0,35.0,23.0,CGT,1,val
3,0.75,0.141953,0.631773,0.845820,0.174419,0.150000,0.208333,0.229562,0.327364,0.637586,764.0,85.0,57.0,15.0,CGT,1,test
4,0.52,0.148312,0.638413,0.690638,0.213793,0.133621,0.534483,0.211391,0.267426,0.604639,478.0,201.0,27.0,31.0,CGT,2,val
5,0.52,0.119305,0.624395,0.663409,0.175532,0.108553,0.458333,0.215262,0.269030,0.612761,578.0,271.0,39.0,33.0,CGT,2,test
6,0.80,0.149455,0.678610,0.852103,0.237762,0.200000,0.293103,0.233368,0.330612,0.656011,611.0,68.0,41.0,17.0,CGT,3,val
7,0.80,0.109271,0.628566,0.840391,0.130178,0.113402,0.152778,0.237649,0.328017,0.665067,763.0,86.0,61.0,11.0,CGT,3,test
8,0.68,0.218075,0.728226,0.777476,0.293103,0.195402,0.586207,0.209179,0.320891,0.595715,539.0,140.0,24.0,34.0,CGT,4,val
9,0.68,0.121214,0.614694,0.752443,0.197183,0.132075,0.388889,0.222119,0.318007,0.625485,665.0,184.0,44.0,28.0,CGT,4,test


,metric,mean,std
0,pr_auc,0.127856,0.016191
1,roc_auc,0.623904,0.006769
2,f1,0.169846,0.024385
3,recall,0.325000,0.135586
4,brier,0.225604,0.008435
5,ece,0.310753,0.024330
6,nll,0.632578,0.020222
7,acc,0.757655,0.084605


✅ Saved: {'pdf': '/kaggle/working/outputs/cgt_stroke_v1/figures/cgt_global_concept_importance.pdf', 'png': '/kaggle/working/outputs/cgt_stroke_v1/figures/cgt_global_concept_importance.png'}
✅ Saved concept outputs: /kaggle/working/outputs/cgt_stroke_v1/predictions/cgt_concept_outputs_test.npy

✅ TASK 5.2 complete.
Next: TASK 6.1 (Uncertainty: Ensembles + calibration + risk-coverage/abstention).


In [9]:
# =========================================================
# TASK 5.3 — Strong FT-Transformer backbone (proper tokens + attention)
# Goal: beat LogReg PR-AUC (~0.151) or get very close
# =========================================================

import os, math
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import average_precision_score

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)

df = pd.read_csv(os.path.join(DIRS["base"], "data_clean.csv"))
TARGET = "stroke"

ft = pd.read_csv(os.path.join(DIRS["tab"], "feature_type_inference.csv"))
bin_cols = ft.loc[ft["type_inferred"]=="binary", "feature"].tolist()
cat_cols = ft.loc[ft["type_inferred"]=="categorical", "feature"].tolist()
cont_cols = ft.loc[ft["type_inferred"]=="continuous", "feature"].tolist()

SPLIT_DIR = os.path.join(DIRS["base"], "splits")
test_idx = np.load(os.path.join(SPLIT_DIR, "test_idx.npy"))

# ---------------------------
# Preprocessor (same robust one)
# ---------------------------
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OrdinalEncoder

class Preprocessor:
    def __init__(self, cont_cols, cat_cols, bin_cols):
        self.cont_cols=list(cont_cols); self.cat_cols=list(cat_cols); self.bin_cols=list(bin_cols)
        self.cont_imputer=SimpleImputer(strategy="median")
        self.scaler=StandardScaler()
        self.cat_imputer=SimpleImputer(strategy="most_frequent")
        self.encoder=OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
        self.bin_maps={}

    def fit(self, df_tr):
        self.scaler.fit(self.cont_imputer.fit_transform(df_tr[self.cont_cols]))
        self.encoder.fit(self.cat_imputer.fit_transform(df_tr[self.cat_cols]))
        for c in self.bin_cols:
            vals = sorted([int(v) for v in pd.unique(df_tr[c].dropna())])
            if len(vals) <= 1:
                self.bin_maps[c] = {vals[0]:0} if len(vals)==1 else {0:0,1:1}
            else:
                self.bin_maps[c] = {vals[0]:0, vals[-1]:1}
        return self

    def transform(self, df_any):
        Xc = self.scaler.transform(self.cont_imputer.transform(df_any[self.cont_cols])).astype(np.float32)

        Xk_imp = self.cat_imputer.transform(df_any[self.cat_cols])
        Xk = self.encoder.transform(Xk_imp).astype(np.int64)
        Xk = Xk + 1  # unknown(-1)->0

        Xb = df_any[self.bin_cols].copy()
        for c in self.bin_cols:
            mp = self.bin_maps[c]
            Xb[c] = Xb[c].map(lambda v: mp.get(int(v), 0))
        Xb = np.clip(Xb.values.astype(np.int64), 0, 1)

        return {
            "cont": torch.tensor(Xc, dtype=torch.float32),
            "cat": torch.tensor(Xk, dtype=torch.long),
            "bin": torch.tensor(Xb, dtype=torch.long),
        }

class TabDataset(Dataset):
    def __init__(self, X, y):
        self.X=X
        self.y=torch.tensor(y, dtype=torch.float32)
    def __len__(self): return len(self.y)
    def __getitem__(self, i):
        return self.X["cont"][i], self.X["cat"][i], self.X["bin"][i], self.y[i]

# ---------------------------
# Focal loss (for imbalance + PR-AUC)
# ---------------------------
class FocalBCEWithLogits(nn.Module):
    def __init__(self, gamma=2.0, pos_weight=None):
        super().__init__()
        self.gamma = gamma
        self.bce = nn.BCEWithLogitsLoss(reduction="none", pos_weight=pos_weight)

    def forward(self, logits, targets):
        bce = self.bce(logits, targets)
        pt = torch.exp(-bce)
        loss = ((1 - pt) ** self.gamma) * bce
        return loss.mean()

# ---------------------------
# Proper FT tokens + Transformer encoder
# ---------------------------
class FTTransformer(nn.Module):
    def __init__(self, n_cont, cat_cards, n_bin,
                 d_token=64, n_heads=4, n_layers=3, dropout=0.15):
        super().__init__()
        self.n_cont = n_cont
        self.n_cat = len(cat_cards)
        self.n_bin = n_bin

        # per-continuous-feature tokenization
        self.cont_weight = nn.Parameter(torch.randn(n_cont, d_token) * 0.02)
        self.cont_bias   = nn.Parameter(torch.zeros(n_cont, d_token))

        # categorical embeddings (shifted indices => size card+1)
        self.cat_embeds = nn.ModuleList([nn.Embedding(c+1, d_token) for c in cat_cards])

        # binary embeddings
        self.bin_embeds = nn.ModuleList([nn.Embedding(2, d_token) for _ in range(n_bin)])

        # CLS token
        self.cls = nn.Parameter(torch.zeros(1, 1, d_token))

        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_token, nhead=n_heads,
            dim_feedforward=4*d_token,
            dropout=dropout,
            batch_first=True,
            activation="gelu",
            norm_first=True
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=n_layers)

        self.head = nn.Sequential(
            nn.LayerNorm(d_token),
            nn.Linear(d_token, 1)
        )

    def forward(self, x_cont, x_cat, x_bin):
        B = x_cont.size(0)

        # cont tokens: (B, n_cont, d)
        cont_tok = x_cont.unsqueeze(-1) * self.cont_weight.unsqueeze(0) + self.cont_bias.unsqueeze(0)

        cat_tok = []
        for i, emb in enumerate(self.cat_embeds):
            cat_tok.append(emb(x_cat[:, i]))
        cat_tok = torch.stack(cat_tok, dim=1) if len(cat_tok)>0 else None

        bin_tok = []
        for i, emb in enumerate(self.bin_embeds):
            bin_tok.append(emb(x_bin[:, i]))
        bin_tok = torch.stack(bin_tok, dim=1) if len(bin_tok)>0 else None

        toks = [cont_tok]
        if cat_tok is not None: toks.append(cat_tok)
        if bin_tok is not None: toks.append(bin_tok)

        x = torch.cat(toks, dim=1)  # (B, T, d)
        cls = self.cls.expand(B, -1, -1)
        x = torch.cat([cls, x], dim=1)  # prepend CLS

        x = self.encoder(x)
        cls_out = x[:, 0, :]
        logit = self.head(cls_out).squeeze(1)
        return logit

# ---------------------------
# Train/eval repeat
# ---------------------------
def run_repeat(r, max_epochs=60, patience=10):
    tr_idx = np.load(os.path.join(SPLIT_DIR, f"train_idx_r{r}.npy"))
    va_idx = np.load(os.path.join(SPLIT_DIR, f"val_idx_r{r}.npy"))
    df_tr = df.iloc[tr_idx].reset_index(drop=True)
    df_va = df.iloc[va_idx].reset_index(drop=True)
    df_te = df.iloc[test_idx].reset_index(drop=True)

    y_tr = df_tr[TARGET].values.astype(int)
    y_va = df_va[TARGET].values.astype(int)
    y_te = df_te[TARGET].values.astype(int)

    pp = Preprocessor(cont_cols, cat_cols, bin_cols).fit(df_tr)
    X_tr = pp.transform(df_tr)
    X_va = pp.transform(df_va)
    X_te = pp.transform(df_te)

    # Use GLOBAL cardinalities (safer across folds): max encoded value in full df + 1 (already shifted)
    # But easiest: number of uniques in full df for each categorical
    cat_cards = [int(df[c].nunique(dropna=True)) for c in cat_cols]

    model = FTTransformer(
        n_cont=len(cont_cols),
        cat_cards=cat_cards,
        n_bin=len(bin_cols),
        d_token=64, n_heads=4, n_layers=3, dropout=0.15
    ).to(DEVICE)

    pos = y_tr.sum(); neg = len(y_tr) - pos
    pos_weight = torch.tensor(neg / max(1.0, pos), device=DEVICE)
    crit = FocalBCEWithLogits(gamma=2.0, pos_weight=pos_weight)

    opt = torch.optim.AdamW(model.parameters(), lr=5e-4, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max_epochs)

    tr_loader = DataLoader(TabDataset(X_tr, y_tr), batch_size=256, shuffle=True)
    va_loader = DataLoader(TabDataset(X_va, y_va), batch_size=512, shuffle=False)
    te_loader = DataLoader(TabDataset(X_te, y_te), batch_size=512, shuffle=False)

    best_pr = -1.0
    best_state = None
    bad = 0
    history = []

    for ep in range(max_epochs):
        model.train()
        losses=[]
        for xc, xk, xb, yb in tr_loader:
            xc, xk, xb, yb = xc.to(DEVICE), xk.to(DEVICE), xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            logit = model(xc, xk, xb)
            loss = crit(logit, yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            losses.append(loss.item())
        sched.step()

        # val PR
        model.eval()
        p_va=[]
        with torch.no_grad():
            for xc, xk, xb, yb in va_loader:
                logit = model(xc.to(DEVICE), xk.to(DEVICE), xb.to(DEVICE))
                p_va.append(torch.sigmoid(logit).cpu().numpy())
        p_va = np.concatenate(p_va)
        pr = float(average_precision_score(y_va, p_va))
        history.append({"epoch": ep, "loss": float(np.mean(losses)), "val_pr_auc": pr})

        print(f"Repeat {r} | Epoch {ep:02d} | loss={np.mean(losses):.4f} | val PR={pr:.4f}")

        if pr > best_pr + 1e-6:
            best_pr = pr
            best_state = {k:v.detach().cpu() for k,v in model.state_dict().items()}
            bad = 0
        else:
            bad += 1
            if bad >= patience:
                break

    model.load_state_dict(best_state)

    # final val probs
    model.eval()
    p_va=[]
    with torch.no_grad():
        for xc, xk, xb, yb in va_loader:
            logit = model(xc.to(DEVICE), xk.to(DEVICE), xb.to(DEVICE))
            p_va.append(torch.sigmoid(logit).cpu().numpy())
    p_va = np.concatenate(p_va)

    # test probs
    p_te=[]
    with torch.no_grad():
        for xc, xk, xb, yb in te_loader:
            logit = model(xc.to(DEVICE), xk.to(DEVICE), xb.to(DEVICE))
            p_te.append(torch.sigmoid(logit).cpu().numpy())
    p_te = np.concatenate(p_te)

    t_best, _ = find_best_threshold(y_va, p_va, metric="f1")

    m_va = evaluate_binary(y_va, p_va, threshold=t_best, ece_bins=cfg.ece_bins)
    m_te = evaluate_binary(y_te, p_te, threshold=t_best, ece_bins=cfg.ece_bins)

    # exports
    fig = plot_reliability_curve(y_va, p_va, n_bins=cfg.ece_bins, title=f"Reliability (VAL) — FTTransformer r{r}")
    save_fig(f"fttransformer_reliability_val_r{r}", fig=fig)

    hist_df = pd.DataFrame(history)
    fig = plt.figure()
    plt.plot(hist_df["epoch"], hist_df["val_pr_auc"], marker="o")
    plt.xlabel("Epoch"); plt.ylabel("Val PR-AUC")
    plt.title(f"FTTransformer learning curve (repeat {r})")
    save_fig(f"fttransformer_learningcurve_r{r}", fig=fig)

    return m_va, m_te

# ---------------------------
# Run repeats
# ---------------------------
seed_everything(cfg.seed)

rows=[]
for r in range(5):
    m_va, m_te = run_repeat(r)
    m_va.update({"model":"FTTransformer", "repeat":r, "split":"val"})
    m_te.update({"model":"FTTransformer", "repeat":r, "split":"test"})
    rows += [m_va, m_te]

df_m = pd.DataFrame(rows)
display(df_m)
save_table(df_m, "fttransformer_per_repeat_val_test", index=False)

test_df = df_m[df_m["split"]=="test"]
summary = test_df[["pr_auc","roc_auc","f1","recall","brier","ece","nll","acc"]].agg(["mean","std"]).T.reset_index()
summary.columns = ["metric","mean","std"]
display(summary)
save_table(summary, "fttransformer_test_summary_meanstd", index=False)

print("\n✅ TASK 5.3 complete.")
print("Compare TEST PR-AUC mean with LogReg (≈0.151).")

Using device: cpu
Repeat 0 | Epoch 00 | loss=0.7798 | val PR=0.1020
Repeat 0 | Epoch 01 | loss=0.7367 | val PR=0.1089
Repeat 0 | Epoch 02 | loss=0.6800 | val PR=0.1107
Repeat 0 | Epoch 03 | loss=0.6762 | val PR=0.1216
Repeat 0 | Epoch 04 | loss=0.6687 | val PR=0.1272
Repeat 0 | Epoch 05 | loss=0.6416 | val PR=0.1280
Repeat 0 | Epoch 06 | loss=0.6273 | val PR=0.1258
Repeat 0 | Epoch 07 | loss=0.6124 | val PR=0.1271
Repeat 0 | Epoch 08 | loss=0.5867 | val PR=0.1314
Repeat 0 | Epoch 09 | loss=0.5791 | val PR=0.1256
Repeat 0 | Epoch 10 | loss=0.5780 | val PR=0.1260
Repeat 0 | Epoch 11 | loss=0.5820 | val PR=0.1233
Repeat 0 | Epoch 12 | loss=0.5744 | val PR=0.1283
Repeat 0 | Epoch 13 | loss=0.5427 | val PR=0.1225
Repeat 0 | Epoch 14 | loss=0.5408 | val PR=0.1224
Repeat 0 | Epoch 15 | loss=0.5355 | val PR=0.1314
Repeat 0 | Epoch 16 | loss=0.5185 | val PR=0.1188
Repeat 0 | Epoch 17 | loss=0.5097 | val PR=0.1417
Repeat 0 | Epoch 18 | loss=0.5177 | val PR=0.1232
Repeat 0 | Epoch 19 | loss=0.488

,threshold,pr_auc,roc_auc,acc,f1,precision,recall,brier,ece,nll,tn,fp,fn,tp,model,repeat,split
0,0.63,0.121154,0.625514,0.774763,0.209524,0.144737,0.379310,0.199385,0.254904,0.593812,549.0,130.0,36.0,22.0,FTTransformer,0,val
1,0.63,0.116389,0.570933,0.762215,0.179775,0.123077,0.333333,0.197330,0.252344,0.584179,678.0,171.0,48.0,24.0,FTTransformer,0,test
2,0.70,0.182136,0.723122,0.814111,0.290155,0.207407,0.482759,0.253386,0.404367,0.708117,572.0,107.0,30.0,28.0,FTTransformer,1,val
3,0.70,0.120538,0.616313,0.801303,0.186667,0.137255,0.291667,0.260672,0.402236,0.722738,717.0,132.0,51.0,21.0,FTTransformer,1,test
4,0.55,0.157617,0.657889,0.712347,0.220588,0.140187,0.517241,0.193771,0.290312,0.553895,495.0,184.0,28.0,30.0,FTTransformer,2,val
5,0.55,0.129183,0.619503,0.701412,0.193548,0.122677,0.458333,0.206382,0.306400,0.586490,613.0,236.0,39.0,33.0,FTTransformer,2,test
6,0.58,0.155154,0.695775,0.607870,0.233422,0.137931,0.758621,0.295313,0.432293,0.818437,404.0,275.0,14.0,44.0,FTTransformer,3,val
7,0.58,0.145559,0.599971,0.561346,0.161826,0.095122,0.541667,0.304062,0.427474,0.836556,478.0,371.0,33.0,39.0,FTTransformer,3,test
8,0.78,0.132788,0.676477,0.834464,0.237500,0.186275,0.327586,0.255174,0.372527,0.717029,596.0,83.0,39.0,19.0,FTTransformer,4,val
9,0.78,0.104849,0.572307,0.812161,0.084656,0.068376,0.111111,0.264262,0.369275,0.744999,740.0,109.0,64.0,8.0,FTTransformer,4,test


,metric,mean,std
0,pr_auc,0.123303,0.015214
1,roc_auc,0.595806,0.023293
2,f1,0.161294,0.044439
3,recall,0.347222,0.165214
4,brier,0.246541,0.044319
5,ece,0.351546,0.071638
6,nll,0.694992,0.108815
7,acc,0.727687,0.102600



✅ TASK 5.3 complete.
Compare TEST PR-AUC mean with LogReg (≈0.151).


In [10]:
# =========================================================
# TASK 5.3 — FT-Transformer v2 (cont tokens + attention)
# Goal: beat LogReg PR-AUC (~0.151) on test
# =========================================================

import os, json, math
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import average_precision_score

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)

df = pd.read_csv(os.path.join(DIRS["base"], "data_clean.csv"))
TARGET = "stroke"

ft = pd.read_csv(os.path.join(DIRS["tab"], "feature_type_inference.csv"))
bin_cols = ft.loc[ft["type_inferred"]=="binary", "feature"].tolist()
cat_cols = ft.loc[ft["type_inferred"]=="categorical", "feature"].tolist()
cont_cols = ft.loc[ft["type_inferred"]=="continuous", "feature"].tolist()

SPLIT_DIR = os.path.join(DIRS["base"], "splits")
test_idx = np.load(os.path.join(SPLIT_DIR, "test_idx.npy"))

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OrdinalEncoder

class Preprocessor:
    def __init__(self, cont_cols, cat_cols, bin_cols):
        self.cont_cols=list(cont_cols); self.cat_cols=list(cat_cols); self.bin_cols=list(bin_cols)
        self.cont_imputer=SimpleImputer(strategy="median")
        self.scaler=StandardScaler()
        self.cat_imputer=SimpleImputer(strategy="most_frequent")
        self.encoder=OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
        self.bin_maps={}

    def fit(self, df_tr):
        self.scaler.fit(self.cont_imputer.fit_transform(df_tr[self.cont_cols]))
        self.encoder.fit(self.cat_imputer.fit_transform(df_tr[self.cat_cols]))
        for c in self.bin_cols:
            vals=sorted([int(v) for v in pd.unique(df_tr[c].dropna())])
            if len(vals)<=1: self.bin_maps[c]={vals[0]:0} if len(vals)==1 else {0:0,1:1}
            else: self.bin_maps[c]={vals[0]:0, vals[-1]:1}
        return self

    def transform(self, df_any):
        Xc=self.scaler.transform(self.cont_imputer.transform(df_any[self.cont_cols])).astype(np.float32)
        Xk=self.encoder.transform(self.cat_imputer.transform(df_any[self.cat_cols])).astype(np.int64) + 1  # unknown->0
        Xb=df_any[self.bin_cols].copy()
        for c in self.bin_cols:
            mp=self.bin_maps[c]
            Xb[c]=Xb[c].map(lambda v: mp.get(int(v),0))
        Xb=np.clip(Xb.values.astype(np.int64),0,1)
        return {
            "cont": torch.tensor(Xc, dtype=torch.float32),
            "cat":  torch.tensor(Xk, dtype=torch.long),
            "bin":  torch.tensor(Xb, dtype=torch.long),
        }

class TabDataset(Dataset):
    def __init__(self, X, y):
        self.X=X
        self.y=torch.tensor(y, dtype=torch.float32)
    def __len__(self): return len(self.y)
    def __getitem__(self,i):
        return self.X["cont"][i], self.X["cat"][i], self.X["bin"][i], self.y[i]

# ---- FT-Transformer v2 ----
class FTTransformerV2(nn.Module):
    def __init__(self, n_cont, cat_cards, n_bin, d=64, n_heads=4, n_layers=3, dropout=0.15):
        super().__init__()
        self.n_cont=n_cont
        self.cls = nn.Parameter(torch.zeros(1,1,d))

        # Per-cont-feature tokenization: t_i = w_i*x_i + b_i
        self.cont_w = nn.Parameter(torch.randn(n_cont, d) * 0.02)
        self.cont_b = nn.Parameter(torch.zeros(n_cont, d))

        self.cat_emb = nn.ModuleList([nn.Embedding(card+1, d) for card in cat_cards])  # indices 0..card
        self.bin_emb = nn.ModuleList([nn.Embedding(2, d) for _ in range(n_bin)])

        enc_layer = nn.TransformerEncoderLayer(
            d_model=d, nhead=n_heads, dim_feedforward=4*d, dropout=dropout,
            activation="gelu", batch_first=True, norm_first=True
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=n_layers)

        self.head = nn.Sequential(
            nn.LayerNorm(d),
            nn.Linear(d, 1)
        )

    def forward(self, x_cont, x_cat, x_bin):
        B = x_cont.size(0)

        # continuous tokens: (B,n_cont,d)
        cont_tokens = x_cont.unsqueeze(-1) * self.cont_w.unsqueeze(0) + self.cont_b.unsqueeze(0)

        cat_tokens=[]
        for j, emb in enumerate(self.cat_emb):
            cat_tokens.append(emb(x_cat[:,j]))
        cat_tokens = torch.stack(cat_tokens, dim=1) if len(cat_tokens)>0 else None

        bin_tokens=[]
        for j, emb in enumerate(self.bin_emb):
            bin_tokens.append(emb(x_bin[:,j]))
        bin_tokens = torch.stack(bin_tokens, dim=1) if len(bin_tokens)>0 else None

        tokens = [self.cls.expand(B,1,-1), cont_tokens]
        if cat_tokens is not None: tokens.append(cat_tokens)
        if bin_tokens is not None: tokens.append(bin_tokens)
        x = torch.cat(tokens, dim=1)

        h = self.encoder(x)
        cls = h[:,0,:]
        logit = self.head(cls).squeeze(1)
        return logit

def run_repeat(r, max_epochs=60, patience=10):
    tr_idx=np.load(os.path.join(SPLIT_DIR,f"train_idx_r{r}.npy"))
    va_idx=np.load(os.path.join(SPLIT_DIR,f"val_idx_r{r}.npy"))

    df_tr=df.iloc[tr_idx].reset_index(drop=True)
    df_va=df.iloc[va_idx].reset_index(drop=True)
    df_te=df.iloc[test_idx].reset_index(drop=True)

    y_tr=df_tr[TARGET].values.astype(int)
    y_va=df_va[TARGET].values.astype(int)
    y_te=df_te[TARGET].values.astype(int)

    pp=Preprocessor(cont_cols,cat_cols,bin_cols).fit(df_tr)
    X_tr=pp.transform(df_tr); X_va=pp.transform(df_va); X_te=pp.transform(df_te)

    cat_cards=[int(df_tr[c].nunique(dropna=True)) for c in cat_cols]

    model=FTTransformerV2(len(cont_cols), cat_cards, len(bin_cols), d=64, n_heads=4, n_layers=3, dropout=0.15).to(DEVICE)

    pos=y_tr.sum(); neg=len(y_tr)-pos
    pos_weight=neg/max(1.0,pos)
    crit=nn.BCEWithLogitsLoss(pos_weight=torch.tensor(pos_weight,device=DEVICE))
    opt=torch.optim.AdamW(model.parameters(), lr=5e-4, weight_decay=3e-4)
    sch=torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max_epochs)

    tr_loader=DataLoader(TabDataset(X_tr,y_tr), batch_size=256, shuffle=True)
    va_loader=DataLoader(TabDataset(X_va,y_va), batch_size=512, shuffle=False)
    te_loader=DataLoader(TabDataset(X_te,y_te), batch_size=512, shuffle=False)

    best_pr=-1.0; best_state=None; bad=0
    hist=[]
    for epoch in range(max_epochs):
        model.train()
        losses=[]
        for xc,xk,xb,yb in tr_loader:
            xc,xk,xb,yb=xc.to(DEVICE),xk.to(DEVICE),xb.to(DEVICE),yb.to(DEVICE)
            opt.zero_grad()
            logit=model(xc,xk,xb)
            loss=crit(logit,yb)
            loss.backward()
            opt.step()
            losses.append(loss.item())
        sch.step()

        # val PR
        model.eval()
        p_va=[]
        with torch.no_grad():
            for xc,xk,xb,yb in va_loader:
                logit=model(xc.to(DEVICE),xk.to(DEVICE),xb.to(DEVICE))
                p_va.append(torch.sigmoid(logit).cpu().numpy())
        p_va=np.concatenate(p_va)
        pr=float(average_precision_score(y_va,p_va))
        hist.append({"epoch":epoch,"loss":float(np.mean(losses)),"val_pr_auc":pr})
        print(f"r{r} ep{epoch:02d} loss={np.mean(losses):.4f} valPR={pr:.4f}")

        if pr>best_pr+1e-6:
            best_pr=pr
            best_state={k:v.detach().cpu() for k,v in model.state_dict().items()}
            bad=0
        else:
            bad+=1
            if bad>=patience:
                break

    model.load_state_dict(best_state)

    # final val probs for threshold
    model.eval()
    p_va=[]
    with torch.no_grad():
        for xc,xk,xb,yb in va_loader:
            logit=model(xc.to(DEVICE),xk.to(DEVICE),xb.to(DEVICE))
            p_va.append(torch.sigmoid(logit).cpu().numpy())
    p_va=np.concatenate(p_va)

    # test probs
    p_te=[]
    with torch.no_grad():
        for xc,xk,xb,yb in te_loader:
            logit=model(xc.to(DEVICE),xk.to(DEVICE),xb.to(DEVICE))
            p_te.append(torch.sigmoid(logit).cpu().numpy())
    p_te=np.concatenate(p_te)

    t_best,_=find_best_threshold(y_va,p_va,metric="f1")
    m_val=evaluate_binary(y_va,p_va,threshold=t_best,ece_bins=cfg.ece_bins)
    m_tst=evaluate_binary(y_te,p_te,threshold=t_best,ece_bins=cfg.ece_bins)

    # plots
    fig=plot_reliability_curve(y_va,p_va,n_bins=cfg.ece_bins,title=f"Reliability (VAL) — FTTransformerV2 r{r}")
    save_fig(f"ftv2_reliability_val_r{r}", fig=fig)

    hdf=pd.DataFrame(hist)
    fig=plt.figure()
    plt.plot(hdf["epoch"], hdf["val_pr_auc"], marker="o")
    plt.xlabel("Epoch"); plt.ylabel("Val PR-AUC")
    plt.title(f"FTTransformerV2 learning curve (repeat {r})")
    save_fig(f"ftv2_learningcurve_r{r}", fig=fig)

    return m_val, m_tst

# ---------------------------
# Run 5 repeats
# ---------------------------
seed_everything(cfg.seed)
rows=[]
for r in range(5):
    mva,mte=run_repeat(r)
    mva.update({"model":"FTTransformerV2","repeat":r,"split":"val"})
    mte.update({"model":"FTTransformerV2","repeat":r,"split":"test"})
    rows += [mva,mte]

df_metrics=pd.DataFrame(rows)
display(df_metrics)
save_table(df_metrics, "fttransformerv2_per_repeat_val_test", index=False)

test_df=df_metrics[df_metrics["split"]=="test"]
summary=test_df[["pr_auc","roc_auc","f1","recall","brier","ece","nll","acc"]].agg(["mean","std"]).T.reset_index()
summary.columns=["metric","mean","std"]
display(summary)
save_table(summary, "fttransformerv2_test_summary_meanstd", index=False)

print("\n✅ TASK 5.3 complete. Compare PR-AUC mean to LogReg ≈ 0.151.")

Using device: cpu
r0 ep00 loss=1.2604 valPR=0.1031
r0 ep01 loss=1.1832 valPR=0.0995
r0 ep02 loss=1.0964 valPR=0.1060
r0 ep03 loss=1.0811 valPR=0.1244
r0 ep04 loss=1.1017 valPR=0.1257
r1 ep06 loss=1.0935 valPR=0.2153
r1 ep07 loss=1.0707 valPR=0.2021
r1 ep08 loss=1.0587 valPR=0.1910
r1 ep09 loss=1.0375 valPR=0.1991
r1 ep10 loss=1.0824 valPR=0.1792
r1 ep11 loss=1.0396 valPR=0.2015
r1 ep12 loss=1.0121 valPR=0.1926
r2 ep00 loss=1.2826 valPR=0.1218
r2 ep01 loss=1.1718 valPR=0.1160
r2 ep02 loss=1.1334 valPR=0.1321
r2 ep03 loss=1.1110 valPR=0.1357
r2 ep04 loss=1.1922 valPR=0.1365
r2 ep05 loss=1.0879 valPR=0.1167
r2 ep06 loss=1.0746 valPR=0.1173
r2 ep07 loss=1.0015 valPR=0.1178
r2 ep08 loss=1.0456 valPR=0.1399
r2 ep09 loss=0.9939 valPR=0.1278
r2 ep10 loss=0.9640 valPR=0.1404
r2 ep11 loss=0.9424 valPR=0.1444
r2 ep12 loss=0.8950 valPR=0.1311
r2 ep13 loss=0.9201 valPR=0.1556
r2 ep14 loss=0.9347 valPR=0.1312
r2 ep15 loss=0.9327 valPR=0.1404
r2 ep16 loss=0.8835 valPR=0.1221
r2 ep17 loss=0.8709 valPR

,threshold,pr_auc,roc_auc,acc,f1,precision,recall,brier,ece,nll,tn,fp,fn,tp,model,repeat,split
0,0.58,0.136759,0.658346,0.719132,0.241758,0.153488,0.568966,0.211124,0.273019,0.610380,497.0,182.0,25.0,33.0,FTTransformerV2,0,val
1,0.58,0.096334,0.592920,0.682953,0.175141,0.109929,0.430556,0.231111,0.290293,0.663415,598.0,251.0,41.0,31.0,FTTransformerV2,0,test
2,0.72,0.192564,0.735184,0.788331,0.264151,0.181818,0.482759,0.236869,0.335091,0.644121,553.0,126.0,30.0,28.0,FTTransformerV2,1,val
3,0.72,0.127575,0.610686,0.735071,0.164384,0.109091,0.333333,0.268538,0.352768,0.719656,653.0,196.0,48.0,24.0,FTTransformerV2,1,test
4,0.79,0.136395,0.646006,0.824966,0.208589,0.161905,0.293103,0.200234,0.227863,0.596445,591.0,88.0,41.0,17.0,FTTransformerV2,2,val
5,0.79,0.117227,0.609737,0.821933,0.211538,0.161765,0.305556,0.214954,0.256044,0.650124,735.0,114.0,50.0,22.0,FTTransformerV2,2,test
6,0.75,0.150539,0.686633,0.842605,0.265823,0.210000,0.362069,0.203648,0.256320,0.573368,600.0,79.0,37.0,21.0,FTTransformerV2,3,val
7,0.75,0.149957,0.641245,0.814332,0.189573,0.143885,0.277778,0.214355,0.272502,0.605117,730.0,119.0,52.0,20.0,FTTransformerV2,3,test
8,0.76,0.142782,0.680869,0.814111,0.259459,0.188976,0.413793,0.190575,0.224857,0.539039,576.0,103.0,34.0,24.0,FTTransformerV2,4,val
9,0.76,0.106258,0.556586,0.782845,0.180328,0.127907,0.305556,0.220562,0.246031,0.619865,699.0,150.0,50.0,22.0,FTTransformerV2,4,test


,metric,mean,std
0,pr_auc,0.119470,0.020676
1,roc_auc,0.602235,0.030903
2,f1,0.184193,0.017791
3,recall,0.330556,0.059252
4,brier,0.229904,0.022620
5,ece,0.283528,0.042199
6,nll,0.651635,0.044557
7,acc,0.767427,0.058291



✅ TASK 5.3 complete. Compare PR-AUC mean to LogReg ≈ 0.151.


In [11]:
# ===== TASK A: Feature audit (10 seconds) =====
import pandas as pd, numpy as np, os

df = pd.read_csv(os.path.join(DIRS["base"], "data_clean.csv"))
TARGET="stroke"

ft = pd.read_csv(os.path.join(DIRS["tab"], "feature_type_inference.csv"))
bin_cols = ft.loc[ft["type_inferred"]=="binary", "feature"].tolist()
cat_cols = ft.loc[ft["type_inferred"]=="categorical", "feature"].tolist()
cont_cols = ft.loc[ft["type_inferred"]=="continuous", "feature"].tolist()

rows=[]
for c in bin_cols+cat_cols+cont_cols:
    s = df[c]
    uniq = pd.unique(s.dropna())
    rows.append({
        "feature": c,
        "type_inferred": ft.loc[ft["feature"]==c, "type_inferred"].values[0],
        "n_unique": len(uniq),
        "min": np.nanmin(s.values) if s.notna().any() else np.nan,
        "max": np.nanmax(s.values) if s.notna().any() else np.nan,
        "sample_uniques": str(list(sorted(uniq)[:10])),
        "missing_frac": float(s.isna().mean())
    })

audit = pd.DataFrame(rows).sort_values(["type_inferred","n_unique"], ascending=[True,True])
display(audit.head(40))
display(audit.tail(20))

# Flag suspicious "binary"
sus_bin = audit[(audit["type_inferred"]=="binary") & (~audit["sample_uniques"].isin(["[0, 1]","[0]","[1]"]))][["feature","sample_uniques","min","max","n_unique"]]
print("\nSuspicious binary columns (not {0,1}):")
display(sus_bin)

,feature,type_inferred,n_unique,min,max,sample_uniques,missing_frac
0,gender,binary,2,1.000,2.000,"[np.int64(1), np.int64(2)]",0.0
1,alcohol,binary,2,0.000,1.000,"[np.int64(0), np.int64(1)]",0.0
2,smoke,binary,2,0.000,1.000,"[np.int64(0), np.int64(1)]",0.0
3,sleep_disorder,binary,2,1.000,2.000,"[np.int64(1), np.int64(2)]",0.0
4,Health_Insurance,binary,2,1.000,2.000,"[np.int64(1), np.int64(2)]",0.0
5,diabetes,binary,2,0.000,1.000,"[np.int64(0), np.int64(1)]",0.0
6,hypertension,binary,2,0.000,1.000,"[np.int64(0), np.int64(1)]",0.0
7,high_cholesterol,binary,2,0.000,1.000,"[np.int64(0), np.int64(1)]",0.0
8,Coronary_Heart_Disease,binary,2,0.000,1.000,"[np.int64(0), np.int64(1)]",0.0
9,age,categorical,3,1.000,3.000,"[np.int64(1), np.int64(2), np.int64(3)]",0.0


,feature,type_inferred,n_unique,min,max,sample_uniques,missing_frac
15,sleep_time,continuous,23,1.000,14.000,"[np.float64(1.0), np.float64(2.0), np.float64(...",0.0
16,Minutes_sedentary_activity,continuous,37,0.000,1200.000,"[np.int64(0), np.int64(1), np.int64(2), np.int...",0.0
19,Diastolic_blood_pressure,continuous,44,32.000,124.000,"[np.int64(32), np.int64(38), np.int64(40), np....",0.0
18,Systolic_blood_pressure,continuous,77,66.000,238.000,"[np.int64(66), np.int64(72), np.int64(80), np....",0.0
24,Glycohemoglobin,continuous,106,2.000,16.400,"[np.float64(2.0), np.float64(3.7), np.float64(...",0.0
20,Highdensity_lipoprotein,continuous,110,0.280,5.840,"[np.float64(0.28), np.float64(0.39), np.float6...",0.0
28,Dietary_fiber,continuous,484,0.000,107.000,"[np.float64(0.0), np.float64(0.1), np.float64(...",0.0
17,Waist_Circumference,continuous,776,63.500,176.000,"[np.float64(63.5), np.float64(64.0), np.float6...",0.0
25,energy,continuous,2387,0.000,13687.000,"[np.int64(0), np.int64(89), np.int64(110), np....",0.0
22,Lowdensity_lipoprotein,continuous,2414,0.388,9.232,"[np.float64(0.388), np.float64(0.621), np.floa...",0.0



Suspicious binary columns (not {0,1}):


,feature,sample_uniques,min,max,n_unique
0,gender,"[np.int64(1), np.int64(2)]",1.0,2.0,2
1,alcohol,"[np.int64(0), np.int64(1)]",0.0,1.0,2
2,smoke,"[np.int64(0), np.int64(1)]",0.0,1.0,2
3,sleep_disorder,"[np.int64(1), np.int64(2)]",1.0,2.0,2
4,Health_Insurance,"[np.int64(1), np.int64(2)]",1.0,2.0,2
5,diabetes,"[np.int64(0), np.int64(1)]",0.0,1.0,2
6,hypertension,"[np.int64(0), np.int64(1)]",0.0,1.0,2
7,high_cholesterol,"[np.int64(0), np.int64(1)]",0.0,1.0,2
8,Coronary_Heart_Disease,"[np.int64(0), np.int64(1)]",0.0,1.0,2


In [12]:
# =========================================================
# TASK B — Fix binary/ordinal typing + skew transform
# Outputs new dataset: data_clean_v2.csv and feature types v2
# =========================================================
import os, numpy as np, pandas as pd

df = pd.read_csv(os.path.join(DIRS["base"], "data_clean.csv"))
TARGET="stroke"

# ---- 1) Fix binary {1,2} -> {0,1}
bin_fix = ["gender", "sleep_disorder", "Health_Insurance"]
for c in bin_fix:
    if c in df.columns:
        df[c] = df[c].map({1:0, 2:1}).astype(int)

# ---- 2) Ordinal-as-numeric: move these to "continuous"
ordinal_cols = ["age", "depression", "Body_Mass_Index", "General_health_condition"]
for c in ordinal_cols:
    if c in df.columns:
        df[c] = df[c].astype(float)  # treat as numeric

# ---- 3) Keep only truly nominal categoricals
nominal_cat = [c for c in ["Race", "Marital_status"] if c in df.columns]

# ---- 4) Identify skewed continuous features and log1p them
# We'll log1p any feature with max > 1000 or (max/min positive) huge; simple heuristic.
cont_candidates = [c for c in df.columns if c not in [TARGET] + nominal_cat]
# exclude obvious binaries
for c in cont_candidates:
    if set(pd.unique(df[c])).issubset({0,1}):
        continue

skew_cols = []
for c in df.columns:
    if c in [TARGET] + nominal_cat: 
        continue
    if df[c].dtype.kind not in "if": 
        continue
    mx = float(df[c].max())
    if mx > 1000:  # heuristic
        skew_cols.append(c)

# log1p only non-negative columns
for c in skew_cols:
    if (df[c] < 0).any():
        continue
    df[c] = np.log1p(df[c].astype(float))

print("Binary fixed:", bin_fix)
print("Ordinal treated as numeric:", ordinal_cols)
print("Nominal categoricals kept:", nominal_cat)
print("log1p applied to:", skew_cols)

# ---- 5) Rebuild feature types file v2
# Rule:
# - nominal_cat => categorical
# - strict {0,1} => binary
# - others numeric => continuous
rows=[]
for c in df.columns:
    if c == TARGET: 
        continue
    if c in nominal_cat:
        t="categorical"
    else:
        uniq=set(pd.unique(df[c]))
        if uniq.issubset({0,1}):
            t="binary"
        else:
            t="continuous"
    rows.append({"feature":c, "type_inferred":t, "n_unique_nonnull": int(df[c].nunique())})

ft2 = pd.DataFrame(rows)
display(ft2["type_inferred"].value_counts())

# Save
out_data = os.path.join(DIRS["base"], "data_clean_v2.csv")
out_ft   = os.path.join(DIRS["tab"],  "feature_type_inference_v2.csv")
df.to_csv(out_data, index=False)
ft2.to_csv(out_ft, index=False)

print("✅ Saved:", out_data)
print("✅ Saved:", out_ft)

Binary fixed: ['gender', 'sleep_disorder', 'Health_Insurance']
Ordinal treated as numeric: ['age', 'depression', 'Body_Mass_Index', 'General_health_condition']
Nominal categoricals kept: ['Race', 'Marital_status']
log1p applied to: ['Minutes_sedentary_activity', 'energy', 'Carbohydrate', 'Potassium', 'Sodium']


type_inferred
continuous     24
binary          9
categorical     2
Name: count, dtype: int64

✅ Saved: /kaggle/working/outputs/cgt_stroke_v1/data_clean_v2.csv
✅ Saved: /kaggle/working/outputs/cgt_stroke_v1/tables/feature_type_inference_v2.csv


In [13]:
# =========================================================
# TASK C.1 — Baseline sanity on v2
# - Load data_clean_v2.csv + feature_type_inference_v2.csv
# - Use SAME split indices as before
# - Train LogReg baseline and Tiny FTTransformerV2
# - Export tables + reliability plots
# =========================================================

import os, json, numpy as np, pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)

TARGET = "stroke"
DATA_V2 = os.path.join(DIRS["base"], "data_clean_v2.csv")
FT_V2   = os.path.join(DIRS["tab"],  "feature_type_inference_v2.csv")

df = pd.read_csv(DATA_V2)
ft = pd.read_csv(FT_V2)

bin_cols = ft.loc[ft["type_inferred"]=="binary", "feature"].tolist()
cat_cols = ft.loc[ft["type_inferred"]=="categorical", "feature"].tolist()
cont_cols = ft.loc[ft["type_inferred"]=="continuous", "feature"].tolist()

print("Counts:", {"bin":len(bin_cols), "cat":len(cat_cols), "cont":len(cont_cols)})

SPLIT_DIR = os.path.join(DIRS["base"], "splits")
test_idx = np.load(os.path.join(SPLIT_DIR, "test_idx.npy"))

# ---------------------------
# Preprocessor (updated: cat only 2 cols now)
# ---------------------------
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OrdinalEncoder

class Preprocessor:
    def __init__(self, cont_cols, cat_cols, bin_cols):
        self.cont_cols=list(cont_cols); self.cat_cols=list(cat_cols); self.bin_cols=list(bin_cols)
        self.cont_imputer=SimpleImputer(strategy="median")
        self.scaler=StandardScaler()
        self.cat_imputer=SimpleImputer(strategy="most_frequent")
        self.encoder=OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)

    def fit(self, df_tr):
        self.scaler.fit(self.cont_imputer.fit_transform(df_tr[self.cont_cols]))
        if len(self.cat_cols)>0:
            self.encoder.fit(self.cat_imputer.fit_transform(df_tr[self.cat_cols]))
        return self

    def transform(self, df_any):
        Xc = self.scaler.transform(self.cont_imputer.transform(df_any[self.cont_cols])).astype(np.float32)

        if len(self.cat_cols)>0:
            Xk = self.encoder.transform(self.cat_imputer.transform(df_any[self.cat_cols])).astype(np.int64) + 1
        else:
            Xk = np.zeros((len(df_any),0), dtype=np.int64)

        # binaries are now true {0,1} numeric; we keep as int64 for embeddings in DL
        Xb = df_any[self.bin_cols].values.astype(np.int64) if len(self.bin_cols)>0 else np.zeros((len(df_any),0),dtype=np.int64)
        Xb = np.clip(Xb,0,1)

        return {
            "cont": torch.tensor(Xc, dtype=torch.float32),
            "cat":  torch.tensor(Xk, dtype=torch.long),
            "bin":  torch.tensor(Xb, dtype=torch.long),
        }

# ---------------------------
# Evaluate helper: for LR we need numeric matrix
# ---------------------------
def to_numpy_matrix(Xdict):
    # cont (float) + cat (int) + bin (int) -> float matrix
    Xc = Xdict["cont"].numpy()
    Xk = Xdict["cat"].numpy().astype(np.float32)
    Xb = Xdict["bin"].numpy().astype(np.float32)
    return np.concatenate([Xc, Xk, Xb], axis=1)

# ---------------------------
# Logistic Regression baseline on v2
# ---------------------------
def run_logreg_repeat(r):
    tr_idx=np.load(os.path.join(SPLIT_DIR,f"train_idx_r{r}.npy"))
    va_idx=np.load(os.path.join(SPLIT_DIR,f"val_idx_r{r}.npy"))

    df_tr=df.iloc[tr_idx].reset_index(drop=True)
    df_va=df.iloc[va_idx].reset_index(drop=True)
    df_te=df.iloc[test_idx].reset_index(drop=True)

    y_tr=df_tr[TARGET].values.astype(int)
    y_va=df_va[TARGET].values.astype(int)
    y_te=df_te[TARGET].values.astype(int)

    pp=Preprocessor(cont_cols,cat_cols,bin_cols).fit(df_tr)
    X_tr=to_numpy_matrix(pp.transform(df_tr))
    X_va=to_numpy_matrix(pp.transform(df_va))
    X_te=to_numpy_matrix(pp.transform(df_te))

    # class_weight balances implicitly
    lr = LogisticRegression(max_iter=2000, class_weight="balanced", solver="liblinear")
    lr.fit(X_tr, y_tr)

    p_va = lr.predict_proba(X_va)[:,1]
    p_te = lr.predict_proba(X_te)[:,1]

    t_best,_ = find_best_threshold(y_va, p_va, metric="f1")
    m_va = evaluate_binary(y_va, p_va, threshold=t_best, ece_bins=cfg.ece_bins)
    m_te = evaluate_binary(y_te, p_te, threshold=t_best, ece_bins=cfg.ece_bins)

    fig = plot_reliability_curve(y_va, p_va, n_bins=cfg.ece_bins, title=f"Reliability (VAL) — LogReg v2 r{r}")
    save_fig(f"logreg_v2_reliability_val_r{r}", fig=fig)

    return m_va, m_te

# ---------------------------
# Tiny FTTransformerV2 (right-sized)
# ---------------------------
class TabDataset(Dataset):
    def __init__(self, X, y):
        self.X=X
        self.y=torch.tensor(y, dtype=torch.float32)
    def __len__(self): return len(self.y)
    def __getitem__(self,i):
        return self.X["cont"][i], self.X["cat"][i], self.X["bin"][i], self.y[i]

class FTTransformerV2Tiny(nn.Module):
    def __init__(self, n_cont, cat_cards, n_bin, d=32, n_heads=2, n_layers=1, dropout=0.35):
        super().__init__()
        self.cls = nn.Parameter(torch.zeros(1,1,d))
        self.cont_w = nn.Parameter(torch.randn(n_cont, d)*0.02)
        self.cont_b = nn.Parameter(torch.zeros(n_cont, d))

        self.cat_emb = nn.ModuleList([nn.Embedding(card+1, d) for card in cat_cards])
        self.bin_emb = nn.ModuleList([nn.Embedding(2, d) for _ in range(n_bin)])

        enc_layer = nn.TransformerEncoderLayer(
            d_model=d, nhead=n_heads, dim_feedforward=4*d, dropout=dropout,
            activation="gelu", batch_first=True, norm_first=True
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=n_layers)
        self.head = nn.Sequential(nn.LayerNorm(d), nn.Dropout(dropout), nn.Linear(d,1))

    def forward(self, x_cont, x_cat, x_bin):
        B = x_cont.size(0)
        cont_tokens = x_cont.unsqueeze(-1) * self.cont_w.unsqueeze(0) + self.cont_b.unsqueeze(0)

        if x_cat.size(1)>0:
            cat_tokens = torch.stack([emb(x_cat[:,j]) for j,emb in enumerate(self.cat_emb)], dim=1)
        else:
            cat_tokens = None

        if x_bin.size(1)>0:
            bin_tokens = torch.stack([emb(x_bin[:,j]) for j,emb in enumerate(self.bin_emb)], dim=1)
        else:
            bin_tokens = None

        tokens = [self.cls.expand(B,1,-1), cont_tokens]
        if cat_tokens is not None: tokens.append(cat_tokens)
        if bin_tokens is not None: tokens.append(bin_tokens)
        x = torch.cat(tokens, dim=1)

        h = self.encoder(x)
        cls = h[:,0,:]
        return self.head(cls).squeeze(1)

def run_fttiny_repeat(r, max_epochs=80, patience=12):
    tr_idx=np.load(os.path.join(SPLIT_DIR,f"train_idx_r{r}.npy"))
    va_idx=np.load(os.path.join(SPLIT_DIR,f"val_idx_r{r}.npy"))

    df_tr=df.iloc[tr_idx].reset_index(drop=True)
    df_va=df.iloc[va_idx].reset_index(drop=True)
    df_te=df.iloc[test_idx].reset_index(drop=True)

    y_tr=df_tr[TARGET].values.astype(int)
    y_va=df_va[TARGET].values.astype(int)
    y_te=df_te[TARGET].values.astype(int)

    pp=Preprocessor(cont_cols,cat_cols,bin_cols).fit(df_tr)
    X_tr=pp.transform(df_tr); X_va=pp.transform(df_va); X_te=pp.transform(df_te)

    cat_cards=[int(df_tr[c].nunique(dropna=True)) for c in cat_cols]

    model=FTTransformerV2Tiny(len(cont_cols), cat_cards, len(bin_cols)).to(DEVICE)

    pos=y_tr.sum(); neg=len(y_tr)-pos
    pos_weight=neg/max(1.0,pos)
    crit=nn.BCEWithLogitsLoss(pos_weight=torch.tensor(pos_weight,device=DEVICE))

    opt=torch.optim.AdamW(model.parameters(), lr=6e-4, weight_decay=1e-3)
    sch=torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max_epochs)

    tr_loader=DataLoader(TabDataset(X_tr,y_tr), batch_size=256, shuffle=True)
    va_loader=DataLoader(TabDataset(X_va,y_va), batch_size=512, shuffle=False)
    te_loader=DataLoader(TabDataset(X_te,y_te), batch_size=512, shuffle=False)

    best_pr=-1.0; best_state=None; bad=0; history=[]
    for epoch in range(max_epochs):
        model.train()
        losses=[]
        for xc,xk,xb,yb in tr_loader:
            xc,xk,xb,yb=xc.to(DEVICE),xk.to(DEVICE),xb.to(DEVICE),yb.to(DEVICE)
            opt.zero_grad()
            logit=model(xc,xk,xb)
            loss=crit(logit,yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            losses.append(loss.item())
        sch.step()

        model.eval()
        p_va=[]
        with torch.no_grad():
            for xc,xk,xb,yb in va_loader:
                logit=model(xc.to(DEVICE),xk.to(DEVICE),xb.to(DEVICE))
                p_va.append(torch.sigmoid(logit).cpu().numpy())
        p_va=np.concatenate(p_va)

        pr=float(average_precision_score(y_va,p_va))
        history.append({"epoch":epoch,"loss":float(np.mean(losses)),"val_pr_auc":pr})
        print(f"FTTiny r{r} ep{epoch:02d} loss={np.mean(losses):.4f} valPR={pr:.4f}")

        if pr>best_pr+1e-6:
            best_pr=pr
            best_state={k:v.detach().cpu() for k,v in model.state_dict().items()}
            bad=0
        else:
            bad+=1
            if bad>=patience:
                break

    model.load_state_dict(best_state)

    # probs
    model.eval()
    p_va=[]
    with torch.no_grad():
        for xc,xk,xb,yb in va_loader:
            logit=model(xc.to(DEVICE),xk.to(DEVICE),xb.to(DEVICE))
            p_va.append(torch.sigmoid(logit).cpu().numpy())
    p_va=np.concatenate(p_va)

    p_te=[]
    with torch.no_grad():
        for xc,xk,xb,yb in te_loader:
            logit=model(xc.to(DEVICE),xk.to(DEVICE),xb.to(DEVICE))
            p_te.append(torch.sigmoid(logit).cpu().numpy())
    p_te=np.concatenate(p_te)

    t_best,_=find_best_threshold(y_va,p_va,metric="f1")
    m_va=evaluate_binary(y_va,p_va,threshold=t_best,ece_bins=cfg.ece_bins)
    m_te=evaluate_binary(y_te,p_te,threshold=t_best,ece_bins=cfg.ece_bins)

    fig=plot_reliability_curve(y_va,p_va,n_bins=cfg.ece_bins,title=f"Reliability (VAL) — FTTiny v2 r{r}")
    save_fig(f"fttiny_v2_reliability_val_r{r}", fig=fig)

    hdf=pd.DataFrame(history)
    fig=plt.figure()
    plt.plot(hdf["epoch"], hdf["val_pr_auc"], marker="o")
    plt.xlabel("Epoch"); plt.ylabel("Val PR-AUC")
    plt.title(f"FTTiny v2 learning curve (repeat {r})")
    save_fig(f"fttiny_v2_learningcurve_r{r}", fig=fig)

    return m_va, m_te

# ---------------------------
# Run 5 repeats for both
# ---------------------------
seed_everything(cfg.seed)

rows=[]

for r in range(5):
    mva,mte = run_logreg_repeat(r)
    mva.update({"model":"LogReg_v2","repeat":r,"split":"val"})
    mte.update({"model":"LogReg_v2","repeat":r,"split":"test"})
    rows += [mva,mte]

for r in range(5):
    mva,mte = run_fttiny_repeat(r)
    mva.update({"model":"FTTiny_v2","repeat":r,"split":"val"})
    mte.update({"model":"FTTiny_v2","repeat":r,"split":"test"})
    rows += [mva,mte]

df_m = pd.DataFrame(rows)
display(df_m)
save_table(df_m, "v2_logreg_fttiny_per_repeat_val_test", index=False)

# summary table (test)
test_df = df_m[df_m["split"]=="test"].copy()
summary = test_df.groupby("model")[["pr_auc","roc_auc","f1","recall","brier","ece","nll","acc"]].agg(["mean","std"])
summary.columns = [f"{a}_{b}" for a,b in summary.columns]
summary = summary.reset_index()
display(summary)
save_table(summary, "v2_summary_meanstd_by_model", index=False)

print("\n✅ TASK C.1 complete. Next: decide Wide&Deep if FTTiny improves.")

Using device: cpu
Counts: {'bin': 9, 'cat': 2, 'cont': 24}
FTTiny r0 ep00 loss=1.4140 valPR=0.1383
FTTiny r0 ep01 loss=1.3080 valPR=0.1434
FTTiny r0 ep02 loss=1.2636 valPR=0.1464
FTTiny r0 ep03 loss=1.2341 valPR=0.1731
FTTiny r0 ep04 loss=1.2169 valPR=0.1741
FTTiny r0 ep05 loss=1.2287 valPR=0.1753
FTTiny r0 ep06 loss=1.2354 valPR=0.1476
FTTiny r0 ep07 loss=1.1815 valPR=0.1459
FTTiny r0 ep08 loss=1.1871 valPR=0.1492
FTTiny r0 ep09 loss=1.1643 valPR=0.1534
FTTiny r0 ep10 loss=1.1980 valPR=0.1357
FTTiny r0 ep11 loss=1.1751 valPR=0.1494
FTTiny r0 ep12 loss=1.1392 valPR=0.1426
FTTiny r0 ep13 loss=1.1530 valPR=0.1364
FTTiny r0 ep14 loss=1.1244 valPR=0.1368
FTTiny r0 ep15 loss=1.1040 valPR=0.1335
FTTiny r0 ep16 loss=1.1552 valPR=0.1389
FTTiny r0 ep17 loss=1.1385 valPR=0.1470
FTTiny r1 ep00 loss=1.3742 valPR=0.1438
FTTiny r1 ep01 loss=1.3625 valPR=0.1451
FTTiny r1 ep02 loss=1.3249 valPR=0.1401
FTTiny r1 ep03 loss=1.3096 valPR=0.1399
FTTiny r1 ep04 loss=1.2688 valPR=0.1491
FTTiny r1 ep05 loss=1

,threshold,pr_auc,roc_auc,acc,f1,precision,recall,brier,ece,nll,tn,fp,fn,tp,model,repeat,split
0,0.72,0.163021,0.681453,0.871099,0.274809,0.246575,0.310345,0.202013,0.316377,0.590900,624.0,55.0,40.0,18.0,LogReg_v2,0,val
1,0.72,0.158503,0.614187,0.847991,0.135802,0.122222,0.152778,0.213385,0.318694,0.609727,770.0,79.0,61.0,11.0,LogReg_v2,0,test
2,0.73,0.232478,0.760728,0.876526,0.345324,0.296296,0.413793,0.206520,0.336274,0.597882,622.0,57.0,34.0,24.0,LogReg_v2,1,val
3,0.73,0.150439,0.618653,0.856678,0.131579,0.125000,0.138889,0.215177,0.328030,0.614535,779.0,70.0,62.0,10.0,LogReg_v2,1,test
4,0.62,0.171860,0.700523,0.780190,0.242991,0.166667,0.448276,0.199893,0.310642,0.581868,549.0,130.0,32.0,26.0,LogReg_v2,2,val
5,0.62,0.147745,0.626554,0.776330,0.155738,0.110465,0.263889,0.209030,0.316645,0.600789,696.0,153.0,53.0,19.0,LogReg_v2,2,test
6,0.72,0.164443,0.697501,0.854817,0.241135,0.204819,0.293103,0.215130,0.336490,0.619642,613.0,66.0,41.0,17.0,LogReg_v2,3,val
7,0.72,0.157524,0.634112,0.847991,0.146341,0.130435,0.166667,0.211645,0.323769,0.609079,769.0,80.0,60.0,12.0,LogReg_v2,3,test
8,0.70,0.180898,0.736885,0.857531,0.304636,0.247312,0.396552,0.204305,0.330492,0.592440,609.0,70.0,35.0,23.0,LogReg_v2,4,val
9,0.70,0.154013,0.619127,0.834962,0.146067,0.122642,0.180556,0.215624,0.328762,0.616085,756.0,93.0,59.0,13.0,LogReg_v2,4,test


,model,pr_auc_mean,pr_auc_std,roc_auc_mean,roc_auc_std,f1_mean,f1_std,recall_mean,recall_std,brier_mean,brier_std,ece_mean,ece_std,nll_mean,nll_std,acc_mean,acc_std
0,FTTiny_v2,0.118346,0.009474,0.608183,0.015733,0.160522,0.012715,0.291667,0.136437,0.236507,0.031060,0.331838,0.044685,0.651554,0.075309,0.762649,0.105724
1,LogReg_v2,0.153645,0.004579,0.622527,0.007851,0.143106,0.009553,0.180556,0.049105,0.212972,0.002709,0.323180,0.005428,0.610043,0.005986,0.832790,0.032501



✅ TASK C.1 complete. Next: decide Wide&Deep if FTTiny improves.


In [14]:
# =========================================================
# TASK D.1 — Wide & Deep (Residual) on v2
# logit_final = logit_wide(LogReg) + logit_residual(FTTiny)
# =========================================================
import os, numpy as np, pandas as pd, json
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)

TARGET = "stroke"
DATA_V2 = os.path.join(DIRS["base"], "data_clean_v2.csv")
FT_V2   = os.path.join(DIRS["tab"],  "feature_type_inference_v2.csv")

df = pd.read_csv(DATA_V2)
ft = pd.read_csv(FT_V2)

bin_cols = ft.loc[ft["type_inferred"]=="binary", "feature"].tolist()
cat_cols = ft.loc[ft["type_inferred"]=="categorical", "feature"].tolist()
cont_cols = ft.loc[ft["type_inferred"]=="continuous", "feature"].tolist()
print("Counts:", {"bin":len(bin_cols), "cat":len(cat_cols), "cont":len(cont_cols)})

SPLIT_DIR = os.path.join(DIRS["base"], "splits")
test_idx = np.load(os.path.join(SPLIT_DIR, "test_idx.npy"))

# ---------------------------
# Preprocessor (same as in C.1)
# ---------------------------
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OrdinalEncoder

class Preprocessor:
    def __init__(self, cont_cols, cat_cols, bin_cols):
        self.cont_cols=list(cont_cols); self.cat_cols=list(cat_cols); self.bin_cols=list(bin_cols)
        self.cont_imputer=SimpleImputer(strategy="median")
        self.scaler=StandardScaler()
        self.cat_imputer=SimpleImputer(strategy="most_frequent")
        self.encoder=OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)

    def fit(self, df_tr):
        self.scaler.fit(self.cont_imputer.fit_transform(df_tr[self.cont_cols]))
        if len(self.cat_cols)>0:
            self.encoder.fit(self.cat_imputer.fit_transform(df_tr[self.cat_cols]))
        return self

    def transform(self, df_any):
        Xc = self.scaler.transform(self.cont_imputer.transform(df_any[self.cont_cols])).astype(np.float32)

        if len(self.cat_cols)>0:
            Xk = self.encoder.transform(self.cat_imputer.transform(df_any[self.cat_cols])).astype(np.int64) + 1
        else:
            Xk = np.zeros((len(df_any),0), dtype=np.int64)

        Xb = df_any[self.bin_cols].values.astype(np.int64) if len(self.bin_cols)>0 else np.zeros((len(df_any),0),dtype=np.int64)
        Xb = np.clip(Xb,0,1)

        return {
            "cont": torch.tensor(Xc, dtype=torch.float32),
            "cat":  torch.tensor(Xk, dtype=torch.long),
            "bin":  torch.tensor(Xb, dtype=torch.long),
        }

def to_numpy_matrix(Xdict):
    Xc = Xdict["cont"].numpy()
    Xk = Xdict["cat"].numpy().astype(np.float32)
    Xb = Xdict["bin"].numpy().astype(np.float32)
    return np.concatenate([Xc, Xk, Xb], axis=1)

# ---------------------------
# Deep residual backbone (same as FTTiny, outputs residual logit)
# ---------------------------
class WideDeepDataset(Dataset):
    def __init__(self, X, y, wide_logit):
        self.X = X
        self.y = torch.tensor(y, dtype=torch.float32)
        self.w = torch.tensor(wide_logit, dtype=torch.float32)
    def __len__(self): return len(self.y)
    def __getitem__(self, i):
        return self.X["cont"][i], self.X["cat"][i], self.X["bin"][i], self.w[i], self.y[i]

class FTTinyResidual(nn.Module):
    def __init__(self, n_cont, cat_cards, n_bin, d=32, n_heads=2, n_layers=1, dropout=0.35):
        super().__init__()
        self.cls = nn.Parameter(torch.zeros(1,1,d))
        self.cont_w = nn.Parameter(torch.randn(n_cont, d)*0.02)
        self.cont_b = nn.Parameter(torch.zeros(n_cont, d))

        self.cat_emb = nn.ModuleList([nn.Embedding(card+1, d) for card in cat_cards])
        self.bin_emb = nn.ModuleList([nn.Embedding(2, d) for _ in range(n_bin)])

        enc_layer = nn.TransformerEncoderLayer(
            d_model=d, nhead=n_heads, dim_feedforward=4*d,
            dropout=dropout, activation="gelu", batch_first=True,
            norm_first=True
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=n_layers)

        # residual head (small)
        self.head = nn.Sequential(
            nn.LayerNorm(d),
            nn.Dropout(dropout),
            nn.Linear(d, 1)
        )

    def forward(self, x_cont, x_cat, x_bin):
        B = x_cont.size(0)

        cont_tokens = x_cont.unsqueeze(-1) * self.cont_w.unsqueeze(0) + self.cont_b.unsqueeze(0)

        if x_cat.size(1)>0:
            cat_tokens = torch.stack([emb(x_cat[:,j]) for j,emb in enumerate(self.cat_emb)], dim=1)
        else:
            cat_tokens = None

        if x_bin.size(1)>0:
            bin_tokens = torch.stack([emb(x_bin[:,j]) for j,emb in enumerate(self.bin_emb)], dim=1)
        else:
            bin_tokens = None

        tokens = [self.cls.expand(B,1,-1), cont_tokens]
        if cat_tokens is not None: tokens.append(cat_tokens)
        if bin_tokens is not None: tokens.append(bin_tokens)

        x = torch.cat(tokens, dim=1)
        h = self.encoder(x)
        cls = h[:,0,:]
        return self.head(cls).squeeze(1)   # residual logit

# ---------------------------
# One repeat: train wide + residual deep
# ---------------------------
def run_widedeep_repeat(r, max_epochs=120, patience=15):
    tr_idx=np.load(os.path.join(SPLIT_DIR,f"train_idx_r{r}.npy"))
    va_idx=np.load(os.path.join(SPLIT_DIR,f"val_idx_r{r}.npy"))

    df_tr=df.iloc[tr_idx].reset_index(drop=True)
    df_va=df.iloc[va_idx].reset_index(drop=True)
    df_te=df.iloc[test_idx].reset_index(drop=True)

    y_tr=df_tr[TARGET].values.astype(int)
    y_va=df_va[TARGET].values.astype(int)
    y_te=df_te[TARGET].values.astype(int)

    # Fit preprocessing on TRAIN only
    pp=Preprocessor(cont_cols,cat_cols,bin_cols).fit(df_tr)
    X_tr=pp.transform(df_tr); X_va=pp.transform(df_va); X_te=pp.transform(df_te)

    # ---------- WIDE: Logistic Regression ----------
    Xtr_np = to_numpy_matrix(X_tr)
    Xva_np = to_numpy_matrix(X_va)
    Xte_np = to_numpy_matrix(X_te)

    wide = LogisticRegression(max_iter=3000, class_weight="balanced", solver="liblinear")
    wide.fit(Xtr_np, y_tr)

    # wide logits (not probs)
    w_tr = wide.decision_function(Xtr_np).astype(np.float32)
    w_va = wide.decision_function(Xva_np).astype(np.float32)
    w_te = wide.decision_function(Xte_np).astype(np.float32)

    # ---------- DEEP RESIDUAL ----------
    cat_cards=[int(df_tr[c].nunique(dropna=True)) for c in cat_cols]
    model = FTTinyResidual(len(cont_cols), cat_cards, len(bin_cols),
                           d=32, n_heads=2, n_layers=1, dropout=0.35).to(DEVICE)

    # imbalance handling
    pos=y_tr.sum(); neg=len(y_tr)-pos
    pos_weight=neg/max(1.0,pos)
    crit = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(pos_weight, device=DEVICE))

    opt = torch.optim.AdamW(model.parameters(), lr=7e-4, weight_decay=2e-3)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max_epochs)

    tr_loader = DataLoader(WideDeepDataset(X_tr, y_tr, w_tr), batch_size=256, shuffle=True)
    va_loader = DataLoader(WideDeepDataset(X_va, y_va, w_va), batch_size=512, shuffle=False)
    te_loader = DataLoader(WideDeepDataset(X_te, y_te, w_te), batch_size=512, shuffle=False)

    best_pr=-1.0; best_state=None; bad=0; history=[]
    for epoch in range(max_epochs):
        model.train()
        losses=[]
        for xc,xk,xb,wl,yb in tr_loader:
            xc,xk,xb,wl,yb = xc.to(DEVICE),xk.to(DEVICE),xb.to(DEVICE),wl.to(DEVICE),yb.to(DEVICE)
            opt.zero_grad()
            res = model(xc,xk,xb)
            logit = wl + res
            loss = crit(logit, yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            losses.append(loss.item())
        sch.step()

        # val PR-AUC on final probability
        model.eval()
        p_va=[]
        with torch.no_grad():
            for xc,xk,xb,wl,yb in va_loader:
                xc,xk,xb,wl = xc.to(DEVICE),xk.to(DEVICE),xb.to(DEVICE),wl.to(DEVICE)
                res = model(xc,xk,xb)
                logit = wl + res
                p_va.append(torch.sigmoid(logit).cpu().numpy())
        p_va=np.concatenate(p_va)
        pr=float(average_precision_score(y_va, p_va))
        history.append({"epoch":epoch,"loss":float(np.mean(losses)),"val_pr_auc":pr})
        print(f"WideDeep r{r} ep{epoch:03d} loss={np.mean(losses):.4f} valPR={pr:.4f}")

        if pr > best_pr + 1e-6:
            best_pr = pr
            best_state = {k:v.detach().cpu() for k,v in model.state_dict().items()}
            bad = 0
        else:
            bad += 1
            if bad >= patience:
                break

    model.load_state_dict(best_state)

    # probs (VAL/TEST) for metrics
    model.eval()
    p_va=[]
    with torch.no_grad():
        for xc,xk,xb,wl,yb in va_loader:
            xc,xk,xb,wl = xc.to(DEVICE),xk.to(DEVICE),xb.to(DEVICE),wl.to(DEVICE)
            res = model(xc,xk,xb)
            logit = wl + res
            p_va.append(torch.sigmoid(logit).cpu().numpy())
    p_va=np.concatenate(p_va)

    p_te=[]
    with torch.no_grad():
        for xc,xk,xb,wl,yb in te_loader:
            xc,xk,xb,wl = xc.to(DEVICE),xk.to(DEVICE),xb.to(DEVICE),wl.to(DEVICE)
            res = model(xc,xk,xb)
            logit = wl + res
            p_te.append(torch.sigmoid(logit).cpu().numpy())
    p_te=np.concatenate(p_te)

    # threshold from VAL only
    t_best,_ = find_best_threshold(y_va, p_va, metric="f1")

    m_va = evaluate_binary(y_va, p_va, threshold=t_best, ece_bins=cfg.ece_bins)
    m_te = evaluate_binary(y_te, p_te, threshold=t_best, ece_bins=cfg.ece_bins)

    # exports: reliability + learning curve
    fig = plot_reliability_curve(y_va, p_va, n_bins=cfg.ece_bins, title=f"Reliability (VAL) — WideDeep r{r}")
    save_fig(f"widedeep_reliability_val_r{r}", fig=fig)

    hist_df = pd.DataFrame(history)
    fig = plt.figure()
    plt.plot(hist_df["epoch"], hist_df["val_pr_auc"], marker="o")
    plt.xlabel("Epoch"); plt.ylabel("Val PR-AUC")
    plt.title(f"WideDeep learning curve (repeat {r})")
    save_fig(f"widedeep_learningcurve_r{r}", fig=fig)

    # return
    return {
        "repeat": r,
        "t_best": float(t_best),
        "val": m_va,
        "test": m_te,
        "p_val": p_va,
        "p_test": p_te,
        "wide_only_val_pr": float(average_precision_score(y_va, 1/(1+np.exp(-w_va)))),
        "wide_only_test_pr": float(average_precision_score(y_te, 1/(1+np.exp(-w_te))))
    }

# ---------------------------
# Run 5 repeats
# ---------------------------
seed_everything(cfg.seed)

outs=[]
rows=[]
for r in range(5):
    out = run_widedeep_repeat(r)
    outs.append(out)

    mv = dict(out["val"]); mt = dict(out["test"])
    mv.update({"model":"WideDeep_v2","repeat":r,"split":"val","t_best":out["t_best"]})
    mt.update({"model":"WideDeep_v2","repeat":r,"split":"test","t_best":out["t_best"]})
    rows += [mv, mt]

df_m = pd.DataFrame(rows)
display(df_m)
save_table(df_m, "widedeep_v2_per_repeat_val_test", index=False)

# summary (test)
test_df = df_m[df_m["split"]=="test"].copy()
summary = test_df[["pr_auc","roc_auc","f1","recall","brier","ece","nll","acc"]].agg(["mean","std"]).T.reset_index()
summary.columns = ["metric","mean","std"]
display(summary)
save_table(summary, "widedeep_v2_test_summary_meanstd", index=False)

# Save predictions (so later tasks can use them for fairness/uncertainty/abstention)
pred_path = os.path.join(DIRS["pred"], "widedeep_v2_predictions.json")
payload=[]
for out in outs:
    payload.append({
        "repeat": out["repeat"],
        "t_best": out["t_best"],
        "wide_only_val_pr": out["wide_only_val_pr"],
        "wide_only_test_pr": out["wide_only_test_pr"],
        "p_val": out["p_val"].astype(float).tolist(),
        "p_test": out["p_test"].astype(float).tolist()
    })
with open(pred_path, "w") as f:
    json.dump(payload, f)
print("✅ Saved predictions:", pred_path)

print("\n✅ TASK D.1 complete.")
print("Next: TASK D.2 (add Concept-Gated residual head + global concept importance + per-group explanations).")

Using device: cpu
Counts: {'bin': 9, 'cat': 2, 'cont': 24}
WideDeep r0 ep000 loss=1.1575 valPR=0.1663
WideDeep r0 ep001 loss=1.0933 valPR=0.1594
WideDeep r0 ep002 loss=1.0514 valPR=0.1601
WideDeep r0 ep003 loss=1.0744 valPR=0.1623
WideDeep r0 ep004 loss=1.0642 valPR=0.1632
WideDeep r0 ep005 loss=1.0588 valPR=0.1642
WideDeep r0 ep006 loss=1.0563 valPR=0.1646
WideDeep r0 ep007 loss=1.0487 valPR=0.1634
WideDeep r0 ep008 loss=1.0579 valPR=0.1620
WideDeep r0 ep009 loss=1.0467 valPR=0.1632
WideDeep r0 ep010 loss=1.0429 valPR=0.1634
WideDeep r0 ep011 loss=1.0376 valPR=0.1657
WideDeep r0 ep012 loss=1.0642 valPR=0.1632
WideDeep r0 ep013 loss=1.0317 valPR=0.1642
WideDeep r0 ep014 loss=1.0532 valPR=0.1650
WideDeep r0 ep015 loss=1.0232 valPR=0.1637
WideDeep r1 ep000 loss=1.1609 valPR=0.2373
WideDeep r1 ep001 loss=1.1320 valPR=0.2340
WideDeep r1 ep002 loss=1.0898 valPR=0.2353
WideDeep r1 ep003 loss=1.0838 valPR=0.2336
WideDeep r1 ep004 loss=1.0745 valPR=0.2352
WideDeep r1 ep005 loss=1.0764 valPR=0.

,threshold,pr_auc,roc_auc,acc,f1,precision,recall,brier,ece,nll,tn,fp,fn,tp,model,repeat,split,t_best
0,0.69,0.163742,0.687522,0.849389,0.274510,0.221053,0.362069,0.199329,0.309408,0.585554,605.0,74.0,37.0,21.0,WideDeep_v2,0,val,0.69
1,0.69,0.162403,0.625769,0.823018,0.137566,0.111111,0.180556,0.208770,0.310967,0.598775,745.0,104.0,59.0,13.0,WideDeep_v2,0,test,0.69
2,0.76,0.252042,0.749200,0.902307,0.368421,0.375000,0.362069,0.183167,0.292694,0.539322,644.0,35.0,37.0,21.0,WideDeep_v2,1,val,0.76
3,0.76,0.139412,0.605189,0.880565,0.140625,0.160714,0.125000,0.198711,0.292029,0.575266,802.0,47.0,63.0,9.0,WideDeep_v2,1,test,0.76
4,0.55,0.167343,0.697146,0.740841,0.232932,0.151832,0.500000,0.197247,0.304771,0.575253,517.0,162.0,29.0,29.0,WideDeep_v2,2,val,0.55
5,0.55,0.149014,0.634995,0.712269,0.184615,0.118577,0.416667,0.204667,0.310275,0.590260,626.0,223.0,42.0,30.0,WideDeep_v2,2,test,0.55
6,0.70,0.176350,0.706719,0.838535,0.260870,0.203883,0.362069,0.206989,0.320216,0.601293,597.0,82.0,37.0,21.0,WideDeep_v2,3,val,0.70
7,0.70,0.171624,0.634930,0.839305,0.212766,0.172414,0.277778,0.208519,0.311657,0.602652,753.0,96.0,52.0,20.0,WideDeep_v2,3,test,0.70
8,0.66,0.172613,0.729013,0.839891,0.280488,0.216981,0.396552,0.182778,0.295584,0.543355,596.0,83.0,35.0,23.0,WideDeep_v2,4,val,0.66
9,0.66,0.143884,0.621826,0.830619,0.161290,0.131579,0.208333,0.193163,0.294666,0.564898,750.0,99.0,57.0,15.0,WideDeep_v2,4,test,0.66


,metric,mean,std
0,pr_auc,0.153268,0.013403
1,roc_auc,0.624542,0.012254
2,f1,0.167373,0.031618
3,recall,0.241667,0.112234
4,brier,0.202766,0.006734
5,ece,0.303919,0.009708
6,nll,0.586370,0.015957
7,acc,0.817155,0.062706


✅ Saved predictions: /kaggle/working/outputs/cgt_stroke_v1/predictions/widedeep_v2_predictions.json

✅ TASK D.1 complete.
Next: TASK D.2 (add Concept-Gated residual head + global concept importance + per-group explanations).


In [15]:
# =========================================================
# TASK D.2 — Concept-Gated Residual Head (CG-WideDeep)
# Residual logit = sum_g gate_g(x_all) * expert_g(x_group)
# =========================================================
import torch.nn.functional as F

# 1) Define concept groups (you already have a concept_groups.json saved earlier)
# We'll load it if available, else define a sane default.
CG_PATH = os.path.join(DIRS["log"], "concept_groups.json")
if os.path.exists(CG_PATH):
    with open(CG_PATH, "r") as f:
        concept_groups = json.load(f)
else:
    concept_groups = {
        "Demographics": ["gender","age","Race","Marital_status"],
        "Lifestyle": ["alcohol","smoke","sleep_disorder","sleep_time","Minutes_sedentary_activity"],
        "Comorbidity": ["diabetes","hypertension","high_cholesterol","Coronary_Heart_Disease","Body_Mass_Index","General_health_condition","depression"],
        "Vitals": ["Systolic_blood_pressure","Diastolic_blood_pressure","Waist_Circumference"],
        "Lab": ["Fasting_Glucose","Glycohemoglobin","Highdensity_lipoprotein","Lowdensity_lipoprotein","Triglyceride"],
        "Nutrition": ["energy","protein","Carbohydrate","Dietary_fiber","Total_fat",
                      "Total_saturated_fatty_acids","Total_monounsaturated_fatty_acids","Total_polyunsaturated_fatty_acids",
                      "Potassium","Sodium"]
    }
print("Concept groups:", {k:len(v) for k,v in concept_groups.items()})

# 2) Map each feature to its token index in the transformer input
# Input token order in FTTinyResidual was: [CLS] + cont_tokens + cat_tokens + bin_tokens
# We'll build indices to extract group tokens and pool them.

def build_token_index_map(cont_cols, cat_cols, bin_cols):
    idx = {}
    # CLS token at 0
    # cont tokens start at 1
    for j,c in enumerate(cont_cols):
        idx[c] = 1 + j
    base = 1 + len(cont_cols)
    for j,c in enumerate(cat_cols):
        idx[c] = base + j
    base = base + len(cat_cols)
    for j,c in enumerate(bin_cols):
        idx[c] = base + j
    return idx

token_idx = build_token_index_map(cont_cols, cat_cols, bin_cols)

# 3) Concept-gated model
class ConceptGatedResidual(nn.Module):
    def __init__(self, n_cont, cat_cards, n_bin, concept_groups,
                 d=32, n_heads=2, n_layers=1, dropout=0.35,
                 gate_temp=1.0, gate_dropout=0.1):
        super().__init__()
        self.concept_names = list(concept_groups.keys())
        self.concept_groups = concept_groups
        self.gate_temp = gate_temp

        # backbone encoder (same tokenization as FTTinyResidual)
        self.cls = nn.Parameter(torch.zeros(1,1,d))
        self.cont_w = nn.Parameter(torch.randn(n_cont, d)*0.02)
        self.cont_b = nn.Parameter(torch.zeros(n_cont, d))
        self.cat_emb = nn.ModuleList([nn.Embedding(card+1, d) for card in cat_cards])
        self.bin_emb = nn.ModuleList([nn.Embedding(2, d) for _ in range(n_bin)])

        enc_layer = nn.TransformerEncoderLayer(
            d_model=d, nhead=n_heads, dim_feedforward=4*d,
            dropout=dropout, activation="gelu", batch_first=True,
            norm_first=True
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=n_layers)

        # gate: uses CLS to weight concepts
        self.gate = nn.Sequential(
            nn.LayerNorm(d),
            nn.Dropout(gate_dropout),
            nn.Linear(d, len(self.concept_names))
        )

        # experts: each concept outputs a residual logit
        self.experts = nn.ModuleDict({
            name: nn.Sequential(
                nn.LayerNorm(d),
                nn.Dropout(dropout),
                nn.Linear(d, d),
                nn.GELU(),
                nn.Dropout(dropout),
                nn.Linear(d, 1)
            ) for name in self.concept_names
        })

    def forward(self, x_cont, x_cat, x_bin, token_idx_map):
        B = x_cont.size(0)

        cont_tokens = x_cont.unsqueeze(-1) * self.cont_w.unsqueeze(0) + self.cont_b.unsqueeze(0)

        if x_cat.size(1)>0:
            cat_tokens = torch.stack([emb(x_cat[:,j]) for j,emb in enumerate(self.cat_emb)], dim=1)
        else:
            cat_tokens = None

        if x_bin.size(1)>0:
            bin_tokens = torch.stack([emb(x_bin[:,j]) for j,emb in enumerate(self.bin_emb)], dim=1)
        else:
            bin_tokens = None

        toks = [self.cls.expand(B,1,-1), cont_tokens]
        if cat_tokens is not None: toks.append(cat_tokens)
        if bin_tokens is not None: toks.append(bin_tokens)
        x = torch.cat(toks, dim=1)

        h = self.encoder(x)          # [B, T, d]
        cls = h[:,0,:]               # [B, d]

        # gates: [B, G]
        g_logits = self.gate(cls) / self.gate_temp
        g = F.softmax(g_logits, dim=1)

        # expert per concept: pool tokens for that group, then map to logit
        expert_logits = []
        for name in self.concept_names:
            feats = self.concept_groups[name]
            idxs = [token_idx_map[f] for f in feats if f in token_idx_map]
            if len(idxs)==0:
                pooled = cls
            else:
                pooled = h[:, idxs, :].mean(dim=1)
            expert_logits.append(self.experts[name](pooled).squeeze(1))
        expert_logits = torch.stack(expert_logits, dim=1)   # [B, G]

        # residual logit = sum g * expert
        res = (g * expert_logits).sum(dim=1)                # [B]
        return res, g.detach(), expert_logits.detach()

# 4) Replace model in run_widedeep_repeat
def run_cg_widedeep_repeat(r, max_epochs=140, patience=18):
    tr_idx=np.load(os.path.join(SPLIT_DIR,f"train_idx_r{r}.npy"))
    va_idx=np.load(os.path.join(SPLIT_DIR,f"val_idx_r{r}.npy"))

    df_tr=df.iloc[tr_idx].reset_index(drop=True)
    df_va=df.iloc[va_idx].reset_index(drop=True)
    df_te=df.iloc[test_idx].reset_index(drop=True)

    y_tr=df_tr[TARGET].values.astype(int)
    y_va=df_va[TARGET].values.astype(int)
    y_te=df_te[TARGET].values.astype(int)

    pp=Preprocessor(cont_cols,cat_cols,bin_cols).fit(df_tr)
    X_tr=pp.transform(df_tr); X_va=pp.transform(df_va); X_te=pp.transform(df_te)

    Xtr_np = to_numpy_matrix(X_tr); Xva_np = to_numpy_matrix(X_va); Xte_np = to_numpy_matrix(X_te)

    wide = LogisticRegression(max_iter=3000, class_weight="balanced", solver="liblinear")
    wide.fit(Xtr_np, y_tr)

    w_tr = wide.decision_function(Xtr_np).astype(np.float32)
    w_va = wide.decision_function(Xva_np).astype(np.float32)
    w_te = wide.decision_function(Xte_np).astype(np.float32)

    cat_cards=[int(df_tr[c].nunique(dropna=True)) for c in cat_cols]
    model = ConceptGatedResidual(len(cont_cols), cat_cards, len(bin_cols),
                                 concept_groups=concept_groups,
                                 d=32, n_heads=2, n_layers=1,
                                 dropout=0.30, gate_dropout=0.10,
                                 gate_temp=1.0).to(DEVICE)

    pos=y_tr.sum(); neg=len(y_tr)-pos
    pos_weight=neg/max(1.0,pos)
    crit = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(pos_weight, device=DEVICE))

    opt = torch.optim.AdamW(model.parameters(), lr=6e-4, weight_decay=1e-3)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max_epochs)

    tr_loader = DataLoader(WideDeepDataset(X_tr, y_tr, w_tr), batch_size=256, shuffle=True)
    va_loader = DataLoader(WideDeepDataset(X_va, y_va, w_va), batch_size=512, shuffle=False)
    te_loader = DataLoader(WideDeepDataset(X_te, y_te, w_te), batch_size=512, shuffle=False)

    best_pr=-1.0; best_state=None; bad=0; history=[]
    for epoch in range(max_epochs):
        model.train()
        losses=[]
        for xc,xk,xb,wl,yb in tr_loader:
            xc,xk,xb,wl,yb = xc.to(DEVICE),xk.to(DEVICE),xb.to(DEVICE),wl.to(DEVICE),yb.to(DEVICE)
            opt.zero_grad()
            res, _, _ = model(xc,xk,xb, token_idx)
            logit = wl + res
            loss = crit(logit, yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            losses.append(loss.item())
        sch.step()

        model.eval()
        p_va=[]
        with torch.no_grad():
            for xc,xk,xb,wl,yb in va_loader:
                xc,xk,xb,wl = xc.to(DEVICE),xk.to(DEVICE),xb.to(DEVICE),wl.to(DEVICE)
                res, _, _ = model(xc,xk,xb, token_idx)
                logit = wl + res
                p_va.append(torch.sigmoid(logit).cpu().numpy())
        p_va=np.concatenate(p_va)
        pr=float(average_precision_score(y_va, p_va))
        history.append({"epoch":epoch,"loss":float(np.mean(losses)),"val_pr_auc":pr})
        print(f"CG-WideDeep r{r} ep{epoch:03d} loss={np.mean(losses):.4f} valPR={pr:.4f}")

        if pr > best_pr + 1e-6:
            best_pr = pr
            best_state = {k:v.detach().cpu() for k,v in model.state_dict().items()}
            bad = 0
        else:
            bad += 1
            if bad >= patience:
                break

    model.load_state_dict(best_state)

    # final probs
    model.eval()
    p_va=[]
    g_va=[]
    with torch.no_grad():
        for xc,xk,xb,wl,yb in va_loader:
            xc,xk,xb,wl = xc.to(DEVICE),xk.to(DEVICE),xb.to(DEVICE),wl.to(DEVICE)
            res, g, _ = model(xc,xk,xb, token_idx)
            logit = wl + res
            p_va.append(torch.sigmoid(logit).cpu().numpy())
            g_va.append(g.cpu().numpy())
    p_va=np.concatenate(p_va)
    g_va=np.concatenate(g_va)

    p_te=[]
    g_te=[]
    with torch.no_grad():
        for xc,xk,xb,wl,yb in te_loader:
            xc,xk,xb,wl = xc.to(DEVICE),xk.to(DEVICE),xb.to(DEVICE),wl.to(DEVICE)
            res, g, _ = model(xc,xk,xb, token_idx)
            logit = wl + res
            p_te.append(torch.sigmoid(logit).cpu().numpy())
            g_te.append(g.cpu().numpy())
    p_te=np.concatenate(p_te)
    g_te=np.concatenate(g_te)

    t_best,_ = find_best_threshold(y_va, p_va, metric="f1")
    m_va = evaluate_binary(y_va, p_va, threshold=t_best, ece_bins=cfg.ece_bins)
    m_te = evaluate_binary(y_te, p_te, threshold=t_best, ece_bins=cfg.ece_bins)

    # exports
    fig = plot_reliability_curve(y_va, p_va, n_bins=cfg.ece_bins, title=f"Reliability (VAL) — CG-WideDeep r{r}")
    save_fig(f"cg_widedeep_reliability_val_r{r}", fig=fig)

    hist_df = pd.DataFrame(history)
    fig = plt.figure()
    plt.plot(hist_df["epoch"], hist_df["val_pr_auc"], marker="o")
    plt.xlabel("Epoch"); plt.ylabel("Val PR-AUC")
    plt.title(f"CG-WideDeep learning curve (repeat {r})")
    save_fig(f"cg_widedeep_learningcurve_r{r}", fig=fig)

    # save gates for analysis (interpretability)
    np.save(os.path.join(DIRS["pred"], f"cg_widedeep_gates_val_r{r}.npy"), g_va)
    np.save(os.path.join(DIRS["pred"], f"cg_widedeep_gates_test_r{r}.npy"), g_te)

    return m_va, m_te

# 5) Run repeats
seed_everything(cfg.seed)

rows=[]
for r in range(5):
    m_va, m_te = run_cg_widedeep_repeat(r)
    m_va.update({"model":"CG_WideDeep_v2","repeat":r,"split":"val"})
    m_te.update({"model":"CG_WideDeep_v2","repeat":r,"split":"test"})
    rows += [m_va, m_te]

df_cg = pd.DataFrame(rows)
display(df_cg)
save_table(df_cg, "cg_widedeep_v2_per_repeat_val_test", index=False)

test_df = df_cg[df_cg["split"]=="test"]
summary = test_df[["pr_auc","roc_auc","f1","recall","brier","ece","nll","acc"]].agg(["mean","std"]).T.reset_index()
summary.columns = ["metric","mean","std"]
display(summary)
save_table(summary, "cg_widedeep_v2_test_summary_meanstd", index=False)

print("\n✅ TASK D.2 complete.")
print("Next: TASK D.3 (global concept importance + per-patient explanations + stability plots).")

Concept groups: {'Demographics': 4, 'Lifestyle': 4, 'Comorbidity': 4, 'Nutrition_Macro': 6, 'Nutrition_FattyAcids': 3, 'Nutrition_Minerals': 2, 'Other': 12}
CG-WideDeep r0 ep000 loss=1.0460 valPR=0.1604
CG-WideDeep r0 ep001 loss=1.0505 valPR=0.1609
CG-WideDeep r0 ep002 loss=1.0328 valPR=0.1618
CG-WideDeep r0 ep003 loss=1.0395 valPR=0.1609
CG-WideDeep r0 ep004 loss=1.0203 valPR=0.1604
CG-WideDeep r0 ep005 loss=1.0334 valPR=0.1622
CG-WideDeep r0 ep006 loss=1.0371 valPR=0.1617
CG-WideDeep r0 ep007 loss=1.0259 valPR=0.1622
CG-WideDeep r0 ep008 loss=1.0318 valPR=0.1598
CG-WideDeep r0 ep009 loss=1.0247 valPR=0.1585
CG-WideDeep r0 ep010 loss=1.0105 valPR=0.1551
CG-WideDeep r0 ep011 loss=1.0222 valPR=0.1519
CG-WideDeep r0 ep012 loss=0.9901 valPR=0.1479
CG-WideDeep r0 ep013 loss=0.9972 valPR=0.1527
CG-WideDeep r0 ep014 loss=0.9962 valPR=0.1539
CG-WideDeep r0 ep015 loss=0.9806 valPR=0.1484
CG-WideDeep r0 ep016 loss=1.0056 valPR=0.1516
CG-WideDeep r0 ep017 loss=0.9937 valPR=0.1517
CG-WideDeep r0 

,threshold,pr_auc,roc_auc,acc,f1,precision,recall,brier,ece,nll,tn,fp,fn,tp,model,repeat,split
0,0.48,0.137294,0.638921,0.713704,0.215613,0.137441,0.500000,0.184578,0.255023,0.555009,497.0,182.0,29.0,29.0,CG_WideDeep_v2,0,val
1,0.48,0.122039,0.597304,0.681868,0.165242,0.103943,0.402778,0.192762,0.255634,0.564986,599.0,250.0,43.0,29.0,CG_WideDeep_v2,0,test
2,0.60,0.219819,0.762353,0.767978,0.318725,0.207254,0.689655,0.204461,0.311908,0.596365,526.0,153.0,18.0,40.0,CG_WideDeep_v2,1,val
3,0.60,0.133899,0.582155,0.719870,0.145695,0.095652,0.305556,0.224456,0.306437,0.644328,641.0,208.0,50.0,22.0,CG_WideDeep_v2,1,test
4,0.55,0.154932,0.668326,0.720488,0.237037,0.150943,0.551724,0.208193,0.282324,0.603512,499.0,180.0,26.0,32.0,CG_WideDeep_v2,2,val
5,0.55,0.143323,0.616182,0.687296,0.186441,0.117021,0.458333,0.218492,0.295023,0.629867,600.0,249.0,39.0,33.0,CG_WideDeep_v2,2,test
6,0.67,0.164884,0.692093,0.824966,0.254335,0.191304,0.379310,0.171100,0.214533,0.510698,586.0,93.0,36.0,22.0,CG_WideDeep_v2,3,val
7,0.67,0.145425,0.615283,0.820847,0.179104,0.139535,0.250000,0.175194,0.211294,0.523842,738.0,111.0,54.0,18.0,CG_WideDeep_v2,3,test
8,0.70,0.189835,0.683028,0.850746,0.256757,0.211111,0.327586,0.156708,0.189702,0.478538,608.0,71.0,39.0,19.0,CG_WideDeep_v2,4,val
9,0.70,0.106888,0.584724,0.829533,0.194872,0.154472,0.263889,0.173424,0.207215,0.531403,745.0,104.0,53.0,19.0,CG_WideDeep_v2,4,test


,metric,mean,std
0,pr_auc,0.130315,0.016028
1,roc_auc,0.599130,0.016207
2,f1,0.174271,0.019326
3,recall,0.336111,0.090757
4,brier,0.196866,0.023796
5,ece,0.255120,0.045939
6,nll,0.578885,0.055586
7,acc,0.747883,0.072118



✅ TASK D.2 complete.
Next: TASK D.3 (global concept importance + per-patient explanations + stability plots).


In [16]:
import numpy as np, glob, os

G = []
for r in range(5):
    path = os.path.join(DIRS["pred"], f"cg_widedeep_gates_test_r{r}.npy")
    if os.path.exists(path):
        G.append(np.load(path))
G = np.concatenate(G, axis=0)   # [N, num_concepts]

concepts = list(concept_groups.keys())
mean_w = G.mean(axis=0)
order = np.argsort(-mean_w)

for i in order:
    print(f"{concepts[i]:20s} mean_gate={mean_w[i]:.4f}")
print("Gate entropy (mean):", (- (G*np.log(G+1e-12)).sum(axis=1)).mean())

Other                mean_gate=0.2714
Nutrition_Macro      mean_gate=0.2653
Nutrition_FattyAcids mean_gate=0.1650
Demographics         mean_gate=0.1581
Nutrition_Minerals   mean_gate=0.0641
Comorbidity          mean_gate=0.0405
Lifestyle            mean_gate=0.0356
Gate entropy (mean): 1.2081965


In [19]:
# ============================================================
# TASK D.2 (UPDATED, FULL RUNNABLE): CG-WideDeep with FIXES
#   ✅ residual scale alpha (prevents ranking collapse)
#   ✅ gate entropy regularization (prevents degenerate gates)
#   ✅ optional "Other" gate penalty (only if it dominates)
#   ✅ saves gates + expert logits for diagnostics
#
# Works in Kaggle CPU.
# Expects: /kaggle/working/outputs/cgt_stroke_v1/data_clean_v2.csv exists
# ============================================================

import os, json, math, random, time
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import (
    roc_auc_score, average_precision_score, f1_score, precision_score, recall_score, accuracy_score,
    log_loss
)

# ---------------------------
# 0) Paths / Config
# ---------------------------
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)

BASE_DIR = "/kaggle/working/outputs/cgt_stroke_v1"
DATA_CSV = os.path.join(BASE_DIR, "data_clean_v2.csv")

DIRS = {
    "base": BASE_DIR,
    "logs": os.path.join(BASE_DIR, "logs"),
    "tables": os.path.join(BASE_DIR, "tables"),
    "fig": os.path.join(BASE_DIR, "figures"),
    "pred": os.path.join(BASE_DIR, "predictions"),
}
for k,v in DIRS.items():
    if k != "base":
        os.makedirs(v, exist_ok=True)

class CFG:
    seed = 42
    repeats = 5

    # training
    batch_size = 256
    lr = 2e-3
    weight_decay = 1e-4
    max_epochs = 60
    patience = 10

    # imbalance handling
    use_pos_weight = True

    # calibration metrics
    ece_bins = 15

    # Gate regularization (IMPORTANT FIX)
    gate_entropy_lambda = 0.01   # try 0.005–0.02
    other_gate_penalty = 0.00    # set 0.02–0.10 ONLY if "Other" dominates

    # Model sizes
    d_embed = 16
    deep_hidden = 64
    deep_layers = 2
    dropout = 0.10

cfg = CFG()

def seed_everything(seed=42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(cfg.seed)

# ---------------------------
# 1) Utils: metrics
# ---------------------------
def sigmoid_np(x): return 1/(1+np.exp(-x))

def expected_calibration_error(y_true, p, n_bins=15):
    y_true = np.asarray(y_true).astype(int)
    p = np.asarray(p).astype(float)
    bins = np.linspace(0.0, 1.0, n_bins+1)
    ece = 0.0
    for i in range(n_bins):
        lo, hi = bins[i], bins[i+1]
        m = (p >= lo) & (p < hi) if i < n_bins-1 else (p >= lo) & (p <= hi)
        if m.sum() == 0:
            continue
        acc = y_true[m].mean()
        conf = p[m].mean()
        ece += (m.sum()/len(p)) * abs(acc - conf)
    return float(ece)

def find_best_threshold(y, p, metric="f1"):
    y = np.asarray(y).astype(int)
    p = np.asarray(p).astype(float)
    # coarse grid is enough; you can refine later
    ts = np.linspace(0.05, 0.95, 181)
    best_t, best_s = 0.5, -1
    for t in ts:
        pred = (p >= t).astype(int)
        if metric == "f1":
            s = f1_score(y, pred, zero_division=0)
        elif metric == "youden":
            # TPR - FPR
            tp = ((pred==1)&(y==1)).sum()
            fn = ((pred==0)&(y==1)).sum()
            fp = ((pred==1)&(y==0)).sum()
            tn = ((pred==0)&(y==0)).sum()
            tpr = tp/(tp+fn+1e-12)
            fpr = fp/(fp+tn+1e-12)
            s = tpr - fpr
        else:
            raise ValueError("Unknown metric")
        if s > best_s:
            best_s, best_t = s, t
    return float(best_t), float(best_s)

def evaluate_binary(y, p, threshold=0.5, ece_bins=15):
    y = np.asarray(y).astype(int)
    p = np.asarray(p).astype(float)

    # sanity
    p = np.clip(p, 1e-7, 1-1e-7)
    pred = (p >= threshold).astype(int)

    out = {}
    out["threshold"] = float(threshold)
    out["pr_auc"] = float(average_precision_score(y, p))
    out["roc_auc"] = float(roc_auc_score(y, p))
    out["acc"] = float(accuracy_score(y, pred))
    out["f1"] = float(f1_score(y, pred, zero_division=0))
    out["precision"] = float(precision_score(y, pred, zero_division=0))
    out["recall"] = float(recall_score(y, pred, zero_division=0))

    # confusion
    tn = int(((pred==0) & (y==0)).sum())
    fp = int(((pred==1) & (y==0)).sum())
    fn = int(((pred==0) & (y==1)).sum())
    tp = int(((pred==1) & (y==1)).sum())
    out.update({"tn":tn, "fp":fp, "fn":fn, "tp":tp})

    # prob-quality
    out["brier"] = float(np.mean((p - y)**2))
    out["ece"] = float(expected_calibration_error(y, p, n_bins=ece_bins))
    out["nll"] = float(log_loss(y, p, labels=[0,1]))
    return out

def save_table(df, name, index=False):
    path = os.path.join(DIRS["tables"], f"{name}.csv")
    df.to_csv(path, index=index)
    print("✅ Saved table:", path)
    return path

# ---------------------------
# 2) Load data
# ---------------------------
assert os.path.exists(DATA_CSV), f"Missing: {DATA_CSV}"
df = pd.read_csv(DATA_CSV)

# You must set the label column name correctly.
# If your label column is different, change here:
LABEL_COL = "stroke" if "stroke" in df.columns else df.columns[-1]
print("Label column:", LABEL_COL)

y_all = df[LABEL_COL].values.astype(int)

# ---------------------------
# 3) Feature typing (use your v2 decisions)
# ---------------------------
bin_cols = ['gender','alcohol','smoke','sleep_disorder','Health_Insurance','diabetes','hypertension','high_cholesterol','Coronary_Heart_Disease']
cat_cols = ['Race','Marital_status']
cont_cols = [c for c in df.columns if c not in ([LABEL_COL] + bin_cols + cat_cols)]

# Note: you treated these as numeric/ordinal already; they are in cont_cols after cleaning.
print("Counts:", {"bin":len(bin_cols), "cat":len(cat_cols), "cont":len(cont_cols)})

# ---------------------------
# 4) Split (fixed test split per repeat with different seeds)
# ---------------------------
def make_split(seed):
    sss = StratifiedShuffleSplit(n_splits=1, test_size=0.20, random_state=seed)
    (tr_idx, te_idx) = next(sss.split(df, y_all))
    # then val from train
    y_tr = y_all[tr_idx]
    sss2 = StratifiedShuffleSplit(n_splits=1, test_size=0.20, random_state=seed+1337)
    (tr2, va2) = next(sss2.split(np.zeros(len(tr_idx)), y_tr))
    tr2_idx = tr_idx[tr2]
    va_idx = tr_idx[va2]
    return tr2_idx, va_idx, te_idx

# ---------------------------
# 5) Tensor datasets
# ---------------------------
def to_long_tensor(x):  return torch.as_tensor(x, dtype=torch.long)
def to_float_tensor(x): return torch.as_tensor(x, dtype=torch.float32)

class TabDS(Dataset):
    def __init__(self, Xc, Xk, Xb, y):
        self.Xc = Xc; self.Xk = Xk; self.Xb = Xb; self.y = y
    def __len__(self): return len(self.y)
    def __getitem__(self, i):
        return self.Xc[i], self.Xk[i], self.Xb[i], self.y[i]

# ---------------------------
# 6) Preprocessing (light; assumes v2 cleaned)
#    - continuous: z-score
#    - categorical: map to 0..K-1
#    - binary: ensure 0/1
# ---------------------------
def fit_preprocess(train_df):
    # bin
    bmap = {}
    for c in bin_cols:
        vals = train_df[c].values
        uniq = np.unique(vals)
        # allow {0,1} or {1,2}
        if set(uniq.tolist()) == {0,1}:
            bmap[c] = ("01", None)
        elif set(uniq.tolist()) == {1,2}:
            bmap[c] = ("12", None)
        else:
            # fallback: map sorted to 0/1 if only 2 unique
            if len(uniq)==2:
                bmap[c] = ("map", {uniq[0]:0, uniq[1]:1})
            else:
                raise ValueError(f"Binary col {c} has >2 unique: {uniq}")

    # cat vocab
    cat_vocab = {}
    for c in cat_cols:
        uniq = pd.Series(train_df[c]).astype(int).unique()
        uniq = np.sort(uniq)
        cat_vocab[c] = {v:i for i,v in enumerate(uniq.tolist())}

    # cont normalization
    mu = train_df[cont_cols].values.astype(np.float32).mean(axis=0)
    sd = train_df[cont_cols].values.astype(np.float32).std(axis=0)
    sd = np.where(sd < 1e-6, 1.0, sd)
    return bmap, cat_vocab, mu, sd

def apply_preprocess(df_part, bmap, cat_vocab, mu, sd):
    # bin
    Xb = np.zeros((len(df_part), len(bin_cols)), dtype=np.int64)
    for j,c in enumerate(bin_cols):
        vals = df_part[c].values
        mode, mapping = bmap[c]
        if mode == "01":
            Xb[:,j] = vals.astype(int)
        elif mode == "12":
            Xb[:,j] = (vals.astype(int) - 1)  # 1/2 -> 0/1
        else:
            Xb[:,j] = np.vectorize(mapping.get)(vals)
        # clip
        Xb[:,j] = np.clip(Xb[:,j], 0, 1)

    # cat
    Xk = np.zeros((len(df_part), len(cat_cols)), dtype=np.int64)
    for j,c in enumerate(cat_cols):
        vocab = cat_vocab[c]
        vals = df_part[c].astype(int).values
        # unknown -> 0 (shouldn't happen in this split)
        Xk[:,j] = np.array([vocab.get(v, 0) for v in vals], dtype=np.int64)

    # cont
    Xc = df_part[cont_cols].values.astype(np.float32)
    Xc = (Xc - mu) / sd
    return Xc, Xk, Xb

# ---------------------------
# 7) Concept groups (EDIT THIS if you want to split "Other")
# ---------------------------
# Use your saved JSON if it exists, else define here.
cg_path = os.path.join(DIRS["logs"], "concept_groups.json")
if os.path.exists(cg_path):
    concept_groups = json.load(open(cg_path, "r"))
else:
    # fallback: a minimal (you should replace with your real grouping)
    concept_groups = {
        "Demographics": ["gender","age","Race","Marital_status"],
        "Lifestyle": ["alcohol","smoke","sleep_time","Minutes_sedentary_activity"],
        "Comorbidity": ["diabetes","hypertension","high_cholesterol","Coronary_Heart_Disease"],
        "Nutrition_Macro": ["energy","Carbohydrate","protein","Total_fat","Dietary_fiber","Body_Mass_Index"],
        "Nutrition_FattyAcids": ["Total_saturated_fatty_acids","Total_monounsaturated_fatty_acids","Total_polyunsaturated_fatty_acids"],
        "Nutrition_Minerals": ["Sodium","Potassium"],
        "Other": [c for c in cont_cols if c not in [
            "age","sleep_time","Minutes_sedentary_activity","energy","Carbohydrate","protein","Total_fat",
            "Dietary_fiber","Body_Mass_Index","Total_saturated_fatty_acids","Total_monounsaturated_fatty_acids",
            "Total_polyunsaturated_fatty_acids","Sodium","Potassium"
        ]]
    }

# map feature name -> index in unified input vector (cont + cat(one-hot-ish as ids) + bin)
# For concept gating we will use ONLY cont+bin+cat-ids embedded outputs (handled in model),
# but the group index will refer to original columns; we will build masks over tokens in the model.
print("Concept groups:", {k:len(v) for k,v in concept_groups.items()})

# ---------------------------
# 8) Model: WideDeep + Concept-Gated Residual (UPDATED)
# ---------------------------
class WideDeep(nn.Module):
    def __init__(self, n_cont, cat_cardinalities, n_bin, d_embed=16, deep_hidden=64, deep_layers=2, dropout=0.1):
        super().__init__()
        self.n_cont = n_cont
        self.n_bin = n_bin
        self.cat_card = cat_cardinalities

        # embeddings for categorical ids
        self.cat_embeds = nn.ModuleList([
            nn.Embedding(card, d_embed) for card in cat_cardinalities
        ])

        # "wide" linear over cont + bin + (cat embeddings pooled)
        wide_in = n_cont + n_bin + len(cat_cardinalities)*d_embed
        self.wide = nn.Linear(wide_in, 1)

        # deep MLP over same vector
        layers = []
        din = wide_in
        for _ in range(deep_layers):
            layers += [nn.Linear(din, deep_hidden), nn.ReLU(), nn.Dropout(dropout)]
            din = deep_hidden
        layers += [nn.Linear(din, 1)]
        self.deep = nn.Sequential(*layers)

    def forward(self, x_cont, x_cat, x_bin):
        # x_cont: [B,n_cont] float
        # x_cat:  [B,n_cat]  long
        # x_bin:  [B,n_bin]  long (0/1)
        cat_vecs = []
        for j, emb in enumerate(self.cat_embeds):
            cat_vecs.append(emb(x_cat[:, j]))
        cat_vec = torch.cat(cat_vecs, dim=1) if len(cat_vecs) else torch.zeros((x_cont.size(0),0), device=x_cont.device)

        x_bin_f = x_bin.float()
        z = torch.cat([x_cont, x_bin_f, cat_vec], dim=1)

        wide_logit = self.wide(z).squeeze(1)
        deep_logit = self.deep(z).squeeze(1)

        # base WideDeep logit
        base = wide_logit + deep_logit
        return base, z

class ConceptGatedResidual(nn.Module):
    """
    Takes z (shared representation) + concept groups (as masks over z parts) and outputs residual logit.
    UPDATED:
      - residual scaling alpha (learned, starts small)
      - gate entropy regularization handled in training loop
      - optional other gate penalty handled in training loop
    """
    def __init__(self, z_dim, n_concepts, hidden=64, dropout=0.1):
        super().__init__()
        self.n_concepts = n_concepts

        # gate network: z -> softmax over concepts
        self.gate = nn.Sequential(
            nn.Linear(z_dim, hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, n_concepts)
        )

        # expert heads: each concept has its own MLP producing a logit
        self.experts = nn.ModuleList([
            nn.Sequential(
                nn.Linear(z_dim, hidden),
                nn.ReLU(),
                nn.Dropout(dropout),
                nn.Linear(hidden, 1)
            ) for _ in range(n_concepts)
        ])

        # ✅ residual scale alpha (init small)
        self.alpha = nn.Parameter(torch.tensor(-2.0))  # sigmoid(-2)=0.119

    def forward(self, z):
        g_logits = self.gate(z)                     # [B,C]
        g = torch.softmax(g_logits, dim=1)          # [B,C]

        expert_logits = []
        for ex in self.experts:
            expert_logits.append(ex(z).squeeze(1))  # [B]
        E = torch.stack(expert_logits, dim=1)       # [B,C]

        res = (g * E).sum(dim=1)                    # [B]
        res = torch.sigmoid(self.alpha) * res       # ✅ scaled residual

        return res, g, E

class CGWideDeep(nn.Module):
    def __init__(self, n_cont, cat_cardinalities, n_bin, concept_names,
                 d_embed=16, deep_hidden=64, deep_layers=2, dropout=0.1):
        super().__init__()
        self.base = WideDeep(n_cont, cat_cardinalities, n_bin, d_embed, deep_hidden, deep_layers, dropout)
        # z_dim = n_cont + n_bin + n_cat*d_embed
        z_dim = n_cont + n_bin + len(cat_cardinalities)*d_embed
        self.concept_names = concept_names
        self.cg = ConceptGatedResidual(z_dim=z_dim, n_concepts=len(concept_names), hidden=deep_hidden, dropout=dropout)

    def forward(self, x_cont, x_cat, x_bin):
        base_logit, z = self.base(x_cont, x_cat, x_bin)
        res, g, E = self.cg(z)
        logit = base_logit + res
        return logit, g, E

# ---------------------------
# 9) Train / eval loop (UPDATED with entropy reg + optional "Other" penalty)
# ---------------------------
@torch.no_grad()
def predict_proba(model, loader):
    model.eval()
    ps, ys = [], []
    gs, es = [], []
    for xc, xk, xb, yb in loader:
        xc = xc.to(DEVICE); xk = xk.to(DEVICE); xb = xb.to(DEVICE)
        logit, g, E = model(xc, xk, xb)
        p = torch.sigmoid(logit).detach().cpu().numpy()
        ps.append(p)
        ys.append(yb.numpy())
        gs.append(g.detach().cpu().numpy())
        es.append(E.detach().cpu().numpy())
    return np.concatenate(ys), np.concatenate(ps), np.concatenate(gs), np.concatenate(es)

def train_one_repeat(r, concept_names, other_idx=None):
    # split
    tr_idx, va_idx, te_idx = make_split(cfg.seed + 1000*r)

    df_tr = df.iloc[tr_idx].copy()
    df_va = df.iloc[va_idx].copy()
    df_te = df.iloc[te_idx].copy()

    # fit preprocess on train
    bmap, cat_vocab, mu, sd = fit_preprocess(df_tr)

    # apply
    Xc_tr, Xk_tr, Xb_tr = apply_preprocess(df_tr, bmap, cat_vocab, mu, sd)
    Xc_va, Xk_va, Xb_va = apply_preprocess(df_va, bmap, cat_vocab, mu, sd)
    Xc_te, Xk_te, Xb_te = apply_preprocess(df_te, bmap, cat_vocab, mu, sd)

    y_tr = df_tr[LABEL_COL].values.astype(int)
    y_va = df_va[LABEL_COL].values.astype(int)
    y_te = df_te[LABEL_COL].values.astype(int)

    # cat cardinalities
    cat_cards = []
    for c in cat_cols:
        cat_cards.append(len(cat_vocab[c]))

    # datasets/loaders
    tr_ds = TabDS(to_float_tensor(Xc_tr), to_long_tensor(Xk_tr), to_long_tensor(Xb_tr), to_long_tensor(y_tr))
    va_ds = TabDS(to_float_tensor(Xc_va), to_long_tensor(Xk_va), to_long_tensor(Xb_va), to_long_tensor(y_va))
    te_ds = TabDS(to_float_tensor(Xc_te), to_long_tensor(Xk_te), to_long_tensor(Xb_te), to_long_tensor(y_te))

    tr_loader = DataLoader(tr_ds, batch_size=cfg.batch_size, shuffle=True, drop_last=False)
    va_loader = DataLoader(va_ds, batch_size=cfg.batch_size, shuffle=False)
    te_loader = DataLoader(te_ds, batch_size=cfg.batch_size, shuffle=False)

    # model
    model = CGWideDeep(
        n_cont=len(cont_cols),
        cat_cardinalities=cat_cards,
        n_bin=len(bin_cols),
        concept_names=concept_names,
        d_embed=cfg.d_embed,
        deep_hidden=cfg.deep_hidden,
        deep_layers=cfg.deep_layers,
        dropout=cfg.dropout
    ).to(DEVICE)

    opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)

    # loss
    if cfg.use_pos_weight:
        pos = (y_tr==1).sum()
        neg = (y_tr==0).sum()
        posw = torch.tensor([neg/(pos+1e-12)], device=DEVICE, dtype=torch.float32)
        crit = nn.BCEWithLogitsLoss(pos_weight=posw)
    else:
        crit = nn.BCEWithLogitsLoss()

    best_pr = -1.0
    best_state = None
    bad = 0
    history = []

    for ep in range(cfg.max_epochs):
        model.train()
        ep_loss = 0.0
        n_seen = 0

        for xc, xk, xb, yb in tr_loader:
            xc = xc.to(DEVICE); xk = xk.to(DEVICE); xb = xb.to(DEVICE)
            yb = yb.to(DEVICE).float()

            opt.zero_grad()
            logit, g, E = model(xc, xk, xb)

            loss = crit(logit, yb)

            # ✅ Gate entropy regularization (maximize entropy slightly)
            if cfg.gate_entropy_lambda > 0:
                ent = -(g * (g + 1e-12).log()).sum(dim=1).mean()
                loss = loss - cfg.gate_entropy_lambda * ent

            # ✅ Optional "Other" penalty (use ONLY if Other dominates)
            if (cfg.other_gate_penalty > 0) and (other_idx is not None):
                loss = loss + cfg.other_gate_penalty * g[:, other_idx].mean()

            loss.backward()
            opt.step()

            bs = yb.size(0)
            ep_loss += loss.item() * bs
            n_seen += bs

        # validation PR-AUC
        y_va_np, p_va, g_va, e_va = predict_proba(model, va_loader)
        val_pr = average_precision_score(y_va_np, p_va)

        history.append({"epoch": ep, "loss": ep_loss/max(n_seen,1), "val_pr_auc": float(val_pr)})

        if ep % 1 == 0:
            print(f"CG-WideDeep (FIX) r{r} ep{ep:03d} loss={ep_loss/max(n_seen,1):.4f} valPR={val_pr:.4f}")

        if val_pr > best_pr + 1e-5:
            best_pr = val_pr
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            bad = 0
        else:
            bad += 1
            if bad >= cfg.patience:
                break

    # load best
    model.load_state_dict(best_state)

    # predict
    y_va_np, p_va, g_va, e_va = predict_proba(model, va_loader)
    y_te_np, p_te, g_te, e_te = predict_proba(model, te_loader)

    # threshold on val (F1)
    t_best, _ = find_best_threshold(y_va_np, p_va, metric="f1")

    m_va = evaluate_binary(y_va_np, p_va, threshold=t_best, ece_bins=cfg.ece_bins)
    m_te = evaluate_binary(y_te_np, p_te, threshold=t_best, ece_bins=cfg.ece_bins)

    # export gates + experts for diagnostics
    np.save(os.path.join(DIRS["pred"], f"cg_widedeep_fix_gates_val_r{r}.npy"), g_va)
    np.save(os.path.join(DIRS["pred"], f"cg_widedeep_fix_gates_test_r{r}.npy"), g_te)
    np.save(os.path.join(DIRS["pred"], f"cg_widedeep_fix_expertlogits_val_r{r}.npy"), e_va)
    np.save(os.path.join(DIRS["pred"], f"cg_widedeep_fix_expertlogits_test_r{r}.npy"), e_te)

    # small json (probs)
    out_json = {
        "repeat": r,
        "t_best": float(t_best),
        "val_pr_auc_best": float(best_pr),
        "metrics_val": m_va,
        "metrics_test": m_te,
    }
    with open(os.path.join(DIRS["pred"], f"cg_widedeep_fix_summary_r{r}.json"), "w") as f:
        json.dump(out_json, f, indent=2)

    return m_va, m_te, history

# ---------------------------
# 10) Run repeats + table
# ---------------------------
concept_names = list(concept_groups.keys())
other_idx = concept_names.index("Other") if "Other" in concept_names else None

rows = []
histories = []
for r in range(cfg.repeats):
    m_va, m_te, hist = train_one_repeat(r, concept_names, other_idx=other_idx)
    m_va.update({"model": "CG_WideDeep_FIX", "repeat": r, "split": "val"})
    m_te.update({"model": "CG_WideDeep_FIX", "repeat": r, "split": "test"})
    rows += [m_va, m_te]
    histories.append(hist)

df_m = pd.DataFrame(rows)
display(df_m)
save_table(df_m, "cg_widedeep_fix_per_repeat_val_test", index=False)

test_df = df_m[df_m["split"]=="test"].copy()
summary = test_df[["pr_auc","roc_auc","f1","recall","brier","ece","nll","acc"]].agg(["mean","std"]).T.reset_index()
summary.columns = ["metric","mean","std"]
display(summary)
save_table(summary, "cg_widedeep_fix_test_summary_meanstd", index=False)

print("\n✅ UPDATED TASK D.2 complete.")
print("Next recommended: run gate-dominance diagnostic using saved gates (*.npy).")

# ---------------------------
# 11) Gate dominance diagnostic (quick)
# ---------------------------
Gs = []
for r in range(cfg.repeats):
    p = os.path.join(DIRS["pred"], f"cg_widedeep_fix_gates_test_r{r}.npy")
    if os.path.exists(p):
        Gs.append(np.load(p))
if len(Gs):
    G = np.concatenate(Gs, axis=0)
    mean_w = G.mean(axis=0)
    order = np.argsort(-mean_w)
    print("\n=== Gate dominance (TEST, mean gate weight) ===")
    for i in order:
        print(f"{concept_names[i]:20s} mean_gate={mean_w[i]:.4f}")
    ent = (-(G*np.log(G+1e-12)).sum(axis=1)).mean()
    print("Mean gate entropy:", float(ent))
else:
    print("No gates found to diagnose.")

Using device: cpu
Label column: stroke
Counts: {'bin': 9, 'cat': 2, 'cont': 24}
Concept groups: {'Demographics': 4, 'Lifestyle': 4, 'Comorbidity': 4, 'Nutrition_Macro': 6, 'Nutrition_FattyAcids': 3, 'Nutrition_Minerals': 2, 'Other': 12}
CG-WideDeep (FIX) r0 ep000 loss=1.3452 valPR=0.1044
CG-WideDeep (FIX) r0 ep001 loss=1.2011 valPR=0.1427
CG-WideDeep (FIX) r0 ep002 loss=1.1482 valPR=0.1682
CG-WideDeep (FIX) r0 ep003 loss=1.0830 valPR=0.1726
CG-WideDeep (FIX) r0 ep004 loss=1.0350 valPR=0.1777
CG-WideDeep (FIX) r0 ep005 loss=1.0065 valPR=0.1898
CG-WideDeep (FIX) r0 ep006 loss=0.9917 valPR=0.1798
CG-WideDeep (FIX) r0 ep007 loss=0.9636 valPR=0.1752
CG-WideDeep (FIX) r0 ep008 loss=0.9303 valPR=0.1872
CG-WideDeep (FIX) r0 ep009 loss=0.8915 valPR=0.1868
CG-WideDeep (FIX) r0 ep010 loss=0.8618 valPR=0.1853
CG-WideDeep (FIX) r0 ep011 loss=0.8385 valPR=0.1938
CG-WideDeep (FIX) r0 ep012 loss=0.8200 valPR=0.1807
CG-WideDeep (FIX) r0 ep013 loss=0.7781 valPR=0.1795
CG-WideDeep (FIX) r0 ep014 loss=0.7

,threshold,pr_auc,roc_auc,acc,f1,precision,recall,tn,fp,fn,tp,brier,ece,nll,model,repeat,split
0,0.685,0.230604,0.709156,0.846676,0.289308,0.227723,0.396552,601,78,35,23,0.160938,0.201480,0.511477,CG_WideDeep_FIX,0,val
1,0.685,0.111741,0.603750,0.792617,0.173160,0.125786,0.277778,710,139,52,20,0.193613,0.235179,0.623640,CG_WideDeep_FIX,0,test
2,0.580,0.154619,0.665279,0.782904,0.238095,0.164474,0.431034,552,127,33,25,0.177676,0.244984,0.531240,CG_WideDeep_FIX,1,val
3,0.580,0.148371,0.707973,0.781759,0.252788,0.172589,0.472222,686,163,38,34,0.173003,0.232892,0.512397,CG_WideDeep_FIX,1,test
4,0.780,0.170207,0.719339,0.890095,0.295652,0.298246,0.293103,639,40,41,17,0.212375,0.324131,0.605559,CG_WideDeep_FIX,2,val
5,0.780,0.181868,0.691124,0.871878,0.213333,0.205128,0.222222,787,62,56,16,0.221115,0.330261,0.628287,CG_WideDeep_FIX,2,test
6,0.625,0.163980,0.687751,0.807327,0.228261,0.166667,0.362069,574,105,37,21,0.189942,0.303734,0.557388,CG_WideDeep_FIX,3,val
7,0.625,0.187775,0.684236,0.804560,0.230769,0.166667,0.375000,714,135,45,27,0.198888,0.319745,0.577201,CG_WideDeep_FIX,3,test
8,0.720,0.221135,0.715479,0.824966,0.302703,0.220472,0.482759,580,99,30,28,0.263620,0.409524,0.729959,CG_WideDeep_FIX,4,val
9,0.720,0.128241,0.652009,0.798046,0.184211,0.134615,0.291667,714,135,51,21,0.268561,0.408863,0.745651,CG_WideDeep_FIX,4,test


✅ Saved table: /kaggle/working/outputs/cgt_stroke_v1/tables/cg_widedeep_fix_per_repeat_val_test.csv


,metric,mean,std
0,pr_auc,0.151599,0.033051
1,roc_auc,0.667818,0.041170
2,f1,0.210852,0.032756
3,recall,0.327778,0.097519
4,brier,0.211036,0.036429
5,ece,0.305388,0.073683
6,nll,0.617435,0.085509
7,acc,0.809772,0.035710


✅ Saved table: /kaggle/working/outputs/cgt_stroke_v1/tables/cg_widedeep_fix_test_summary_meanstd.csv

✅ UPDATED TASK D.2 complete.
Next recommended: run gate-dominance diagnostic using saved gates (*.npy).

=== Gate dominance (TEST, mean gate weight) ===
Other                mean_gate=0.1791
Nutrition_Macro      mean_gate=0.1572
Nutrition_Minerals   mean_gate=0.1533
Lifestyle            mean_gate=0.1468
Demographics         mean_gate=0.1342
Nutrition_FattyAcids mean_gate=0.1263
Comorbidity          mean_gate=0.1031
Mean gate entropy: 1.7107281684875488


In [21]:
# ============================================================
# TASK D.2 (UPDATED, FULL RUNNABLE): CG-WideDeep with FIXES
#   ✅ residual scale alpha (prevents ranking collapse)
#   ✅ gate entropy schedule (explore early -> sharpen later)
#   ✅ optional "Other" gate penalty (only if it dominates)
#   ✅ saves gates + expert logits for diagnostics
#
# Works in Kaggle CPU/GPU.
# Expects: /kaggle/working/outputs/cgt_stroke_v1/data_clean_v2.csv exists
# ============================================================

import os, json, math, random, time
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import (
    roc_auc_score, average_precision_score, f1_score, precision_score, recall_score, accuracy_score,
    log_loss
)

# ---------------------------
# 0) Paths / Config
# ---------------------------
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)

BASE_DIR = "/kaggle/working/outputs/cgt_stroke_v1"
DATA_CSV = os.path.join(BASE_DIR, "data_clean_v2.csv")

DIRS = {
    "base": BASE_DIR,
    "logs": os.path.join(BASE_DIR, "logs"),
    "tables": os.path.join(BASE_DIR, "tables"),
    "fig": os.path.join(BASE_DIR, "figures"),
    "pred": os.path.join(BASE_DIR, "predictions"),
}
for k,v in DIRS.items():
    if k != "base":
        os.makedirs(v, exist_ok=True)

class CFG:
    seed = 42
    repeats = 5

    # training
    batch_size = 256
    lr = 2e-3
    weight_decay = 1e-4
    max_epochs = 60
    patience = 10

    # imbalance handling
    use_pos_weight = True

    # calibration metrics
    ece_bins = 15

    # -----------------------
    # Gate regularization (UPDATED)
    # We will use a SCHEDULE, not constant entropy maximization.
    # -----------------------
    gate_ent_warmup = 2          # epochs with no entropy reg
    gate_ent_flip_epoch = 12     # after this: encourage SHARP gates
    gate_ent_max_mag = 0.02      # try 0.01–0.03

    other_gate_penalty = 0.00    # set 0.02–0.10 ONLY if "Other" dominates

    # Model sizes
    d_embed = 16
    deep_hidden = 64
    deep_layers = 2
    dropout = 0.10

cfg = CFG()

def seed_everything(seed=42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(cfg.seed)

# ---------------------------
# 1) Utils: metrics
# ---------------------------
def sigmoid_np(x): return 1/(1+np.exp(-x))

def expected_calibration_error(y_true, p, n_bins=15):
    y_true = np.asarray(y_true).astype(int)
    p = np.asarray(p).astype(float)
    bins = np.linspace(0.0, 1.0, n_bins+1)
    ece = 0.0
    for i in range(n_bins):
        lo, hi = bins[i], bins[i+1]
        m = (p >= lo) & (p < hi) if i < n_bins-1 else (p >= lo) & (p <= hi)
        if m.sum() == 0:
            continue
        acc = y_true[m].mean()
        conf = p[m].mean()
        ece += (m.sum()/len(p)) * abs(acc - conf)
    return float(ece)

def find_best_threshold(y, p, metric="f1"):
    y = np.asarray(y).astype(int)
    p = np.asarray(p).astype(float)
    ts = np.linspace(0.05, 0.95, 181)
    best_t, best_s = 0.5, -1
    for t in ts:
        pred = (p >= t).astype(int)
        if metric == "f1":
            s = f1_score(y, pred, zero_division=0)
        elif metric == "youden":
            tp = ((pred==1)&(y==1)).sum()
            fn = ((pred==0)&(y==1)).sum()
            fp = ((pred==1)&(y==0)).sum()
            tn = ((pred==0)&(y==0)).sum()
            tpr = tp/(tp+fn+1e-12)
            fpr = fp/(fp+tn+1e-12)
            s = tpr - fpr
        else:
            raise ValueError("Unknown metric")
        if s > best_s:
            best_s, best_t = s, t
    return float(best_t), float(best_s)

def evaluate_binary(y, p, threshold=0.5, ece_bins=15):
    y = np.asarray(y).astype(int)
    p = np.asarray(p).astype(float)
    p = np.clip(p, 1e-7, 1-1e-7)
    pred = (p >= threshold).astype(int)

    out = {}
    out["threshold"] = float(threshold)
    out["pr_auc"] = float(average_precision_score(y, p))
    out["roc_auc"] = float(roc_auc_score(y, p))
    out["acc"] = float(accuracy_score(y, pred))
    out["f1"] = float(f1_score(y, pred, zero_division=0))
    out["precision"] = float(precision_score(y, pred, zero_division=0))
    out["recall"] = float(recall_score(y, pred, zero_division=0))

    tn = int(((pred==0) & (y==0)).sum())
    fp = int(((pred==1) & (y==0)).sum())
    fn = int(((pred==0) & (y==1)).sum())
    tp = int(((pred==1) & (y==1)).sum())
    out.update({"tn":tn, "fp":fp, "fn":fn, "tp":tp})

    out["brier"] = float(np.mean((p - y)**2))
    out["ece"] = float(expected_calibration_error(y, p, n_bins=ece_bins))
    out["nll"] = float(log_loss(y, p, labels=[0,1]))
    return out

def save_table(df, name, index=False):
    path = os.path.join(DIRS["tables"], f"{name}.csv")
    df.to_csv(path, index=index)
    print("✅ Saved table:", path)
    return path

# ---------------------------
# 1b) Gate entropy schedule (NEW)
# ---------------------------
def gate_entropy_torch(g, eps=1e-12):
    return -(g * (g + eps).log()).sum(dim=1).mean()

def gate_reg_coeff(epoch, warmup=2, flip_epoch=12, max_mag=0.02):
    """
    Returns coefficient c for: loss += c * entropy(g)

    c < 0  => maximize entropy (more uniform)
    c > 0  => minimize entropy (sharper)
    """
    if epoch < warmup:
        return 0.0

    # ramp magnitude negative until flip_epoch
    if epoch < flip_epoch:
        t = (epoch - warmup) / max(1, (flip_epoch - warmup))
        return -max_mag * float(t)

    # after flip: ramp positive to +max_mag over 10 epochs
    t2 = min(1.0, (epoch - flip_epoch) / 10.0)
    return +max_mag * float(t2)

# ---------------------------
# 2) Load data
# ---------------------------
assert os.path.exists(DATA_CSV), f"Missing: {DATA_CSV}"
df = pd.read_csv(DATA_CSV)

LABEL_COL = "stroke" if "stroke" in df.columns else df.columns[-1]
print("Label column:", LABEL_COL)

y_all = df[LABEL_COL].values.astype(int)

# ---------------------------
# 3) Feature typing (use your v2 decisions)
# ---------------------------
bin_cols = ['gender','alcohol','smoke','sleep_disorder','Health_Insurance','diabetes','hypertension','high_cholesterol','Coronary_Heart_Disease']
cat_cols = ['Race','Marital_status']
cont_cols = [c for c in df.columns if c not in ([LABEL_COL] + bin_cols + cat_cols)]
print("Counts:", {"bin":len(bin_cols), "cat":len(cat_cols), "cont":len(cont_cols)})

# ---------------------------
# 4) Split
# ---------------------------
def make_split(seed):
    sss = StratifiedShuffleSplit(n_splits=1, test_size=0.20, random_state=seed)
    (tr_idx, te_idx) = next(sss.split(df, y_all))
    y_tr = y_all[tr_idx]
    sss2 = StratifiedShuffleSplit(n_splits=1, test_size=0.20, random_state=seed+1337)
    (tr2, va2) = next(sss2.split(np.zeros(len(tr_idx)), y_tr))
    tr2_idx = tr_idx[tr2]
    va_idx = tr_idx[va2]
    return tr2_idx, va_idx, te_idx

# ---------------------------
# 5) Tensor datasets
# ---------------------------
def to_long_tensor(x):  return torch.as_tensor(x, dtype=torch.long)
def to_float_tensor(x): return torch.as_tensor(x, dtype=torch.float32)

class TabDS(Dataset):
    def __init__(self, Xc, Xk, Xb, y):
        self.Xc = Xc; self.Xk = Xk; self.Xb = Xb; self.y = y
    def __len__(self): return len(self.y)
    def __getitem__(self, i):
        return self.Xc[i], self.Xk[i], self.Xb[i], self.y[i]

# ---------------------------
# 6) Preprocessing
# ---------------------------
def fit_preprocess(train_df):
    bmap = {}
    for c in bin_cols:
        vals = train_df[c].values
        uniq = np.unique(vals)
        if set(uniq.tolist()) == {0,1}:
            bmap[c] = ("01", None)
        elif set(uniq.tolist()) == {1,2}:
            bmap[c] = ("12", None)
        else:
            if len(uniq)==2:
                bmap[c] = ("map", {uniq[0]:0, uniq[1]:1})
            else:
                raise ValueError(f"Binary col {c} has >2 unique: {uniq}")

    cat_vocab = {}
    for c in cat_cols:
        uniq = pd.Series(train_df[c]).astype(int).unique()
        uniq = np.sort(uniq)
        cat_vocab[c] = {v:i for i,v in enumerate(uniq.tolist())}

    mu = train_df[cont_cols].values.astype(np.float32).mean(axis=0)
    sd = train_df[cont_cols].values.astype(np.float32).std(axis=0)
    sd = np.where(sd < 1e-6, 1.0, sd)
    return bmap, cat_vocab, mu, sd

def apply_preprocess(df_part, bmap, cat_vocab, mu, sd):
    Xb = np.zeros((len(df_part), len(bin_cols)), dtype=np.int64)
    for j,c in enumerate(bin_cols):
        vals = df_part[c].values
        mode, mapping = bmap[c]
        if mode == "01":
            Xb[:,j] = vals.astype(int)
        elif mode == "12":
            Xb[:,j] = (vals.astype(int) - 1)
        else:
            Xb[:,j] = np.vectorize(mapping.get)(vals)
        Xb[:,j] = np.clip(Xb[:,j], 0, 1)

    Xk = np.zeros((len(df_part), len(cat_cols)), dtype=np.int64)
    for j,c in enumerate(cat_cols):
        vocab = cat_vocab[c]
        vals = df_part[c].astype(int).values
        Xk[:,j] = np.array([vocab.get(v, 0) for v in vals], dtype=np.int64)

    Xc = df_part[cont_cols].values.astype(np.float32)
    Xc = (Xc - mu) / sd
    return Xc, Xk, Xb

# ---------------------------
# 7) Concept groups
# ---------------------------
cg_path = os.path.join(DIRS["logs"], "concept_groups.json")
if os.path.exists(cg_path):
    concept_groups = json.load(open(cg_path, "r"))
else:
    concept_groups = {
        "Demographics": ["gender","age","Race","Marital_status"],
        "Lifestyle": ["alcohol","smoke","sleep_time","Minutes_sedentary_activity"],
        "Comorbidity": ["diabetes","hypertension","high_cholesterol","Coronary_Heart_Disease"],
        "Nutrition_Macro": ["energy","Carbohydrate","protein","Total_fat","Dietary_fiber","Body_Mass_Index"],
        "Nutrition_FattyAcids": ["Total_saturated_fatty_acids","Total_monounsaturated_fatty_acids","Total_polyunsaturated_fatty_acids"],
        "Nutrition_Minerals": ["Sodium","Potassium"],
        "Other": [c for c in cont_cols if c not in [
            "age","sleep_time","Minutes_sedentary_activity","energy","Carbohydrate","protein","Total_fat",
            "Dietary_fiber","Body_Mass_Index","Total_saturated_fatty_acids","Total_monounsaturated_fatty_acids",
            "Total_polyunsaturated_fatty_acids","Sodium","Potassium"
        ]]
    }

print("Concept groups:", {k:len(v) for k,v in concept_groups.items()})

# ---------------------------
# 8) Model: WideDeep + Concept-Gated Residual
# ---------------------------
class WideDeep(nn.Module):
    def __init__(self, n_cont, cat_cardinalities, n_bin, d_embed=16, deep_hidden=64, deep_layers=2, dropout=0.1):
        super().__init__()
        self.n_cont = n_cont
        self.n_bin = n_bin
        self.cat_card = cat_cardinalities

        self.cat_embeds = nn.ModuleList([nn.Embedding(card, d_embed) for card in cat_cardinalities])

        wide_in = n_cont + n_bin + len(cat_cardinalities)*d_embed
        self.wide = nn.Linear(wide_in, 1)

        layers = []
        din = wide_in
        for _ in range(deep_layers):
            layers += [nn.Linear(din, deep_hidden), nn.ReLU(), nn.Dropout(dropout)]
            din = deep_hidden
        layers += [nn.Linear(din, 1)]
        self.deep = nn.Sequential(*layers)

    def forward(self, x_cont, x_cat, x_bin):
        cat_vecs = []
        for j, emb in enumerate(self.cat_embeds):
            cat_vecs.append(emb(x_cat[:, j]))
        cat_vec = torch.cat(cat_vecs, dim=1) if len(cat_vecs) else torch.zeros((x_cont.size(0),0), device=x_cont.device)

        x_bin_f = x_bin.float()
        z = torch.cat([x_cont, x_bin_f, cat_vec], dim=1)

        wide_logit = self.wide(z).squeeze(1)
        deep_logit = self.deep(z).squeeze(1)
        base = wide_logit + deep_logit
        return base, z

class ConceptGatedResidual(nn.Module):
    def __init__(self, z_dim, n_concepts, hidden=64, dropout=0.1):
        super().__init__()
        self.n_concepts = n_concepts

        self.gate = nn.Sequential(
            nn.Linear(z_dim, hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, n_concepts)
        )

        self.experts = nn.ModuleList([
            nn.Sequential(
                nn.Linear(z_dim, hidden),
                nn.ReLU(),
                nn.Dropout(dropout),
                nn.Linear(hidden, 1)
            ) for _ in range(n_concepts)
        ])

        # residual scale alpha (init small)
        self.alpha = nn.Parameter(torch.tensor(-2.0))  # sigmoid(-2)=0.119

    def forward(self, z):
        g_logits = self.gate(z)                # [B,C]
        g = torch.softmax(g_logits, dim=1)     # [B,C]

        expert_logits = []
        for ex in self.experts:
            expert_logits.append(ex(z).squeeze(1))  # [B]
        E = torch.stack(expert_logits, dim=1)        # [B,C]

        res = (g * E).sum(dim=1)                     # [B]
        res = torch.sigmoid(self.alpha) * res        # scaled residual
        return res, g, E

class CGWideDeep(nn.Module):
    def __init__(self, n_cont, cat_cardinalities, n_bin, concept_names,
                 d_embed=16, deep_hidden=64, deep_layers=2, dropout=0.1):
        super().__init__()
        self.base = WideDeep(n_cont, cat_cardinalities, n_bin, d_embed, deep_hidden, deep_layers, dropout)
        z_dim = n_cont + n_bin + len(cat_cardinalities)*d_embed
        self.concept_names = concept_names
        self.cg = ConceptGatedResidual(z_dim=z_dim, n_concepts=len(concept_names), hidden=deep_hidden, dropout=dropout)

    def forward(self, x_cont, x_cat, x_bin):
        base_logit, z = self.base(x_cont, x_cat, x_bin)
        res, g, E = self.cg(z)
        logit = base_logit + res
        return logit, g, E

# ---------------------------
# 9) Train / eval loop (UPDATED entropy schedule)
# ---------------------------
@torch.no_grad()
def predict_proba(model, loader):
    model.eval()
    ps, ys = [], []
    gs, es = [], []
    for xc, xk, xb, yb in loader:
        xc = xc.to(DEVICE); xk = xk.to(DEVICE); xb = xb.to(DEVICE)
        logit, g, E = model(xc, xk, xb)
        p = torch.sigmoid(logit).detach().cpu().numpy()
        ps.append(p)
        ys.append(yb.numpy())
        gs.append(g.detach().cpu().numpy())
        es.append(E.detach().cpu().numpy())
    return np.concatenate(ys), np.concatenate(ps), np.concatenate(gs), np.concatenate(es)

def train_one_repeat(r, concept_names, other_idx=None):
    tr_idx, va_idx, te_idx = make_split(cfg.seed + 1000*r)

    df_tr = df.iloc[tr_idx].copy()
    df_va = df.iloc[va_idx].copy()
    df_te = df.iloc[te_idx].copy()

    bmap, cat_vocab, mu, sd = fit_preprocess(df_tr)

    Xc_tr, Xk_tr, Xb_tr = apply_preprocess(df_tr, bmap, cat_vocab, mu, sd)
    Xc_va, Xk_va, Xb_va = apply_preprocess(df_va, bmap, cat_vocab, mu, sd)
    Xc_te, Xk_te, Xb_te = apply_preprocess(df_te, bmap, cat_vocab, mu, sd)

    y_tr = df_tr[LABEL_COL].values.astype(int)
    y_va = df_va[LABEL_COL].values.astype(int)
    y_te = df_te[LABEL_COL].values.astype(int)

    cat_cards = [len(cat_vocab[c]) for c in cat_cols]

    tr_ds = TabDS(to_float_tensor(Xc_tr), to_long_tensor(Xk_tr), to_long_tensor(Xb_tr), to_long_tensor(y_tr))
    va_ds = TabDS(to_float_tensor(Xc_va), to_long_tensor(Xk_va), to_long_tensor(Xb_va), to_long_tensor(y_va))
    te_ds = TabDS(to_float_tensor(Xc_te), to_long_tensor(Xk_te), to_long_tensor(Xb_te), to_long_tensor(y_te))

    tr_loader = DataLoader(tr_ds, batch_size=cfg.batch_size, shuffle=True, drop_last=False)
    va_loader = DataLoader(va_ds, batch_size=cfg.batch_size, shuffle=False)
    te_loader = DataLoader(te_ds, batch_size=cfg.batch_size, shuffle=False)

    model = CGWideDeep(
        n_cont=len(cont_cols),
        cat_cardinalities=cat_cards,
        n_bin=len(bin_cols),
        concept_names=concept_names,
        d_embed=cfg.d_embed,
        deep_hidden=cfg.deep_hidden,
        deep_layers=cfg.deep_layers,
        dropout=cfg.dropout
    ).to(DEVICE)

    opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)

    if cfg.use_pos_weight:
        pos = (y_tr==1).sum()
        neg = (y_tr==0).sum()
        posw = torch.tensor([neg/(pos+1e-12)], device=DEVICE, dtype=torch.float32)
        crit = nn.BCEWithLogitsLoss(pos_weight=posw)
    else:
        crit = nn.BCEWithLogitsLoss()

    best_pr = -1.0
    best_state = None
    bad = 0
    history = []

    for ep in range(cfg.max_epochs):
        model.train()
        ep_loss = 0.0
        n_seen = 0

        # schedule coefficient for this epoch
        c_ent_epoch = gate_reg_coeff(
            epoch=ep,
            warmup=cfg.gate_ent_warmup,
            flip_epoch=cfg.gate_ent_flip_epoch,
            max_mag=cfg.gate_ent_max_mag
        )

        last_ent = None

        for xc, xk, xb, yb in tr_loader:
            xc = xc.to(DEVICE); xk = xk.to(DEVICE); xb = xb.to(DEVICE)
            yb = yb.to(DEVICE).float()

            opt.zero_grad()
            logit, g, E = model(xc, xk, xb)

            loss = crit(logit, yb)

            # ✅ Gate entropy schedule: loss += c * entropy(g)
            ent = gate_entropy_torch(g)
            loss = loss + c_ent_epoch * ent
            last_ent = ent.detach()

            # ✅ Optional "Other" penalty (use ONLY if Other dominates)
            if (cfg.other_gate_penalty > 0) and (other_idx is not None):
                loss = loss + cfg.other_gate_penalty * g[:, other_idx].mean()

            loss.backward()
            opt.step()

            bs = yb.size(0)
            ep_loss += loss.item() * bs
            n_seen += bs

        # validation PR-AUC
        y_va_np, p_va, g_va, e_va = predict_proba(model, va_loader)
        val_pr = average_precision_score(y_va_np, p_va)

        history.append({"epoch": ep, "loss": ep_loss/max(n_seen,1), "val_pr_auc": float(val_pr), "c_ent": float(c_ent_epoch)})

        # logging
        ent_val = float(last_ent.cpu()) if last_ent is not None else float("nan")
        print(f"CG-WideDeep (FIX+EntSched) r{r} ep{ep:03d} "
              f"loss={ep_loss/max(n_seen,1):.4f} valPR={val_pr:.4f} "
              f"ent={ent_val:.4f} c_ent={c_ent_epoch:+.4f}")

        if val_pr > best_pr + 1e-5:
            best_pr = val_pr
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            bad = 0
        else:
            bad += 1
            if bad >= cfg.patience:
                break

    model.load_state_dict(best_state)

    y_va_np, p_va, g_va, e_va = predict_proba(model, va_loader)
    y_te_np, p_te, g_te, e_te = predict_proba(model, te_loader)

    t_best, _ = find_best_threshold(y_va_np, p_va, metric="f1")

    m_va = evaluate_binary(y_va_np, p_va, threshold=t_best, ece_bins=cfg.ece_bins)
    m_te = evaluate_binary(y_te_np, p_te, threshold=t_best, ece_bins=cfg.ece_bins)

    np.save(os.path.join(DIRS["pred"], f"cg_widedeep_fix_gates_val_r{r}.npy"), g_va)
    np.save(os.path.join(DIRS["pred"], f"cg_widedeep_fix_gates_test_r{r}.npy"), g_te)
    np.save(os.path.join(DIRS["pred"], f"cg_widedeep_fix_expertlogits_val_r{r}.npy"), e_va)
    np.save(os.path.join(DIRS["pred"], f"cg_widedeep_fix_expertlogits_test_r{r}.npy"), e_te)

    out_json = {
        "repeat": r,
        "t_best": float(t_best),
        "val_pr_auc_best": float(best_pr),
        "metrics_val": m_va,
        "metrics_test": m_te,
    }
    with open(os.path.join(DIRS["pred"], f"cg_widedeep_fix_summary_r{r}.json"), "w") as f:
        json.dump(out_json, f, indent=2)

    return m_va, m_te, history

# ---------------------------
# 10) Run repeats + table
# ---------------------------
concept_names = list(concept_groups.keys())
other_idx = concept_names.index("Other") if "Other" in concept_names else None

rows = []
histories = []
for r in range(cfg.repeats):
    m_va, m_te, hist = train_one_repeat(r, concept_names, other_idx=other_idx)
    m_va.update({"model": "CG_WideDeep_FIX_EntSched", "repeat": r, "split": "val"})
    m_te.update({"model": "CG_WideDeep_FIX_EntSched", "repeat": r, "split": "test"})
    rows += [m_va, m_te]
    histories.append(hist)

df_m = pd.DataFrame(rows)
display(df_m)
save_table(df_m, "cg_widedeep_fix_entsched_per_repeat_val_test", index=False)

test_df = df_m[df_m["split"]=="test"].copy()
summary = test_df[["pr_auc","roc_auc","f1","recall","brier","ece","nll","acc"]].agg(["mean","std"]).T.reset_index()
summary.columns = ["metric","mean","std"]
display(summary)
save_table(summary, "cg_widedeep_fix_entsched_test_summary_meanstd", index=False)

print("\n✅ UPDATED TASK D.2 (Entropy Schedule) complete.")
print("Next recommended: run gate-dominance diagnostic using saved gates (*.npy).")

# ---------------------------
# 11) Gate dominance diagnostic (quick)
# ---------------------------
Gs = []
for r in range(cfg.repeats):
    p = os.path.join(DIRS["pred"], f"cg_widedeep_fix_gates_test_r{r}.npy")
    if os.path.exists(p):
        Gs.append(np.load(p))
if len(Gs):
    G = np.concatenate(Gs, axis=0)
    mean_w = G.mean(axis=0)
    order = np.argsort(-mean_w)
    print("\n=== Gate dominance (TEST, mean gate weight) ===")
    for i in order:
        print(f"{concept_names[i]:20s} mean_gate={mean_w[i]:.4f}")
    ent = (-(G*np.log(G+1e-12)).sum(axis=1)).mean()
    print("Mean gate entropy:", float(ent))
else:
    print("No gates found to diagnose.")

Using device: cpu
Label column: stroke
Counts: {'bin': 9, 'cat': 2, 'cont': 24}
Concept groups: {'Demographics': 4, 'Lifestyle': 4, 'Comorbidity': 4, 'Nutrition_Macro': 6, 'Nutrition_FattyAcids': 3, 'Nutrition_Minerals': 2, 'Other': 12}
CG-WideDeep (FIX+EntSched) r0 ep000 loss=1.3643 valPR=0.1041 ent=1.8904 c_ent=+0.0000
CG-WideDeep (FIX+EntSched) r0 ep001 loss=1.2197 valPR=0.1417 ent=1.7943 c_ent=+0.0000
CG-WideDeep (FIX+EntSched) r0 ep002 loss=1.1669 valPR=0.1670 ent=1.6332 c_ent=-0.0000
CG-WideDeep (FIX+EntSched) r0 ep003 loss=1.0979 valPR=0.1728 ent=1.6331 c_ent=-0.0020
CG-WideDeep (FIX+EntSched) r0 ep004 loss=1.0468 valPR=0.1764 ent=1.5456 c_ent=-0.0040
CG-WideDeep (FIX+EntSched) r0 ep005 loss=1.0161 valPR=0.1873 ent=1.6692 c_ent=-0.0060
CG-WideDeep (FIX+EntSched) r0 ep006 loss=0.9973 valPR=0.1807 ent=1.6683 c_ent=-0.0080
CG-WideDeep (FIX+EntSched) r0 ep007 loss=0.9659 valPR=0.1753 ent=1.7233 c_ent=-0.0100
CG-WideDeep (FIX+EntSched) r0 ep008 loss=0.9295 valPR=0.1857 ent=1.6479 c_e

,threshold,pr_auc,roc_auc,acc,f1,precision,recall,tn,fp,fn,tp,brier,ece,nll,model,repeat,split
0,0.695,0.218739,0.703012,0.848033,0.291139,0.230000,0.396552,602,77,35,23,0.168901,0.212210,0.526400,CG_WideDeep_FIX_EntSched,0,val
1,0.695,0.112006,0.604420,0.794788,0.181818,0.132075,0.291667,711,138,51,21,0.194816,0.240525,0.623496,CG_WideDeep_FIX_EntSched,0,test
2,0.540,0.161628,0.668046,0.740841,0.245059,0.158974,0.534483,515,164,27,31,0.183558,0.261830,0.540773,CG_WideDeep_FIX_EntSched,1,val
3,0.540,0.151054,0.711409,0.765472,0.250000,0.166667,0.500000,669,180,36,36,0.173267,0.248605,0.509709,CG_WideDeep_FIX_EntSched,1,test
4,0.655,0.167490,0.720024,0.843962,0.267516,0.212121,0.362069,601,78,37,21,0.192538,0.318579,0.564737,CG_WideDeep_FIX_EntSched,2,val
5,0.655,0.214297,0.725314,0.844734,0.274112,0.216000,0.375000,751,98,45,27,0.192507,0.322241,0.563438,CG_WideDeep_FIX_EntSched,2,test
6,0.570,0.150233,0.671119,0.690638,0.213793,0.133621,0.534483,478,201,27,31,0.228408,0.359255,0.647639,CG_WideDeep_FIX_EntSched,3,val
7,0.570,0.207372,0.683926,0.679696,0.209115,0.129568,0.541667,587,262,33,39,0.240364,0.377883,0.673855,CG_WideDeep_FIX_EntSched,3,test
8,0.665,0.202107,0.700675,0.829037,0.267442,0.201754,0.396552,588,91,35,23,0.250630,0.412567,0.695817,CG_WideDeep_FIX_EntSched,4,val
9,0.665,0.138916,0.653432,0.823018,0.197044,0.152672,0.277778,738,111,52,20,0.249906,0.406917,0.695500,CG_WideDeep_FIX_EntSched,4,test


✅ Saved table: /kaggle/working/outputs/cgt_stroke_v1/tables/cg_widedeep_fix_entsched_per_repeat_val_test.csv


,metric,mean,std
0,pr_auc,0.164729,0.044465
1,roc_auc,0.675700,0.048428
2,f1,0.222418,0.038413
3,recall,0.397222,0.119719
4,brier,0.210172,0.033167
5,ece,0.319234,0.074700
6,nll,0.613200,0.077047
7,acc,0.781542,0.064265


✅ Saved table: /kaggle/working/outputs/cgt_stroke_v1/tables/cg_widedeep_fix_entsched_test_summary_meanstd.csv

✅ UPDATED TASK D.2 (Entropy Schedule) complete.
Next recommended: run gate-dominance diagnostic using saved gates (*.npy).

=== Gate dominance (TEST, mean gate weight) ===
Other                mean_gate=0.1999
Nutrition_Minerals   mean_gate=0.1704
Nutrition_Macro      mean_gate=0.1686
Nutrition_FattyAcids mean_gate=0.1470
Demographics         mean_gate=0.1325
Comorbidity          mean_gate=0.0955
Lifestyle            mean_gate=0.0861
Mean gate entropy: 1.581109642982483


In [22]:
# ============================================================
# FULL RUNNABLE: CG-WideDeep (FIX) + Entropy Schedule + Pos-Focal Addon
#   ✅ residual scale alpha
#   ✅ gate entropy schedule (early explore -> later sharpen)
#   ✅ positive-only focal add-on (BCE + beta * pos_focal)
#   ✅ optional "Other" gate penalty (off by default)
#   ✅ saves gates + expert logits + tables
#
# Expects:
#   /kaggle/working/outputs/cgt_stroke_v1/data_clean_v2.csv
# ============================================================

import os, json, random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import (
    roc_auc_score, average_precision_score, f1_score, precision_score, recall_score, accuracy_score, log_loss
)

# ---------------------------
# 0) Paths / Config
# ---------------------------
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)

BASE_DIR = "/kaggle/working/outputs/cgt_stroke_v1"
DATA_CSV = os.path.join(BASE_DIR, "data_clean_v2.csv")

DIRS = {
    "base": BASE_DIR,
    "logs": os.path.join(BASE_DIR, "logs"),
    "tables": os.path.join(BASE_DIR, "tables"),
    "fig": os.path.join(BASE_DIR, "figures"),
    "pred": os.path.join(BASE_DIR, "predictions"),
}
for k, v in DIRS.items():
    if k != "base":
        os.makedirs(v, exist_ok=True)

class CFG:
    seed = 42
    repeats = 5

    # training
    batch_size = 256
    lr = 2e-3
    weight_decay = 1e-4
    max_epochs = 60
    patience = 10

    # imbalance handling
    use_pos_weight = True

    # calibration metrics
    ece_bins = 15

    # ---- Entropy schedule ----
    gate_ent_warmup = 2
    gate_ent_flip_epoch = 12
    gate_ent_max_mag = 0.02   # try 0.01–0.03

    # ---- Optional "Other" penalty (OFF by default) ----
    other_gate_penalty = 0.00  # set 0.02–0.10 ONLY if Other dominates badly

    # ---- Step-2: Positive-only focal add-on ----
    use_pos_focal = True
    focal_gamma = 1.5          # try 1.0–2.0
    focal_beta = 0.25          # try 0.10–0.50

    # Model sizes
    d_embed = 16
    deep_hidden = 64
    deep_layers = 2
    dropout = 0.10

cfg = CFG()

def seed_everything(seed=42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(cfg.seed)

# ---------------------------
# 1) Utils: metrics
# ---------------------------
def expected_calibration_error(y_true, p, n_bins=15):
    y_true = np.asarray(y_true).astype(int)
    p = np.asarray(p).astype(float)
    bins = np.linspace(0.0, 1.0, n_bins+1)
    ece = 0.0
    for i in range(n_bins):
        lo, hi = bins[i], bins[i+1]
        m = (p >= lo) & (p < hi) if i < n_bins-1 else (p >= lo) & (p <= hi)
        if m.sum() == 0:
            continue
        acc = y_true[m].mean()
        conf = p[m].mean()
        ece += (m.sum()/len(p)) * abs(acc - conf)
    return float(ece)

def find_best_threshold(y, p, metric="f1"):
    y = np.asarray(y).astype(int)
    p = np.asarray(p).astype(float)
    ts = np.linspace(0.05, 0.95, 181)
    best_t, best_s = 0.5, -1
    for t in ts:
        pred = (p >= t).astype(int)
        if metric == "f1":
            s = f1_score(y, pred, zero_division=0)
        elif metric == "youden":
            tp = ((pred==1)&(y==1)).sum()
            fn = ((pred==0)&(y==1)).sum()
            fp = ((pred==1)&(y==0)).sum()
            tn = ((pred==0)&(y==0)).sum()
            tpr = tp/(tp+fn+1e-12)
            fpr = fp/(fp+tn+1e-12)
            s = tpr - fpr
        else:
            raise ValueError("Unknown metric")
        if s > best_s:
            best_s, best_t = s, t
    return float(best_t), float(best_s)

def evaluate_binary(y, p, threshold=0.5, ece_bins=15):
    y = np.asarray(y).astype(int)
    p = np.asarray(p).astype(float)
    p = np.clip(p, 1e-7, 1-1e-7)
    pred = (p >= threshold).astype(int)

    out = {}
    out["threshold"] = float(threshold)
    out["pr_auc"] = float(average_precision_score(y, p))
    out["roc_auc"] = float(roc_auc_score(y, p))
    out["acc"] = float(accuracy_score(y, pred))
    out["f1"] = float(f1_score(y, pred, zero_division=0))
    out["precision"] = float(precision_score(y, pred, zero_division=0))
    out["recall"] = float(recall_score(y, pred, zero_division=0))

    tn = int(((pred==0) & (y==0)).sum())
    fp = int(((pred==1) & (y==0)).sum())
    fn = int(((pred==0) & (y==1)).sum())
    tp = int(((pred==1) & (y==1)).sum())
    out.update({"tn":tn, "fp":fp, "fn":fn, "tp":tp})

    out["brier"] = float(np.mean((p - y)**2))
    out["ece"] = float(expected_calibration_error(y, p, n_bins=ece_bins))
    out["nll"] = float(log_loss(y, p, labels=[0,1]))
    return out

def save_table(df, name, index=False):
    path = os.path.join(DIRS["tables"], f"{name}.csv")
    df.to_csv(path, index=index)
    print("✅ Saved table:", path)
    return path

# ---------------------------
# 1b) Gate entropy schedule
# ---------------------------
def gate_entropy_torch(g, eps=1e-12):
    return -(g * (g + eps).log()).sum(dim=1).mean()

def gate_reg_coeff(epoch, warmup=2, flip_epoch=12, max_mag=0.02):
    """
    coefficient c for: loss += c * entropy(g)
      c < 0 => maximize entropy (explore)
      c > 0 => minimize entropy (sharpen)
    """
    if epoch < warmup:
        return 0.0
    if epoch < flip_epoch:
        t = (epoch - warmup) / max(1, (flip_epoch - warmup))
        return -max_mag * float(t)
    t2 = min(1.0, (epoch - flip_epoch) / 10.0)
    return +max_mag * float(t2)

# ---------------------------
# 1c) Step-2: Positive-only focal add-on
# ---------------------------
def pos_only_focal_addon_from_logits(logit, y, gamma=1.5):
    """
    Positive-only focal term (mean over positives in batch).
    """
    p = torch.sigmoid(logit)
    bce_pos = -torch.log(p + 1e-12)
    w = (1.0 - p).pow(gamma)

    pos_mask = (y > 0.5)
    if pos_mask.sum() == 0:
        return logit.new_tensor(0.0)
    return (w[pos_mask] * bce_pos[pos_mask]).mean()

# ---------------------------
# 2) Load data
# ---------------------------
assert os.path.exists(DATA_CSV), f"Missing: {DATA_CSV}"
df = pd.read_csv(DATA_CSV)

LABEL_COL = "stroke" if "stroke" in df.columns else df.columns[-1]
print("Label column:", LABEL_COL)

y_all = df[LABEL_COL].values.astype(int)

# ---------------------------
# 3) Feature typing (your v2)
# ---------------------------
bin_cols = ['gender','alcohol','smoke','sleep_disorder','Health_Insurance','diabetes','hypertension','high_cholesterol','Coronary_Heart_Disease']
cat_cols = ['Race','Marital_status']
cont_cols = [c for c in df.columns if c not in ([LABEL_COL] + bin_cols + cat_cols)]
print("Counts:", {"bin":len(bin_cols), "cat":len(cat_cols), "cont":len(cont_cols)})

# ---------------------------
# 4) Split
# ---------------------------
def make_split(seed):
    sss = StratifiedShuffleSplit(n_splits=1, test_size=0.20, random_state=seed)
    (tr_idx, te_idx) = next(sss.split(df, y_all))
    y_tr = y_all[tr_idx]
    sss2 = StratifiedShuffleSplit(n_splits=1, test_size=0.20, random_state=seed+1337)
    (tr2, va2) = next(sss2.split(np.zeros(len(tr_idx)), y_tr))
    tr2_idx = tr_idx[tr2]
    va_idx = tr_idx[va2]
    return tr2_idx, va_idx, te_idx

# ---------------------------
# 5) Tensor datasets
# ---------------------------
def to_long_tensor(x):  return torch.as_tensor(x, dtype=torch.long)
def to_float_tensor(x): return torch.as_tensor(x, dtype=torch.float32)

class TabDS(Dataset):
    def __init__(self, Xc, Xk, Xb, y):
        self.Xc = Xc; self.Xk = Xk; self.Xb = Xb; self.y = y
    def __len__(self): return len(self.y)
    def __getitem__(self, i):
        return self.Xc[i], self.Xk[i], self.Xb[i], self.y[i]

# ---------------------------
# 6) Preprocessing
# ---------------------------
def fit_preprocess(train_df):
    # bin mapping
    bmap = {}
    for c in bin_cols:
        vals = train_df[c].values
        uniq = np.unique(vals)
        if set(uniq.tolist()) == {0,1}:
            bmap[c] = ("01", None)
        elif set(uniq.tolist()) == {1,2}:
            bmap[c] = ("12", None)
        else:
            if len(uniq)==2:
                bmap[c] = ("map", {uniq[0]:0, uniq[1]:1})
            else:
                raise ValueError(f"Binary col {c} has >2 unique: {uniq}")

    # cat vocab
    cat_vocab = {}
    for c in cat_cols:
        uniq = pd.Series(train_df[c]).astype(int).unique()
        uniq = np.sort(uniq)
        cat_vocab[c] = {v:i for i,v in enumerate(uniq.tolist())}

    # cont zscore
    mu = train_df[cont_cols].values.astype(np.float32).mean(axis=0)
    sd = train_df[cont_cols].values.astype(np.float32).std(axis=0)
    sd = np.where(sd < 1e-6, 1.0, sd)
    return bmap, cat_vocab, mu, sd

def apply_preprocess(df_part, bmap, cat_vocab, mu, sd):
    # bin
    Xb = np.zeros((len(df_part), len(bin_cols)), dtype=np.int64)
    for j,c in enumerate(bin_cols):
        vals = df_part[c].values
        mode, mapping = bmap[c]
        if mode == "01":
            Xb[:,j] = vals.astype(int)
        elif mode == "12":
            Xb[:,j] = (vals.astype(int) - 1)
        else:
            Xb[:,j] = np.vectorize(mapping.get)(vals)
        Xb[:,j] = np.clip(Xb[:,j], 0, 1)

    # cat
    Xk = np.zeros((len(df_part), len(cat_cols)), dtype=np.int64)
    for j,c in enumerate(cat_cols):
        vocab = cat_vocab[c]
        vals = df_part[c].astype(int).values
        Xk[:,j] = np.array([vocab.get(v, 0) for v in vals], dtype=np.int64)

    # cont
    Xc = df_part[cont_cols].values.astype(np.float32)
    Xc = (Xc - mu) / sd
    return Xc, Xk, Xb

# ---------------------------
# 7) Concept groups
# ---------------------------
cg_path = os.path.join(DIRS["logs"], "concept_groups.json")
if os.path.exists(cg_path):
    concept_groups = json.load(open(cg_path, "r"))
else:
    # fallback (you likely have the json already)
    concept_groups = {
        "Demographics": ["gender","age","Race","Marital_status"],
        "Lifestyle": ["alcohol","smoke","sleep_time","Minutes_sedentary_activity"],
        "Comorbidity": ["diabetes","hypertension","high_cholesterol","Coronary_Heart_Disease"],
        "Nutrition_Macro": ["energy","Carbohydrate","protein","Total_fat","Dietary_fiber","Body_Mass_Index"],
        "Nutrition_FattyAcids": ["Total_saturated_fatty_acids","Total_monounsaturated_fatty_acids","Total_polyunsaturated_fatty_acids"],
        "Nutrition_Minerals": ["Sodium","Potassium"],
        "Other": [c for c in cont_cols if c not in [
            "age","sleep_time","Minutes_sedentary_activity","energy","Carbohydrate","protein","Total_fat",
            "Dietary_fiber","Body_Mass_Index","Total_saturated_fatty_acids","Total_monounsaturated_fatty_acids",
            "Total_polyunsaturated_fatty_acids","Sodium","Potassium"
        ]]
    }
print("Concept groups:", {k:len(v) for k,v in concept_groups.items()})

# ---------------------------
# 8) Model: WideDeep + Concept-Gated Residual
# ---------------------------
class WideDeep(nn.Module):
    def __init__(self, n_cont, cat_cardinalities, n_bin, d_embed=16, deep_hidden=64, deep_layers=2, dropout=0.1):
        super().__init__()
        self.cat_embeds = nn.ModuleList([nn.Embedding(card, d_embed) for card in cat_cardinalities])

        wide_in = n_cont + n_bin + len(cat_cardinalities)*d_embed
        self.wide = nn.Linear(wide_in, 1)

        layers = []
        din = wide_in
        for _ in range(deep_layers):
            layers += [nn.Linear(din, deep_hidden), nn.ReLU(), nn.Dropout(dropout)]
            din = deep_hidden
        layers += [nn.Linear(din, 1)]
        self.deep = nn.Sequential(*layers)

    def forward(self, x_cont, x_cat, x_bin):
        cat_vecs = [emb(x_cat[:, j]) for j, emb in enumerate(self.cat_embeds)]
        cat_vec = torch.cat(cat_vecs, dim=1) if len(cat_vecs) else torch.zeros((x_cont.size(0),0), device=x_cont.device)

        z = torch.cat([x_cont, x_bin.float(), cat_vec], dim=1)
        base = self.wide(z).squeeze(1) + self.deep(z).squeeze(1)
        return base, z

class ConceptGatedResidual(nn.Module):
    def __init__(self, z_dim, n_concepts, hidden=64, dropout=0.1):
        super().__init__()
        self.gate = nn.Sequential(
            nn.Linear(z_dim, hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, n_concepts)
        )
        self.experts = nn.ModuleList([
            nn.Sequential(
                nn.Linear(z_dim, hidden),
                nn.ReLU(),
                nn.Dropout(dropout),
                nn.Linear(hidden, 1)
            ) for _ in range(n_concepts)
        ])
        self.alpha = nn.Parameter(torch.tensor(-2.0))  # sigmoid(-2)=0.119

    def forward(self, z):
        g = torch.softmax(self.gate(z), dim=1)          # [B,C]
        E = torch.stack([ex(z).squeeze(1) for ex in self.experts], dim=1)  # [B,C]
        res = (g * E).sum(dim=1)
        res = torch.sigmoid(self.alpha) * res
        return res, g, E

class CGWideDeep(nn.Module):
    def __init__(self, n_cont, cat_cardinalities, n_bin, concept_names,
                 d_embed=16, deep_hidden=64, deep_layers=2, dropout=0.1):
        super().__init__()
        self.base = WideDeep(n_cont, cat_cardinalities, n_bin, d_embed, deep_hidden, deep_layers, dropout)
        z_dim = n_cont + n_bin + len(cat_cardinalities)*d_embed
        self.cg = ConceptGatedResidual(z_dim=z_dim, n_concepts=len(concept_names), hidden=deep_hidden, dropout=dropout)

    def forward(self, x_cont, x_cat, x_bin):
        base_logit, z = self.base(x_cont, x_cat, x_bin)
        res, g, E = self.cg(z)
        return base_logit + res, g, E

# ---------------------------
# 9) Train / eval
# ---------------------------
@torch.no_grad()
def predict_proba(model, loader):
    model.eval()
    ps, ys, gs, es = [], [], [], []
    for xc, xk, xb, yb in loader:
        xc = xc.to(DEVICE); xk = xk.to(DEVICE); xb = xb.to(DEVICE)
        logit, g, E = model(xc, xk, xb)
        p = torch.sigmoid(logit).detach().cpu().numpy()
        ps.append(p)
        ys.append(yb.numpy())
        gs.append(g.detach().cpu().numpy())
        es.append(E.detach().cpu().numpy())
    return np.concatenate(ys), np.concatenate(ps), np.concatenate(gs), np.concatenate(es)

def train_one_repeat(r, concept_names, other_idx=None):
    tr_idx, va_idx, te_idx = make_split(cfg.seed + 1000*r)
    df_tr, df_va, df_te = df.iloc[tr_idx].copy(), df.iloc[va_idx].copy(), df.iloc[te_idx].copy()

    bmap, cat_vocab, mu, sd = fit_preprocess(df_tr)
    Xc_tr, Xk_tr, Xb_tr = apply_preprocess(df_tr, bmap, cat_vocab, mu, sd)
    Xc_va, Xk_va, Xb_va = apply_preprocess(df_va, bmap, cat_vocab, mu, sd)
    Xc_te, Xk_te, Xb_te = apply_preprocess(df_te, bmap, cat_vocab, mu, sd)

    y_tr = df_tr[LABEL_COL].values.astype(int)
    y_va = df_va[LABEL_COL].values.astype(int)
    y_te = df_te[LABEL_COL].values.astype(int)

    cat_cards = [len(cat_vocab[c]) for c in cat_cols]

    tr_ds = TabDS(to_float_tensor(Xc_tr), to_long_tensor(Xk_tr), to_long_tensor(Xb_tr), to_long_tensor(y_tr))
    va_ds = TabDS(to_float_tensor(Xc_va), to_long_tensor(Xk_va), to_long_tensor(Xb_va), to_long_tensor(y_va))
    te_ds = TabDS(to_float_tensor(Xc_te), to_long_tensor(Xk_te), to_long_tensor(Xb_te), to_long_tensor(y_te))

    tr_loader = DataLoader(tr_ds, batch_size=cfg.batch_size, shuffle=True, drop_last=False)
    va_loader = DataLoader(va_ds, batch_size=cfg.batch_size, shuffle=False)
    te_loader = DataLoader(te_ds, batch_size=cfg.batch_size, shuffle=False)

    model = CGWideDeep(
        n_cont=len(cont_cols),
        cat_cardinalities=cat_cards,
        n_bin=len(bin_cols),
        concept_names=concept_names,
        d_embed=cfg.d_embed,
        deep_hidden=cfg.deep_hidden,
        deep_layers=cfg.deep_layers,
        dropout=cfg.dropout
    ).to(DEVICE)

    opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)

    if cfg.use_pos_weight:
        pos = (y_tr==1).sum()
        neg = (y_tr==0).sum()
        posw = torch.tensor([neg/(pos+1e-12)], device=DEVICE, dtype=torch.float32)
        crit = nn.BCEWithLogitsLoss(pos_weight=posw)
    else:
        crit = nn.BCEWithLogitsLoss()

    best_pr, best_state, bad = -1.0, None, 0
    history = []

    for ep in range(cfg.max_epochs):
        model.train()
        ep_loss, n_seen = 0.0, 0

        c_ent_epoch = gate_reg_coeff(
            epoch=ep,
            warmup=cfg.gate_ent_warmup,
            flip_epoch=cfg.gate_ent_flip_epoch,
            max_mag=cfg.gate_ent_max_mag
        )

        last_ent = None
        last_focal = None

        for xc, xk, xb, yb in tr_loader:
            xc = xc.to(DEVICE); xk = xk.to(DEVICE); xb = xb.to(DEVICE)
            yb = yb.to(DEVICE).float()

            opt.zero_grad(set_to_none=True)
            logit, g, E = model(xc, xk, xb)

            loss = crit(logit, yb)

            # Step-2: positive-only focal add-on
            if cfg.use_pos_focal:
                focal_term = pos_only_focal_addon_from_logits(logit, yb, gamma=cfg.focal_gamma)
                loss = loss + cfg.focal_beta * focal_term
                last_focal = focal_term.detach()

            # Entropy schedule
            ent = gate_entropy_torch(g)
            loss = loss + c_ent_epoch * ent
            last_ent = ent.detach()

            # Optional "Other" penalty
            if (cfg.other_gate_penalty > 0) and (other_idx is not None):
                loss = loss + cfg.other_gate_penalty * g[:, other_idx].mean()

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()

            bs = yb.size(0)
            ep_loss += loss.item() * bs
            n_seen += bs

        # val
        y_va_np, p_va, g_va, e_va = predict_proba(model, va_loader)
        val_pr = average_precision_score(y_va_np, p_va)

        ent_val = float(last_ent.cpu()) if last_ent is not None else float("nan")
        focal_val = float(last_focal.cpu()) if last_focal is not None else float("nan")

        print(f"CG-WideDeep (EntSched+PosFocal) r{r} ep{ep:03d} "
              f"loss={ep_loss/max(n_seen,1):.4f} valPR={val_pr:.4f} "
              f"ent={ent_val:.4f} c_ent={c_ent_epoch:+.4f} "
              f"focal={focal_val:.4f} beta={cfg.focal_beta:.2f} gamma={cfg.focal_gamma:.1f}")

        history.append({"epoch": ep, "loss": ep_loss/max(n_seen,1), "val_pr_auc": float(val_pr),
                        "c_ent": float(c_ent_epoch), "ent": ent_val, "focal": focal_val})

        if val_pr > best_pr + 1e-5:
            best_pr = val_pr
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            bad = 0
        else:
            bad += 1
            if bad >= cfg.patience:
                break

    # load best
    model.load_state_dict(best_state)

    # predict
    y_va_np, p_va, g_va, e_va = predict_proba(model, va_loader)
    y_te_np, p_te, g_te, e_te = predict_proba(model, te_loader)

    # threshold on val (F1)
    t_best, _ = find_best_threshold(y_va_np, p_va, metric="f1")

    m_va = evaluate_binary(y_va_np, p_va, threshold=t_best, ece_bins=cfg.ece_bins)
    m_te = evaluate_binary(y_te_np, p_te, threshold=t_best, ece_bins=cfg.ece_bins)

    # save gates/expert logits
    np.save(os.path.join(DIRS["pred"], f"cg_widedeep_gates_val_r{r}.npy"), g_va)
    np.save(os.path.join(DIRS["pred"], f"cg_widedeep_gates_test_r{r}.npy"), g_te)
    np.save(os.path.join(DIRS["pred"], f"cg_widedeep_expertlogits_val_r{r}.npy"), e_va)
    np.save(os.path.join(DIRS["pred"], f"cg_widedeep_expertlogits_test_r{r}.npy"), e_te)

    out_json = {"repeat": r, "t_best": float(t_best), "val_pr_auc_best": float(best_pr),
                "metrics_val": m_va, "metrics_test": m_te}
    with open(os.path.join(DIRS["pred"], f"cg_widedeep_summary_r{r}.json"), "w") as f:
        json.dump(out_json, f, indent=2)

    return m_va, m_te, history

# ---------------------------
# 10) Run repeats + tables
# ---------------------------
concept_names = list(concept_groups.keys())
other_idx = concept_names.index("Other") if "Other" in concept_names else None

rows, histories = [], []
for r in range(cfg.repeats):
    m_va, m_te, hist = train_one_repeat(r, concept_names, other_idx=other_idx)
    m_va.update({"model": "CG_WideDeep_EntSched_PosFocal", "repeat": r, "split": "val"})
    m_te.update({"model": "CG_WideDeep_EntSched_PosFocal", "repeat": r, "split": "test"})
    rows += [m_va, m_te]
    histories.append(hist)

df_m = pd.DataFrame(rows)
display(df_m)
save_table(df_m, "cg_widedeep_entsched_posfocal_per_repeat_val_test", index=False)

test_df = df_m[df_m["split"]=="test"].copy()
summary = test_df[["pr_auc","roc_auc","f1","recall","brier","ece","nll","acc"]].agg(["mean","std"]).T.reset_index()
summary.columns = ["metric","mean","std"]
display(summary)
save_table(summary, "cg_widedeep_entsched_posfocal_test_summary_meanstd", index=False)

print("\n✅ Run complete: Entropy Schedule + Positive-only Focal Add-on")

# ---------------------------
# 11) Gate dominance diagnostic
# ---------------------------
Gs = []
for r in range(cfg.repeats):
    p = os.path.join(DIRS["pred"], f"cg_widedeep_gates_test_r{r}.npy")
    if os.path.exists(p):
        Gs.append(np.load(p))
if len(Gs):
    G = np.concatenate(Gs, axis=0)
    mean_w = G.mean(axis=0)
    order = np.argsort(-mean_w)
    print("\n=== Gate dominance (TEST, mean gate weight) ===")
    for i in order:
        print(f"{concept_names[i]:20s} mean_gate={mean_w[i]:.4f}")
    ent = (-(G*np.log(G+1e-12)).sum(axis=1)).mean()
    print("Mean gate entropy:", float(ent))
else:
    print("No gates found to diagnose.")

Using device: cpu
Label column: stroke
Counts: {'bin': 9, 'cat': 2, 'cont': 24}
Concept groups: {'Demographics': 4, 'Lifestyle': 4, 'Comorbidity': 4, 'Nutrition_Macro': 6, 'Nutrition_FattyAcids': 3, 'Nutrition_Minerals': 2, 'Other': 12}
CG-WideDeep (EntSched+PosFocal) r0 ep000 loss=1.4569 valPR=0.1058 ent=1.8842 c_ent=+0.0000 focal=0.1410 beta=0.25 gamma=1.5
CG-WideDeep (EntSched+PosFocal) r0 ep001 loss=1.2739 valPR=0.1462 ent=1.8229 c_ent=+0.0000 focal=0.1946 beta=0.25 gamma=1.5
CG-WideDeep (EntSched+PosFocal) r0 ep002 loss=1.2152 valPR=0.1781 ent=1.6045 c_ent=-0.0000 focal=0.1383 beta=0.25 gamma=1.5
CG-WideDeep (EntSched+PosFocal) r0 ep003 loss=1.1407 valPR=0.1709 ent=1.5737 c_ent=-0.0020 focal=0.1861 beta=0.25 gamma=1.5
CG-WideDeep (EntSched+PosFocal) r0 ep004 loss=1.0849 valPR=0.1814 ent=1.5014 c_ent=-0.0040 focal=0.1839 beta=0.25 gamma=1.5
CG-WideDeep (EntSched+PosFocal) r0 ep005 loss=1.0536 valPR=0.1812 ent=1.6616 c_ent=-0.0060 focal=0.1902 beta=0.25 gamma=1.5
CG-WideDeep (EntSch

,threshold,pr_auc,roc_auc,acc,f1,precision,recall,tn,fp,fn,tp,brier,ece,nll,model,repeat,split
0,0.730,0.223129,0.689833,0.831750,0.270588,0.205357,0.396552,590,89,35,23,0.190995,0.236843,0.583330,CG_WideDeep_EntSched_PosFocal,0,val
1,0.730,0.107885,0.602457,0.790445,0.171674,0.124224,0.277778,708,141,52,20,0.215473,0.263460,0.678286,CG_WideDeep_EntSched_PosFocal,0,test
2,0.745,0.170972,0.638363,0.849389,0.244898,0.202247,0.310345,608,71,40,18,0.182320,0.227417,0.589157,CG_WideDeep_EntSched_PosFocal,1,val
3,0.745,0.156791,0.713012,0.853420,0.219653,0.188119,0.263889,767,82,53,19,0.165375,0.207296,0.507539,CG_WideDeep_EntSched_PosFocal,1,test
4,0.740,0.157084,0.689274,0.849389,0.244898,0.202247,0.310345,608,71,40,18,0.245060,0.381832,0.684297,CG_WideDeep_EntSched_PosFocal,2,val
5,0.740,0.174007,0.719114,0.840391,0.238342,0.190083,0.319444,751,98,49,23,0.247374,0.388938,0.689400,CG_WideDeep_EntSched_PosFocal,2,test
6,0.600,0.156917,0.672363,0.757123,0.238298,0.158192,0.482759,530,149,30,28,0.210469,0.333698,0.603742,CG_WideDeep_EntSched_PosFocal,3,val
7,0.600,0.209976,0.699810,0.737242,0.234177,0.151639,0.513889,642,207,35,37,0.223634,0.357476,0.631269,CG_WideDeep_EntSched_PosFocal,3,test
8,0.710,0.208463,0.708395,0.791045,0.273585,0.188312,0.500000,554,125,29,29,0.271014,0.414446,0.747340,CG_WideDeep_EntSched_PosFocal,4,val
9,0.710,0.150587,0.649130,0.781759,0.179592,0.127168,0.305556,698,151,50,22,0.273943,0.413273,0.754239,CG_WideDeep_EntSched_PosFocal,4,test


✅ Saved table: /kaggle/working/outputs/cgt_stroke_v1/tables/cg_widedeep_entsched_posfocal_per_repeat_val_test.csv


,metric,mean,std
0,pr_auc,0.159849,0.037110
1,roc_auc,0.676705,0.049809
2,f1,0.208688,0.031088
3,recall,0.336111,0.101778
4,brier,0.225160,0.040446
5,ece,0.326089,0.087422
6,nll,0.652147,0.091980
7,acc,0.800651,0.047026


✅ Saved table: /kaggle/working/outputs/cgt_stroke_v1/tables/cg_widedeep_entsched_posfocal_test_summary_meanstd.csv

✅ Run complete: Entropy Schedule + Positive-only Focal Add-on

=== Gate dominance (TEST, mean gate weight) ===
Demographics         mean_gate=0.1880
Comorbidity          mean_gate=0.1739
Lifestyle            mean_gate=0.1407
Nutrition_Macro      mean_gate=0.1406
Nutrition_Minerals   mean_gate=0.1358
Other                mean_gate=0.1349
Nutrition_FattyAcids mean_gate=0.0861
Mean gate entropy: 1.4225071668624878


In [23]:
# ============================================================
# FULL RUNNABLE: CG-WideDeep (FIX) + Entropy Schedule + Balanced Batching
#   ✅ residual scale alpha
#   ✅ gate entropy schedule (early explore -> later sharpen)
#   ✅ Step-3: WeightedRandomSampler for positive-aware batches
#   ✅ optional "Other" gate penalty (off by default)
#   ✅ saves gates + expert logits + tables
#
# Expects:
#   /kaggle/working/outputs/cgt_stroke_v1/data_clean_v2.csv
# ============================================================

import os, json, random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import (
    roc_auc_score, average_precision_score, f1_score, precision_score, recall_score, accuracy_score, log_loss
)

# ---------------------------
# 0) Paths / Config
# ---------------------------
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)

BASE_DIR = "/kaggle/working/outputs/cgt_stroke_v1"
DATA_CSV = os.path.join(BASE_DIR, "data_clean_v2.csv")

DIRS = {
    "base": BASE_DIR,
    "logs": os.path.join(BASE_DIR, "logs"),
    "tables": os.path.join(BASE_DIR, "tables"),
    "fig": os.path.join(BASE_DIR, "figures"),
    "pred": os.path.join(BASE_DIR, "predictions"),
}
for k, v in DIRS.items():
    if k != "base":
        os.makedirs(v, exist_ok=True)

class CFG:
    seed = 42
    repeats = 5

    # training
    batch_size = 256
    lr = 2e-3
    weight_decay = 1e-4
    max_epochs = 60
    patience = 10

    # imbalance handling
    use_pos_weight = True

    # Step-3: Balanced batching
    use_balanced_sampler = True

    # calibration metrics
    ece_bins = 15

    # ---- Entropy schedule ----
    gate_ent_warmup = 2
    gate_ent_flip_epoch = 12
    gate_ent_max_mag = 0.02   # try 0.01–0.03

    # ---- Optional "Other" penalty ----
    other_gate_penalty = 0.00  # set 0.02–0.10 ONLY if Other dominates badly

    # Model sizes
    d_embed = 16
    deep_hidden = 64
    deep_layers = 2
    dropout = 0.10

cfg = CFG()

def seed_everything(seed=42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(cfg.seed)

# ---------------------------
# 1) Utils: metrics
# ---------------------------
def expected_calibration_error(y_true, p, n_bins=15):
    y_true = np.asarray(y_true).astype(int)
    p = np.asarray(p).astype(float)
    bins = np.linspace(0.0, 1.0, n_bins+1)
    ece = 0.0
    for i in range(n_bins):
        lo, hi = bins[i], bins[i+1]
        m = (p >= lo) & (p < hi) if i < n_bins-1 else (p >= lo) & (p <= hi)
        if m.sum() == 0:
            continue
        acc = y_true[m].mean()
        conf = p[m].mean()
        ece += (m.sum()/len(p)) * abs(acc - conf)
    return float(ece)

def find_best_threshold(y, p, metric="f1"):
    y = np.asarray(y).astype(int)
    p = np.asarray(p).astype(float)
    ts = np.linspace(0.05, 0.95, 181)
    best_t, best_s = 0.5, -1
    for t in ts:
        pred = (p >= t).astype(int)
        if metric == "f1":
            s = f1_score(y, pred, zero_division=0)
        elif metric == "youden":
            tp = ((pred==1)&(y==1)).sum()
            fn = ((pred==0)&(y==1)).sum()
            fp = ((pred==1)&(y==0)).sum()
            tn = ((pred==0)&(y==0)).sum()
            tpr = tp/(tp+fn+1e-12)
            fpr = fp/(fp+tn+1e-12)
            s = tpr - fpr
        else:
            raise ValueError("Unknown metric")
        if s > best_s:
            best_s, best_t = s, t
    return float(best_t), float(best_s)

def evaluate_binary(y, p, threshold=0.5, ece_bins=15):
    y = np.asarray(y).astype(int)
    p = np.asarray(p).astype(float)
    p = np.clip(p, 1e-7, 1-1e-7)
    pred = (p >= threshold).astype(int)

    out = {}
    out["threshold"] = float(threshold)
    out["pr_auc"] = float(average_precision_score(y, p))
    out["roc_auc"] = float(roc_auc_score(y, p))
    out["acc"] = float(accuracy_score(y, pred))
    out["f1"] = float(f1_score(y, pred, zero_division=0))
    out["precision"] = float(precision_score(y, pred, zero_division=0))
    out["recall"] = float(recall_score(y, pred, zero_division=0))

    tn = int(((pred==0) & (y==0)).sum())
    fp = int(((pred==1) & (y==0)).sum())
    fn = int(((pred==0) & (y==1)).sum())
    tp = int(((pred==1) & (y==1)).sum())
    out.update({"tn":tn, "fp":fp, "fn":fn, "tp":tp})

    out["brier"] = float(np.mean((p - y)**2))
    out["ece"] = float(expected_calibration_error(y, p, n_bins=ece_bins))
    out["nll"] = float(log_loss(y, p, labels=[0,1]))
    return out

def save_table(df, name, index=False):
    path = os.path.join(DIRS["tables"], f"{name}.csv")
    df.to_csv(path, index=index)
    print("✅ Saved table:", path)
    return path

# ---------------------------
# 1b) Gate entropy schedule
# ---------------------------
def gate_entropy_torch(g, eps=1e-12):
    return -(g * (g + eps).log()).sum(dim=1).mean()

def gate_reg_coeff(epoch, warmup=2, flip_epoch=12, max_mag=0.02):
    if epoch < warmup:
        return 0.0
    if epoch < flip_epoch:
        t = (epoch - warmup) / max(1, (flip_epoch - warmup))
        return -max_mag * float(t)  # explore: maximize entropy
    t2 = min(1.0, (epoch - flip_epoch) / 10.0)
    return +max_mag * float(t2)    # sharpen: minimize entropy

# ---------------------------
# 2) Load data
# ---------------------------
assert os.path.exists(DATA_CSV), f"Missing: {DATA_CSV}"
df = pd.read_csv(DATA_CSV)

LABEL_COL = "stroke" if "stroke" in df.columns else df.columns[-1]
print("Label column:", LABEL_COL)

y_all = df[LABEL_COL].values.astype(int)

# ---------------------------
# 3) Feature typing (your v2)
# ---------------------------
bin_cols = ['gender','alcohol','smoke','sleep_disorder','Health_Insurance','diabetes','hypertension','high_cholesterol','Coronary_Heart_Disease']
cat_cols = ['Race','Marital_status']
cont_cols = [c for c in df.columns if c not in ([LABEL_COL] + bin_cols + cat_cols)]
print("Counts:", {"bin":len(bin_cols), "cat":len(cat_cols), "cont":len(cont_cols)})

# ---------------------------
# 4) Split
# ---------------------------
def make_split(seed):
    sss = StratifiedShuffleSplit(n_splits=1, test_size=0.20, random_state=seed)
    (tr_idx, te_idx) = next(sss.split(df, y_all))
    y_tr = y_all[tr_idx]
    sss2 = StratifiedShuffleSplit(n_splits=1, test_size=0.20, random_state=seed+1337)
    (tr2, va2) = next(sss2.split(np.zeros(len(tr_idx)), y_tr))
    tr2_idx = tr_idx[tr2]
    va_idx = tr_idx[va2]
    return tr2_idx, va_idx, te_idx

# ---------------------------
# 5) Tensor datasets
# ---------------------------
def to_long_tensor(x):  return torch.as_tensor(x, dtype=torch.long)
def to_float_tensor(x): return torch.as_tensor(x, dtype=torch.float32)

class TabDS(Dataset):
    def __init__(self, Xc, Xk, Xb, y):
        self.Xc = Xc; self.Xk = Xk; self.Xb = Xb; self.y = y
    def __len__(self): return len(self.y)
    def __getitem__(self, i):
        return self.Xc[i], self.Xk[i], self.Xb[i], self.y[i]

# ---------------------------
# 6) Preprocessing
# ---------------------------
def fit_preprocess(train_df):
    bmap = {}
    for c in bin_cols:
        vals = train_df[c].values
        uniq = np.unique(vals)
        if set(uniq.tolist()) == {0,1}:
            bmap[c] = ("01", None)
        elif set(uniq.tolist()) == {1,2}:
            bmap[c] = ("12", None)
        else:
            if len(uniq)==2:
                bmap[c] = ("map", {uniq[0]:0, uniq[1]:1})
            else:
                raise ValueError(f"Binary col {c} has >2 unique: {uniq}")

    cat_vocab = {}
    for c in cat_cols:
        uniq = pd.Series(train_df[c]).astype(int).unique()
        uniq = np.sort(uniq)
        cat_vocab[c] = {v:i for i,v in enumerate(uniq.tolist())}

    mu = train_df[cont_cols].values.astype(np.float32).mean(axis=0)
    sd = train_df[cont_cols].values.astype(np.float32).std(axis=0)
    sd = np.where(sd < 1e-6, 1.0, sd)
    return bmap, cat_vocab, mu, sd

def apply_preprocess(df_part, bmap, cat_vocab, mu, sd):
    Xb = np.zeros((len(df_part), len(bin_cols)), dtype=np.int64)
    for j,c in enumerate(bin_cols):
        vals = df_part[c].values
        mode, mapping = bmap[c]
        if mode == "01":
            Xb[:,j] = vals.astype(int)
        elif mode == "12":
            Xb[:,j] = (vals.astype(int) - 1)
        else:
            Xb[:,j] = np.vectorize(mapping.get)(vals)
        Xb[:,j] = np.clip(Xb[:,j], 0, 1)

    Xk = np.zeros((len(df_part), len(cat_cols)), dtype=np.int64)
    for j,c in enumerate(cat_cols):
        vocab = cat_vocab[c]
        vals = df_part[c].astype(int).values
        Xk[:,j] = np.array([vocab.get(v, 0) for v in vals], dtype=np.int64)

    Xc = df_part[cont_cols].values.astype(np.float32)
    Xc = (Xc - mu) / sd
    return Xc, Xk, Xb

# ---------------------------
# 7) Concept groups
# ---------------------------
cg_path = os.path.join(DIRS["logs"], "concept_groups.json")
if os.path.exists(cg_path):
    concept_groups = json.load(open(cg_path, "r"))
else:
    raise RuntimeError("Missing concept_groups.json in logs; please ensure it exists.")
print("Concept groups:", {k:len(v) for k,v in concept_groups.items()})

# ---------------------------
# 8) Model: WideDeep + Concept-Gated Residual
# ---------------------------
class WideDeep(nn.Module):
    def __init__(self, n_cont, cat_cardinalities, n_bin, d_embed=16, deep_hidden=64, deep_layers=2, dropout=0.1):
        super().__init__()
        self.cat_embeds = nn.ModuleList([nn.Embedding(card, d_embed) for card in cat_cardinalities])

        wide_in = n_cont + n_bin + len(cat_cardinalities)*d_embed
        self.wide = nn.Linear(wide_in, 1)

        layers = []
        din = wide_in
        for _ in range(deep_layers):
            layers += [nn.Linear(din, deep_hidden), nn.ReLU(), nn.Dropout(dropout)]
            din = deep_hidden
        layers += [nn.Linear(din, 1)]
        self.deep = nn.Sequential(*layers)

    def forward(self, x_cont, x_cat, x_bin):
        cat_vecs = [emb(x_cat[:, j]) for j, emb in enumerate(self.cat_embeds)]
        cat_vec = torch.cat(cat_vecs, dim=1) if len(cat_vecs) else torch.zeros((x_cont.size(0),0), device=x_cont.device)

        z = torch.cat([x_cont, x_bin.float(), cat_vec], dim=1)
        base = self.wide(z).squeeze(1) + self.deep(z).squeeze(1)
        return base, z

class ConceptGatedResidual(nn.Module):
    def __init__(self, z_dim, n_concepts, hidden=64, dropout=0.1):
        super().__init__()
        self.gate = nn.Sequential(
            nn.Linear(z_dim, hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, n_concepts)
        )
        self.experts = nn.ModuleList([
            nn.Sequential(
                nn.Linear(z_dim, hidden),
                nn.ReLU(),
                nn.Dropout(dropout),
                nn.Linear(hidden, 1)
            ) for _ in range(n_concepts)
        ])
        self.alpha = nn.Parameter(torch.tensor(-2.0))

    def forward(self, z):
        g = torch.softmax(self.gate(z), dim=1)
        E = torch.stack([ex(z).squeeze(1) for ex in self.experts], dim=1)
        res = (g * E).sum(dim=1)
        res = torch.sigmoid(self.alpha) * res
        return res, g, E

class CGWideDeep(nn.Module):
    def __init__(self, n_cont, cat_cardinalities, n_bin, concept_names,
                 d_embed=16, deep_hidden=64, deep_layers=2, dropout=0.1):
        super().__init__()
        self.base = WideDeep(n_cont, cat_cardinalities, n_bin, d_embed, deep_hidden, deep_layers, dropout)
        z_dim = n_cont + n_bin + len(cat_cardinalities)*d_embed
        self.cg = ConceptGatedResidual(z_dim=z_dim, n_concepts=len(concept_names), hidden=deep_hidden, dropout=dropout)

    def forward(self, x_cont, x_cat, x_bin):
        base_logit, z = self.base(x_cont, x_cat, x_bin)
        res, g, E = self.cg(z)
        return base_logit + res, g, E

# ---------------------------
# 9) Train / eval
# ---------------------------
@torch.no_grad()
def predict_proba(model, loader):
    model.eval()
    ps, ys, gs, es = [], [], [], []
    for xc, xk, xb, yb in loader:
        xc = xc.to(DEVICE); xk = xk.to(DEVICE); xb = xb.to(DEVICE)
        logit, g, E = model(xc, xk, xb)
        p = torch.sigmoid(logit).detach().cpu().numpy()
        ps.append(p)
        ys.append(yb.numpy())
        gs.append(g.detach().cpu().numpy())
        es.append(E.detach().cpu().numpy())
    return np.concatenate(ys), np.concatenate(ps), np.concatenate(gs), np.concatenate(es)

def train_one_repeat(r, concept_names, other_idx=None):
    tr_idx, va_idx, te_idx = make_split(cfg.seed + 1000*r)
    df_tr, df_va, df_te = df.iloc[tr_idx].copy(), df.iloc[va_idx].copy(), df.iloc[te_idx].copy()

    bmap, cat_vocab, mu, sd = fit_preprocess(df_tr)
    Xc_tr, Xk_tr, Xb_tr = apply_preprocess(df_tr, bmap, cat_vocab, mu, sd)
    Xc_va, Xk_va, Xb_va = apply_preprocess(df_va, bmap, cat_vocab, mu, sd)
    Xc_te, Xk_te, Xb_te = apply_preprocess(df_te, bmap, cat_vocab, mu, sd)

    y_tr = df_tr[LABEL_COL].values.astype(int)
    y_va = df_va[LABEL_COL].values.astype(int)
    y_te = df_te[LABEL_COL].values.astype(int)

    cat_cards = [len(cat_vocab[c]) for c in cat_cols]

    tr_ds = TabDS(to_float_tensor(Xc_tr), to_long_tensor(Xk_tr), to_long_tensor(Xb_tr), to_long_tensor(y_tr))
    va_ds = TabDS(to_float_tensor(Xc_va), to_long_tensor(Xk_va), to_long_tensor(Xb_va), to_long_tensor(y_va))
    te_ds = TabDS(to_float_tensor(Xc_te), to_long_tensor(Xk_te), to_long_tensor(Xb_te), to_long_tensor(y_te))

    # ✅ Balanced sampler (positive-aware batches)
    if cfg.use_balanced_sampler:
        y_tr_t = np.asarray(y_tr, dtype=np.int64)
        class_count = np.bincount(y_tr_t, minlength=2)
        # inverse frequency weights
        class_w = 1.0 / np.maximum(class_count, 1)
        sample_w = class_w[y_tr_t].astype(np.float64)
        sampler = WeightedRandomSampler(
            weights=torch.as_tensor(sample_w, dtype=torch.double),
            num_samples=len(sample_w),
            replacement=True
        )
        tr_loader = DataLoader(tr_ds, batch_size=cfg.batch_size, sampler=sampler, drop_last=False)
    else:
        tr_loader = DataLoader(tr_ds, batch_size=cfg.batch_size, shuffle=True, drop_last=False)

    va_loader = DataLoader(va_ds, batch_size=cfg.batch_size, shuffle=False)
    te_loader = DataLoader(te_ds, batch_size=cfg.batch_size, shuffle=False)

    model = CGWideDeep(
        n_cont=len(cont_cols),
        cat_cardinalities=cat_cards,
        n_bin=len(bin_cols),
        concept_names=concept_names,
        d_embed=cfg.d_embed,
        deep_hidden=cfg.deep_hidden,
        deep_layers=cfg.deep_layers,
        dropout=cfg.dropout
    ).to(DEVICE)

    opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)

    if cfg.use_pos_weight:
        pos = (y_tr==1).sum()
        neg = (y_tr==0).sum()
        posw = torch.tensor([neg/(pos+1e-12)], device=DEVICE, dtype=torch.float32)
        crit = nn.BCEWithLogitsLoss(pos_weight=posw)
    else:
        crit = nn.BCEWithLogitsLoss()

    best_pr, best_state, bad = -1.0, None, 0
    history = []

    for ep in range(cfg.max_epochs):
        model.train()
        ep_loss, n_seen = 0.0, 0

        c_ent_epoch = gate_reg_coeff(
            epoch=ep,
            warmup=cfg.gate_ent_warmup,
            flip_epoch=cfg.gate_ent_flip_epoch,
            max_mag=cfg.gate_ent_max_mag
        )

        last_ent = None

        for xc, xk, xb, yb in tr_loader:
            xc = xc.to(DEVICE); xk = xk.to(DEVICE); xb = xb.to(DEVICE)
            yb = yb.to(DEVICE).float()

            opt.zero_grad(set_to_none=True)
            logit, g, E = model(xc, xk, xb)

            loss = crit(logit, yb)

            ent = gate_entropy_torch(g)
            loss = loss + c_ent_epoch * ent
            last_ent = ent.detach()

            if (cfg.other_gate_penalty > 0) and (other_idx is not None):
                loss = loss + cfg.other_gate_penalty * g[:, other_idx].mean()

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()

            bs = yb.size(0)
            ep_loss += loss.item() * bs
            n_seen += bs

        y_va_np, p_va, _, _ = predict_proba(model, va_loader)
        val_pr = average_precision_score(y_va_np, p_va)

        ent_val = float(last_ent.cpu()) if last_ent is not None else float("nan")
        print(f"CG-WideDeep (EntSched+BalSampler) r{r} ep{ep:03d} "
              f"loss={ep_loss/max(n_seen,1):.4f} valPR={val_pr:.4f} "
              f"ent={ent_val:.4f} c_ent={c_ent_epoch:+.4f}")

        history.append({"epoch": ep, "loss": ep_loss/max(n_seen,1), "val_pr_auc": float(val_pr),
                        "c_ent": float(c_ent_epoch), "ent": ent_val})

        if val_pr > best_pr + 1e-5:
            best_pr = val_pr
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            bad = 0
        else:
            bad += 1
            if bad >= cfg.patience:
                break

    model.load_state_dict(best_state)

    y_va_np, p_va, g_va, e_va = predict_proba(model, va_loader)
    y_te_np, p_te, g_te, e_te = predict_proba(model, te_loader)

    t_best, _ = find_best_threshold(y_va_np, p_va, metric="f1")

    m_va = evaluate_binary(y_va_np, p_va, threshold=t_best, ece_bins=cfg.ece_bins)
    m_te = evaluate_binary(y_te_np, p_te, threshold=t_best, ece_bins=cfg.ece_bins)

    np.save(os.path.join(DIRS["pred"], f"cg_bal_gates_val_r{r}.npy"), g_va)
    np.save(os.path.join(DIRS["pred"], f"cg_bal_gates_test_r{r}.npy"), g_te)
    np.save(os.path.join(DIRS["pred"], f"cg_bal_expertlogits_val_r{r}.npy"), e_va)
    np.save(os.path.join(DIRS["pred"], f"cg_bal_expertlogits_test_r{r}.npy"), e_te)

    out_json = {"repeat": r, "t_best": float(t_best), "val_pr_auc_best": float(best_pr),
                "metrics_val": m_va, "metrics_test": m_te}
    with open(os.path.join(DIRS["pred"], f"cg_bal_summary_r{r}.json"), "w") as f:
        json.dump(out_json, f, indent=2)

    return m_va, m_te, history

# ---------------------------
# 10) Run repeats + tables
# ---------------------------
concept_names = list(concept_groups.keys())
other_idx = concept_names.index("Other") if "Other" in concept_names else None

rows, histories = [], []
for r in range(cfg.repeats):
    m_va, m_te, hist = train_one_repeat(r, concept_names, other_idx=other_idx)
    m_va.update({"model": "CG_WideDeep_EntSched_BalSampler", "repeat": r, "split": "val"})
    m_te.update({"model": "CG_WideDeep_EntSched_BalSampler", "repeat": r, "split": "test"})
    rows += [m_va, m_te]
    histories.append(hist)

df_m = pd.DataFrame(rows)
display(df_m)
save_table(df_m, "cg_widedeep_entsched_balsampler_per_repeat_val_test", index=False)

test_df = df_m[df_m["split"]=="test"].copy()
summary = test_df[["pr_auc","roc_auc","f1","recall","brier","ece","nll","acc"]].agg(["mean","std"]).T.reset_index()
summary.columns = ["metric","mean","std"]
display(summary)
save_table(summary, "cg_widedeep_entsched_balsampler_test_summary_meanstd", index=False)

print("\n✅ Run complete: Entropy Schedule + Balanced Sampler")

# ---------------------------
# 11) Gate dominance diagnostic
# ---------------------------
Gs = []
for r in range(cfg.repeats):
    p = os.path.join(DIRS["pred"], f"cg_bal_gates_test_r{r}.npy")
    if os.path.exists(p):
        Gs.append(np.load(p))
if len(Gs):
    G = np.concatenate(Gs, axis=0)
    mean_w = G.mean(axis=0)
    order = np.argsort(-mean_w)
    print("\n=== Gate dominance (TEST, mean gate weight) ===")
    for i in order:
        print(f"{concept_names[i]:20s} mean_gate={mean_w[i]:.4f}")
    ent = (-(G*np.log(G+1e-12)).sum(axis=1)).mean()
    print("Mean gate entropy:", float(ent))
else:
    print("No gates found to diagnose.")

Using device: cpu
Label column: stroke
Counts: {'bin': 9, 'cat': 2, 'cont': 24}
Concept groups: {'Demographics': 4, 'Lifestyle': 4, 'Comorbidity': 4, 'Nutrition_Macro': 6, 'Nutrition_FattyAcids': 3, 'Nutrition_Minerals': 2, 'Other': 12}
CG-WideDeep (EntSched+BalSampler) r0 ep000 loss=4.3334 valPR=0.1062 ent=1.7815 c_ent=+0.0000
CG-WideDeep (EntSched+BalSampler) r0 ep001 loss=1.7485 valPR=0.1497 ent=1.5596 c_ent=+0.0000
CG-WideDeep (EntSched+BalSampler) r0 ep002 loss=1.6022 valPR=0.1868 ent=1.5499 c_ent=-0.0000
CG-WideDeep (EntSched+BalSampler) r0 ep003 loss=1.5135 valPR=0.1952 ent=1.5686 c_ent=-0.0020
CG-WideDeep (EntSched+BalSampler) r0 ep004 loss=1.4583 valPR=0.1906 ent=1.5348 c_ent=-0.0040
CG-WideDeep (EntSched+BalSampler) r0 ep005 loss=1.4166 valPR=0.2040 ent=1.3758 c_ent=-0.0060
CG-WideDeep (EntSched+BalSampler) r0 ep006 loss=1.3557 valPR=0.1849 ent=1.3220 c_ent=-0.0080
CG-WideDeep (EntSched+BalSampler) r0 ep007 loss=1.2532 valPR=0.1705 ent=1.3938 c_ent=-0.0100
CG-WideDeep (EntSch

,threshold,pr_auc,roc_auc,acc,f1,precision,recall,tn,fp,fn,tp,brier,ece,nll,model,repeat,split
0,0.950,0.204028,0.747677,0.808684,0.298507,0.209790,0.517241,566,113,28,30,0.606948,0.720491,1.834801,CG_WideDeep_EntSched_BalSampler,0,val
1,0.950,0.128025,0.606637,0.761129,0.160305,0.110526,0.291667,680,169,51,21,0.609149,0.713002,1.868316,CG_WideDeep_EntSched_BalSampler,0,test
2,0.940,0.155551,0.657128,0.766621,0.225225,0.152439,0.431034,540,139,33,25,0.446064,0.525547,1.475480,CG_WideDeep_EntSched_BalSampler,1,val
3,0.940,0.179332,0.732136,0.779587,0.250923,0.170854,0.472222,684,165,38,34,0.415373,0.493979,1.390358,CG_WideDeep_EntSched_BalSampler,1,test
4,0.950,0.149031,0.698822,0.751696,0.234310,0.154696,0.482759,526,153,30,28,0.673919,0.771340,2.091478,CG_WideDeep_EntSched_BalSampler,2,val
5,0.950,0.207551,0.721911,0.755700,0.257426,0.168831,0.541667,657,192,33,39,0.678145,0.774779,2.093112,CG_WideDeep_EntSched_BalSampler,2,test
6,0.945,0.169692,0.673544,0.757123,0.225108,0.150289,0.448276,532,147,32,26,0.621137,0.730083,1.902051,CG_WideDeep_EntSched_BalSampler,3,val
7,0.945,0.197989,0.681995,0.749186,0.222222,0.146667,0.458333,657,192,39,33,0.642348,0.747421,1.974829,CG_WideDeep_EntSched_BalSampler,3,test
8,0.935,0.216380,0.711696,0.757123,0.281124,0.183246,0.603448,523,156,23,35,0.613617,0.726742,1.840784,CG_WideDeep_EntSched_BalSampler,4,val
9,0.935,0.133538,0.670822,0.750271,0.201389,0.134259,0.402778,662,187,43,29,0.611038,0.725588,1.833288,CG_WideDeep_EntSched_BalSampler,4,test


✅ Saved table: /kaggle/working/outputs/cgt_stroke_v1/tables/cg_widedeep_entsched_balsampler_per_repeat_val_test.csv


,metric,mean,std
0,pr_auc,0.169287,0.036638
1,roc_auc,0.682700,0.049771
2,f1,0.218453,0.039559
3,recall,0.433333,0.093376
4,brier,0.591211,0.102233
5,ece,0.690954,0.112583
6,nll,1.831981,0.266903
7,acc,0.759175,0.012365


✅ Saved table: /kaggle/working/outputs/cgt_stroke_v1/tables/cg_widedeep_entsched_balsampler_test_summary_meanstd.csv

✅ Run complete: Entropy Schedule + Balanced Sampler

=== Gate dominance (TEST, mean gate weight) ===
Nutrition_FattyAcids mean_gate=0.1886
Comorbidity          mean_gate=0.1874
Nutrition_Macro      mean_gate=0.1605
Other                mean_gate=0.1604
Nutrition_Minerals   mean_gate=0.1503
Lifestyle            mean_gate=0.0830
Demographics         mean_gate=0.0698
Mean gate entropy: 1.227538824081421


In [24]:
# ============================================================
# MULTI-VARIANT RUNNER: CG-WideDeep (FIX) Ablations
# Compares multiple improvements in ONE run:
#   - Entropy schedule
#   - Balanced sampler
#   - pos_weight on/off
#   - Calibration: None / Platt / Temperature / Isotonic
#
# Outputs:
#   - per-variant per-repeat val/test table
#   - per-variant test mean±std summary
#
# Expects:
#   /kaggle/working/outputs/cgt_stroke_v1/data_clean_v2.csv
# ============================================================

import os, json, random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, precision_score, recall_score, accuracy_score, log_loss
from sklearn.linear_model import LogisticRegression
from sklearn.isotonic import IsotonicRegression

# ---------------------------
# 0) Paths / Global Config
# ---------------------------
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)

BASE_DIR = "/kaggle/working/outputs/cgt_stroke_v1"
DATA_CSV  = os.path.join(BASE_DIR, "data_clean_v2.csv")

OUT_DIR = os.path.join(BASE_DIR, "ablation_runs")
os.makedirs(OUT_DIR, exist_ok=True)
for sub in ["tables", "predictions"]:
    os.makedirs(os.path.join(OUT_DIR, sub), exist_ok=True)

class CFG:
    seed = 42
    repeats = 5

    # training
    batch_size = 256
    lr = 2e-3
    weight_decay = 1e-4
    max_epochs = 60
    patience = 10

    # calibration metrics
    ece_bins = 15

    # Entropy schedule
    gate_ent_warmup = 2
    gate_ent_flip_epoch = 12
    gate_ent_max_mag = 0.02

    # optional (keep off for ablation)
    other_gate_penalty = 0.00

    # Model sizes
    d_embed = 16
    deep_hidden = 64
    deep_layers = 2
    dropout = 0.10

cfg = CFG()

def seed_everything(seed=42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(cfg.seed)

# ---------------------------
# 1) Metrics + helpers
# ---------------------------
def expected_calibration_error(y_true, p, n_bins=15):
    y_true = np.asarray(y_true).astype(int)
    p = np.asarray(p).astype(float)
    bins = np.linspace(0.0, 1.0, n_bins+1)
    ece = 0.0
    for i in range(n_bins):
        lo, hi = bins[i], bins[i+1]
        m = (p >= lo) & (p < hi) if i < n_bins-1 else (p >= lo) & (p <= hi)
        if m.sum() == 0:
            continue
        acc = y_true[m].mean()
        conf = p[m].mean()
        ece += (m.sum()/len(p)) * abs(acc - conf)
    return float(ece)

def find_best_threshold(y, p, metric="f1"):
    y = np.asarray(y).astype(int)
    p = np.asarray(p).astype(float)
    ts = np.linspace(0.05, 0.95, 181)
    best_t, best_s = 0.5, -1
    for t in ts:
        pred = (p >= t).astype(int)
        s = f1_score(y, pred, zero_division=0)
        if s > best_s:
            best_s, best_t = s, t
    return float(best_t), float(best_s)

def evaluate_binary(y, p, threshold=0.5, ece_bins=15):
    y = np.asarray(y).astype(int)
    p = np.asarray(p).astype(float)
    p = np.clip(p, 1e-7, 1-1e-7)
    pred = (p >= threshold).astype(int)

    out = {}
    out["threshold"]  = float(threshold)
    out["pr_auc"]     = float(average_precision_score(y, p))
    out["roc_auc"]    = float(roc_auc_score(y, p))
    out["acc"]        = float(accuracy_score(y, pred))
    out["f1"]         = float(f1_score(y, pred, zero_division=0))
    out["precision"]  = float(precision_score(y, pred, zero_division=0))
    out["recall"]     = float(recall_score(y, pred, zero_division=0))

    tn = int(((pred==0) & (y==0)).sum())
    fp = int(((pred==1) & (y==0)).sum())
    fn = int(((pred==0) & (y==1)).sum())
    tp = int(((pred==1) & (y==1)).sum())
    out.update({"tn":tn, "fp":fp, "fn":fn, "tp":tp})

    out["brier"] = float(np.mean((p - y)**2))
    out["ece"]   = float(expected_calibration_error(y, p, n_bins=ece_bins))
    out["nll"]   = float(log_loss(y, p, labels=[0,1]))
    return out

def gate_entropy_torch(g, eps=1e-12):
    return -(g * (g + eps).log()).sum(dim=1).mean()

def gate_reg_coeff(epoch, warmup=2, flip_epoch=12, max_mag=0.02):
    if epoch < warmup:
        return 0.0
    if epoch < flip_epoch:
        t = (epoch - warmup) / max(1, (flip_epoch - warmup))
        return -max_mag * float(t)  # explore
    t2 = min(1.0, (epoch - flip_epoch) / 10.0)
    return +max_mag * float(t2)    # sharpen

def save_csv(df, name):
    path = os.path.join(OUT_DIR, "tables", f"{name}.csv")
    df.to_csv(path, index=False)
    print("✅ Saved:", path)
    return path

# ---------------------------
# 2) Load data + columns
# ---------------------------
assert os.path.exists(DATA_CSV), f"Missing: {DATA_CSV}"
df = pd.read_csv(DATA_CSV)
LABEL_COL = "stroke" if "stroke" in df.columns else df.columns[-1]
print("Label column:", LABEL_COL)
y_all = df[LABEL_COL].values.astype(int)

bin_cols = ['gender','alcohol','smoke','sleep_disorder','Health_Insurance','diabetes','hypertension','high_cholesterol','Coronary_Heart_Disease']
cat_cols = ['Race','Marital_status']
cont_cols = [c for c in df.columns if c not in ([LABEL_COL] + bin_cols + cat_cols)]
print("Counts:", {"bin":len(bin_cols), "cat":len(cat_cols), "cont":len(cont_cols)})

# concept groups
cg_path = os.path.join(BASE_DIR, "logs", "concept_groups.json")
assert os.path.exists(cg_path), "Missing concept_groups.json at outputs/cgt_stroke_v1/logs/"
concept_groups = json.load(open(cg_path, "r"))
concept_names = list(concept_groups.keys())
other_idx = concept_names.index("Other") if "Other" in concept_names else None
print("Concept groups:", {k:len(v) for k,v in concept_groups.items()})

# ---------------------------
# 3) Split
# ---------------------------
def make_split(seed):
    sss = StratifiedShuffleSplit(n_splits=1, test_size=0.20, random_state=seed)
    (tr_idx, te_idx) = next(sss.split(df, y_all))
    y_tr = y_all[tr_idx]
    sss2 = StratifiedShuffleSplit(n_splits=1, test_size=0.20, random_state=seed+1337)
    (tr2, va2) = next(sss2.split(np.zeros(len(tr_idx)), y_tr))
    tr2_idx = tr_idx[tr2]
    va_idx  = tr_idx[va2]
    return tr2_idx, va_idx, te_idx

# ---------------------------
# 4) Dataset + preprocessing
# ---------------------------
def to_long_tensor(x):  return torch.as_tensor(x, dtype=torch.long)
def to_float_tensor(x): return torch.as_tensor(x, dtype=torch.float32)

class TabDS(Dataset):
    def __init__(self, Xc, Xk, Xb, y):
        self.Xc = Xc; self.Xk = Xk; self.Xb = Xb; self.y = y
    def __len__(self): return len(self.y)
    def __getitem__(self, i):
        return self.Xc[i], self.Xk[i], self.Xb[i], self.y[i]

def fit_preprocess(train_df):
    bmap = {}
    for c in bin_cols:
        vals = train_df[c].values
        uniq = np.unique(vals)
        if set(uniq.tolist()) == {0,1}:
            bmap[c] = ("01", None)
        elif set(uniq.tolist()) == {1,2}:
            bmap[c] = ("12", None)
        else:
            if len(uniq)==2:
                bmap[c] = ("map", {uniq[0]:0, uniq[1]:1})
            else:
                raise ValueError(f"Binary col {c} has >2 unique: {uniq}")

    cat_vocab = {}
    for c in cat_cols:
        uniq = pd.Series(train_df[c]).astype(int).unique()
        uniq = np.sort(uniq)
        cat_vocab[c] = {v:i for i,v in enumerate(uniq.tolist())}

    mu = train_df[cont_cols].values.astype(np.float32).mean(axis=0)
    sd = train_df[cont_cols].values.astype(np.float32).std(axis=0)
    sd = np.where(sd < 1e-6, 1.0, sd)
    return bmap, cat_vocab, mu, sd

def apply_preprocess(df_part, bmap, cat_vocab, mu, sd):
    Xb = np.zeros((len(df_part), len(bin_cols)), dtype=np.int64)
    for j,c in enumerate(bin_cols):
        vals = df_part[c].values
        mode, mapping = bmap[c]
        if mode == "01":
            Xb[:,j] = vals.astype(int)
        elif mode == "12":
            Xb[:,j] = (vals.astype(int) - 1)
        else:
            Xb[:,j] = np.vectorize(mapping.get)(vals)
        Xb[:,j] = np.clip(Xb[:,j], 0, 1)

    Xk = np.zeros((len(df_part), len(cat_cols)), dtype=np.int64)
    for j,c in enumerate(cat_cols):
        vocab = cat_vocab[c]
        vals = df_part[c].astype(int).values
        Xk[:,j] = np.array([vocab.get(v, 0) for v in vals], dtype=np.int64)

    Xc = df_part[cont_cols].values.astype(np.float32)
    Xc = (Xc - mu) / sd
    return Xc, Xk, Xb

# ---------------------------
# 5) Model
# ---------------------------
class WideDeep(nn.Module):
    def __init__(self, n_cont, cat_cardinalities, n_bin, d_embed=16, deep_hidden=64, deep_layers=2, dropout=0.1):
        super().__init__()
        self.cat_embeds = nn.ModuleList([nn.Embedding(card, d_embed) for card in cat_cardinalities])
        wide_in = n_cont + n_bin + len(cat_cardinalities)*d_embed
        self.wide = nn.Linear(wide_in, 1)

        layers = []
        din = wide_in
        for _ in range(deep_layers):
            layers += [nn.Linear(din, deep_hidden), nn.ReLU(), nn.Dropout(dropout)]
            din = deep_hidden
        layers += [nn.Linear(din, 1)]
        self.deep = nn.Sequential(*layers)

    def forward(self, x_cont, x_cat, x_bin):
        cat_vecs = [emb(x_cat[:, j]) for j, emb in enumerate(self.cat_embeds)]
        cat_vec = torch.cat(cat_vecs, dim=1) if len(cat_vecs) else torch.zeros((x_cont.size(0),0), device=x_cont.device)
        z = torch.cat([x_cont, x_bin.float(), cat_vec], dim=1)
        base = self.wide(z).squeeze(1) + self.deep(z).squeeze(1)
        return base, z

class ConceptGatedResidual(nn.Module):
    def __init__(self, z_dim, n_concepts, hidden=64, dropout=0.1):
        super().__init__()
        self.gate = nn.Sequential(
            nn.Linear(z_dim, hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, n_concepts)
        )
        self.experts = nn.ModuleList([
            nn.Sequential(
                nn.Linear(z_dim, hidden),
                nn.ReLU(),
                nn.Dropout(dropout),
                nn.Linear(hidden, 1)
            ) for _ in range(n_concepts)
        ])
        self.alpha = nn.Parameter(torch.tensor(-2.0))

    def forward(self, z):
        g = torch.softmax(self.gate(z), dim=1)
        E = torch.stack([ex(z).squeeze(1) for ex in self.experts], dim=1)
        res = (g * E).sum(dim=1)
        res = torch.sigmoid(self.alpha) * res
        return res, g, E

class CGWideDeep(nn.Module):
    def __init__(self, n_cont, cat_cardinalities, n_bin, concept_names,
                 d_embed=16, deep_hidden=64, deep_layers=2, dropout=0.1):
        super().__init__()
        self.base = WideDeep(n_cont, cat_cardinalities, n_bin, d_embed, deep_hidden, deep_layers, dropout)
        z_dim = n_cont + n_bin + len(cat_cardinalities)*d_embed
        self.cg = ConceptGatedResidual(z_dim=z_dim, n_concepts=len(concept_names), hidden=deep_hidden, dropout=dropout)

    def forward(self, x_cont, x_cat, x_bin):
        base_logit, z = self.base(x_cont, x_cat, x_bin)
        res, g, E = self.cg(z)
        return base_logit + res, g, E

@torch.no_grad()
def predict_probs_logits(model, loader):
    model.eval()
    ys, ps, ls = [], [], []
    gs, es = [], []
    for xc, xk, xb, yb in loader:
        xc = xc.to(DEVICE); xk = xk.to(DEVICE); xb = xb.to(DEVICE)
        logit, g, E = model(xc, xk, xb)
        p = torch.sigmoid(logit).detach().cpu().numpy()
        ys.append(yb.numpy()); ps.append(p); ls.append(logit.detach().cpu().numpy())
        gs.append(g.detach().cpu().numpy()); es.append(E.detach().cpu().numpy())
    return np.concatenate(ys), np.concatenate(ps), np.concatenate(ls), np.concatenate(gs), np.concatenate(es)

# ---------------------------
# 6) Calibration modules
# ---------------------------
def cal_none(logits_val, y_val, logits_test):
    p_val = 1/(1+np.exp(-logits_val))
    p_te  = 1/(1+np.exp(-logits_test))
    return p_val, p_te

def cal_platt(logits_val, y_val, logits_test):
    lr = LogisticRegression(solver="lbfgs", max_iter=200)
    lr.fit(logits_val.reshape(-1,1), y_val.astype(int))
    p_val = lr.predict_proba(logits_val.reshape(-1,1))[:,1]
    p_te  = lr.predict_proba(logits_test.reshape(-1,1))[:,1]
    return p_val, p_te

def cal_temperature(logits_val, y_val, logits_test):
    # fit temperature T by minimizing NLL on val (simple 1D grid)
    y = y_val.astype(int)
    Ts = np.linspace(0.5, 5.0, 91)
    bestT, bestN = 1.0, 1e18
    for T in Ts:
        p = 1/(1+np.exp(-(logits_val / T)))
        p = np.clip(p, 1e-7, 1-1e-7)
        nll = log_loss(y, p, labels=[0,1])
        if nll < bestN:
            bestN, bestT = nll, T
    p_val = 1/(1+np.exp(-(logits_val / bestT)))
    p_te  = 1/(1+np.exp(-(logits_test / bestT)))
    return p_val, p_te

def cal_isotonic(logits_val, y_val, logits_test):
    p_val0 = 1/(1+np.exp(-logits_val))
    p_te0  = 1/(1+np.exp(-logits_test))
    iso = IsotonicRegression(out_of_bounds="clip")
    iso.fit(p_val0, y_val.astype(int))
    p_val = iso.transform(p_val0)
    p_te  = iso.transform(p_te0)
    return p_val, p_te

CALIBRATORS = {
    "none": cal_none,
    "platt": cal_platt,
    "temp": cal_temperature,
    "iso": cal_isotonic
}

# ---------------------------
# 7) Variant definitions (what to compare)
# ---------------------------
VARIANTS = [
    # name, balanced_sampler, use_pos_weight, calibrator
    ("BASE_EntSched",              False, True,  "none"),
    ("BAL_EntSched",               True,  True,  "none"),
    ("BAL_NoPosW",                 True,  False, "none"),
    ("BAL_NoPosW_Platt",           True,  False, "platt"),
    ("BAL_NoPosW_Temp",            True,  False, "temp"),
    ("BAL_NoPosW_Iso",             True,  False, "iso"),
]

# ---------------------------
# 8) Train one repeat for one variant
# ---------------------------
def make_train_loader(tr_ds, y_tr, balanced_sampler, batch_size):
    if not balanced_sampler:
        return DataLoader(tr_ds, batch_size=batch_size, shuffle=True, drop_last=False)
    y_tr = np.asarray(y_tr, dtype=np.int64)
    class_count = np.bincount(y_tr, minlength=2)
    class_w = 1.0 / np.maximum(class_count, 1)
    sample_w = class_w[y_tr].astype(np.float64)
    sampler = WeightedRandomSampler(
        weights=torch.as_tensor(sample_w, dtype=torch.double),
        num_samples=len(sample_w),
        replacement=True
    )
    return DataLoader(tr_ds, batch_size=batch_size, sampler=sampler, drop_last=False)

def train_eval_variant_repeat(variant_name, balanced_sampler, use_pos_weight, cal_name, r):
    tr_idx, va_idx, te_idx = make_split(cfg.seed + 1000*r)

    df_tr, df_va, df_te = df.iloc[tr_idx].copy(), df.iloc[va_idx].copy(), df.iloc[te_idx].copy()
    bmap, cat_vocab, mu, sd = fit_preprocess(df_tr)

    Xc_tr, Xk_tr, Xb_tr = apply_preprocess(df_tr, bmap, cat_vocab, mu, sd)
    Xc_va, Xk_va, Xb_va = apply_preprocess(df_va, bmap, cat_vocab, mu, sd)
    Xc_te, Xk_te, Xb_te = apply_preprocess(df_te, bmap, cat_vocab, mu, sd)

    y_tr = df_tr[LABEL_COL].values.astype(int)
    y_va = df_va[LABEL_COL].values.astype(int)
    y_te = df_te[LABEL_COL].values.astype(int)

    cat_cards = [len(cat_vocab[c]) for c in cat_cols]

    tr_ds = TabDS(to_float_tensor(Xc_tr), to_long_tensor(Xk_tr), to_long_tensor(Xb_tr), to_long_tensor(y_tr))
    va_ds = TabDS(to_float_tensor(Xc_va), to_long_tensor(Xk_va), to_long_tensor(Xb_va), to_long_tensor(y_va))
    te_ds = TabDS(to_float_tensor(Xc_te), to_long_tensor(Xk_te), to_long_tensor(Xb_te), to_long_tensor(y_te))

    tr_loader = make_train_loader(tr_ds, y_tr, balanced_sampler, cfg.batch_size)
    va_loader = DataLoader(va_ds, batch_size=cfg.batch_size, shuffle=False)
    te_loader = DataLoader(te_ds, batch_size=cfg.batch_size, shuffle=False)

    model = CGWideDeep(
        n_cont=len(cont_cols),
        cat_cardinalities=cat_cards,
        n_bin=len(bin_cols),
        concept_names=concept_names,
        d_embed=cfg.d_embed,
        deep_hidden=cfg.deep_hidden,
        deep_layers=cfg.deep_layers,
        dropout=cfg.dropout
    ).to(DEVICE)

    opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)

    if use_pos_weight:
        pos = (y_tr==1).sum()
        neg = (y_tr==0).sum()
        posw = torch.tensor([neg/(pos+1e-12)], device=DEVICE, dtype=torch.float32)
        crit = nn.BCEWithLogitsLoss(pos_weight=posw)
    else:
        crit = nn.BCEWithLogitsLoss()

    best_pr, best_state, bad = -1.0, None, 0

    for ep in range(cfg.max_epochs):
        model.train()
        ep_loss, n_seen = 0.0, 0

        c_ent = gate_reg_coeff(ep, cfg.gate_ent_warmup, cfg.gate_ent_flip_epoch, cfg.gate_ent_max_mag)

        for xc, xk, xb, yb in tr_loader:
            xc = xc.to(DEVICE); xk = xk.to(DEVICE); xb = xb.to(DEVICE)
            yb = yb.to(DEVICE).float()

            opt.zero_grad(set_to_none=True)
            logit, g, E = model(xc, xk, xb)

            loss = crit(logit, yb)
            loss = loss + c_ent * gate_entropy_torch(g)

            if (cfg.other_gate_penalty > 0) and (other_idx is not None):
                loss = loss + cfg.other_gate_penalty * g[:, other_idx].mean()

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()

            bs = yb.size(0)
            ep_loss += loss.item() * bs
            n_seen += bs

        # early stop by VAL PR-AUC (uncalibrated is fine for early stop)
        yv, pv, lv, _, _ = predict_probs_logits(model, va_loader)
        val_pr = average_precision_score(yv, pv)

        if val_pr > best_pr + 1e-5:
            best_pr = val_pr
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            bad = 0
        else:
            bad += 1
            if bad >= cfg.patience:
                break

    model.load_state_dict(best_state)

    # get logits for calibration + gates for diagnostics
    yv, pv, lv, gv, ev = predict_probs_logits(model, va_loader)
    yt, pt, lt, gt, et = predict_probs_logits(model, te_loader)

    # calibration
    cal_fn = CALIBRATORS[cal_name]
    pv_cal, pt_cal = cal_fn(lv, yv, lt)

    # threshold on calibrated val
    t_best, _ = find_best_threshold(yv, pv_cal, metric="f1")

    m_va = evaluate_binary(yv, pv_cal, threshold=t_best, ece_bins=cfg.ece_bins)
    m_te = evaluate_binary(yt, pt_cal, threshold=t_best, ece_bins=cfg.ece_bins)

    # save gates (optional)
    np.save(os.path.join(OUT_DIR, "predictions", f"{variant_name}_gates_test_r{r}.npy"), gt)

    return m_va, m_te

# ---------------------------
# 9) Run all variants
# ---------------------------
rows = []
for (vname, bal, posw, cal) in VARIANTS:
    print("\n" + "="*80)
    print(f"VARIANT: {vname} | balanced={bal} | pos_weight={posw} | cal={cal}")
    print("="*80)

    for r in range(cfg.repeats):
        m_va, m_te = train_eval_variant_repeat(vname, bal, posw, cal, r)

        m_va.update({"variant": vname, "repeat": r, "split": "val",  "balanced": bal, "pos_weight": posw, "cal": cal})
        m_te.update({"variant": vname, "repeat": r, "split": "test", "balanced": bal, "pos_weight": posw, "cal": cal})

        rows += [m_va, m_te]
        print(f"  r{r} TEST pr_auc={m_te['pr_auc']:.4f} roc_auc={m_te['roc_auc']:.4f} "
              f"brier={m_te['brier']:.4f} ece={m_te['ece']:.4f} nll={m_te['nll']:.4f}")

df_all = pd.DataFrame(rows)
display(df_all)
save_csv(df_all, "ablation_all_variants_per_repeat_val_test")

# summary per variant (TEST only)
test_df = df_all[df_all["split"]=="test"].copy()
summary = test_df.groupby("variant")[["pr_auc","roc_auc","f1","recall","brier","ece","nll","acc"]].agg(["mean","std"])
summary.columns = [f"{a}_{b}" for a,b in summary.columns]
summary = summary.reset_index().sort_values("pr_auc_mean", ascending=False)
display(summary)
save_csv(summary, "ablation_test_summary_by_variant")

print("\n✅ Ablation complete. Check tables in:", os.path.join(OUT_DIR, "tables"))

Using device: cpu
Label column: stroke
Counts: {'bin': 9, 'cat': 2, 'cont': 24}
Concept groups: {'Demographics': 4, 'Lifestyle': 4, 'Comorbidity': 4, 'Nutrition_Macro': 6, 'Nutrition_FattyAcids': 3, 'Nutrition_Minerals': 2, 'Other': 12}

VARIANT: BASE_EntSched | balanced=False | pos_weight=True | cal=none
  r0 TEST pr_auc=0.1114 roc_auc=0.5979 brier=0.1712 ece=0.2018 nll=0.5945
  r1 TEST pr_auc=0.1335 roc_auc=0.6765 brier=0.1486 ece=0.1765 nll=0.4666
  r2 TEST pr_auc=0.1473 roc_auc=0.6791 brier=0.1682 ece=0.2254 nll=0.5010
  r3 TEST pr_auc=0.2114 roc_auc=0.7059 brier=0.2147 ece=0.3440 nll=0.6158
  r4 TEST pr_auc=0.1501 roc_auc=0.6375 brier=0.2198 ece=0.3785 nll=0.6307

VARIANT: BAL_EntSched | balanced=True | pos_weight=True | cal=none
  r0 TEST pr_auc=0.1081 roc_auc=0.6001 brier=0.2027 ece=0.2247 nll=0.9940
  r1 TEST pr_auc=0.1481 roc_auc=0.7054 brier=0.3018 ece=0.3651 nll=0.9701
  r2 TEST pr_auc=0.2121 roc_auc=0.7305 brier=0.6405 ece=0.7490 nll=1.9250
  r3 TEST pr_auc=0.1627 roc_auc=0

,threshold,pr_auc,roc_auc,acc,f1,precision,recall,tn,fp,fn,tp,brier,ece,nll,variant,repeat,split,balanced,pos_weight,cal
0,0.640,0.212296,0.691179,0.850746,0.303797,0.240000,0.413793,603,76,34,24,0.140703,1.640010e-01,0.482507,BASE_EntSched,0,val,False,True,none
1,0.640,0.111389,0.597893,0.800217,0.155963,0.116438,0.236111,720,129,55,17,0.171223,2.018048e-01,0.594491,BASE_EntSched,0,test,False,True,none
2,0.560,0.184380,0.616881,0.793758,0.232323,0.164286,0.396552,562,117,35,23,0.164533,2.188933e-01,0.527449,BASE_EntSched,1,val,False,True,none
3,0.560,0.133506,0.676548,0.805646,0.197309,0.145695,0.305556,720,129,50,22,0.148649,1.764674e-01,0.466602,BASE_EntSched,1,test,False,True,none
4,0.680,0.165138,0.705221,0.860244,0.237037,0.207792,0.275862,618,61,42,16,0.160974,2.203563e-01,0.474936,BASE_EntSched,2,val,False,True,none
5,0.680,0.147275,0.679051,0.839305,0.229167,0.183333,0.305556,751,98,50,22,0.168155,2.254060e-01,0.500993,BASE_EntSched,2,test,False,True,none
6,0.620,0.141177,0.674217,0.784261,0.224390,0.156463,0.396552,555,124,35,23,0.203497,3.203652e-01,0.591097,BASE_EntSched,3,val,False,True,none
7,0.620,0.211403,0.705863,0.763301,0.243056,0.162037,0.486111,668,181,37,35,0.214711,3.440325e-01,0.615757,BASE_EntSched,3,test,False,True,none
8,0.675,0.209275,0.649738,0.911805,0.252874,0.379310,0.189655,661,18,47,11,0.224237,3.860973e-01,0.639447,BASE_EntSched,4,val,False,True,none
9,0.675,0.150135,0.637531,0.908795,0.142857,0.269231,0.097222,830,19,65,7,0.219843,3.784882e-01,0.630728,BASE_EntSched,4,test,False,True,none


✅ Saved: /kaggle/working/outputs/cgt_stroke_v1/ablation_runs/tables/ablation_all_variants_per_repeat_val_test.csv


,variant,pr_auc_mean,pr_auc_std,roc_auc_mean,roc_auc_std,f1_mean,f1_std,recall_mean,recall_std,brier_mean,brier_std,ece_mean,ece_std,nll_mean,nll_std,acc_mean,acc_std
4,BAL_NoPosW_Temp,0.161491,0.024517,0.672883,0.040336,0.210778,0.020910,0.405556,0.140064,0.200002,0.027260,0.317206,0.060769,0.578841,0.064057,0.763518,0.073582
3,BAL_NoPosW_Platt,0.150805,0.043069,0.668388,0.049787,0.192667,0.052456,0.394444,0.198762,0.070411,0.001609,0.014269,0.003319,0.264036,0.009760,0.751574,0.080777
5,BASE_EntSched,0.150741,0.037208,0.659377,0.042132,0.193670,0.043920,0.286111,0.140477,0.184516,0.031187,0.265240,0.090173,0.561714,0.073299,0.823453,0.054795
1,BAL_NoPosW,0.150194,0.022683,0.679073,0.039916,0.185853,0.039637,0.252778,0.133723,0.209551,0.047689,0.315724,0.103675,0.598363,0.115346,0.834745,0.048444
0,BAL_EntSched,0.148254,0.042844,0.657748,0.062740,0.206114,0.032651,0.419444,0.114699,0.533430,0.264533,0.599636,0.284792,1.735364,0.724744,0.739631,0.090837
2,BAL_NoPosW_Iso,0.131882,0.031353,0.655747,0.048255,0.170737,0.062082,0.341667,0.222569,0.073073,0.004393,0.022961,0.009792,0.293928,0.052909,0.769381,0.095093


✅ Saved: /kaggle/working/outputs/cgt_stroke_v1/ablation_runs/tables/ablation_test_summary_by_variant.csv

✅ Ablation complete. Check tables in: /kaggle/working/outputs/cgt_stroke_v1/ablation_runs/tables


In [25]:
# ============================================================
# Robust calibration study (no leakage):
# Training recipe fixed: BALANCED + NoPosW + EntSched
# Then do: VAL -> (CalFit, CalEval) for calibration + threshold
# Report metrics on TEST only
# ============================================================

import os, json, random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, precision_score, recall_score, accuracy_score, log_loss
from sklearn.linear_model import LogisticRegression
from sklearn.isotonic import IsotonicRegression

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BASE_DIR = "/kaggle/working/outputs/cgt_stroke_v1"
DATA_CSV = os.path.join(BASE_DIR, "data_clean_v2.csv")

OUT_DIR = os.path.join(BASE_DIR, "robust_calibration")
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(os.path.join(OUT_DIR, "tables"), exist_ok=True)

class CFG:
    seed = 42
    repeats = 5

    batch_size = 256
    lr = 2e-3
    weight_decay = 1e-4
    max_epochs = 60
    patience = 10

    ece_bins = 15
    gate_ent_warmup = 2
    gate_ent_flip_epoch = 12
    gate_ent_max_mag = 0.02

    d_embed = 16
    deep_hidden = 64
    deep_layers = 2
    dropout = 0.10

cfg = CFG()

def seed_everything(seed=42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(cfg.seed)

# ---------------------------
# Metrics
# ---------------------------
def expected_calibration_error(y_true, p, n_bins=15):
    y_true = np.asarray(y_true).astype(int)
    p = np.asarray(p).astype(float)
    bins = np.linspace(0.0, 1.0, n_bins+1)
    ece = 0.0
    for i in range(n_bins):
        lo, hi = bins[i], bins[i+1]
        m = (p >= lo) & (p < hi) if i < n_bins-1 else (p >= lo) & (p <= hi)
        if m.sum() == 0:
            continue
        acc = y_true[m].mean()
        conf = p[m].mean()
        ece += (m.sum()/len(p)) * abs(acc - conf)
    return float(ece)

def find_best_threshold(y, p):
    y = np.asarray(y).astype(int)
    p = np.asarray(p).astype(float)
    ts = np.linspace(0.05, 0.95, 181)
    best_t, best_s = 0.5, -1
    for t in ts:
        pred = (p >= t).astype(int)
        s = f1_score(y, pred, zero_division=0)
        if s > best_s:
            best_s, best_t = s, t
    return float(best_t), float(best_s)

def evaluate_binary(y, p, threshold=0.5, ece_bins=15):
    y = np.asarray(y).astype(int)
    p = np.asarray(p).astype(float)
    p = np.clip(p, 1e-7, 1-1e-7)
    pred = (p >= threshold).astype(int)

    out = {}
    out["threshold"]  = float(threshold)
    out["pr_auc"]     = float(average_precision_score(y, p))
    out["roc_auc"]    = float(roc_auc_score(y, p))
    out["acc"]        = float(accuracy_score(y, pred))
    out["f1"]         = float(f1_score(y, pred, zero_division=0))
    out["precision"]  = float(precision_score(y, pred, zero_division=0))
    out["recall"]     = float(recall_score(y, pred, zero_division=0))

    tn = int(((pred==0) & (y==0)).sum())
    fp = int(((pred==1) & (y==0)).sum())
    fn = int(((pred==0) & (y==1)).sum())
    tp = int(((pred==1) & (y==1)).sum())
    out.update({"tn":tn, "fp":fp, "fn":fn, "tp":tp})

    out["brier"] = float(np.mean((p - y)**2))
    out["ece"]   = float(expected_calibration_error(y, p, n_bins=ece_bins))
    out["nll"]   = float(log_loss(y, p, labels=[0,1]))
    return out

def gate_entropy_torch(g, eps=1e-12):
    return -(g * (g + eps).log()).sum(dim=1).mean()

def gate_reg_coeff(epoch, warmup=2, flip_epoch=12, max_mag=0.02):
    if epoch < warmup:
        return 0.0
    if epoch < flip_epoch:
        t = (epoch - warmup) / max(1, (flip_epoch - warmup))
        return -max_mag * float(t)
    t2 = min(1.0, (epoch - flip_epoch) / 10.0)
    return +max_mag * float(t2)

# ---------------------------
# Load data
# ---------------------------
df = pd.read_csv(DATA_CSV)
LABEL_COL = "stroke" if "stroke" in df.columns else df.columns[-1]
y_all = df[LABEL_COL].values.astype(int)

bin_cols = ['gender','alcohol','smoke','sleep_disorder','Health_Insurance','diabetes','hypertension','high_cholesterol','Coronary_Heart_Disease']
cat_cols = ['Race','Marital_status']
cont_cols = [c for c in df.columns if c not in ([LABEL_COL] + bin_cols + cat_cols)]

cg_path = os.path.join(BASE_DIR, "logs", "concept_groups.json")
concept_groups = json.load(open(cg_path, "r"))
concept_names = list(concept_groups.keys())

# ---------------------------
# Split
# ---------------------------
def make_split(seed):
    sss = StratifiedShuffleSplit(n_splits=1, test_size=0.20, random_state=seed)
    tr_idx, te_idx = next(sss.split(df, y_all))
    y_tr = y_all[tr_idx]
    sss2 = StratifiedShuffleSplit(n_splits=1, test_size=0.20, random_state=seed+1337)
    tr2, va2 = next(sss2.split(np.zeros(len(tr_idx)), y_tr))
    tr2_idx = tr_idx[tr2]
    va_idx  = tr_idx[va2]
    return tr2_idx, va_idx, te_idx

# ---------------------------
# Dataset + preprocessing
# ---------------------------
def to_long_tensor(x):  return torch.as_tensor(x, dtype=torch.long)
def to_float_tensor(x): return torch.as_tensor(x, dtype=torch.float32)

class TabDS(Dataset):
    def __init__(self, Xc, Xk, Xb, y):
        self.Xc = Xc; self.Xk = Xk; self.Xb = Xb; self.y = y
    def __len__(self): return len(self.y)
    def __getitem__(self, i):
        return self.Xc[i], self.Xk[i], self.Xb[i], self.y[i]

def fit_preprocess(train_df):
    bmap = {}
    for c in bin_cols:
        vals = train_df[c].values
        uniq = np.unique(vals)
        if set(uniq.tolist()) == {0,1}:
            bmap[c] = ("01", None)
        elif set(uniq.tolist()) == {1,2}:
            bmap[c] = ("12", None)
        else:
            if len(uniq)==2:
                bmap[c] = ("map", {uniq[0]:0, uniq[1]:1})
            else:
                raise ValueError(f"Binary col {c} has >2 unique: {uniq}")

    cat_vocab = {}
    for c in cat_cols:
        uniq = pd.Series(train_df[c]).astype(int).unique()
        uniq = np.sort(uniq)
        cat_vocab[c] = {v:i for i,v in enumerate(uniq.tolist())}

    mu = train_df[cont_cols].values.astype(np.float32).mean(axis=0)
    sd = train_df[cont_cols].values.astype(np.float32).std(axis=0)
    sd = np.where(sd < 1e-6, 1.0, sd)
    return bmap, cat_vocab, mu, sd

def apply_preprocess(df_part, bmap, cat_vocab, mu, sd):
    Xb = np.zeros((len(df_part), len(bin_cols)), dtype=np.int64)
    for j,c in enumerate(bin_cols):
        vals = df_part[c].values
        mode, mapping = bmap[c]
        if mode == "01":
            Xb[:,j] = vals.astype(int)
        elif mode == "12":
            Xb[:,j] = (vals.astype(int) - 1)
        else:
            Xb[:,j] = np.vectorize(mapping.get)(vals)
        Xb[:,j] = np.clip(Xb[:,j], 0, 1)

    Xk = np.zeros((len(df_part), len(cat_cols)), dtype=np.int64)
    for j,c in enumerate(cat_cols):
        vocab = cat_vocab[c]
        vals = df_part[c].astype(int).values
        Xk[:,j] = np.array([vocab.get(v, 0) for v in vals], dtype=np.int64)

    Xc = df_part[cont_cols].values.astype(np.float32)
    Xc = (Xc - mu) / sd
    return Xc, Xk, Xb

def make_balanced_loader(tr_ds, y_tr):
    y_tr = np.asarray(y_tr, dtype=np.int64)
    cc = np.bincount(y_tr, minlength=2)
    cw = 1.0 / np.maximum(cc, 1)
    sw = cw[y_tr].astype(np.float64)
    sampler = WeightedRandomSampler(torch.as_tensor(sw, dtype=torch.double), num_samples=len(sw), replacement=True)
    return DataLoader(tr_ds, batch_size=cfg.batch_size, sampler=sampler, drop_last=False)

# ---------------------------
# Model
# ---------------------------
class WideDeep(nn.Module):
    def __init__(self, n_cont, cat_cards, n_bin, d_embed=16, deep_hidden=64, deep_layers=2, dropout=0.1):
        super().__init__()
        self.cat_embeds = nn.ModuleList([nn.Embedding(card, d_embed) for card in cat_cards])
        zdim = n_cont + n_bin + len(cat_cards)*d_embed
        self.wide = nn.Linear(zdim, 1)

        layers, din = [], zdim
        for _ in range(deep_layers):
            layers += [nn.Linear(din, deep_hidden), nn.ReLU(), nn.Dropout(dropout)]
            din = deep_hidden
        layers += [nn.Linear(din, 1)]
        self.deep = nn.Sequential(*layers)

    def forward(self, xc, xk, xb):
        cat_vec = torch.cat([emb(xk[:,j]) for j, emb in enumerate(self.cat_embeds)], dim=1)
        z = torch.cat([xc, xb.float(), cat_vec], dim=1)
        base = self.wide(z).squeeze(1) + self.deep(z).squeeze(1)
        return base, z

class ConceptGatedResidual(nn.Module):
    def __init__(self, z_dim, n_concepts, hidden=64, dropout=0.1):
        super().__init__()
        self.gate = nn.Sequential(nn.Linear(z_dim, hidden), nn.ReLU(), nn.Dropout(dropout), nn.Linear(hidden, n_concepts))
        self.experts = nn.ModuleList([
            nn.Sequential(nn.Linear(z_dim, hidden), nn.ReLU(), nn.Dropout(dropout), nn.Linear(hidden, 1))
            for _ in range(n_concepts)
        ])
        self.alpha = nn.Parameter(torch.tensor(-2.0))

    def forward(self, z):
        g = torch.softmax(self.gate(z), dim=1)
        E = torch.stack([ex(z).squeeze(1) for ex in self.experts], dim=1)
        res = (g * E).sum(dim=1)
        res = torch.sigmoid(self.alpha) * res
        return res, g, E

class CGWideDeep(nn.Module):
    def __init__(self, n_cont, cat_cards, n_bin, concept_names, d_embed=16, deep_hidden=64, deep_layers=2, dropout=0.1):
        super().__init__()
        self.base = WideDeep(n_cont, cat_cards, n_bin, d_embed, deep_hidden, deep_layers, dropout)
        zdim = n_cont + n_bin + len(cat_cards)*d_embed
        self.cg = ConceptGatedResidual(zdim, len(concept_names), deep_hidden, dropout)

    def forward(self, xc, xk, xb):
        base, z = self.base(xc, xk, xb)
        res, g, E = self.cg(z)
        return base + res, g, E

@torch.no_grad()
def predict_logits(model, loader):
    model.eval()
    ys, logits = [], []
    for xc, xk, xb, yb in loader:
        xc = xc.to(DEVICE); xk = xk.to(DEVICE); xb = xb.to(DEVICE)
        logit, _, _ = model(xc, xk, xb)
        ys.append(yb.numpy())
        logits.append(logit.detach().cpu().numpy())
    return np.concatenate(ys), np.concatenate(logits)

# ---------------------------
# Calibrators (fit on CalFit logits, pick threshold on CalEval)
# ---------------------------
def cal_none_fit_predict(l_fit, y_fit, l_eval, l_test):
    p_eval = 1/(1+np.exp(-l_eval))
    p_test = 1/(1+np.exp(-l_test))
    return p_eval, p_test

def cal_platt_fit_predict(l_fit, y_fit, l_eval, l_test):
    lr = LogisticRegression(solver="lbfgs", max_iter=200)
    lr.fit(l_fit.reshape(-1,1), y_fit.astype(int))
    p_eval = lr.predict_proba(l_eval.reshape(-1,1))[:,1]
    p_test = lr.predict_proba(l_test.reshape(-1,1))[:,1]
    return p_eval, p_test

def cal_temp_fit_predict(l_fit, y_fit, l_eval, l_test):
    y = y_fit.astype(int)
    Ts = np.linspace(0.5, 5.0, 91)
    bestT, bestN = 1.0, 1e18
    for T in Ts:
        p = 1/(1+np.exp(-(l_fit / T)))
        p = np.clip(p, 1e-7, 1-1e-7)
        nll = log_loss(y, p, labels=[0,1])
        if nll < bestN:
            bestN, bestT = nll, T
    p_eval = 1/(1+np.exp(-(l_eval / bestT)))
    p_test = 1/(1+np.exp(-(l_test / bestT)))
    return p_eval, p_test

def cal_iso_fit_predict(l_fit, y_fit, l_eval, l_test):
    p_fit0  = 1/(1+np.exp(-l_fit))
    p_eval0 = 1/(1+np.exp(-l_eval))
    p_test0 = 1/(1+np.exp(-l_test))
    iso = IsotonicRegression(out_of_bounds="clip")
    iso.fit(p_fit0, y_fit.astype(int))
    return iso.transform(p_eval0), iso.transform(p_test0)

CAL = {
    "none": cal_none_fit_predict,
    "platt": cal_platt_fit_predict,
    "temp": cal_temp_fit_predict,
    "iso": cal_iso_fit_predict
}

# ---------------------------
# Train + robust calibration evaluation
# ---------------------------
def run_repeat(r, cal_name):
    tr_idx, va_idx, te_idx = make_split(cfg.seed + 1000*r)
    df_tr, df_va, df_te = df.iloc[tr_idx].copy(), df.iloc[va_idx].copy(), df.iloc[te_idx].copy()

    bmap, cat_vocab, mu, sd = fit_preprocess(df_tr)
    Xc_tr, Xk_tr, Xb_tr = apply_preprocess(df_tr, bmap, cat_vocab, mu, sd)
    Xc_va, Xk_va, Xb_va = apply_preprocess(df_va, bmap, cat_vocab, mu, sd)
    Xc_te, Xk_te, Xb_te = apply_preprocess(df_te, bmap, cat_vocab, mu, sd)

    y_tr = df_tr[LABEL_COL].values.astype(int)
    y_va = df_va[LABEL_COL].values.astype(int)
    y_te = df_te[LABEL_COL].values.astype(int)

    cat_cards = [len(cat_vocab[c]) for c in cat_cols]

    tr_ds = TabDS(to_float_tensor(Xc_tr), to_long_tensor(Xk_tr), to_long_tensor(Xb_tr), to_long_tensor(y_tr))
    va_ds = TabDS(to_float_tensor(Xc_va), to_long_tensor(Xk_va), to_long_tensor(Xb_va), to_long_tensor(y_va))
    te_ds = TabDS(to_float_tensor(Xc_te), to_long_tensor(Xk_te), to_long_tensor(Xb_te), to_long_tensor(y_te))

    tr_loader = make_balanced_loader(tr_ds, y_tr)
    va_loader = DataLoader(va_ds, batch_size=cfg.batch_size, shuffle=False)
    te_loader = DataLoader(te_ds, batch_size=cfg.batch_size, shuffle=False)

    model = CGWideDeep(len(cont_cols), cat_cards, len(bin_cols), concept_names,
                       cfg.d_embed, cfg.deep_hidden, cfg.deep_layers, cfg.dropout).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    crit = nn.BCEWithLogitsLoss()  # No pos_weight (important)

    best_pr, best_state, bad = -1.0, None, 0

    for ep in range(cfg.max_epochs):
        model.train()
        c_ent = gate_reg_coeff(ep, cfg.gate_ent_warmup, cfg.gate_ent_flip_epoch, cfg.gate_ent_max_mag)

        for xc, xk, xb, yb in tr_loader:
            xc = xc.to(DEVICE); xk = xk.to(DEVICE); xb = xb.to(DEVICE)
            yb = yb.to(DEVICE).float()

            opt.zero_grad(set_to_none=True)
            logit, g, _ = model(xc, xk, xb)

            loss = crit(logit, yb) + c_ent * gate_entropy_torch(g)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()

        # early stop on VAL PR-AUC (uncalibrated)
        yv, lv = predict_logits(model, va_loader)
        pv = 1/(1+np.exp(-lv))
        val_pr = average_precision_score(yv, pv)

        if val_pr > best_pr + 1e-5:
            best_pr = val_pr
            best_state = {k: v.detach().cpu().clone() for k,v in model.state_dict().items()}
            bad = 0
        else:
            bad += 1
            if bad >= cfg.patience:
                break

    model.load_state_dict(best_state)

    # logits
    yv, lv = predict_logits(model, va_loader)
    yt, lt = predict_logits(model, te_loader)

    # split VAL into CalFit/CalEval (stratified)
    sss = StratifiedShuffleSplit(n_splits=1, test_size=0.5, random_state=cfg.seed + 999 + r)
    fit_idx, eval_idx = next(sss.split(np.zeros(len(yv)), yv))
    y_fit, l_fit = yv[fit_idx], lv[fit_idx]
    y_eval, l_eval = yv[eval_idx], lv[eval_idx]

    # calibrate using CalFit, choose threshold on CalEval
    p_eval, p_test = CAL[cal_name](l_fit, y_fit, l_eval, lt)
    t_best, _ = find_best_threshold(y_eval, p_eval)

    # evaluate on TEST
    m_te = evaluate_binary(yt, p_test, threshold=t_best, ece_bins=cfg.ece_bins)
    return m_te

rows = []
for cal_name in ["none","temp","platt","iso"]:
    for r in range(cfg.repeats):
        m = run_repeat(r, cal_name)
        m.update({"cal": cal_name, "repeat": r})
        rows.append(m)
        print(f"cal={cal_name:5s} r{r} test pr={m['pr_auc']:.4f} brier={m['brier']:.4f} ece={m['ece']:.4f} nll={m['nll']:.4f}")

df_res = pd.DataFrame(rows)
display(df_res)

summary = df_res.groupby("cal")[["pr_auc","roc_auc","f1","recall","brier","ece","nll","acc"]].agg(["mean","std"])
summary.columns = [f"{a}_{b}" for a,b in summary.columns]
summary = summary.reset_index().sort_values("pr_auc_mean", ascending=False)
display(summary)

df_res.to_csv(os.path.join(OUT_DIR, "tables", "robust_calibration_per_repeat_test.csv"), index=False)
summary.to_csv(os.path.join(OUT_DIR, "tables", "robust_calibration_summary_test.csv"), index=False)
print("✅ Saved robust calibration tables to:", os.path.join(OUT_DIR, "tables"))

cal=none  r0 test pr=0.1179 brier=0.1958 ece=0.2468 nll=0.5947
cal=none  r1 test pr=0.1422 brier=0.1699 ece=0.2189 nll=0.4994
cal=none  r2 test pr=0.2105 brier=0.1989 ece=0.3261 nll=0.5782
cal=none  r3 test pr=0.1749 brier=0.2197 ece=0.3466 nll=0.6231
cal=none  r4 test pr=0.1330 brier=0.2514 ece=0.3871 nll=0.7053
cal=temp  r0 test pr=0.1033 brier=0.1593 ece=0.2013 nll=0.4933
cal=temp  r1 test pr=0.1820 brier=0.1932 ece=0.3246 nll=0.5656
cal=temp  r2 test pr=0.2147 brier=0.1789 ece=0.2858 nll=0.5295
cal=temp  r3 test pr=0.2101 brier=0.1931 ece=0.3098 nll=0.5601
cal=temp  r4 test pr=0.1321 brier=0.2084 ece=0.3320 nll=0.6005
cal=platt r0 test pr=0.1119 brier=0.0720 ece=0.0223 nll=0.2761
cal=platt r1 test pr=0.1514 brier=0.0696 ece=0.0114 nll=0.2548
cal=platt r2 test pr=0.1471 brier=0.0703 ece=0.0129 nll=0.2645
cal=platt r3 test pr=0.1944 brier=0.0691 ece=0.0141 nll=0.2559
cal=platt r4 test pr=0.1307 brier=0.0714 ece=0.0203 nll=0.2670
cal=iso   r0 test pr=0.0982 brier=0.0767 ece=0.0278 nll

,threshold,pr_auc,roc_auc,acc,f1,precision,recall,tn,fp,fn,tp,brier,ece,nll,cal,repeat
0,0.625,0.117948,0.596748,0.764387,0.174905,0.120419,0.319444,681,168,49,23,0.195790,0.246761,0.594721,none,0
1,0.760,0.142172,0.697634,0.854506,0.141026,0.130952,0.152778,776,73,61,11,0.169949,0.218902,0.499366,none,1
2,0.575,0.210504,0.716169,0.754615,0.251656,0.165217,0.527778,657,192,34,38,0.198936,0.326116,0.578195,none,2
3,0.605,0.174875,0.682273,0.736156,0.208469,0.136170,0.444444,646,203,40,32,0.219722,0.346564,0.623128,none,3
4,0.640,0.133025,0.654545,0.742671,0.202020,0.133333,0.416667,654,195,42,30,0.251434,0.387062,0.705272,none,4
5,0.800,0.103332,0.590139,0.896851,0.077670,0.129032,0.055556,822,27,68,4,0.159305,0.201346,0.493317,temp,0
6,0.635,0.181952,0.719441,0.843648,0.280000,0.218750,0.388889,749,100,44,28,0.193236,0.324609,0.565586,temp,1
7,0.680,0.214742,0.733543,0.849077,0.264550,0.213675,0.347222,757,92,47,25,0.178928,0.285771,0.529479,temp,2
8,0.690,0.210123,0.693856,0.858849,0.252874,0.215686,0.305556,769,80,50,22,0.193103,0.309842,0.560052,temp,3
9,0.525,0.132112,0.648884,0.687296,0.191011,0.119718,0.472222,599,250,38,34,0.208404,0.331979,0.600501,temp,4


,cal,pr_auc_mean,pr_auc_std,roc_auc_mean,roc_auc_std,f1_mean,f1_std,recall_mean,recall_std,brier_mean,brier_std,ece_mean,ece_std,nll_mean,nll_std,acc_mean,acc_std
3,temp,0.168452,0.049048,0.677172,0.058347,0.213221,0.082967,0.313889,0.157012,0.186595,0.018477,0.290709,0.052982,0.549787,0.040399,0.827144,0.080894
1,none,0.155705,0.037059,0.669474,0.046501,0.195615,0.041097,0.372222,0.143466,0.207166,0.030412,0.305081,0.070200,0.600137,0.074578,0.770467,0.048220
2,platt,0.147109,0.030671,0.665103,0.044462,0.175842,0.050947,0.325000,0.215748,0.070480,0.001216,0.016186,0.004803,0.263658,0.008748,0.783496,0.112326
0,iso,0.129613,0.038804,0.654186,0.061164,0.187527,0.054002,0.313889,0.196811,0.073876,0.004681,0.023166,0.012853,0.332037,0.062936,0.795440,0.100399


✅ Saved robust calibration tables to: /kaggle/working/outputs/cgt_stroke_v1/robust_calibration/tables


In [26]:
# ============================================================
# MULTI-IMPROVEMENT SWEEP (single script)
# Focus: improve PR-AUC while keeping calibration reasonable
#
# Variants tested (training):
#   V0: BAL + NoPosW + EntSched  (baseline best recipe)
#   V1: V0 + EMA
#   V2: V0 + SWA
#   V3: V0 + LogitAdjust (prior correction during training)
#
# Calibrations:
#   none, temp  (optionally platt)
#
# Output: per-repeat test table + summary by variant+cal
# ============================================================

import os, json, random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, precision_score, recall_score, accuracy_score, log_loss
from sklearn.linear_model import LogisticRegression

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BASE_DIR = "/kaggle/working/outputs/cgt_stroke_v1"
DATA_CSV = os.path.join(BASE_DIR, "data_clean_v2.csv")

OUT_DIR = os.path.join(BASE_DIR, "sweep_multi_improvements")
os.makedirs(os.path.join(OUT_DIR, "tables"), exist_ok=True)

class CFG:
    seed = 42
    repeats = 3      # <-- increase to 5 after quick check
    batch_size = 256
    lr = 2e-3
    weight_decay = 1e-4
    max_epochs = 60
    patience = 10
    ece_bins = 15

    # entropy schedule
    gate_ent_warmup = 2
    gate_ent_flip_epoch = 12
    gate_ent_max_mag = 0.02

    # model sizes
    d_embed = 16
    deep_hidden = 64
    deep_layers = 2
    dropout = 0.10

    # EMA / SWA
    use_ema = True
    ema_decay = 0.995

    use_swa = True
    swa_start = 20
    swa_lr = 1e-3

cfg = CFG()

def seed_everything(seed=42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(cfg.seed)

# ---------------------------
# Metrics
# ---------------------------
def expected_calibration_error(y_true, p, n_bins=15):
    y_true = np.asarray(y_true).astype(int)
    p = np.asarray(p).astype(float)
    bins = np.linspace(0.0, 1.0, n_bins+1)
    ece = 0.0
    for i in range(n_bins):
        lo, hi = bins[i], bins[i+1]
        m = (p >= lo) & (p < hi) if i < n_bins-1 else (p >= lo) & (p <= hi)
        if m.sum() == 0:
            continue
        acc = y_true[m].mean()
        conf = p[m].mean()
        ece += (m.sum()/len(p)) * abs(acc - conf)
    return float(ece)

def find_best_threshold(y, p):
    y = np.asarray(y).astype(int)
    p = np.asarray(p).astype(float)
    ts = np.linspace(0.05, 0.95, 181)
    best_t, best_s = 0.5, -1
    for t in ts:
        pred = (p >= t).astype(int)
        s = f1_score(y, pred, zero_division=0)
        if s > best_s:
            best_s, best_t = s, t
    return float(best_t), float(best_s)

def evaluate_binary(y, p, threshold=0.5, ece_bins=15):
    y = np.asarray(y).astype(int)
    p = np.asarray(p).astype(float)
    p = np.clip(p, 1e-7, 1-1e-7)
    pred = (p >= threshold).astype(int)

    out = {}
    out["threshold"]  = float(threshold)
    out["pr_auc"]     = float(average_precision_score(y, p))
    out["roc_auc"]    = float(roc_auc_score(y, p))
    out["acc"]        = float(accuracy_score(y, pred))
    out["f1"]         = float(f1_score(y, pred, zero_division=0))
    out["precision"]  = float(precision_score(y, pred, zero_division=0))
    out["recall"]     = float(recall_score(y, pred, zero_division=0))

    tn = int(((pred==0) & (y==0)).sum())
    fp = int(((pred==1) & (y==0)).sum())
    fn = int(((pred==0) & (y==1)).sum())
    tp = int(((pred==1) & (y==1)).sum())
    out.update({"tn":tn, "fp":fp, "fn":fn, "tp":tp})

    out["brier"] = float(np.mean((p - y)**2))
    out["ece"]   = float(expected_calibration_error(y, p, n_bins=ece_bins))
    out["nll"]   = float(log_loss(y, p, labels=[0,1]))
    return out

def gate_entropy_torch(g, eps=1e-12):
    return -(g * (g + eps).log()).sum(dim=1).mean()

def gate_reg_coeff(epoch, warmup=2, flip_epoch=12, max_mag=0.02):
    if epoch < warmup:
        return 0.0
    if epoch < flip_epoch:
        t = (epoch - warmup) / max(1, (flip_epoch - warmup))
        return -max_mag * float(t)
    t2 = min(1.0, (epoch - flip_epoch) / 10.0)
    return +max_mag * float(t2)

# ---------------------------
# Load data + columns
# ---------------------------
df = pd.read_csv(DATA_CSV)
LABEL_COL = "stroke" if "stroke" in df.columns else df.columns[-1]
y_all = df[LABEL_COL].values.astype(int)

bin_cols = ['gender','alcohol','smoke','sleep_disorder','Health_Insurance','diabetes','hypertension','high_cholesterol','Coronary_Heart_Disease']
cat_cols = ['Race','Marital_status']
cont_cols = [c for c in df.columns if c not in ([LABEL_COL] + bin_cols + cat_cols)]

cg_path = os.path.join(BASE_DIR, "logs", "concept_groups.json")
concept_groups = json.load(open(cg_path, "r"))
concept_names = list(concept_groups.keys())

# ---------------------------
# Split
# ---------------------------
def make_split(seed):
    sss = StratifiedShuffleSplit(n_splits=1, test_size=0.20, random_state=seed)
    tr_idx, te_idx = next(sss.split(df, y_all))
    y_tr = y_all[tr_idx]
    sss2 = StratifiedShuffleSplit(n_splits=1, test_size=0.20, random_state=seed+1337)
    tr2, va2 = next(sss2.split(np.zeros(len(tr_idx)), y_tr))
    tr2_idx = tr_idx[tr2]
    va_idx  = tr_idx[va2]
    return tr2_idx, va_idx, te_idx

# ---------------------------
# Preprocess + dataset
# ---------------------------
def to_long_tensor(x):  return torch.as_tensor(x, dtype=torch.long)
def to_float_tensor(x): return torch.as_tensor(x, dtype=torch.float32)

class TabDS(Dataset):
    def __init__(self, Xc, Xk, Xb, y):
        self.Xc = Xc; self.Xk = Xk; self.Xb = Xb; self.y = y
    def __len__(self): return len(self.y)
    def __getitem__(self, i):
        return self.Xc[i], self.Xk[i], self.Xb[i], self.y[i]

def fit_preprocess(train_df):
    bmap = {}
    for c in bin_cols:
        vals = train_df[c].values
        uniq = np.unique(vals)
        if set(uniq.tolist()) == {0,1}:
            bmap[c] = ("01", None)
        elif set(uniq.tolist()) == {1,2}:
            bmap[c] = ("12", None)
        else:
            if len(uniq)==2:
                bmap[c] = ("map", {uniq[0]:0, uniq[1]:1})
            else:
                raise ValueError(f"Binary col {c} has >2 unique: {uniq}")

    cat_vocab = {}
    for c in cat_cols:
        uniq = pd.Series(train_df[c]).astype(int).unique()
        uniq = np.sort(uniq)
        cat_vocab[c] = {v:i for i,v in enumerate(uniq.tolist())}

    mu = train_df[cont_cols].values.astype(np.float32).mean(axis=0)
    sd = train_df[cont_cols].values.astype(np.float32).std(axis=0)
    sd = np.where(sd < 1e-6, 1.0, sd)
    return bmap, cat_vocab, mu, sd

def apply_preprocess(df_part, bmap, cat_vocab, mu, sd):
    Xb = np.zeros((len(df_part), len(bin_cols)), dtype=np.int64)
    for j,c in enumerate(bin_cols):
        vals = df_part[c].values
        mode, mapping = bmap[c]
        if mode == "01":
            Xb[:,j] = vals.astype(int)
        elif mode == "12":
            Xb[:,j] = (vals.astype(int) - 1)
        else:
            Xb[:,j] = np.vectorize(mapping.get)(vals)
        Xb[:,j] = np.clip(Xb[:,j], 0, 1)

    Xk = np.zeros((len(df_part), len(cat_cols)), dtype=np.int64)
    for j,c in enumerate(cat_cols):
        vocab = cat_vocab[c]
        vals = df_part[c].astype(int).values
        Xk[:,j] = np.array([vocab.get(v, 0) for v in vals], dtype=np.int64)

    Xc = df_part[cont_cols].values.astype(np.float32)
    Xc = (Xc - mu) / sd
    return Xc, Xk, Xb

def make_balanced_loader(tr_ds, y_tr):
    y_tr = np.asarray(y_tr, dtype=np.int64)
    cc = np.bincount(y_tr, minlength=2)
    cw = 1.0 / np.maximum(cc, 1)
    sw = cw[y_tr].astype(np.float64)
    sampler = WeightedRandomSampler(torch.as_tensor(sw, dtype=torch.double), num_samples=len(sw), replacement=True)
    return DataLoader(tr_ds, batch_size=cfg.batch_size, sampler=sampler, drop_last=False)

# ---------------------------
# Model
# ---------------------------
class WideDeep(nn.Module):
    def __init__(self, n_cont, cat_cards, n_bin, d_embed=16, deep_hidden=64, deep_layers=2, dropout=0.1):
        super().__init__()
        self.cat_embeds = nn.ModuleList([nn.Embedding(card, d_embed) for card in cat_cards])
        zdim = n_cont + n_bin + len(cat_cards)*d_embed
        self.wide = nn.Linear(zdim, 1)
        layers, din = [], zdim
        for _ in range(deep_layers):
            layers += [nn.Linear(din, deep_hidden), nn.ReLU(), nn.Dropout(dropout)]
            din = deep_hidden
        layers += [nn.Linear(din, 1)]
        self.deep = nn.Sequential(*layers)

    def forward(self, xc, xk, xb):
        cat_vec = torch.cat([emb(xk[:,j]) for j, emb in enumerate(self.cat_embeds)], dim=1)
        z = torch.cat([xc, xb.float(), cat_vec], dim=1)
        base = self.wide(z).squeeze(1) + self.deep(z).squeeze(1)
        return base, z

class ConceptGatedResidual(nn.Module):
    def __init__(self, z_dim, n_concepts, hidden=64, dropout=0.1):
        super().__init__()
        self.gate = nn.Sequential(nn.Linear(z_dim, hidden), nn.ReLU(), nn.Dropout(dropout), nn.Linear(hidden, n_concepts))
        self.experts = nn.ModuleList([
            nn.Sequential(nn.Linear(z_dim, hidden), nn.ReLU(), nn.Dropout(dropout), nn.Linear(hidden, 1))
            for _ in range(n_concepts)
        ])
        self.alpha = nn.Parameter(torch.tensor(-2.0))

    def forward(self, z):
        g = torch.softmax(self.gate(z), dim=1)
        E = torch.stack([ex(z).squeeze(1) for ex in self.experts], dim=1)
        res = (g * E).sum(dim=1)
        res = torch.sigmoid(self.alpha) * res
        return res, g, E

class CGWideDeep(nn.Module):
    def __init__(self, n_cont, cat_cards, n_bin, concept_names, d_embed=16, deep_hidden=64, deep_layers=2, dropout=0.1):
        super().__init__()
        self.base = WideDeep(n_cont, cat_cards, n_bin, d_embed, deep_hidden, deep_layers, dropout)
        zdim = n_cont + n_bin + len(cat_cards)*d_embed
        self.cg = ConceptGatedResidual(zdim, len(concept_names), deep_hidden, dropout)

    def forward(self, xc, xk, xb):
        base, z = self.base(xc, xk, xb)
        res, g, E = self.cg(z)
        return base + res, g, E

@torch.no_grad()
def predict_logits(model, loader):
    model.eval()
    ys, logits = [], []
    for xc, xk, xb, yb in loader:
        xc = xc.to(DEVICE); xk = xk.to(DEVICE); xb = xb.to(DEVICE)
        logit, _, _ = model(xc, xk, xb)
        ys.append(yb.numpy())
        logits.append(logit.detach().cpu().numpy())
    return np.concatenate(ys), np.concatenate(logits)

# ---------------------------
# EMA helper
# ---------------------------
class EMA:
    def __init__(self, model, decay=0.995):
        self.decay = decay
        self.shadow = {k: v.detach().clone() for k,v in model.state_dict().items()}
    @torch.no_grad()
    def update(self, model):
        for k,v in model.state_dict().items():
            self.shadow[k].mul_(self.decay).add_(v.detach(), alpha=1.0-self.decay)
    def apply_to(self, model):
        self.backup = {k: v.detach().clone() for k,v in model.state_dict().items()}
        model.load_state_dict(self.shadow)
    def restore(self, model):
        model.load_state_dict(self.backup)

# ---------------------------
# Calibrators
# ---------------------------
def cal_none(l_fit, y_fit, l_eval, l_test):
    p_eval = 1/(1+np.exp(-l_eval))
    p_test = 1/(1+np.exp(-l_test))
    return p_eval, p_test

def cal_temp(l_fit, y_fit, l_eval, l_test):
    y = y_fit.astype(int)
    Ts = np.linspace(0.5, 5.0, 91)
    bestT, bestN = 1.0, 1e18
    for T in Ts:
        p = 1/(1+np.exp(-(l_fit / T)))
        p = np.clip(p, 1e-7, 1-1e-7)
        nll = log_loss(y, p, labels=[0,1])
        if nll < bestN:
            bestN, bestT = nll, T
    p_eval = 1/(1+np.exp(-(l_eval / bestT)))
    p_test = 1/(1+np.exp(-(l_test / bestT)))
    return p_eval, p_test

def cal_platt(l_fit, y_fit, l_eval, l_test):
    lr = LogisticRegression(solver="lbfgs", max_iter=200)
    lr.fit(l_fit.reshape(-1,1), y_fit.astype(int))
    p_eval = lr.predict_proba(l_eval.reshape(-1,1))[:,1]
    p_test = lr.predict_proba(l_test.reshape(-1,1))[:,1]
    return p_eval, p_test

CALS = {"none": cal_none, "temp": cal_temp, "platt": cal_platt}

# ---------------------------
# Training variants
# ---------------------------
def train_and_eval_variant(variant_name, r, cal_name, use_ema=False, use_swa=False, use_logit_adjust=False):
    tr_idx, va_idx, te_idx = make_split(cfg.seed + 1000*r)
    df_tr, df_va, df_te = df.iloc[tr_idx].copy(), df.iloc[va_idx].copy(), df.iloc[te_idx].copy()

    bmap, cat_vocab, mu, sd = fit_preprocess(df_tr)
    Xc_tr, Xk_tr, Xb_tr = apply_preprocess(df_tr, bmap, cat_vocab, mu, sd)
    Xc_va, Xk_va, Xb_va = apply_preprocess(df_va, bmap, cat_vocab, mu, sd)
    Xc_te, Xk_te, Xb_te = apply_preprocess(df_te, bmap, cat_vocab, mu, sd)

    y_tr = df_tr[LABEL_COL].values.astype(int)
    y_va = df_va[LABEL_COL].values.astype(int)
    y_te = df_te[LABEL_COL].values.astype(int)

    cat_cards = [len(cat_vocab[c]) for c in cat_cols]

    tr_ds = TabDS(to_float_tensor(Xc_tr), to_long_tensor(Xk_tr), to_long_tensor(Xb_tr), to_long_tensor(y_tr))
    va_ds = TabDS(to_float_tensor(Xc_va), to_long_tensor(Xk_va), to_long_tensor(Xb_va), to_long_tensor(y_va))
    te_ds = TabDS(to_float_tensor(Xc_te), to_long_tensor(Xk_te), to_long_tensor(Xb_te), to_long_tensor(y_te))

    tr_loader = make_balanced_loader(tr_ds, y_tr)
    va_loader = DataLoader(va_ds, batch_size=cfg.batch_size, shuffle=False)
    te_loader = DataLoader(te_ds, batch_size=cfg.batch_size, shuffle=False)

    model = CGWideDeep(len(cont_cols), cat_cards, len(bin_cols), concept_names,
                       cfg.d_embed, cfg.deep_hidden, cfg.deep_layers, cfg.dropout).to(DEVICE)

    opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    crit = nn.BCEWithLogitsLoss()

    # logit adjustment prior term
    if use_logit_adjust:
        pos = (y_tr==1).sum()
        neg = (y_tr==0).sum()
        pi = pos / max(1, pos+neg)
        logit_prior = float(np.log(pi/(1-pi+1e-12)))
    else:
        logit_prior = 0.0

    ema = EMA(model, decay=cfg.ema_decay) if use_ema else None

    # SWA
    if use_swa:
        from torch.optim.swa_utils import AveragedModel, SWALR
        swa_model = AveragedModel(model)
        swa_sched = SWALR(opt, swa_lr=cfg.swa_lr)
    else:
        swa_model = None
        swa_sched = None

    best_pr, best_state, bad = -1.0, None, 0

    for ep in range(cfg.max_epochs):
        model.train()
        c_ent = gate_reg_coeff(ep, cfg.gate_ent_warmup, cfg.gate_ent_flip_epoch, cfg.gate_ent_max_mag)

        for xc, xk, xb, yb in tr_loader:
            xc = xc.to(DEVICE); xk = xk.to(DEVICE); xb = xb.to(DEVICE)
            yb = yb.to(DEVICE).float()

            opt.zero_grad(set_to_none=True)
            logit, g, _ = model(xc, xk, xb)

            # logit adjust (train-time)
            if use_logit_adjust:
                logit = logit - logit_prior

            loss = crit(logit, yb) + c_ent * gate_entropy_torch(g)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()

            if ema is not None:
                ema.update(model)

        if use_swa and ep >= cfg.swa_start:
            swa_model.update_parameters(model)
            swa_sched.step()

        # early stop on VAL PR-AUC (uncalibrated)
        yv, lv = predict_logits(model, va_loader)
        pv = 1/(1+np.exp(-lv))
        val_pr = average_precision_score(yv, pv)

        if val_pr > best_pr + 1e-5:
            best_pr = val_pr
            best_state = {k: v.detach().cpu().clone() for k,v in model.state_dict().items()}
            bad = 0
        else:
            bad += 1
            if bad >= cfg.patience:
                break

    model.load_state_dict(best_state)

    # apply EMA weights for evaluation
    if ema is not None:
        ema.apply_to(model)

    # if SWA used, evaluate SWA weights instead of base
    if use_swa and swa_model is not None:
        model_eval = swa_model.to(DEVICE)
    else:
        model_eval = model

    # logits
    yv, lv = predict_logits(model_eval, va_loader)
    yt, lt = predict_logits(model_eval, te_loader)

    # split VAL into CalFit/CalEval (stratified)
    sss = StratifiedShuffleSplit(n_splits=1, test_size=0.5, random_state=cfg.seed + 999 + r)
    fit_idx, eval_idx = next(sss.split(np.zeros(len(yv)), yv))
    y_fit, l_fit = yv[fit_idx], lv[fit_idx]
    y_eval, l_eval = yv[eval_idx], lv[eval_idx]

    # calibrate & pick threshold on CalEval
    p_eval, p_test = CALS[cal_name](l_fit, y_fit, l_eval, lt)
    t_best, _ = find_best_threshold(y_eval, p_eval)

    m_te = evaluate_binary(yt, p_test, threshold=t_best, ece_bins=cfg.ece_bins)
    m_te.update({"variant": variant_name, "cal": cal_name, "repeat": r})
    return m_te

# ---------------------------
# Run sweep
# ---------------------------
VARIANTS = [
    ("V0_BASE", dict(use_ema=False, use_swa=False, use_logit_adjust=False)),
    ("V1_EMA",  dict(use_ema=True,  use_swa=False, use_logit_adjust=False)),
    ("V2_SWA",  dict(use_ema=False, use_swa=True,  use_logit_adjust=False)),
    ("V3_LOGITADJ", dict(use_ema=False, use_swa=False, use_logit_adjust=True)),
]

CAL_LIST = ["none", "temp"]  # add "platt" if you want

rows = []
for vname, vkw in VARIANTS:
    for cal_name in CAL_LIST:
        for r in range(cfg.repeats):
            m = train_and_eval_variant(vname, r, cal_name, **vkw)
            rows.append(m)
            print(f"{vname:10s} cal={cal_name:5s} r{r} PR={m['pr_auc']:.4f} Brier={m['brier']:.4f} ECE={m['ece']:.4f}")

df_out = pd.DataFrame(rows)
display(df_out)

summary = df_out.groupby(["variant","cal"])[["pr_auc","roc_auc","f1","recall","brier","ece","nll","acc"]].agg(["mean","std"])
summary.columns = [f"{a}_{b}" for a,b in summary.columns]
summary = summary.reset_index().sort_values("pr_auc_mean", ascending=False)
display(summary)

df_out.to_csv(os.path.join(OUT_DIR, "tables", "sweep_per_repeat_test.csv"), index=False)
summary.to_csv(os.path.join(OUT_DIR, "tables", "sweep_summary_test.csv"), index=False)
print("✅ Saved sweep tables to:", os.path.join(OUT_DIR, "tables"))

V0_BASE    cal=none  r0 PR=0.1179 Brier=0.1958 ECE=0.2468
V0_BASE    cal=none  r1 PR=0.1422 Brier=0.1699 ECE=0.2189
V0_BASE    cal=none  r2 PR=0.2105 Brier=0.1989 ECE=0.3261
V0_BASE    cal=temp  r0 PR=0.1178 Brier=0.2055 ECE=0.2943
V0_BASE    cal=temp  r1 PR=0.1468 Brier=0.1694 ECE=0.2563
V0_BASE    cal=temp  r2 PR=0.2062 Brier=0.2187 ECE=0.3704
V1_EMA     cal=none  r0 PR=0.1520 Brier=0.1953 ECE=0.3403
V1_EMA     cal=none  r1 PR=0.1657 Brier=0.1476 ECE=0.2545
V1_EMA     cal=none  r2 PR=0.2231 Brier=0.2186 ECE=0.3767
V1_EMA     cal=temp  r0 PR=0.1181 Brier=0.1760 ECE=0.2622
V1_EMA     cal=temp  r1 PR=0.1430 Brier=0.1735 ECE=0.2743
V1_EMA     cal=temp  r2 PR=0.1945 Brier=0.2005 ECE=0.3193
V2_SWA     cal=none  r0 PR=0.0956 Brier=0.2923 ECE=0.4558
V2_SWA     cal=none  r1 PR=0.0935 Brier=0.2734 ECE=0.4383
V2_SWA     cal=none  r2 PR=0.0599 Brier=0.2681 ECE=0.4154
V2_SWA     cal=temp  r0 PR=0.0813 Brier=0.2619 ECE=0.4350
V2_SWA     cal=temp  r1 PR=0.0936 Brier=0.2137 ECE=0.3404
V2_SWA     cal

,threshold,pr_auc,roc_auc,acc,f1,precision,recall,tn,fp,fn,tp,brier,ece,nll,variant,cal,repeat
0,0.625,0.117948,0.596748,0.764387,0.174905,0.120419,0.319444,681,168,49,23,0.195790,0.246761,0.594721,V0_BASE,none,0
1,0.760,0.142172,0.697634,0.854506,0.141026,0.130952,0.152778,776,73,61,11,0.169949,0.218902,0.499366,V0_BASE,none,1
2,0.575,0.210504,0.716169,0.754615,0.251656,0.165217,0.527778,657,192,34,38,0.198936,0.326116,0.578195,V0_BASE,none,2
3,0.535,0.117824,0.613581,0.698154,0.177515,0.112782,0.416667,613,236,42,30,0.205485,0.294334,0.590665,V0_BASE,temp,0
4,0.740,0.146829,0.700366,0.876221,0.123077,0.137931,0.111111,799,50,64,8,0.169371,0.256338,0.502853,V0_BASE,temp,1
5,0.590,0.206154,0.712554,0.769815,0.242857,0.163462,0.472222,675,174,38,34,0.218703,0.370368,0.624998,V0_BASE,temp,2
6,0.495,0.152025,0.658994,0.703583,0.199413,0.126394,0.472222,614,235,38,34,0.195337,0.340318,0.576728,V1_EMA,none,0
7,0.470,0.165736,0.738581,0.774159,0.267606,0.179245,0.527778,675,174,34,38,0.147637,0.254539,0.459181,V1_EMA,none,1
8,0.570,0.223064,0.712767,0.778502,0.260870,0.176471,0.500000,681,168,36,36,0.218574,0.376663,0.624689,V1_EMA,none,2
9,0.650,0.118115,0.582106,0.851249,0.116129,0.108434,0.125000,775,74,63,9,0.175979,0.262222,0.521735,V1_EMA,temp,0


,variant,cal,pr_auc_mean,pr_auc_std,roc_auc_mean,roc_auc_std,f1_mean,f1_std,recall_mean,recall_std,brier_mean,brier_std,ece_mean,ece_std,nll_mean,nll_std,acc_mean,acc_std
2,V1_EMA,none,0.180275,0.037685,0.703447,0.040604,0.242630,0.037577,0.500000,0.027778,0.187183,0.036165,0.323840,0.062708,0.553533,0.085158,0.752081,0.042057
7,V3_LOGITADJ,temp,0.158031,0.023803,0.667834,0.053867,0.210969,0.039511,0.388889,0.083333,0.073257,0.004061,0.036879,0.014646,0.277932,0.026665,0.771263,0.038623
1,V0_BASE,temp,0.156936,0.045024,0.675501,0.053969,0.181150,0.059973,0.333333,0.194444,0.197853,0.025536,0.307013,0.058063,0.572839,0.062994,0.781397,0.089597
0,V0_BASE,none,0.156875,0.047998,0.670184,0.064269,0.189195,0.056683,0.333333,0.187885,0.188225,0.015906,0.263927,0.055630,0.557428,0.050957,0.791169,0.055069
3,V1_EMA,temp,0.151894,0.038968,0.658482,0.066896,0.180125,0.055426,0.342593,0.191613,0.183316,0.014951,0.285248,0.030057,0.538622,0.034758,0.775606,0.068845
6,V3_LOGITADJ,none,0.129910,0.018116,0.644604,0.041170,0.111011,0.065589,0.101852,0.080188,0.081429,0.006624,0.071639,0.018469,0.479404,0.200212,0.882374,0.013748
5,V2_SWA,temp,0.094799,0.014137,0.531453,0.063622,0.145429,0.023231,0.537037,0.354189,0.201663,0.067029,0.312430,0.138712,0.583922,0.151128,0.518277,0.319636
4,V2_SWA,none,0.083019,0.020020,0.471587,0.078433,0.108575,0.046450,0.587963,0.493072,0.277925,0.012729,0.436504,0.020217,0.752829,0.027832,0.390879,0.436937


✅ Saved sweep tables to: /kaggle/working/outputs/cgt_stroke_v1/sweep_multi_improvements/tables


In [27]:
# ============================================================
# RUN ONLY NEW COMBO VARIANTS: EMA + LOGITADJ
# (append to existing sweep script after definitions)
# ============================================================

NEW_VARIANTS = [
    ("V4_EMA_LOGITADJ", dict(use_ema=True, use_swa=False, use_logit_adjust=True)),
]

NEW_CALS = ["none", "temp"]  # add "platt" if you want

new_rows = []
for vname, vkw in NEW_VARIANTS:
    for cal_name in NEW_CALS:
        for r in range(cfg.repeats):
            m = train_and_eval_variant(vname, r, cal_name, **vkw)
            new_rows.append(m)
            print(f"{vname:14s} cal={cal_name:5s} r{r} PR={m['pr_auc']:.4f} "
                  f"ROC={m['roc_auc']:.4f} F1={m['f1']:.4f} "
                  f"Brier={m['brier']:.4f} ECE={m['ece']:.4f}")

df_new = pd.DataFrame(new_rows)
display(df_new)

# append to previous results if df_out exists
try:
    df_out2 = pd.concat([df_out, df_new], ignore_index=True)
except NameError:
    df_out2 = df_new.copy()

summary2 = df_out2.groupby(["variant","cal"])[["pr_auc","roc_auc","f1","recall","brier","ece","nll","acc"]].agg(["mean","std"])
summary2.columns = [f"{a}_{b}" for a,b in summary2.columns]
summary2 = summary2.reset_index().sort_values("pr_auc_mean", ascending=False)
display(summary2)

# save updated tables
df_out2.to_csv(os.path.join(OUT_DIR, "tables", "sweep_per_repeat_test_UPDATED.csv"), index=False)
summary2.to_csv(os.path.join(OUT_DIR, "tables", "sweep_summary_test_UPDATED.csv"), index=False)
print("✅ Saved UPDATED sweep tables to:", os.path.join(OUT_DIR, "tables"))

V4_EMA_LOGITADJ cal=none  r0 PR=0.1140 ROC=0.5891 F1=0.1613 Brier=0.0790 ECE=0.0758
V4_EMA_LOGITADJ cal=none  r1 PR=0.1644 ROC=0.6890 F1=0.2538 Brier=0.0724 ECE=0.0380
V4_EMA_LOGITADJ cal=none  r2 PR=0.1923 ROC=0.6914 F1=0.2214 Brier=0.0861 ECE=0.1223
V4_EMA_LOGITADJ cal=temp  r0 PR=0.1288 ROC=0.6310 F1=0.1684 Brier=0.0771 ECE=0.0492
V4_EMA_LOGITADJ cal=temp  r1 PR=0.1532 ROC=0.7001 F1=0.1389 Brier=0.0735 ECE=0.0282
V4_EMA_LOGITADJ cal=temp  r2 PR=0.1566 ROC=0.6629 F1=0.2271 Brier=0.0702 ECE=0.0105


,threshold,pr_auc,roc_auc,acc,f1,precision,recall,tn,fp,fn,tp,brier,ece,nll,variant,cal,repeat
0,0.070,0.114034,0.589108,0.774159,0.161290,0.113636,0.277778,693,156,52,20,0.078960,0.075811,0.407841,V4_EMA_LOGITADJ,none,0
1,0.120,0.164436,0.689046,0.789359,0.253846,0.175532,0.458333,694,155,39,33,0.072420,0.037998,0.287694,V4_EMA_LOGITADJ,none,1
2,0.270,0.192316,0.691434,0.763301,0.221429,0.149038,0.430556,672,177,41,31,0.086104,0.122262,0.316860,V4_EMA_LOGITADJ,none,2
3,0.165,0.128755,0.631004,0.731813,0.168350,0.111111,0.347222,649,200,47,25,0.077120,0.049162,0.294029,V4_EMA_LOGITADJ,temp,0
4,0.275,0.153226,0.700056,0.865364,0.138889,0.138889,0.138889,787,62,62,10,0.073495,0.028246,0.269126,V4_EMA_LOGITADJ,temp,1
5,0.135,0.156646,0.662888,0.807818,0.227074,0.165605,0.361111,718,131,46,26,0.070154,0.010524,0.265873,V4_EMA_LOGITADJ,temp,2


,variant,cal,pr_auc_mean,pr_auc_std,roc_auc_mean,roc_auc_std,f1_mean,f1_std,recall_mean,recall_std,brier_mean,brier_std,ece_mean,ece_std,nll_mean,nll_std,acc_mean,acc_std
2,V1_EMA,none,0.180275,0.037685,0.703447,0.040604,0.242630,0.037577,0.500000,0.027778,0.187183,0.036165,0.323840,0.062708,0.553533,0.085158,0.752081,0.042057
7,V3_LOGITADJ,temp,0.158031,0.023803,0.667834,0.053867,0.210969,0.039511,0.388889,0.083333,0.073257,0.004061,0.036879,0.014646,0.277932,0.026665,0.771263,0.038623
1,V0_BASE,temp,0.156936,0.045024,0.675501,0.053969,0.181150,0.059973,0.333333,0.194444,0.197853,0.025536,0.307013,0.058063,0.572839,0.062994,0.781397,0.089597
8,V4_EMA_LOGITADJ,none,0.156929,0.039677,0.656529,0.058401,0.212188,0.046965,0.388889,0.097222,0.079161,0.006844,0.078690,0.042206,0.337465,0.062667,0.775606,0.013089
0,V0_BASE,none,0.156875,0.047998,0.670184,0.064269,0.189195,0.056683,0.333333,0.187885,0.188225,0.015906,0.263927,0.055630,0.557428,0.050957,0.791169,0.055069
3,V1_EMA,temp,0.151894,0.038968,0.658482,0.066896,0.180125,0.055426,0.342593,0.191613,0.183316,0.014951,0.285248,0.030057,0.538622,0.034758,0.775606,0.068845
9,V4_EMA_LOGITADJ,temp,0.146209,0.015212,0.664649,0.034560,0.178104,0.044895,0.282407,0.124485,0.073589,0.003484,0.029311,0.019341,0.276342,0.015403,0.801665,0.066988
6,V3_LOGITADJ,none,0.129910,0.018116,0.644604,0.041170,0.111011,0.065589,0.101852,0.080188,0.081429,0.006624,0.071639,0.018469,0.479404,0.200212,0.882374,0.013748
5,V2_SWA,temp,0.094799,0.014137,0.531453,0.063622,0.145429,0.023231,0.537037,0.354189,0.201663,0.067029,0.312430,0.138712,0.583922,0.151128,0.518277,0.319636
4,V2_SWA,none,0.083019,0.020020,0.471587,0.078433,0.108575,0.046450,0.587963,0.493072,0.277925,0.012729,0.436504,0.020217,0.752829,0.027832,0.390879,0.436937


✅ Saved UPDATED sweep tables to: /kaggle/working/outputs/cgt_stroke_v1/sweep_multi_improvements/tables


In [30]:
# ============================================================
# FULL RUNNABLE (Kaggle CPU): CG-WideDeep + Multi-Improvement Sweep
# - Loads: /kaggle/working/outputs/cgt_stroke_v1/data_clean_v2.csv
# - Trains CG-WideDeep with:
#     V0_BASE         : entropy schedule + pos_weight BCE
#     V1_EMA          : V0 + EMA weights for eval
#     V3_LOGITADJ     : V0 + logit adjustment (class prior)
#     V4_EMA_LOGITADJ : V0 + EMA + logit adjustment
# - Optional calibration on VAL: none / temperature
# - Saves tables under: /kaggle/working/outputs/cgt_stroke_v1/sweep_multi_improvements/
# ============================================================

import os, json, math, random, time
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import (
    roc_auc_score, average_precision_score, f1_score, precision_score, recall_score, accuracy_score, log_loss
)

# ---------------------------
# 0) Paths / Config
# ---------------------------
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)

BASE_DIR = "/kaggle/working/outputs/cgt_stroke_v1"
DATA_CSV = os.path.join(BASE_DIR, "data_clean_v2.csv")

OUT_DIR = os.path.join(BASE_DIR, "sweep_multi_improvements")
DIRS = {
    "base": OUT_DIR,
    "tables": os.path.join(OUT_DIR, "tables"),
    "pred": os.path.join(OUT_DIR, "pred"),
    "logs": os.path.join(OUT_DIR, "logs"),
}
for k,v in DIRS.items():
    if k != "base":
        os.makedirs(v, exist_ok=True)

class CFG:
    seed = 42
    repeats = 3  # increase to 5 when stable

    batch_size = 256
    lr = 2e-3
    weight_decay = 1e-4
    max_epochs = 60
    patience = 10

    use_pos_weight = True
    ece_bins = 15

    # model
    d_embed = 16
    deep_hidden = 64
    deep_layers = 2
    dropout = 0.10

    # gate reg (base)
    gate_entropy_lambda_max = 0.02   # scheduled
    other_gate_penalty = 0.00        # use only if "Other" dominates strongly

    # EMA
    use_ema = True
    ema_decay = 0.995

    # logit adjustment strength (1.0 is typical; try 0.5–2.0)
    logit_adj_tau = 1.0

cfg = CFG()

def seed_everything(seed=42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(cfg.seed)

# ---------------------------
# 1) Utils
# ---------------------------
def expected_calibration_error(y_true, p, n_bins=15):
    y_true = np.asarray(y_true).astype(int)
    p = np.asarray(p).astype(float)
    bins = np.linspace(0.0, 1.0, n_bins+1)
    ece = 0.0
    for i in range(n_bins):
        lo, hi = bins[i], bins[i+1]
        m = (p >= lo) & (p < hi) if i < n_bins-1 else (p >= lo) & (p <= hi)
        if m.sum() == 0:
            continue
        acc = y_true[m].mean()
        conf = p[m].mean()
        ece += (m.sum()/len(p)) * abs(acc - conf)
    return float(ece)

def find_best_threshold(y, p, metric="f1"):
    y = np.asarray(y).astype(int)
    p = np.asarray(p).astype(float)
    ts = np.linspace(0.05, 0.95, 181)
    best_t, best_s = 0.5, -1
    for t in ts:
        pred = (p >= t).astype(int)
        if metric == "f1":
            s = f1_score(y, pred, zero_division=0)
        else:
            raise ValueError("Unknown metric")
        if s > best_s:
            best_s, best_t = s, t
    return float(best_t), float(best_s)

def evaluate_binary(y, p, threshold=0.5, ece_bins=15):
    y = np.asarray(y).astype(int)
    p = np.asarray(p).astype(float)
    p = np.clip(p, 1e-7, 1-1e-7)
    pred = (p >= threshold).astype(int)

    out = {}
    out["threshold"] = float(threshold)
    out["pr_auc"] = float(average_precision_score(y, p))
    out["roc_auc"] = float(roc_auc_score(y, p))
    out["acc"] = float(accuracy_score(y, pred))
    out["f1"] = float(f1_score(y, pred, zero_division=0))
    out["precision"] = float(precision_score(y, pred, zero_division=0))
    out["recall"] = float(recall_score(y, pred, zero_division=0))

    tn = int(((pred==0) & (y==0)).sum())
    fp = int(((pred==1) & (y==0)).sum())
    fn = int(((pred==0) & (y==1)).sum())
    tp = int(((pred==1) & (y==1)).sum())
    out.update({"tn":tn, "fp":fp, "fn":fn, "tp":tp})

    out["brier"] = float(np.mean((p - y)**2))
    out["ece"] = float(expected_calibration_error(y, p, n_bins=ece_bins))
    out["nll"] = float(log_loss(y, p, labels=[0,1]))
    return out

def save_table(df, name, index=False):
    path = os.path.join(DIRS["tables"], f"{name}.csv")
    df.to_csv(path, index=index)
    print("✅ Saved:", path)
    return path

# ---------------------------
# 2) Load data
# ---------------------------
assert os.path.exists(DATA_CSV), f"Missing: {DATA_CSV}"
df = pd.read_csv(DATA_CSV)

LABEL_COL = "stroke" if "stroke" in df.columns else df.columns[-1]
print("Label column:", LABEL_COL)
y_all = df[LABEL_COL].values.astype(int)

# ---------------------------
# 3) Feature typing
# ---------------------------
bin_cols = ['gender','alcohol','smoke','sleep_disorder','Health_Insurance','diabetes','hypertension','high_cholesterol','Coronary_Heart_Disease']
cat_cols = ['Race','Marital_status']
cont_cols = [c for c in df.columns if c not in ([LABEL_COL] + bin_cols + cat_cols)]
print("Counts:", {"bin":len(bin_cols), "cat":len(cat_cols), "cont":len(cont_cols)})

# ---------------------------
# 4) Split per repeat
# ---------------------------
def make_split(seed):
    sss = StratifiedShuffleSplit(n_splits=1, test_size=0.20, random_state=seed)
    (tr_idx, te_idx) = next(sss.split(df, y_all))
    y_tr = y_all[tr_idx]
    sss2 = StratifiedShuffleSplit(n_splits=1, test_size=0.20, random_state=seed+1337)
    (tr2, va2) = next(sss2.split(np.zeros(len(tr_idx)), y_tr))
    tr2_idx = tr_idx[tr2]
    va_idx = tr_idx[va2]
    return tr2_idx, va_idx, te_idx

# ---------------------------
# 5) Dataset
# ---------------------------
def to_long_tensor(x):  return torch.as_tensor(x, dtype=torch.long)
def to_float_tensor(x): return torch.as_tensor(x, dtype=torch.float32)

class TabDS(Dataset):
    def __init__(self, Xc, Xk, Xb, y):
        self.Xc = Xc; self.Xk = Xk; self.Xb = Xb; self.y = y
    def __len__(self): return len(self.y)
    def __getitem__(self, i):
        return self.Xc[i], self.Xk[i], self.Xb[i], self.y[i]

# ---------------------------
# 6) Preprocess (assumes v2 cleaned)
# ---------------------------
def fit_preprocess(train_df):
    bmap = {}
    for c in bin_cols:
        vals = train_df[c].values
        uniq = np.unique(vals)
        if set(uniq.tolist()) == {0,1}:
            bmap[c] = ("01", None)
        elif set(uniq.tolist()) == {1,2}:
            bmap[c] = ("12", None)
        else:
            if len(uniq)==2:
                bmap[c] = ("map", {uniq[0]:0, uniq[1]:1})
            else:
                raise ValueError(f"Binary col {c} has >2 unique: {uniq}")

    cat_vocab = {}
    for c in cat_cols:
        uniq = pd.Series(train_df[c]).astype(int).unique()
        uniq = np.sort(uniq)
        cat_vocab[c] = {v:i for i,v in enumerate(uniq.tolist())}

    mu = train_df[cont_cols].values.astype(np.float32).mean(axis=0)
    sd = train_df[cont_cols].values.astype(np.float32).std(axis=0)
    sd = np.where(sd < 1e-6, 1.0, sd)
    return bmap, cat_vocab, mu, sd

def apply_preprocess(df_part, bmap, cat_vocab, mu, sd):
    Xb = np.zeros((len(df_part), len(bin_cols)), dtype=np.int64)
    for j,c in enumerate(bin_cols):
        vals = df_part[c].values
        mode, mapping = bmap[c]
        if mode == "01":
            Xb[:,j] = vals.astype(int)
        elif mode == "12":
            Xb[:,j] = (vals.astype(int) - 1)
        else:
            Xb[:,j] = np.vectorize(mapping.get)(vals)
        Xb[:,j] = np.clip(Xb[:,j], 0, 1)

    Xk = np.zeros((len(df_part), len(cat_cols)), dtype=np.int64)
    for j,c in enumerate(cat_cols):
        vocab = cat_vocab[c]
        vals = df_part[c].astype(int).values
        Xk[:,j] = np.array([vocab.get(v, 0) for v in vals], dtype=np.int64)

    Xc = df_part[cont_cols].values.astype(np.float32)
    Xc = (Xc - mu) / sd
    return Xc, Xk, Xb

# ---------------------------
# 7) Concept groups (only used for naming; gating is global here)
# ---------------------------
cg_path = os.path.join(BASE_DIR, "logs", "concept_groups.json")
if os.path.exists(cg_path):
    concept_groups = json.load(open(cg_path, "r"))
else:
    concept_groups = {
        "Demographics": ["gender","age","Race","Marital_status"],
        "Lifestyle": ["alcohol","smoke","sleep_time","Minutes_sedentary_activity"],
        "Comorbidity": ["diabetes","hypertension","high_cholesterol","Coronary_Heart_Disease"],
        "Nutrition_Macro": ["energy","Carbohydrate","protein","Total_fat","Dietary_fiber","Body_Mass_Index"],
        "Nutrition_FattyAcids": ["Total_saturated_fatty_acids","Total_monounsaturated_fatty_acids","Total_polyunsaturated_fatty_acids"],
        "Nutrition_Minerals": ["Sodium","Potassium"],
        "Other": [c for c in cont_cols if c not in [
            "age","sleep_time","Minutes_sedentary_activity","energy","Carbohydrate","protein","Total_fat",
            "Dietary_fiber","Body_Mass_Index","Total_saturated_fatty_acids","Total_monounsaturated_fatty_acids",
            "Total_polyunsaturated_fatty_acids","Sodium","Potassium"
        ]]
    }
concept_names = list(concept_groups.keys())
print("Concept groups:", {k:len(v) for k,v in concept_groups.items()})
other_idx = concept_names.index("Other") if "Other" in concept_names else None

# ---------------------------
# 8) Model
# ---------------------------
class WideDeep(nn.Module):
    def __init__(self, n_cont, cat_cardinalities, n_bin, d_embed=16, deep_hidden=64, deep_layers=2, dropout=0.1):
        super().__init__()
        self.cat_embeds = nn.ModuleList([nn.Embedding(card, d_embed) for card in cat_cardinalities])
        wide_in = n_cont + n_bin + len(cat_cardinalities)*d_embed
        self.wide = nn.Linear(wide_in, 1)

        layers = []
        din = wide_in
        for _ in range(deep_layers):
            layers += [nn.Linear(din, deep_hidden), nn.ReLU(), nn.Dropout(dropout)]
            din = deep_hidden
        layers += [nn.Linear(din, 1)]
        self.deep = nn.Sequential(*layers)

    def forward(self, x_cont, x_cat, x_bin):
        cat_vecs = [emb(x_cat[:, j]) for j, emb in enumerate(self.cat_embeds)]
        cat_vec = torch.cat(cat_vecs, dim=1) if len(cat_vecs) else torch.zeros((x_cont.size(0),0), device=x_cont.device)
        z = torch.cat([x_cont, x_bin.float(), cat_vec], dim=1)
        base = self.wide(z).squeeze(1) + self.deep(z).squeeze(1)
        return base, z

class ConceptGatedResidual(nn.Module):
    def __init__(self, z_dim, n_concepts, hidden=64, dropout=0.1):
        super().__init__()
        self.gate = nn.Sequential(
            nn.Linear(z_dim, hidden), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden, n_concepts)
        )
        self.experts = nn.ModuleList([
            nn.Sequential(
                nn.Linear(z_dim, hidden), nn.ReLU(), nn.Dropout(dropout),
                nn.Linear(hidden, 1)
            ) for _ in range(n_concepts)
        ])
        self.alpha = nn.Parameter(torch.tensor(-2.0))  # sigmoid(-2)=0.119

    def forward(self, z):
        g_logits = self.gate(z)
        g = torch.softmax(g_logits, dim=1)
        E = torch.stack([ex(z).squeeze(1) for ex in self.experts], dim=1)
        res = (g * E).sum(dim=1)
        res = torch.sigmoid(self.alpha) * res
        return res, g, E

class CGWideDeep(nn.Module):
    def __init__(self, n_cont, cat_cardinalities, n_bin, concept_names,
                 d_embed=16, deep_hidden=64, deep_layers=2, dropout=0.1):
        super().__init__()
        self.base = WideDeep(n_cont, cat_cardinalities, n_bin, d_embed, deep_hidden, deep_layers, dropout)
        z_dim = n_cont + n_bin + len(cat_cardinalities)*d_embed
        self.concept_names = concept_names
        self.cg = ConceptGatedResidual(z_dim=z_dim, n_concepts=len(concept_names), hidden=deep_hidden, dropout=dropout)

    def forward(self, x_cont, x_cat, x_bin):
        base_logit, z = self.base(x_cont, x_cat, x_bin)
        res, g, E = self.cg(z)
        return base_logit + res, g, E

# ---------------------------
# 9) EMA helper
# ---------------------------
class EMA:
    def __init__(self, model, decay=0.995):
        self.decay = decay
        self.shadow = {}
        self.backup = {}
        for n, p in model.named_parameters():
            if p.requires_grad:
                self.shadow[n] = p.detach().clone()

    @torch.no_grad()
    def update(self, model):
        for n, p in model.named_parameters():
            if p.requires_grad:
                self.shadow[n].mul_(self.decay).add_(p.detach(), alpha=(1.0 - self.decay))

    def apply_shadow(self, model):
        self.backup = {}
        for n, p in model.named_parameters():
            if p.requires_grad:
                self.backup[n] = p.detach().clone()
                p.data.copy_(self.shadow[n].data)

    def restore(self, model):
        for n, p in model.named_parameters():
            if p.requires_grad:
                p.data.copy_(self.backup[n].data)
        self.backup = {}

# ---------------------------
# 10) Prediction
# ---------------------------
@torch.no_grad()
def predict_proba(model, loader, logit_adjust=0.0, use_temp=False, temp=1.0):
    model.eval()
    ps, ys = [], []
    for xc, xk, xb, yb in loader:
        xc = xc.to(DEVICE); xk = xk.to(DEVICE); xb = xb.to(DEVICE)
        logit, g, E = model(xc, xk, xb)
        logit = logit + logit_adjust
        if use_temp:
            logit = logit / max(temp, 1e-6)
        p = torch.sigmoid(logit).detach().cpu().numpy()
        ps.append(p)
        ys.append(yb.numpy())
    return np.concatenate(ys), np.concatenate(ps)

# ---------------------------
# 11) Temperature scaling fit on VAL (simple 1D search)
# ---------------------------
def fit_temperature_on_val(model, va_loader, logit_adjust=0.0):
    # grid search (fast enough on CPU)
    temps = np.linspace(0.7, 3.0, 60)
    best_t, best_nll = 1.0, 1e18
    yv, pv = predict_proba(model, va_loader, logit_adjust=logit_adjust, use_temp=False, temp=1.0)
    # convert prob->logit for nll evaluation under T:
    pv = np.clip(pv, 1e-7, 1-1e-7)
    lv = np.log(pv/(1-pv))
    for t in temps:
        p = 1/(1+np.exp(-(lv/t)))
        nll = log_loss(yv, np.clip(p,1e-7,1-1e-7), labels=[0,1])
        if nll < best_nll:
            best_nll = nll
            best_t = float(t)
    return best_t

# ---------------------------
# 12) Train one run (variant + calibration)
# ---------------------------
def train_one(repeat, variant="V0_BASE", cal="none"):
    tr_idx, va_idx, te_idx = make_split(cfg.seed + 1000*repeat)
    df_tr = df.iloc[tr_idx].copy()
    df_va = df.iloc[va_idx].copy()
    df_te = df.iloc[te_idx].copy()

    bmap, cat_vocab, mu, sd = fit_preprocess(df_tr)

    Xc_tr, Xk_tr, Xb_tr = apply_preprocess(df_tr, bmap, cat_vocab, mu, sd)
    Xc_va, Xk_va, Xb_va = apply_preprocess(df_va, bmap, cat_vocab, mu, sd)
    Xc_te, Xk_te, Xb_te = apply_preprocess(df_te, bmap, cat_vocab, mu, sd)

    y_tr = df_tr[LABEL_COL].values.astype(int)
    y_va = df_va[LABEL_COL].values.astype(int)
    y_te = df_te[LABEL_COL].values.astype(int)

    cat_cards = [len(cat_vocab[c]) for c in cat_cols]

    tr_ds = TabDS(to_float_tensor(Xc_tr), to_long_tensor(Xk_tr), to_long_tensor(Xb_tr), to_long_tensor(y_tr))
    va_ds = TabDS(to_float_tensor(Xc_va), to_long_tensor(Xk_va), to_long_tensor(Xb_va), to_long_tensor(y_va))
    te_ds = TabDS(to_float_tensor(Xc_te), to_long_tensor(Xk_te), to_long_tensor(Xb_te), to_long_tensor(y_te))

    tr_loader = DataLoader(tr_ds, batch_size=cfg.batch_size, shuffle=True, drop_last=False)
    va_loader = DataLoader(va_ds, batch_size=cfg.batch_size, shuffle=False)
    te_loader = DataLoader(te_ds, batch_size=cfg.batch_size, shuffle=False)

    model = CGWideDeep(
        n_cont=len(cont_cols),
        cat_cardinalities=cat_cards,
        n_bin=len(bin_cols),
        concept_names=concept_names,
        d_embed=cfg.d_embed,
        deep_hidden=cfg.deep_hidden,
        deep_layers=cfg.deep_layers,
        dropout=cfg.dropout
    ).to(DEVICE)

    opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)

    # pos_weight BCE
    if cfg.use_pos_weight:
        pos = (y_tr==1).sum()
        neg = (y_tr==0).sum()
        posw = torch.tensor([neg/(pos+1e-12)], device=DEVICE, dtype=torch.float32)
        crit = nn.BCEWithLogitsLoss(pos_weight=posw)
    else:
        crit = nn.BCEWithLogitsLoss()

    # Variant controls
    use_ema = ("EMA" in variant)
    use_logitadj = ("LOGITADJ" in variant)

    # logit adjustment: add tau * log(pi/(1-pi))
    pi = float((y_tr==1).mean())
    logit_adj = cfg.logit_adj_tau * math.log((pi+1e-12)/(1.0-pi+1e-12)) if use_logitadj else 0.0

    ema = EMA(model, decay=cfg.ema_decay) if use_ema else None

    best_pr = -1.0
    best_state = None
    best_ema_shadow = None
    bad = 0

    # entropy schedule: push entropy early (negative sign), then release later
    # We implement: loss = bce - lam(ep)*entropy
    # lam(ep) ramps to max by ~1/3 epochs and then decays to 0 by ~2/3.
    def entropy_lambda(ep, max_epochs):
        t = ep / max(1, max_epochs-1)
        if t < 0.33:
            return cfg.gate_entropy_lambda_max * (t/0.33)
        elif t < 0.66:
            # decay to 0
            return cfg.gate_entropy_lambda_max * (1.0 - (t-0.33)/0.33)
        else:
            return 0.0

    for ep in range(cfg.max_epochs):
        model.train()
        ep_loss = 0.0
        n_seen = 0

        lam_ent = entropy_lambda(ep, cfg.max_epochs)

        for xc, xk, xb, yb in tr_loader:
            xc = xc.to(DEVICE); xk = xk.to(DEVICE); xb = xb.to(DEVICE)
            yb = yb.to(DEVICE).float()

            opt.zero_grad()
            logit, g, E = model(xc, xk, xb)
            logit = logit + logit_adj

            loss = crit(logit, yb)

            # gate entropy regularization (maximize entropy slightly early)
            if lam_ent > 0:
                ent = -(g * (g + 1e-12).log()).sum(dim=1).mean()
                loss = loss - lam_ent * ent

            # optional other penalty
            if (cfg.other_gate_penalty > 0) and (other_idx is not None):
                loss = loss + cfg.other_gate_penalty * g[:, other_idx].mean()

            loss.backward()
            opt.step()

            if ema is not None:
                ema.update(model)

            bs = yb.size(0)
            ep_loss += loss.item() * bs
            n_seen += bs

        # eval on val PR using (EMA if enabled)
        if ema is not None:
            ema.apply_shadow(model)

        yv, pv = predict_proba(model, va_loader, logit_adjust=0.0, use_temp=False, temp=1.0)
        # note: pv already includes logit_adj during training above; for selection we keep consistent:
        # so re-run with logit_adj applied:
        yv, pv = predict_proba(model, va_loader, logit_adjust=logit_adj, use_temp=False, temp=1.0)
        val_pr = average_precision_score(yv, pv)

        if ema is not None:
            ema.restore(model)

        if (ep % 1) == 0:
            print(f"{variant} r{repeat} ep{ep:03d} loss={ep_loss/max(n_seen,1):.4f} valPR={val_pr:.4f} (lam_ent={lam_ent:.4f})")

        if val_pr > best_pr + 1e-5:
            best_pr = val_pr
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            if ema is not None:
                best_ema_shadow = {k: v.detach().cpu().clone() for k, v in ema.shadow.items()}
            bad = 0
        else:
            bad += 1
            if bad >= cfg.patience:
                break

    # load best weights
    model.load_state_dict(best_state)
    if ema is not None and best_ema_shadow is not None:
        ema.shadow = {k: v.clone() for k,v in best_ema_shadow.items()}

    # apply EMA for final eval (if enabled)
    if ema is not None:
        ema.apply_shadow(model)

    # calibration fit on VAL (optional)
    temp_val = 1.0
    use_temp = (cal == "temp")
    if use_temp:
        temp_val = fit_temperature_on_val(model, va_loader, logit_adjust=logit_adj)

    # VAL predictions (with calibration)
    y_va_np, p_va = predict_proba(model, va_loader, logit_adjust=logit_adj, use_temp=use_temp, temp=temp_val)
    # threshold on val f1
    t_best, _ = find_best_threshold(y_va_np, p_va, metric="f1")

    # TEST predictions
    y_te_np, p_te = predict_proba(model, te_loader, logit_adjust=logit_adj, use_temp=use_temp, temp=temp_val)

    m_te = evaluate_binary(y_te_np, p_te, threshold=t_best, ece_bins=cfg.ece_bins)
    m_te.update({"variant": variant, "cal": cal, "repeat": repeat})

    if ema is not None:
        ema.restore(model)

    # save a small json for reproducibility
    with open(os.path.join(DIRS["pred"], f"{variant}_{cal}_r{repeat}.json"), "w") as f:
        json.dump({"t_best": t_best, "best_val_pr": float(best_pr), "temp": float(temp_val), "metrics_test": m_te}, f, indent=2)

    return m_te

# ---------------------------
# 13) Run sweep
# ---------------------------
VARIANTS = [
    ("V0_BASE", dict()),
    ("V1_EMA", dict()),
    ("V3_LOGITADJ", dict()),
    ("V4_EMA_LOGITADJ", dict()),
]
CALS = ["none", "temp"]  # add "none" only if you want faster

rows = []
for variant, _kw in VARIANTS:
    for cal in CALS:
        for r in range(cfg.repeats):
            m = train_one(r, variant=variant, cal=cal)
            rows.append(m)
            print(f"--> DONE {variant} {cal} r{r}: PR={m['pr_auc']:.4f} ROC={m['roc_auc']:.4f} F1={m['f1']:.4f}")

df_res = pd.DataFrame(rows)
display(df_res)
save_table(df_res, "sweep_per_repeat_test", index=False)

# summary by (variant, cal)
agg_cols = ["pr_auc","roc_auc","f1","recall","brier","ece","nll","acc"]
summary = df_res.groupby(["variant","cal"])[agg_cols].agg(["mean","std"])
summary.columns = ["_".join(c).strip() for c in summary.columns.values]
summary = summary.reset_index()
display(summary)
save_table(summary, "sweep_test_summary_by_variant_cal", index=False)

print("\n✅ Saved sweep tables to:", DIRS["tables"])

Using device: cpu
Label column: stroke
Counts: {'bin': 9, 'cat': 2, 'cont': 24}
Concept groups: {'Demographics': 4, 'Lifestyle': 4, 'Comorbidity': 4, 'Nutrition_Macro': 6, 'Nutrition_FattyAcids': 3, 'Nutrition_Minerals': 2, 'Other': 12}
V0_BASE r0 ep000 loss=1.3643 valPR=0.1041 (lam_ent=0.0000)
V0_BASE r0 ep001 loss=1.2309 valPR=0.1451 (lam_ent=0.0010)
V0_BASE r0 ep002 loss=1.1424 valPR=0.1705 (lam_ent=0.0021)
V0_BASE r0 ep003 loss=1.0917 valPR=0.1735 (lam_ent=0.0031)
V0_BASE r0 ep004 loss=1.0456 valPR=0.1696 (lam_ent=0.0041)
V0_BASE r0 ep005 loss=1.0096 valPR=0.1820 (lam_ent=0.0051)
V0_BASE r0 ep006 loss=0.9805 valPR=0.1720 (lam_ent=0.0062)
V0_BASE r0 ep007 loss=0.9497 valPR=0.1831 (lam_ent=0.0072)
V0_BASE r0 ep008 loss=0.9231 valPR=0.1675 (lam_ent=0.0082)
V0_BASE r0 ep009 loss=0.8970 valPR=0.1776 (lam_ent=0.0092)
V0_BASE r0 ep010 loss=0.8409 valPR=0.1824 (lam_ent=0.0103)
V0_BASE r0 ep011 loss=0.8271 valPR=0.1712 (lam_ent=0.0113)
V0_BASE r0 ep012 loss=0.7825 valPR=0.1724 (lam_ent=0.01

,threshold,pr_auc,roc_auc,acc,f1,precision,recall,tn,fp,fn,tp,brier,ece,nll,variant,cal,repeat
0,0.420,0.092456,0.570999,0.771987,0.125000,0.089286,0.208333,696,153,57,15,0.152196,0.165368,0.650993,V0_BASE,none,0
1,0.785,0.141214,0.695459,0.843648,0.181818,0.153846,0.222222,761,88,56,16,0.214205,0.288861,0.611717,V0_BASE,none,1
2,0.695,0.183682,0.723596,0.845820,0.260417,0.208333,0.347222,754,95,47,25,0.201048,0.319937,0.578435,V0_BASE,none,2
3,0.445,0.125740,0.616510,0.689468,0.182857,0.115108,0.444444,603,246,40,32,0.168310,0.238422,0.503858,V0_BASE,temp,0
4,0.640,0.162987,0.721650,0.838219,0.235897,0.186992,0.319444,749,100,49,23,0.165579,0.256734,0.496361,V0_BASE,temp,1
5,0.600,0.175574,0.703409,0.775244,0.241758,0.164179,0.458333,681,168,39,33,0.188188,0.296156,0.547195,V0_BASE,temp,2
6,0.440,0.123675,0.609753,0.820847,0.126984,0.102564,0.166667,744,105,60,12,0.131611,0.134180,0.591894,V1_EMA,none,0
7,0.750,0.156817,0.710754,0.889251,0.163934,0.200000,0.138889,809,40,62,10,0.149983,0.201956,0.446802,V1_EMA,none,1
8,0.640,0.181249,0.707712,0.833876,0.238806,0.186047,0.333333,744,105,48,24,0.216029,0.358666,0.615635,V1_EMA,none,2
9,0.510,0.131809,0.616166,0.757872,0.177122,0.120603,0.333333,674,175,48,24,0.160802,0.231370,0.494552,V1_EMA,temp,0


✅ Saved: /kaggle/working/outputs/cgt_stroke_v1/sweep_multi_improvements/tables/sweep_per_repeat_test.csv


,variant,cal,pr_auc_mean,pr_auc_std,roc_auc_mean,roc_auc_std,f1_mean,f1_std,recall_mean,recall_std,brier_mean,brier_std,ece_mean,ece_std,nll_mean,nll_std,acc_mean,acc_std
0,V0_BASE,none,0.139118,0.045649,0.663351,0.081208,0.189078,0.068000,0.259259,0.076494,0.189150,0.032672,0.258055,0.081760,0.613715,0.036320,0.820485,0.042015
1,V0_BASE,temp,0.154767,0.025914,0.680523,0.056182,0.220171,0.032447,0.407407,0.076494,0.174025,0.012341,0.263771,0.029504,0.515805,0.027442,0.767644,0.074666
2,V1_EMA,none,0.153914,0.028897,0.676073,0.057455,0.176575,0.056973,0.212963,0.105165,0.165874,0.044396,0.231601,0.115142,0.551444,0.091396,0.847991,0.036321
3,V1_EMA,temp,0.135025,0.008201,0.650907,0.035974,0.186333,0.044421,0.263889,0.086736,0.146632,0.016270,0.201409,0.035560,0.451784,0.050290,0.819399,0.053295
4,V3_LOGITADJ,none,0.150667,0.022001,0.669284,0.042844,0.157018,0.078837,0.212963,0.136319,0.212080,0.028629,0.307154,0.064077,0.617492,0.055179,0.838581,0.044521
5,V3_LOGITADJ,temp,0.173770,0.039547,0.697765,0.063985,0.231021,0.028943,0.458333,0.210177,0.202272,0.010653,0.332452,0.023440,0.589268,0.026009,0.768006,0.077466
6,V4_EMA_LOGITADJ,none,0.138390,0.029188,0.661912,0.050090,0.182147,0.045326,0.296296,0.105165,0.117159,0.006884,0.177639,0.016009,0.388493,0.009953,0.791893,0.044904
7,V4_EMA_LOGITADJ,temp,0.162579,0.037944,0.681619,0.055098,0.207677,0.036352,0.495370,0.105165,0.101353,0.013837,0.126720,0.018033,0.341791,0.037908,0.705393,0.030813


✅ Saved: /kaggle/working/outputs/cgt_stroke_v1/sweep_multi_improvements/tables/sweep_test_summary_by_variant_cal.csv

✅ Saved sweep tables to: /kaggle/working/outputs/cgt_stroke_v1/sweep_multi_improvements/tables


In [31]:
# ============================================================
# Kaggle FULL RUNNABLE: Attention + Imbalance Techniques Sweep
# - Data: /kaggle/working/outputs/cgt_stroke_v1/data_clean_v2.csv
# - Tries: MLP, MLP+AFL, FT-Transformer, FT-Transformer+AUPRC-Rank
# - Saves: tables to /kaggle/working/outputs/cgt_stroke_v1/attn_imbalance_sweep/tables
# ============================================================

import os, math, random, json
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, precision_score, recall_score, accuracy_score, log_loss

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE)

BASE_DIR = "/kaggle/working/outputs/cgt_stroke_v1"
DATA_CSV = os.path.join(BASE_DIR, "data_clean_v2.csv")
assert os.path.exists(DATA_CSV), f"Missing {DATA_CSV}"
df = pd.read_csv(DATA_CSV)

LABEL_COL = "stroke" if "stroke" in df.columns else df.columns[-1]
y_all = df[LABEL_COL].values.astype(int)

OUT_DIR = os.path.join(BASE_DIR, "attn_imbalance_sweep")
TAB_DIR = os.path.join(OUT_DIR, "tables")
os.makedirs(TAB_DIR, exist_ok=True)

# ---- feature lists (match your setup) ----
bin_cols = ['gender','alcohol','smoke','sleep_disorder','Health_Insurance','diabetes','hypertension','high_cholesterol','Coronary_Heart_Disease']
cat_cols = ['Race','Marital_status']
cont_cols = [c for c in df.columns if c not in ([LABEL_COL] + bin_cols + cat_cols)]
print("Counts:", {"bin":len(bin_cols), "cat":len(cat_cols), "cont":len(cont_cols)})

# ---------------------------
# Utils
# ---------------------------
def seed_everything(seed=42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def expected_calibration_error(y_true, p, n_bins=15):
    y_true = np.asarray(y_true).astype(int)
    p = np.asarray(p).astype(float)
    bins = np.linspace(0.0, 1.0, n_bins+1)
    ece = 0.0
    for i in range(n_bins):
        lo, hi = bins[i], bins[i+1]
        m = (p >= lo) & (p < hi) if i < n_bins-1 else (p >= lo) & (p <= hi)
        if m.sum() == 0: 
            continue
        acc = y_true[m].mean()
        conf = p[m].mean()
        ece += (m.sum()/len(p)) * abs(acc - conf)
    return float(ece)

def find_best_threshold(y, p):
    y = np.asarray(y).astype(int)
    p = np.asarray(p).astype(float)
    ts = np.linspace(0.05, 0.95, 181)
    best_t, best_f1 = 0.5, -1
    for t in ts:
        pred = (p >= t).astype(int)
        f1 = f1_score(y, pred, zero_division=0)
        if f1 > best_f1:
            best_f1 = f1
            best_t = t
    return float(best_t)

def evaluate_binary(y, p, threshold=0.5, ece_bins=15):
    y = np.asarray(y).astype(int)
    p = np.asarray(p).astype(float)
    p = np.clip(p, 1e-7, 1-1e-7)
    pred = (p >= threshold).astype(int)
    out = {}
    out["threshold"] = float(threshold)
    out["pr_auc"] = float(average_precision_score(y, p))
    out["roc_auc"] = float(roc_auc_score(y, p))
    out["acc"] = float(accuracy_score(y, pred))
    out["f1"] = float(f1_score(y, pred, zero_division=0))
    out["precision"] = float(precision_score(y, pred, zero_division=0))
    out["recall"] = float(recall_score(y, pred, zero_division=0))
    tn = int(((pred==0) & (y==0)).sum())
    fp = int(((pred==1) & (y==0)).sum())
    fn = int(((pred==0) & (y==1)).sum())
    tp = int(((pred==1) & (y==1)).sum())
    out.update({"tn":tn,"fp":fp,"fn":fn,"tp":tp})
    out["brier"] = float(np.mean((p - y)**2))
    out["ece"] = float(expected_calibration_error(y, p, n_bins=ece_bins))
    out["nll"] = float(log_loss(y, p, labels=[0,1]))
    return out

def save_csv(df_, name):
    path = os.path.join(TAB_DIR, f"{name}.csv")
    df_.to_csv(path, index=False)
    print("✅ Saved:", path)
    return path

# ---------------------------
# Split per repeat
# ---------------------------
def make_split(seed):
    sss = StratifiedShuffleSplit(n_splits=1, test_size=0.20, random_state=seed)
    tr_idx, te_idx = next(sss.split(df, y_all))
    y_tr = y_all[tr_idx]
    sss2 = StratifiedShuffleSplit(n_splits=1, test_size=0.20, random_state=seed+1337)
    tr2, va2 = next(sss2.split(np.zeros(len(tr_idx)), y_tr))
    tr2_idx = tr_idx[tr2]
    va_idx = tr_idx[va2]
    return tr2_idx, va_idx, te_idx

# ---------------------------
# Preprocess (simple, assumes v2 cleaned)
# ---------------------------
def fit_preprocess(train_df):
    # bin mapping
    bmap = {}
    for c in bin_cols:
        uniq = np.unique(train_df[c].values)
        if set(uniq.tolist()) == {0,1}:
            bmap[c] = ("01", None)
        elif set(uniq.tolist()) == {1,2}:
            bmap[c] = ("12", None)
        else:
            if len(uniq)==2:
                bmap[c] = ("map", {uniq[0]:0, uniq[1]:1})
            else:
                raise ValueError(f"Binary col {c} has >2 unique: {uniq}")

    # cat vocab
    cat_vocab = {}
    for c in cat_cols:
        uniq = pd.Series(train_df[c]).astype(int).unique()
        uniq = np.sort(uniq)
        cat_vocab[c] = {v:i for i,v in enumerate(uniq.tolist())}

    # cont norm
    Xc = train_df[cont_cols].values.astype(np.float32)
    mu = Xc.mean(axis=0)
    sd = Xc.std(axis=0)
    sd = np.where(sd < 1e-6, 1.0, sd)
    return bmap, cat_vocab, mu, sd

def apply_preprocess(df_part, bmap, cat_vocab, mu, sd):
    Xb = np.zeros((len(df_part), len(bin_cols)), dtype=np.int64)
    for j,c in enumerate(bin_cols):
        vals = df_part[c].values
        mode, mapping = bmap[c]
        if mode == "01":
            Xb[:,j] = vals.astype(int)
        elif mode == "12":
            Xb[:,j] = (vals.astype(int) - 1)
        else:
            Xb[:,j] = np.array([mapping.get(v,0) for v in vals], dtype=np.int64)
        Xb[:,j] = np.clip(Xb[:,j], 0, 1)

    Xk = np.zeros((len(df_part), len(cat_cols)), dtype=np.int64)
    for j,c in enumerate(cat_cols):
        vocab = cat_vocab[c]
        vals = df_part[c].astype(int).values
        Xk[:,j] = np.array([vocab.get(v,0) for v in vals], dtype=np.int64)

    Xc = df_part[cont_cols].values.astype(np.float32)
    Xc = (Xc - mu) / sd
    return Xc, Xk, Xb

class TabDS(Dataset):
    def __init__(self, Xc, Xk, Xb, y):
        self.Xc = torch.tensor(Xc, dtype=torch.float32)
        self.Xk = torch.tensor(Xk, dtype=torch.long)
        self.Xb = torch.tensor(Xb, dtype=torch.long)
        self.y  = torch.tensor(y, dtype=torch.float32)
    def __len__(self): return len(self.y)
    def __getitem__(self, i):
        return self.Xc[i], self.Xk[i], self.Xb[i], self.y[i]

# ---------------------------
# Losses
# ---------------------------
class AsymmetricFocalLoss(nn.Module):
    """
    AFL for binary:
      - uses different focusing for pos/neg
      - optional clipping for stable negatives (from ASL paper family)
    """
    def __init__(self, gamma_pos=0.0, gamma_neg=2.0, clip=0.05, eps=1e-8):
        super().__init__()
        self.gp = gamma_pos
        self.gn = gamma_neg
        self.clip = clip
        self.eps = eps

    def forward(self, logits, targets):
        targets = targets.float()
        p = torch.sigmoid(logits)
        # optional clipping on negatives
        if self.clip is not None and self.clip > 0:
            p = torch.clamp(p, self.eps, 1.0 - self.eps)
            p = torch.where(targets < 0.5, torch.clamp(p + self.clip, max=1.0), p)

        pt = torch.where(targets > 0.5, p, 1 - p)
        w = torch.where(targets > 0.5, (1-pt).pow(self.gp), (1-pt).pow(self.gn))
        bce = F.binary_cross_entropy_with_logits(logits, targets, reduction="none")
        return (w * bce).mean()

def pairwise_auprc_surrogate(logits, y, n_pairs=256):
    """
    Simple pairwise ranking term:
      encourage positives > negatives via softplus margin.
    Not a perfect AUPRC surrogate, but helps PR-AUC in practice.
    """
    y = y.detach()
    pos_idx = (y > 0.5).nonzero(as_tuple=True)[0]
    neg_idx = (y < 0.5).nonzero(as_tuple=True)[0]
    if len(pos_idx) == 0 or len(neg_idx) == 0:
        return logits.new_tensor(0.0)

    # sample pairs
    pi = pos_idx[torch.randint(0, len(pos_idx), (n_pairs,), device=logits.device)]
    ni = neg_idx[torch.randint(0, len(neg_idx), (n_pairs,), device=logits.device)]
    lp = logits[pi]
    ln = logits[ni]
    # want lp > ln
    return F.softplus(ln - lp).mean()

# ---------------------------
# Models: MLP and FT-Transformer (attention)
# ---------------------------
class MLPModel(nn.Module):
    def __init__(self, n_cont, cat_cards, n_bin, d_embed=16, hidden=128, depth=2, dropout=0.1):
        super().__init__()
        self.embs = nn.ModuleList([nn.Embedding(card, d_embed) for card in cat_cards])
        in_dim = n_cont + n_bin + len(cat_cards)*d_embed
        layers = []
        dim = in_dim
        for _ in range(depth):
            layers += [nn.Linear(dim, hidden), nn.ReLU(), nn.Dropout(dropout)]
            dim = hidden
        layers += [nn.Linear(dim, 1)]
        self.net = nn.Sequential(*layers)

    def forward(self, xc, xk, xb):
        cat_vecs = [emb(xk[:,j]) for j,emb in enumerate(self.embs)]
        cat_vec = torch.cat(cat_vecs, dim=1) if len(cat_vecs) else torch.zeros((xc.size(0),0), device=xc.device)
        z = torch.cat([xc, xb.float(), cat_vec], dim=1)
        return self.net(z).squeeze(1)

class FTTransformer(nn.Module):
    """
    Lightweight FT-Transformer style:
      - embed each numeric and categorical into tokens
      - transformer encoder over tokens
      - [CLS] token pooled -> logit
    """
    def __init__(self, n_cont, cat_cards, n_bin, d_token=32, n_heads=4, n_layers=2, dropout=0.1):
        super().__init__()
        self.n_cont = n_cont
        self.n_bin = n_bin
        self.cat_cards = cat_cards

        self.cls = nn.Parameter(torch.zeros(1, 1, d_token))

        # numeric tokenizers: per-feature linear (x -> token)
        self.cont_w = nn.Parameter(torch.randn(n_cont, d_token) * 0.02)
        self.cont_b = nn.Parameter(torch.zeros(n_cont, d_token))

        # bin tokenizers as embeddings (0/1) per bin feature
        self.bin_emb = nn.ModuleList([nn.Embedding(2, d_token) for _ in range(n_bin)])

        # categorical embeddings (per feature)
        self.cat_emb = nn.ModuleList([nn.Embedding(card, d_token) for card in cat_cards])

        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_token, nhead=n_heads, dim_feedforward=d_token*4,
            dropout=dropout, batch_first=True, activation="gelu"
        )
        self.tr = nn.TransformerEncoder(enc_layer, num_layers=n_layers)

        self.head = nn.Sequential(
            nn.LayerNorm(d_token),
            nn.Linear(d_token, 1)
        )

    def forward(self, xc, xk, xb):
        B = xc.size(0)

        # cont tokens: (B, n_cont, d)
        cont_tokens = xc.unsqueeze(-1) * self.cont_w.unsqueeze(0) + self.cont_b.unsqueeze(0)

        # bin tokens: list -> (B, n_bin, d)
        if self.n_bin > 0:
            bin_tokens = torch.stack([emb(xb[:,j]) for j,emb in enumerate(self.bin_emb)], dim=1)
        else:
            bin_tokens = torch.zeros((B,0,self.cls.size(-1)), device=xc.device)

        # cat tokens: (B, n_cat, d)
        if len(self.cat_emb) > 0:
            cat_tokens = torch.stack([emb(xk[:,j]) for j,emb in enumerate(self.cat_emb)], dim=1)
        else:
            cat_tokens = torch.zeros((B,0,self.cls.size(-1)), device=xc.device)

        tokens = torch.cat([cont_tokens, bin_tokens, cat_tokens], dim=1)
        cls = self.cls.expand(B, -1, -1)
        tokens = torch.cat([cls, tokens], dim=1)

        h = self.tr(tokens)
        cls_h = h[:,0,:]
        logit = self.head(cls_h).squeeze(1)
        return logit

# ---------------------------
# Train / Predict
# ---------------------------
@torch.no_grad()
def predict_proba(model, loader):
    model.eval()
    ps, ys = [], []
    for xc, xk, xb, y in loader:
        xc = xc.to(DEVICE); xk = xk.to(DEVICE); xb = xb.to(DEVICE)
        logit = model(xc, xk, xb)
        p = torch.sigmoid(logit).cpu().numpy()
        ps.append(p); ys.append(y.numpy())
    return np.concatenate(ys), np.concatenate(ps)

def train_run(repeat, model_name):
    seed_everything(1000 + repeat)

    tr_idx, va_idx, te_idx = make_split(1000 + repeat)
    df_tr = df.iloc[tr_idx].copy()
    df_va = df.iloc[va_idx].copy()
    df_te = df.iloc[te_idx].copy()

    bmap, cat_vocab, mu, sd = fit_preprocess(df_tr)

    Xc_tr, Xk_tr, Xb_tr = apply_preprocess(df_tr, bmap, cat_vocab, mu, sd)
    Xc_va, Xk_va, Xb_va = apply_preprocess(df_va, bmap, cat_vocab, mu, sd)
    Xc_te, Xk_te, Xb_te = apply_preprocess(df_te, bmap, cat_vocab, mu, sd)

    y_tr = df_tr[LABEL_COL].values.astype(int)
    y_va = df_va[LABEL_COL].values.astype(int)
    y_te = df_te[LABEL_COL].values.astype(int)

    cat_cards = [len(cat_vocab[c]) for c in cat_cols]

    tr_loader = DataLoader(TabDS(Xc_tr, Xk_tr, Xb_tr, y_tr), batch_size=256, shuffle=True)
    va_loader = DataLoader(TabDS(Xc_va, Xk_va, Xb_va, y_va), batch_size=512, shuffle=False)
    te_loader = DataLoader(TabDS(Xc_te, Xk_te, Xb_te, y_te), batch_size=512, shuffle=False)

    # pos_weight for BCE baseline
    pos = (y_tr==1).sum()
    neg = (y_tr==0).sum()
    posw = torch.tensor([neg/(pos+1e-12)], device=DEVICE, dtype=torch.float32)

    # choose model + loss recipe
    if model_name == "MLP_BCE":
        model = MLPModel(len(cont_cols), cat_cards, len(bin_cols), d_embed=16, hidden=128, depth=2, dropout=0.1).to(DEVICE)
        bce = nn.BCEWithLogitsLoss(pos_weight=posw)
        rank_lambda = 0.0

        def loss_fn(logits, y):
            return bce(logits, y)

    elif model_name == "MLP_AFL":
        model = MLPModel(len(cont_cols), cat_cards, len(bin_cols), d_embed=16, hidden=128, depth=2, dropout=0.1).to(DEVICE)
        afl = AsymmetricFocalLoss(gamma_pos=0.0, gamma_neg=2.0, clip=0.05)
        rank_lambda = 0.0

        def loss_fn(logits, y):
            return afl(logits, y)

    elif model_name == "FTT_BCE":
        model = FTTransformer(len(cont_cols), cat_cards, len(bin_cols), d_token=32, n_heads=4, n_layers=2, dropout=0.1).to(DEVICE)
        bce = nn.BCEWithLogitsLoss(pos_weight=posw)
        rank_lambda = 0.0

        def loss_fn(logits, y):
            return bce(logits, y)

    elif model_name == "FTT_BCE_RANK":
        model = FTTransformer(len(cont_cols), cat_cards, len(bin_cols), d_token=32, n_heads=4, n_layers=2, dropout=0.1).to(DEVICE)
        bce = nn.BCEWithLogitsLoss(pos_weight=posw)
        rank_lambda = 0.2  # try 0.1–0.5

        def loss_fn(logits, y):
            return bce(logits, y) + rank_lambda * pairwise_auprc_surrogate(logits, y, n_pairs=256)

    else:
        raise ValueError("Unknown model_name")

    opt = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=1e-4)

    best_pr = -1
    best_state = None
    patience = 10
    bad = 0

    for ep in range(60):
        model.train()
        tot = 0.0; n=0
        for xc, xk, xb, y in tr_loader:
            xc = xc.to(DEVICE); xk = xk.to(DEVICE); xb = xb.to(DEVICE)
            y  = y.to(DEVICE)
            opt.zero_grad()
            logit = model(xc, xk, xb)
            loss = loss_fn(logit, y)
            loss.backward()
            opt.step()
            tot += loss.item()*y.size(0); n += y.size(0)

        # val PR
        yv, pv = predict_proba(model, va_loader)
        pr = average_precision_score(yv, pv)
        if pr > best_pr + 1e-5:
            best_pr = pr
            best_state = {k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
            bad = 0
        else:
            bad += 1
            if bad >= patience:
                break

        if ep % 1 == 0:
            print(f"{model_name:12s} r{repeat} ep{ep:03d} loss={tot/max(n,1):.4f} valPR={pr:.4f}")

    model.load_state_dict(best_state)

    # choose threshold on val
    yv, pv = predict_proba(model, va_loader)
    t = find_best_threshold(yv, pv)

    # test metrics
    yt, pt = predict_proba(model, te_loader)
    m = evaluate_binary(yt, pt, threshold=t, ece_bins=15)
    m.update({"variant": model_name, "repeat": repeat})
    return m

# ---------------------------
# Run sweep
# ---------------------------
MODELS = ["MLP_BCE", "MLP_AFL", "FTT_BCE", "FTT_BCE_RANK"]
REPEATS = 3  # set 5 later

rows = []
for name in MODELS:
    for r in range(REPEATS):
        m = train_run(r, name)
        rows.append(m)
        print(f"--> DONE {name} r{r}: PR={m['pr_auc']:.4f} ROC={m['roc_auc']:.4f} F1={m['f1']:.4f}")

df_res = pd.DataFrame(rows)
display(df_res)
save_csv(df_res, "attn_imbalance_per_repeat_test")

agg_cols = ["pr_auc","roc_auc","f1","recall","brier","ece","nll","acc"]
summary = df_res.groupby(["variant"])[agg_cols].agg(["mean","std"])
summary.columns = ["_".join(c) for c in summary.columns.values]
summary = summary.reset_index()
display(summary)
save_csv(summary, "attn_imbalance_test_summary_by_variant")

print("\n✅ Tables saved in:", TAB_DIR)

DEVICE: cpu
Counts: {'bin': 9, 'cat': 2, 'cont': 24}
MLP_BCE      r0 ep000 loss=1.2502 valPR=0.1674
MLP_BCE      r0 ep001 loss=1.1644 valPR=0.1672
MLP_BCE      r0 ep002 loss=1.0955 valPR=0.1901
MLP_BCE      r0 ep003 loss=1.0636 valPR=0.1979
MLP_BCE      r0 ep004 loss=1.0191 valPR=0.2071
MLP_BCE      r0 ep005 loss=0.9944 valPR=0.1907
MLP_BCE      r0 ep006 loss=0.9681 valPR=0.1782
MLP_BCE      r0 ep007 loss=0.9479 valPR=0.1953
MLP_BCE      r0 ep008 loss=0.9141 valPR=0.1886
MLP_BCE      r0 ep009 loss=0.8798 valPR=0.1822
MLP_BCE      r0 ep010 loss=0.8190 valPR=0.1921
MLP_BCE      r0 ep011 loss=0.7915 valPR=0.1727
MLP_BCE      r0 ep012 loss=0.7586 valPR=0.1669
MLP_BCE      r0 ep013 loss=0.7437 valPR=0.1904
--> DONE MLP_BCE r0: PR=0.1304 ROC=0.6637 F1=0.1250
MLP_BCE      r1 ep000 loss=1.2492 valPR=0.1194
MLP_BCE      r1 ep001 loss=1.1507 valPR=0.1236
MLP_BCE      r1 ep002 loss=1.0763 valPR=0.1441
MLP_BCE      r1 ep003 loss=1.0700 valPR=0.1487
MLP_BCE      r1 ep004 loss=1.0237 valPR=0.1483
ML

,threshold,pr_auc,roc_auc,acc,f1,precision,recall,tn,fp,fn,tp,brier,ece,nll,variant,repeat
0,0.735,0.130394,0.663657,0.847991,0.125000,0.113636,0.138889,771,78,62,10,0.199239,0.297736,0.572361,MLP_BCE,0
1,0.615,0.146616,0.676858,0.834962,0.232323,0.182540,0.319444,746,103,49,23,0.162464,0.238420,0.482107,MLP_BCE,1
2,0.670,0.152582,0.674372,0.817590,0.207547,0.157143,0.305556,731,118,50,22,0.215201,0.336175,0.614891,MLP_BCE,2
3,0.280,0.129494,0.652761,0.710098,0.173375,0.111554,0.388889,626,223,44,28,0.095030,0.150396,0.346423,MLP_AFL,0
4,0.300,0.148260,0.675730,0.780673,0.223077,0.154255,0.402778,690,159,43,29,0.091416,0.139178,0.334816,MLP_AFL,1
5,0.300,0.149200,0.661710,0.813246,0.218182,0.162162,0.333333,725,124,48,24,0.092078,0.144925,0.340542,MLP_AFL,2
6,0.775,0.170987,0.706436,0.877307,0.175182,0.184615,0.166667,796,53,60,12,0.223764,0.360563,0.639561,FTT_BCE,0
7,0.610,0.169053,0.688277,0.864278,0.193548,0.180723,0.208333,781,68,57,15,0.149804,0.251355,0.465751,FTT_BCE,1
8,0.715,0.160877,0.692350,0.849077,0.232044,0.192661,0.291667,761,88,51,21,0.215182,0.306723,0.595171,FTT_BCE,2
9,0.650,0.167222,0.677840,0.850163,0.197674,0.170000,0.236111,766,83,55,17,0.206340,0.352960,0.601309,FTT_BCE_RANK,0


✅ Saved: /kaggle/working/outputs/cgt_stroke_v1/attn_imbalance_sweep/tables/attn_imbalance_per_repeat_test.csv


,variant,pr_auc_mean,pr_auc_std,roc_auc_mean,roc_auc_std,f1_mean,f1_std,recall_mean,recall_std,brier_mean,brier_std,ece_mean,ece_std,nll_mean,nll_std,acc_mean,acc_std
0,FTT_BCE,0.166973,0.005366,0.695688,0.009528,0.200258,0.029019,0.222222,0.063647,0.196250,0.040452,0.306214,0.054606,0.566828,0.090305,0.863554,0.014129
1,FTT_BCE_RANK,0.163065,0.009931,0.687759,0.012304,0.204901,0.015069,0.222222,0.013889,0.247106,0.039406,0.385219,0.029721,0.686819,0.082511,0.864640,0.014129
2,MLP_AFL,0.142318,0.011116,0.663400,0.011577,0.204878,0.027392,0.375000,0.036747,0.092841,0.001924,0.144833,0.005610,0.340594,0.005804,0.768006,0.052728
3,MLP_BCE,0.143197,0.011482,0.671629,0.007015,0.188290,0.056193,0.254630,0.100475,0.192302,0.027044,0.290777,0.049248,0.556453,0.067806,0.833514,0.015252


✅ Saved: /kaggle/working/outputs/cgt_stroke_v1/attn_imbalance_sweep/tables/attn_imbalance_test_summary_by_variant.csv

✅ Tables saved in: /kaggle/working/outputs/cgt_stroke_v1/attn_imbalance_sweep/tables


In [ ]:
# ============================================================
# TRUST-STROKE (CPU) — ADVANCED SWEEP (FULL, RUNNABLE, SAFE)
# ------------------------------------------------------------
# What this script gives you (CPU-friendly):
# ✅ Full loading + typing (safe)
# ✅ Robust fold-preprocessing (train-only):
#    - Winsorization (clip tails)
#    - Quantile transform (optional, strong for skewed features)
#    - Missingness indicators (optional)
#    - Categorical mapping with UNK=0
# ✅ FT-Transformer-like backbone (TabTransformer)
# ✅ Imbalance tools:
#    - Balanced batches (pos_frac)
#    - Loss choices: BCE, AFL (asymmetric focal), Logit-Adjusted BCE
#    - Optional ranking auxiliary loss (PR-oriented)
# ✅ Bagging (keep all positives, subsample negatives per bag)
# ✅ Early stopping on VAL PR-AUC
# ✅ Temperature scaling calibration (per fold, fit on validation)
# ✅ Saves:
#    - per-fold metrics CSV
#    - summary by variant CSV
#    - optional plots (reliability + PR curve)
#
# You can add more variants by editing VARIANTS below.
# ============================================================

# -------------------------
# 0) Imports
# -------------------------
import os
import gc
import math
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader, Sampler

from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit
from sklearn.metrics import average_precision_score, roc_auc_score, brier_score_loss, precision_recall_curve
from sklearn.calibration import calibration_curve

# QuantileTransformer is CPU and very useful for skewed continuous features
from sklearn.preprocessing import QuantileTransformer

import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

# -------------------------
# 1) Global Config
# -------------------------
SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE)

# Data
DATA_PATH = "/kaggle/input/stroke-dataset/Stroke.csv"   # <-- change if needed
TARGET_COL = "stroke"

# CV
N_SPLITS = 5
N_REPEATS = 1          # start with 1; later increase to 2
INNER_VAL_FRAC = 0.25

# Training (CPU-safe defaults)
EPOCHS = 20
BATCH_SIZE = 256
POS_FRAC = 0.30
LR = 2e-3
WD = 1e-5
PATIENCE = 5

# Model
D_MODEL = 64
N_HEADS = 4
N_LAYERS = 3
DROPOUT = 0.20

# Bagging
BAGS = 1               # set 3 for stronger results (CPU slower)
NEG_SUBSAMPLE = 1.0    # 1.0 = keep all negatives; 0.5 = subsample negatives per bag

# Calibration
ECE_BINS = 15
TEMP_MAX_ITER = 200
TEMP_LR = 0.05

# Output
OUTDIR = "/kaggle/working/outputs/trust_stroke_adv_sweep"
os.makedirs(OUTDIR, exist_ok=True)
os.makedirs(os.path.join(OUTDIR, "tables"), exist_ok=True)
os.makedirs(os.path.join(OUTDIR, "figs"), exist_ok=True)

# -------------------------
# 2) Repro
# -------------------------
def set_seed(seed=SEED):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)

# -------------------------
# 3) Load Data (SAFE)
# -------------------------
df = pd.read_csv(DATA_PATH)
df.columns = [c.strip() for c in df.columns]

if TARGET_COL not in df.columns:
    raise ValueError(f"TARGET_COL='{TARGET_COL}' not found. Columns: {df.columns.tolist()[:20]}...")

# common cleanup
if "id" in df.columns:
    df = df.drop(columns=["id"])

y = df[TARGET_COL].astype(int).values
X = df.drop(columns=[TARGET_COL]).copy()

assert isinstance(X, pd.DataFrame)
y = np.asarray(y).astype(int)
assert len(X) == len(y)

print("N:", len(X), "Pos rate:", float(y.mean()))
print("Columns:", X.shape[1])

# -------------------------
# 4) Typing (SAFE)
# -------------------------
def infer_cat_cols(df_: pd.DataFrame, max_unique_int_as_cat=10):
    cat_cols_ = []
    for c in df_.columns:
        if df_[c].dtype == "object":
            cat_cols_.append(c)
        elif str(df_[c].dtype).startswith("category"):
            cat_cols_.append(c)
        else:
            if pd.api.types.is_integer_dtype(df_[c]) and df_[c].nunique(dropna=True) <= max_unique_int_as_cat:
                cat_cols_.append(c)
    return cat_cols_

cat_cols = infer_cat_cols(X)
num_cols = [c for c in X.columns if c not in cat_cols]

print("n_cat:", len(cat_cols), "n_num:", len(num_cols))
if len(cat_cols) > 0:
    print("cat_cols:", cat_cols)
if len(num_cols) > 0:
    print("num_cols (first 12):", num_cols[:12])

# -------------------------
# 5) Helpers: sigmoid/logit
# -------------------------
def sigmoid_np(x):
    return 1.0 / (1.0 + np.exp(-x))

def logit_np(p):
    p = np.clip(p, 1e-6, 1 - 1e-6)
    return np.log(p / (1 - p))

# -------------------------
# 6) Fold Preprocessor (robust, train-only)
# -------------------------
class FoldPreprocessor:
    """
    Categorical:
      - fit mapping on train only (UNK=0)
    Numerical:
      - median impute (train median)
      - winsorize by train quantiles (clip tails)
      - optional quantile transform (train-fit only)
      - standardize (train mean/std) AFTER winsor/quantile
      - optional missingness indicators per numeric column
    """
    def __init__(self, cat_cols, num_cols,
                 winsor_q=(0.005, 0.995),
                 use_quantile=False,
                 add_missing_indicators=True,
                 quantile_n=2000):
        self.cat_cols = list(cat_cols)
        self.num_cols = list(num_cols)
        self.winsor_q = winsor_q
        self.use_quantile = bool(use_quantile)
        self.add_missing_indicators = bool(add_missing_indicators)
        self.quantile_n = int(quantile_n)

        self.cat_maps = {}
        self.cat_sizes = []

        self.num_median = {}
        self.num_clip = {}
        self.num_meanstd = {}
        self.qt = None  # QuantileTransformer

    def fit(self, X_tr: pd.DataFrame):
        Xtr = X_tr.copy()

        # categorical
        self.cat_maps = {}
        self.cat_sizes = []
        for c in self.cat_cols:
            s = Xtr[c].astype("object").fillna("MISSING").astype(str)
            uniq = pd.Index(s.unique())
            mp = {k: i + 1 for i, k in enumerate(uniq)}
            self.cat_maps[c] = mp
            self.cat_sizes.append(len(mp) + 1)

        # numerical: compute medians + clip bounds
        self.num_median = {}
        self.num_clip = {}
        for c in self.num_cols:
            s = pd.to_numeric(Xtr[c], errors="coerce")
            med = float(s.median())
            s_imp = s.fillna(med).astype(np.float32)

            lo = float(np.quantile(s_imp.values, self.winsor_q[0]))
            hi = float(np.quantile(s_imp.values, self.winsor_q[1]))
            if hi <= lo:
                hi = lo + 1e-6
            self.num_median[c] = med
            self.num_clip[c] = (lo, hi)

        # fit quantile transformer on winsorized numeric matrix (optional)
        if self.use_quantile and len(self.num_cols) > 0:
            Xnum = []
            for c in self.num_cols:
                s = pd.to_numeric(Xtr[c], errors="coerce")
                med = self.num_median[c]
                lo, hi = self.num_clip[c]
                s = s.fillna(med).astype(np.float32)
                s = np.clip(s.values, lo, hi)
                Xnum.append(s)
            Xnum = np.stack(Xnum, axis=1).astype(np.float32)

            self.qt = QuantileTransformer(
                n_quantiles=min(self.quantile_n, max(10, Xnum.shape[0])),
                output_distribution="normal",
                subsample=int(1e9),
                random_state=SEED,
                copy=True,
            )
            self.qt.fit(Xnum)
        else:
            self.qt = None

        # compute mean/std AFTER optional quantile
        self.num_meanstd = {}
        if len(self.num_cols) > 0:
            Xnum = []
            for c in self.num_cols:
                s = pd.to_numeric(Xtr[c], errors="coerce")
                med = self.num_median[c]
                lo, hi = self.num_clip[c]
                s = s.fillna(med).astype(np.float32)
                s = np.clip(s.values, lo, hi)
                Xnum.append(s)
            Xnum = np.stack(Xnum, axis=1).astype(np.float32)
            if self.qt is not None:
                Xnum = self.qt.transform(Xnum).astype(np.float32)

            mu = Xnum.mean(axis=0)
            sd = Xnum.std(axis=0) + 1e-6
            for j, c in enumerate(self.num_cols):
                self.num_meanstd[c] = (float(mu[j]), float(sd[j]))

        return self

    def transform(self, X_any: pd.DataFrame):
        Xa = X_any.copy()

        # categorical -> ids
        Xcat = None
        if len(self.cat_cols) > 0:
            arr = []
            for c in self.cat_cols:
                s = Xa[c].astype("object").fillna("MISSING").astype(str)
                mp = self.cat_maps[c]
                codes = s.map(mp).fillna(0).astype(np.int64).values
                arr.append(codes)
            Xcat = np.stack(arr, axis=1)

        # numerical -> float32 matrix
        Xnum = None
        if len(self.num_cols) > 0:
            Xnum_list = []
            miss_list = []
            for c in self.num_cols:
                raw = pd.to_numeric(Xa[c], errors="coerce")
                miss = raw.isna().astype(np.float32).values  # 1 if missing
                med = self.num_median[c]
                lo, hi = self.num_clip[c]

                s = raw.fillna(med).astype(np.float32).values
                s = np.clip(s, lo, hi)

                Xnum_list.append(s.astype(np.float32))
                miss_list.append(miss)

            Xnum = np.stack(Xnum_list, axis=1).astype(np.float32)

            # quantile transform (optional)
            if self.qt is not None:
                Xnum = self.qt.transform(Xnum).astype(np.float32)

            # standardize
            for j, c in enumerate(self.num_cols):
                mu, sd = self.num_meanstd[c]
                Xnum[:, j] = (Xnum[:, j] - mu) / sd

            # missingness indicators
            if self.add_missing_indicators:
                Miss = np.stack(miss_list, axis=1).astype(np.float32)
                Xnum = np.concatenate([Xnum, Miss], axis=1)

        return Xcat, Xnum

# -------------------------
# 7) Dataset + Balanced Sampler
# -------------------------
class TabDataset(Dataset):
    def __init__(self, X_cat, X_num, y):
        self.X_cat = X_cat
        self.X_num = X_num
        self.y = np.asarray(y).astype(np.float32)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, i):
        yi = torch.tensor(self.y[i], dtype=torch.float32)
        x_cat = torch.tensor(self.X_cat[i], dtype=torch.long) if self.X_cat is not None else None
        x_num = torch.tensor(self.X_num[i], dtype=torch.float32) if self.X_num is not None else None
        return x_cat, x_num, yi

class BalancedBatchSampler(Sampler):
    """
    For each batch: ~pos_frac positives and the rest negatives (with replacement).
    """
    def __init__(self, y, batch_size=256, pos_frac=0.30, seed=0):
        self.y = np.asarray(y).astype(int)
        self.batch_size = int(batch_size)
        self.pos_bs = max(1, int(self.batch_size * pos_frac))
        self.neg_bs = self.batch_size - self.pos_bs
        self.pos_idx = np.where(self.y == 1)[0]
        self.neg_idx = np.where(self.y == 0)[0]
        self.rng = np.random.RandomState(seed)

    def __iter__(self):
        n_batches = int(np.ceil(len(self.y) / self.batch_size))
        for _ in range(n_batches):
            pos = self.rng.choice(self.pos_idx, size=self.pos_bs, replace=True)
            neg = self.rng.choice(self.neg_idx, size=self.neg_bs, replace=True)
            idx = np.concatenate([pos, neg])
            self.rng.shuffle(idx)
            yield from idx.tolist()

    def __len__(self):
        return int(np.ceil(len(self.y) / self.batch_size)) * self.batch_size

# -------------------------
# 8) Model: FT-Transformer-like
# -------------------------
class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, dropout):
        super().__init__()
        self.attn = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.ln1 = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(
            nn.Linear(d_model, 4 * d_model),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(4 * d_model, d_model),
        )
        self.ln2 = nn.LayerNorm(d_model)
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        h, _ = self.attn(x, x, x, need_weights=False)
        x = self.ln1(x + self.drop(h))
        h = self.ff(x)
        x = self.ln2(x + self.drop(h))
        return x

class TabTransformer(nn.Module):
    def __init__(self, cat_sizes, n_num, d_model=64, n_heads=4, n_layers=3, dropout=0.20):
        super().__init__()
        self.n_cat = len(cat_sizes)
        self.n_num = int(n_num)
        self.d_model = int(d_model)

        self.cat_embeds = nn.ModuleList([nn.Embedding(sz, d_model) for sz in cat_sizes]) if self.n_cat > 0 else None
        self.num_proj = nn.Linear(1, d_model) if self.n_num > 0 else None

        self.cls = nn.Parameter(torch.zeros(1, 1, d_model))
        nn.init.normal_(self.cls, std=0.02)

        self.blocks = nn.ModuleList([TransformerBlock(d_model, n_heads, dropout) for _ in range(n_layers)])
        self.drop = nn.Dropout(dropout)
        self.head = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Dropout(dropout),
            nn.Linear(d_model, 1),
        )

    def forward(self, x_cat, x_num):
        tokens = []
        if self.n_cat > 0:
            for j, emb in enumerate(self.cat_embeds):
                tokens.append(emb(x_cat[:, j]))  # (B,D)

        if self.n_num > 0:
            for j in range(self.n_num):
                v = x_num[:, j:j+1]              # (B,1)
                tokens.append(self.num_proj(v))  # (B,D)

        if len(tokens) == 0:
            B = x_cat.size(0) if x_cat is not None else x_num.size(0)
            x = torch.zeros((B, 0, self.d_model), device=(x_cat.device if x_cat is not None else x_num.device))
        else:
            x = torch.stack(tokens, dim=1)       # (B,T,D)

        B = x.shape[0]
        cls = self.cls.expand(B, -1, -1)
        x = torch.cat([cls, x], dim=1)           # (B,1+T,D)

        x = self.drop(x)
        for blk in self.blocks:
            x = blk(x)

        z = x[:, 0, :]
        logit = self.head(z).squeeze(-1)
        return logit

# -------------------------
# 9) Losses: BCE, AFL, Logit-Adjusted + optional ranking auxiliary
# -------------------------
class AsymmetricFocalLoss(nn.Module):
    """
    AFL for binary:
      - gamma_pos focuses hard positives
      - gamma_neg focuses hard negatives
    """
    def __init__(self, gamma_pos=0.0, gamma_neg=2.0, clip=0.0, eps=1e-8):
        super().__init__()
        self.gamma_pos = float(gamma_pos)
        self.gamma_neg = float(gamma_neg)
        self.clip = float(clip)
        self.eps = float(eps)

    def forward(self, logits, y):
        y = y.float()
        p = torch.sigmoid(logits)
        if self.clip > 0:
            p = torch.clamp(p, min=self.clip, max=1 - self.clip)

        # pos
        pt_pos = p
        loss_pos = -y * torch.log(torch.clamp(pt_pos, min=self.eps)) * (1 - pt_pos).pow(self.gamma_pos)

        # neg
        pt_neg = 1 - p
        loss_neg = -(1 - y) * torch.log(torch.clamp(pt_neg, min=self.eps)) * (1 - pt_neg).pow(self.gamma_neg)

        return (loss_pos + loss_neg).mean()

def logit_adjusted_bce_with_logits(logits, y, pi_pos, tau=1.0):
    """
    Logit adjusted loss: logit' = logit + tau*log(pi_pos/pi_neg)
    pi_pos from TRAIN prevalence.
    """
    pi_pos = float(np.clip(pi_pos, 1e-6, 1 - 1e-6))
    adj = tau * math.log(pi_pos / (1 - pi_pos))
    logits_adj = logits + adj
    return F.binary_cross_entropy_with_logits(logits_adj, y)

def batch_pairwise_ranking_loss(logits, y, max_pairs=256):
    """
    Simple PR-oriented auxiliary:
      Encourage logits(pos) > logits(neg).
    Uses random pairs within batch.
    """
    with torch.no_grad():
        pos_idx = torch.where(y > 0.5)[0]
        neg_idx = torch.where(y <= 0.5)[0]
        if len(pos_idx) == 0 or len(neg_idx) == 0:
            return logits.new_tensor(0.0)

        # sample pairs
        rng = torch.randperm(len(pos_idx), device=logits.device)
        pos_idx = pos_idx[rng[:min(len(pos_idx), max_pairs)]]
        rng = torch.randperm(len(neg_idx), device=logits.device)
        neg_idx = neg_idx[rng[:min(len(neg_idx), max_pairs)]]

    # all combinations (bounded)
    # build pairs by cycling smaller set
    P = len(pos_idx)
    N = len(neg_idx)
    K = min(max_pairs, P * N)
    if K <= 0:
        return logits.new_tensor(0.0)

    # create K pairs
    p_sel = pos_idx.repeat_interleave(min(N, max_pairs))[:K]
    n_sel = neg_idx.repeat(min(P, max_pairs))[:K]

    s = logits[p_sel] - logits[n_sel]
    # logistic ranking loss
    return F.softplus(-s).mean()

# -------------------------
# 10) Predict / Metrics / Calibration
# -------------------------
def predict_proba(model, Xcat, Xnum):
    n = len(Xcat) if Xcat is not None else len(Xnum)
    ds = TabDataset(Xcat, Xnum, np.zeros(n, dtype=np.float32))
    dl = DataLoader(ds, batch_size=2048, shuffle=False, num_workers=0)

    model.eval()
    ps = []
    with torch.no_grad():
        for x_cat, x_num, _ in dl:
            x_cat = x_cat.to(DEVICE) if x_cat is not None else None
            x_num = x_num.to(DEVICE) if x_num is not None else None
            logits = model(x_cat, x_num).detach().cpu().numpy()
            ps.append(sigmoid_np(logits))
    return np.concatenate(ps)

def ece_score(y_true, p, n_bins=15):
    y_true = np.asarray(y_true).astype(int)
    p = np.asarray(p).astype(float)
    bins = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    for i in range(n_bins):
        lo, hi = bins[i], bins[i + 1]
        if i < n_bins - 1:
            m = (p >= lo) & (p < hi)
        else:
            m = (p >= lo) & (p <= hi)
        if m.sum() == 0:
            continue
        acc = y_true[m].mean()
        conf = p[m].mean()
        ece += (m.sum() / len(p)) * abs(acc - conf)
    return float(ece)

def fit_temperature_scaling(p_val, y_val, max_iter=200, lr=0.05):
    """
    Fit scalar T>0 on validation logits to minimize NLL:
      p_cal = sigmoid(logit(p)/T)
    """
    z = torch.tensor(logit_np(p_val).astype(np.float32), device=DEVICE)
    y = torch.tensor(y_val.astype(np.float32), device=DEVICE)

    logT = torch.zeros((), device=DEVICE, requires_grad=True)
    opt = torch.optim.Adam([logT], lr=lr)

    for _ in range(max_iter):
        T = torch.exp(logT) + 1e-6
        logits = z / T
        loss = F.binary_cross_entropy_with_logits(logits, y)
        opt.zero_grad()
        loss.backward()
        opt.step()

    T = float((torch.exp(logT) + 1e-6).detach().cpu().item())
    return T

def apply_temperature(p, T):
    return sigmoid_np(logit_np(p) / max(1e-6, float(T)))

# -------------------------
# 11) Training (one bag)
# -------------------------
def train_one_model(
    Xtr_cat, Xtr_num, ytr,
    Xva_cat, Xva_num, yva,
    cat_sizes,
    variant_cfg,
    seed=0
):
    set_seed(seed)

    ds_tr = TabDataset(Xtr_cat, Xtr_num, ytr)
    ds_va = TabDataset(Xva_cat, Xva_num, yva)

    sampler = BalancedBatchSampler(ytr, batch_size=BATCH_SIZE, pos_frac=variant_cfg["pos_frac"], seed=seed)
    dl_tr = DataLoader(ds_tr, batch_size=BATCH_SIZE, sampler=sampler, drop_last=True, num_workers=0)
    dl_va = DataLoader(ds_va, batch_size=2048, shuffle=False, num_workers=0)

    model = TabTransformer(
        cat_sizes=cat_sizes,
        n_num=(0 if Xtr_num is None else Xtr_num.shape[1]),
        d_model=variant_cfg["d_model"],
        n_heads=variant_cfg["n_heads"],
        n_layers=variant_cfg["n_layers"],
        dropout=variant_cfg["dropout"],
    ).to(DEVICE)

    opt = torch.optim.AdamW(model.parameters(), lr=variant_cfg["lr"], weight_decay=variant_cfg["wd"])

    loss_name = variant_cfg["loss"]
    pi_pos = float(np.mean(ytr))

    if loss_name == "bce":
        primary_loss = None
    elif loss_name == "afl":
        primary_loss = AsymmetricFocalLoss(
            gamma_pos=variant_cfg.get("gamma_pos", 0.0),
            gamma_neg=variant_cfg.get("gamma_neg", 2.0),
            clip=variant_cfg.get("afl_clip", 0.0),
        )
    elif loss_name == "logitadj":
        primary_loss = None
    else:
        raise ValueError(f"Unknown loss: {loss_name}")

    best_pr = -1.0
    best_state = None
    bad = 0

    for ep in range(1, variant_cfg["epochs"] + 1):
        model.train()
        for x_cat, x_num, yb in dl_tr:
            yb = yb.to(DEVICE)
            x_cat = x_cat.to(DEVICE) if x_cat is not None else None
            x_num = x_num.to(DEVICE) if x_num is not None else None

            opt.zero_grad(set_to_none=True)
            logits = model(x_cat, x_num)

            if loss_name == "bce":
                loss = F.binary_cross_entropy_with_logits(logits, yb)
            elif loss_name == "afl":
                loss = primary_loss(logits, yb)
            elif loss_name == "logitadj":
                loss = logit_adjusted_bce_with_logits(
                    logits, yb, pi_pos=pi_pos, tau=variant_cfg.get("tau", 1.0)
                )

            # optional ranking auxiliary (PR-oriented)
            rank_w = float(variant_cfg.get("rank_w", 0.0))
            if rank_w > 0:
                loss_rank = batch_pairwise_ranking_loss(logits, yb, max_pairs=variant_cfg.get("rank_pairs", 256))
                loss = loss + rank_w * loss_rank

            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()

        # early stop on VAL PR-AUC
        model.eval()
        p_list, y_list = [], []
        with torch.no_grad():
            for x_cat, x_num, yb in dl_va:
                x_cat = x_cat.to(DEVICE) if x_cat is not None else None
                x_num = x_num.to(DEVICE) if x_num is not None else None
                logits = model(x_cat, x_num).detach().cpu().numpy()
                p_list.append(sigmoid_np(logits))
                y_list.append(yb.numpy())

        pva = np.concatenate(p_list)
        yva_np = np.concatenate(y_list).astype(int)
        pr = average_precision_score(yva_np, pva)

        if pr > best_pr + 1e-4:
            best_pr = pr
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            bad = 0
        else:
            bad += 1
            if bad >= PATIENCE:
                break

    if best_state is not None:
        model.load_state_dict(best_state)

    return model

# -------------------------
# 12) Bagging indices (keep all positives, subsample negatives)
# -------------------------
def make_bag_indices(y, rng, neg_subsample=1.0):
    y = np.asarray(y).astype(int)
    pos_idx = np.where(y == 1)[0]
    neg_idx = np.where(y == 0)[0]

    if neg_subsample >= 1.0:
        bag_neg = neg_idx
    else:
        k = max(1, int(len(neg_idx) * float(neg_subsample)))
        bag_neg = rng.choice(neg_idx, size=k, replace=False)

    bag_idx = np.concatenate([pos_idx, bag_neg])
    rng.shuffle(bag_idx)
    return bag_idx

# -------------------------
# 13) Plot helpers (optional)
# -------------------------
def save_reliability_plot(y_true, p, path, n_bins=15, title="Reliability"):
    frac_pos, mean_pred = calibration_curve(y_true, p, n_bins=n_bins, strategy="quantile")
    plt.figure()
    plt.plot(mean_pred, frac_pos, marker="o")
    plt.plot([0, 1], [0, 1], linestyle="--")
    plt.xlabel("Mean predicted probability")
    plt.ylabel("Fraction of positives")
    plt.title(title)
    plt.tight_layout()
    plt.savefig(path, dpi=200)
    plt.close()

def save_pr_curve(y_true, p, path, title="PR Curve"):
    prec, rec, _ = precision_recall_curve(y_true, p)
    ap = average_precision_score(y_true, p)
    plt.figure()
    plt.plot(rec, prec)
    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.title(f"{title} (AP={ap:.4f})")
    plt.tight_layout()
    plt.savefig(path, dpi=200)
    plt.close()

# -------------------------
# 14) Variants to try (CPU-feasible, advanced but controlled)
# -------------------------
BASE_CFG = dict(
    epochs=EPOCHS,
    lr=LR,
    wd=WD,
    d_model=D_MODEL,
    n_heads=N_HEADS,
    n_layers=N_LAYERS,
    dropout=DROPOUT,
    pos_frac=POS_FRAC,
)

VARIANTS = [
    # Strong baseline: quantile + miss-ind + BCE
    ("V0_Q_BCE", dict(**BASE_CFG,
        use_quantile=True, add_miss=True,
        loss="bce",
        rank_w=0.0
    )),
    # Asymmetric focal (good for rare events), with quantile
    ("V1_Q_AFL", dict(**BASE_CFG,
        use_quantile=True, add_miss=True,
        loss="afl", gamma_pos=0.0, gamma_neg=3.0, afl_clip=0.0,
        rank_w=0.0
    )),
    # Logit-adjusted BCE (class-prior correction), quantile
    ("V2_Q_LOGITADJ", dict(**BASE_CFG,
        use_quantile=True, add_miss=True,
        loss="logitadj", tau=1.0,
        rank_w=0.0
    )),
    # PR push: add small ranking auxiliary (often improves AP)
    ("V3_Q_AFL_RANK", dict(**BASE_CFG,
        use_quantile=True, add_miss=True,
        loss="afl", gamma_pos=0.0, gamma_neg=3.0, afl_clip=0.0,
        rank_w=0.10, rank_pairs=256
    )),
    # No-quantile control (sometimes quantile hurts; we check)
    ("V4_Z_BCE", dict(**BASE_CFG,
        use_quantile=False, add_miss=True,
        loss="bce",
        rank_w=0.0
    )),
]

# -------------------------
# 15) Runner: repeated CV + bags + temp scaling
# -------------------------
def run_sweep(X, y, cat_cols, num_cols, variants, n_splits=5, n_repeats=1, bags=1, neg_subsample=1.0):
    rng_global = np.random.RandomState(SEED)

    rows = []
    for vname, vcfg in variants:
        print("\n" + "="*70)
        print("VARIANT:", vname, "| bags:", bags, "| neg_subsample:", neg_subsample)
        print("CFG:", {k: vcfg[k] for k in ["use_quantile","add_miss","loss","pos_frac","epochs"] if k in vcfg})

        for rep in range(n_repeats):
            skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=int(rng_global.randint(0, 10_000_000)))

            for fold, (tr_idx, te_idx) in enumerate(skf.split(X, y), start=1):
                X_tr_full, y_tr_full = X.iloc[tr_idx], y[tr_idx]
                X_te, y_te = X.iloc[te_idx], y[te_idx]

                # inner validation
                sss = StratifiedShuffleSplit(
                    n_splits=1, test_size=INNER_VAL_FRAC,
                    random_state=int(rng_global.randint(0, 10_000_000))
                )
                tr_sub_idx, va_idx = next(sss.split(X_tr_full, y_tr_full))
                X_tr, y_tr = X_tr_full.iloc[tr_sub_idx], y_tr_full[tr_sub_idx]
                X_va, y_va = X_tr_full.iloc[va_idx], y_tr_full[va_idx]

                # preprocess per fold
                prep = FoldPreprocessor(
                    cat_cols, num_cols,
                    winsor_q=(0.005, 0.995),
                    use_quantile=vcfg["use_quantile"],
                    add_missing_indicators=vcfg["add_miss"],
                    quantile_n=2000
                ).fit(X_tr)

                Xtr_cat, Xtr_num = prep.transform(X_tr)
                Xva_cat, Xva_num = prep.transform(X_va)
                Xte_cat, Xte_num = prep.transform(X_te)

                # bagging predictions
                p_va_bags = []
                p_te_bags = []
                for b in range(bags):
                    rng_bag = np.random.RandomState(int(rng_global.randint(0, 10_000_000)))

                    # bag indices on TRAIN subset only
                    bag_idx = make_bag_indices(y_tr, rng_bag, neg_subsample=neg_subsample)
                    X_tr_b = X_tr.iloc[bag_idx]
                    y_tr_b = y_tr[bag_idx]

                    # re-transform from same prep (important: no leakage)
                    Xtrb_cat, Xtrb_num = prep.transform(X_tr_b)

                    seed_model = int(rng_global.randint(0, 10_000_000))
                    model = train_one_model(
                        Xtrb_cat, Xtrb_num, y_tr_b,
                        Xva_cat, Xva_num, y_va,
                        cat_sizes=prep.cat_sizes,
                        variant_cfg=vcfg,
                        seed=seed_model
                    )

                    p_va_bags.append(predict_proba(model, Xva_cat, Xva_num))
                    p_te_bags.append(predict_proba(model, Xte_cat, Xte_num))

                    del model
                    gc.collect()
                    if torch.cuda.is_available():
                        torch.cuda.empty_cache()

                p_va = np.mean(np.stack(p_va_bags, axis=0), axis=0)
                p_te = np.mean(np.stack(p_te_bags, axis=0), axis=0)

                # temperature scaling
                T = fit_temperature_scaling(p_va, y_va, max_iter=TEMP_MAX_ITER, lr=TEMP_LR)
                p_te_cal = apply_temperature(p_te, T)

                # metrics
                pr = average_precision_score(y_te, p_te_cal)
                roc = roc_auc_score(y_te, p_te_cal)
                brier = brier_score_loss(y_te, p_te_cal)
                ece = ece_score(y_te, p_te_cal, n_bins=ECE_BINS)
                nll = float(np.mean(-(y_te * np.log(np.clip(p_te_cal,1e-9,1)) + (1-y_te) * np.log(np.clip(1-p_te_cal,1e-9,1)))))

                rows.append(dict(
                    variant=vname,
                    rep=int(rep),
                    fold=int(fold),
                    pr_auc=float(pr),
                    roc_auc=float(roc),
                    brier=float(brier),
                    ece=float(ece),
                    nll=float(nll),
                    T=float(T),
                    bags=int(bags),
                    neg_subsample=float(neg_subsample),
                    use_quantile=bool(vcfg["use_quantile"]),
                    add_miss=bool(vcfg["add_miss"]),
                    loss=str(vcfg["loss"]),
                    rank_w=float(vcfg.get("rank_w", 0.0)),
                ))

                print(f"[rep {rep+1}/{n_repeats} fold {fold}/{n_splits}] "
                      f"PR={pr:.4f} ROC={roc:.4f} Brier={brier:.4f} ECE={ece:.4f} T={T:.3f}")

    df = pd.DataFrame(rows)

    # summary
    summ = df.groupby("variant").agg(
        pr_auc_mean=("pr_auc","mean"),
        pr_auc_std=("pr_auc","std"),
        roc_auc_mean=("roc_auc","mean"),
        roc_auc_std=("roc_auc","std"),
        brier_mean=("brier","mean"),
        brier_std=("brier","std"),
        ece_mean=("ece","mean"),
        ece_std=("ece","std"),
        nll_mean=("nll","mean"),
        nll_std=("nll","std"),
    ).reset_index().sort_values("pr_auc_mean", ascending=False)

    return df, summ

# -------------------------
# 16) RUN SWEEP
# -------------------------
df_per_fold, df_summary = run_sweep(
    X, y, cat_cols, num_cols,
    variants=VARIANTS,
    n_splits=N_SPLITS,
    n_repeats=N_REPEATS,
    bags=BAGS,
    neg_subsample=NEG_SUBSAMPLE
)

# Save tables
per_fold_path = os.path.join(OUTDIR, "tables", "adv_sweep_per_fold.csv")
summary_path = os.path.join(OUTDIR, "tables", "adv_sweep_summary_by_variant.csv")
df_per_fold.to_csv(per_fold_path, index=False)
df_summary.to_csv(summary_path, index=False)

print("\n✅ Saved per-fold:", per_fold_path)
print("✅ Saved summary :", summary_path)
print("\n=== SUMMARY (sorted by PR-AUC mean) ===")
print(df_summary)

# -------------------------
# 17) Optional: quick plots on the best variant (one fold only)
# -------------------------
# To keep it simple and CPU-light, we re-train just once on one split
# and produce PR + reliability plots for paper-quality figures.
BEST = df_summary.iloc[0]["variant"]
print("\nBest variant by mean PR-AUC:", BEST)

best_cfg = None
for name, cfg in VARIANTS:
    if name == BEST:
        best_cfg = cfg
        break

if best_cfg is not None:
    # Use a deterministic split for plotting
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
    tr_idx, te_idx = next(skf.split(X, y))
    X_tr_full, y_tr_full = X.iloc[tr_idx], y[tr_idx]
    X_te, y_te = X.iloc[te_idx], y[te_idx]

    sss = StratifiedShuffleSplit(n_splits=1, test_size=INNER_VAL_FRAC, random_state=SEED)
    tr_sub_idx, va_idx = next(sss.split(X_tr_full, y_tr_full))
    X_tr, y_tr = X_tr_full.iloc[tr_sub_idx], y_tr_full[tr_sub_idx]
    X_va, y_va = X_tr_full.iloc[va_idx], y_tr_full[va_idx]

    prep = FoldPreprocessor(
        cat_cols, num_cols,
        winsor_q=(0.005, 0.995),
        use_quantile=best_cfg["use_quantile"],
        add_missing_indicators=best_cfg["add_miss"],
        quantile_n=2000
    ).fit(X_tr)

    Xtr_cat, Xtr_num = prep.transform(X_tr)
    Xva_cat, Xva_num = prep.transform(X_va)
    Xte_cat, Xte_num = prep.transform(X_te)

    # bagging for plots
    rng_plot = np.random.RandomState(SEED)
    p_va_bags, p_te_bags = [], []
    for b in range(BAGS):
        bag_idx = make_bag_indices(y_tr, rng_plot, neg_subsample=NEG_SUBSAMPLE)
        X_tr_b = X_tr.iloc[bag_idx]
        y_tr_b = y_tr[bag_idx]
        Xtrb_cat, Xtrb_num = prep.transform(X_tr_b)

        model = train_one_model(
            Xtrb_cat, Xtrb_num, y_tr_b,
            Xva_cat, Xva_num, y_va,
            cat_sizes=prep.cat_sizes,
            variant_cfg=best_cfg,
            seed=SEED + 1000 + b
        )
        p_va_bags.append(predict_proba(model, Xva_cat, Xva_num))
        p_te_bags.append(predict_proba(model, Xte_cat, Xte_num))
        del model
        gc.collect()

    p_va = np.mean(np.stack(p_va_bags, axis=0), axis=0)
    p_te = np.mean(np.stack(p_te_bags, axis=0), axis=0)

    T = fit_temperature_scaling(p_va, y_va, max_iter=TEMP_MAX_ITER, lr=TEMP_LR)
    p_te_cal = apply_temperature(p_te, T)

    pr = average_precision_score(y_te, p_te_cal)
    roc = roc_auc_score(y_te, p_te_cal)
    brier = brier_score_loss(y_te, p_te_cal)
    ece = ece_score(y_te, p_te_cal, n_bins=ECE_BINS)

    print(f"\n[PLOT SPLIT] PR={pr:.4f} ROC={roc:.4f} Brier={brier:.4f} ECE={ece:.4f} T={T:.3f}")

    pr_path = os.path.join(OUTDIR, "figs", f"{BEST}_pr_curve.png")
    rel_path = os.path.join(OUTDIR, "figs", f"{BEST}_reliability.png")

    save_pr_curve(y_te, p_te_cal, pr_path, title=f"{BEST} PR Curve")
    save_reliability_plot(y_te, p_te_cal, rel_path, n_bins=ECE_BINS, title=f"{BEST} Reliability")

    print("✅ Saved PR curve:", pr_path)
    print("✅ Saved reliability plot:", rel_path)

print("\nDONE ✅  (Advanced sweep complete)")

DEVICE: cpu
N: 4603 Pos rate: 0.07864436237236584
Columns: 35
n_cat: 15 n_num: 20
cat_cols: ['gender', 'age', 'Race', 'Marital status', 'alcohol', 'smoke', 'sleep disorder', 'Health Insurance', 'General health condition', 'depression', 'diabetes', 'hypertension', 'high cholesterol', 'Coronary Heart Disease', 'Body Mass Index']
num_cols (first 12): ['sleep time', 'Minutes sedentary activity', 'Waist Circumference', 'Systolic blood pressure', 'Diastolic blood pressure', 'High-density lipoprotein', 'Triglyceride', 'Low-density lipoprotein', 'Fasting Glucose', 'Glycohemoglobin', 'energy', 'protein']

VARIANT: V0_Q_BCE | bags: 1 | neg_subsample: 1.0
CFG: {'use_quantile': True, 'add_miss': True, 'loss': 'bce', 'pos_frac': 0.3, 'epochs': 20}
[rep 1/1 fold 1/5] PR=0.1920 ROC=0.7413 Brier=0.0917 ECE=0.0895 T=0.693
[rep 1/1 fold 2/5] PR=0.1746 ROC=0.6930 Brier=0.1128 ECE=0.1385 T=0.627
[rep 1/1 fold 3/5] PR=0.1617 ROC=0.6725 Brier=0.0998 ECE=0.0992 T=0.500
[rep 1/1 fold 4/5] PR=0.1443 ROC=0.6406